# Part 1 — Data exploration, audits and geometry

This notebook contains the original work through Step 23. Attach the Solar Filament Segmentation competition data and the existing `filament-feature-cache` dataset before running the analysis.

Part 2 begins at Step 30 and rebuilds its small setup from the competition data and the same pinned scoring reference. Its setup can run independently.


# Solar Filament Segmentation Challenge 2026
## Phase 1: Dataset Forensics, Image Analysis & Segmentation Baseline

### 1. Project Overview

This notebook initiates a computer vision pipeline for the **Solar Filament Segmentation Challenge 2026**, hosted on Kaggle.

The objective is to detect and individually segment solar filaments in GONG H-alpha observations while preserving filament morphology, connectivity, and instance-level separation.

### 2. Technical Objectives

- Examine image characteristics, dimensions, intensity distributions, and observational variability.
- Investigate the annotation format and individual filament masks.
- Identify potential preprocessing requirements and image-quality issues.
- Evaluate filament morphology, class imbalance, and object-size distributions.
- Establish leakage-resistant training and validation partitions.
- Prepare a reproducible foundation for classical segmentation, convolutional neural networks, and advanced instance segmentation architectures.

### 3. Evaluation Strategy

The primary competition metric is **Panoptic Quality (PQ)**, which evaluates segmentation quality together with instance detection and matching.

Additional diagnostic metrics may include:

- Intersection over Union (IoU)
- Dice coefficient
- Precision and recall
- Instance fragmentation and over-merging rates

### 4. Research Principles

All experiments will prioritize reproducibility, transparent validation, computational efficiency, and scientific integrity.

Competition-provided annotations will be used in accordance with the organizers' data restrictions. Test-set annotations will not be reconstructed or obtained from external sources.

In [ ]:
import os
import sys
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 65)
print("SOLAR FILAMENT SEGMENTATION — ENVIRONMENT AUDIT")
print("=" * 65)

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU device     : {torch.cuda.get_device_name(0)}")

except ImportError:
    print("PyTorch is not installed.")

SEED = 42
np.random.seed(SEED)

if "torch" in sys.modules:
    torch.manual_seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print(f"\nInput directory exists: {INPUT_ROOT.exists()}")
print(f"Output directory      : {OUTPUT_ROOT}")


### 5. Dataset Discovery and File Inventory

Before performing preprocessing or segmentation, we inspect the complete dataset directory structure.

The purpose of this step is to determine:

- Which datasets are attached to the notebook.
- How training images, annotations, and test images are organized.
- Which file formats are used for images and segmentation targets.
- Whether metadata, sample submissions, or additional documentation are provided.

This inspection prevents incorrect assumptions about the dataset schema and supports a reproducible data-loading pipeline.

In [ ]:
from collections import Counter

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "Dataset not found."
    )

all_files = sorted(
    path
    for path in INPUT_ROOT.rglob("*")
    if path.is_file()
)

print("=" * 65)
print("DATASET INVENTORY")
print("=" * 65)

print(f"\nTotal files: {len(all_files):,}")

print("\nAttached datasets:")

for directory in sorted(INPUT_ROOT.iterdir()):
    if directory.is_dir():
        count = sum(
            1 for path in directory.rglob("*")
            if path.is_file()
        )
        print(f"  {directory.name}: {count:,} files")

extensions = Counter(
    path.suffix.lower() if path.suffix else "<no extension>"
    for path in all_files
)

print("\nFile formats:")
for extension, count in extensions.most_common():
    print(f"  {extension:15s}: {count:,}")

print("\nSample file paths:")
for path in all_files[:40]:
    print(f"  {path.relative_to(INPUT_ROOT)}")

if len(all_files) > 40:
    print(f"\n... {len(all_files) - 40:,} additional files")

if not all_files:
    print("\nNo files found. Attach the competition dataset.")

In [ ]:

records = []

for path in all_files:
    stat = path.stat()

    records.append({
        "relative_path": str(path.relative_to(INPUT_ROOT)),
        "extension": path.suffix.lower(),
        "size_mb": stat.st_size / (1024 ** 2),
        "parent_folder": path.parent.name,
    })

file_inventory = pd.DataFrame(records)

if not file_inventory.empty:
    display(
        file_inventory
        .sort_values("size_mb", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

    print("\nTotal attached data size:")
    print(f"{file_inventory['size_mb'].sum() / 1024:.3f} GB")
else:
    print("File inventory is empty.")


### 6. Annotation Schema Investigation

The dataset provides solar observations alongside structured annotations describing individual filament instances.

Before decoding segmentation masks, we inspect the annotation schema to identify:

- Image identifiers, filenames, and image dimensions.
- Annotation identifiers and their relationships to images.
- Segmentation representations, including polygons or run-length encoding.
- Filament categories and any additional metadata.
- Dataset consistency and annotation coverage.

Understanding the annotation format is essential for constructing accurate instance masks, preventing image-annotation mismatches, and implementing the competition's evaluation procedure.

In [ ]:
import json
from pathlib import Path

json_files = [
    path for path in all_files
    if (
        path.suffix.lower() == ".json"
        and path.name == "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
        and "competitions" in path.parts
    )
]

print("Training annotation files found:", len(json_files))

for path in json_files:
    print(" -", path.relative_to(INPUT_ROOT))

if len(json_files) != 1:
    raise ValueError(
        "Expected exactly one official training annotation JSON. "
        "Check that the Solar Filament Segmentation competition "
        "dataset is attached correctly."
    )

ANNOTATION_PATH = json_files[0]

with open(ANNOTATION_PATH, "r", encoding="utf-8") as f:
    annotations_data = json.load(f)

print("\nANNOTATION SCHEMA")
print("=" * 60)

print("Top-level data type:", type(annotations_data).__name__)

if isinstance(annotations_data, dict):
    print("Top-level keys:", list(annotations_data.keys()))

    for key, value in annotations_data.items():
        if isinstance(value, list):
            print(f"\n{key}: {len(value):,} entries")

            if value:
                print("First entry:")
                print(json.dumps(value[0], indent=2)[:2500])

        elif isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} keys")
            print("Sample keys:", list(value.keys())[:10])

        else:
            print(f"\n{key}: {str(value)[:200]}")

else:
    print("JSON root is not a dictionary.")
    print(str(annotations_data)[:2000])

### 7. Solar Image and Instance Annotation Visualization

This section examines the spatial relationship between solar observations and their manually annotated filament instances.

The visualization serves four purposes:

- Verify that annotation identifiers correctly map to their source images.
- Assess the morphology, connectivity, and contrast of solar filament structures.
- Examine the distribution and separation of individual filament instances.
- Identify potential difficulties arising from overlapping structures, fine boundaries, and low-contrast regions.

Each filament instance is displayed using a distinct color to make its spatial extent identifiable.

This visualization constitutes an initial annotation-integrity check before mask rasterization, model training, and quantitative evaluation.

In [ ]:
from collections import defaultdict
from PIL import Image
from matplotlib.patches import Polygon
import matplotlib.pyplot as plt
import numpy as np

annotations_by_image = defaultdict(list)

for ann in annotations_data["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

train_image_dirs = sorted({
    path.parent
    for path in all_files
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and path.parent.name == "train_images"
})

if len(train_image_dirs) != 1:
    raise ValueError(
        f"Expected one train_images directory, "
        f"found {len(train_image_dirs)}"
    )

TRAIN_IMAGE_DIR = train_image_dirs[0]

sample_images = sorted(
    annotations_data["images"],
    key=lambda img: len(annotations_by_image[img["id"]]),
    reverse=True
)[:3]

fig, axes = plt.subplots(
    len(sample_images), 2,
    figsize=(14, 7 * len(sample_images)),
    squeeze=False
)

colors = plt.cm.tab20(np.linspace(0, 1, 20))

for row, img_info in enumerate(sample_images):

    image_path = TRAIN_IMAGE_DIR / img_info["file_name"]

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    with Image.open(image_path) as image:
        image_array = np.asarray(
            image.convert("L")
        )

    image_annotations = annotations_by_image[img_info["id"]]

    axes[row, 0].imshow(image_array, cmap="gray")
    axes[row, 0].set_title(
        f"Original: {img_info['file_name']}"
    )

    axes[row, 1].imshow(image_array, cmap="gray")

    for index, ann in enumerate(image_annotations):

        segmentation = ann.get("segmentation", [])

        if not isinstance(segmentation, list):
            continue

        if segmentation and isinstance(segmentation[0], (int, float)):
            segmentation = [segmentation]

        color = colors[index % len(colors)]

        for polygon in segmentation:
            coords = np.asarray(polygon, dtype=float)

            if coords.size < 6 or coords.size % 2 != 0:
                continue

            coords = coords.reshape(-1, 2)

            patch = Polygon(
                coords,
                closed=True,
                facecolor=color,
                edgecolor=color,
                linewidth=1,
                alpha=0.4
            )

            axes[row, 1].add_patch(patch)

    axes[row, 1].set_title(
        f"Annotated instances: {len(image_annotations)}"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
categories_df = pd.DataFrame(
    annotations_data["categories"]
)

display(categories_df)

category_counts = (
    pd.Series(
        [ann["category_id"]
         for ann in annotations_data["annotations"]]
    )
    .value_counts()
    .rename_axis("category_id")
    .reset_index(name="instance_count")
)

category_counts = category_counts.merge(
    categories_df[["id", "name"]],
    left_on="category_id",
    right_on="id",
    how="left"
).drop(columns="id")

display(category_counts)


### 8. Instance Distribution and Dataset Integrity Analysis

This section evaluates the structural consistency of the MAGFiLO training annotations and quantifies the distribution of filament instances across solar observations.

The analysis examines annotation coverage, image-to-instance relationships, identifier uniqueness, and the variability of filament counts per observation.

These checks are important because instance segmentation performance may vary significantly between sparsely populated observations and images containing many thin, closely spaced structures.

The findings will guide sampling decisions, validation stratification, and the design of subsequent segmentation experiments.

In [ ]:
images = annotations_data["images"]
annotations = annotations_data["annotations"]

image_ids = [img["id"] for img in images]
annotation_ids = [ann.get("id") for ann in annotations]

instance_counts = Counter(
    ann["image_id"] for ann in annotations
)

image_summary = pd.DataFrame([
    {
        "image_id": img["id"],
        "file_name": img["file_name"],
        "date_captured": img.get("date_captured"),
        "instance_count": instance_counts.get(img["id"], 0),
        "file_exists": (
            TRAIN_IMAGE_DIR / img["file_name"]
        ).is_file()
    }
    for img in images
])

print("DATASET INTEGRITY")
print("=" * 60)

print("Images:", len(images))
print("Annotations:", len(annotations))

print("Unique image IDs:", len(set(image_ids)))
print("Unique annotation IDs:", len(set(annotation_ids)))

print("Images with missing files:",
      (~image_summary["file_exists"]).sum())

print("Images with no annotations:",
      (image_summary["instance_count"] == 0).sum())

orphan_annotations = sum(
    ann["image_id"] not in set(image_ids)
    for ann in annotations
)

print("Annotations with invalid image references:",
      orphan_annotations)

print("\nINSTANCE COUNT STATISTICS")
display(
    image_summary["instance_count"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .to_frame("Filaments per image")
)

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(
    image_summary["instance_count"],
    bins=np.arange(
        0,
        image_summary["instance_count"].max() + 2
    ) - 0.5,
    edgecolor="black",
    alpha=0.8
)

ax.set_title("Distribution of Filament Instances per Image")
ax.set_xlabel("Number of annotated filament instances")
ax.set_ylabel("Number of solar images")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


### 9. Filament Geometry and Morphological Distribution

Solar filaments exhibit significant variability in their apparent length, width, shape, and orientation. These properties can influence segmentation accuracy, especially when structures are narrow, irregular, or closely spaced.

This analysis extracts geometric characteristics from the annotated polygons, including bounding-box width, height, area, and aspect ratio.

The objectives are to identify scale variation across filament instances, assess potential difficulties associated with thin structures, and guide future model-resolution, patch-sampling, and instance-separation decisions.

Bounding-box measurements are treated as approximate geometric descriptors rather than true segmentation-mask areas.

In [ ]:
geometry_records = []
skipped_annotations = 0

for ann in annotations:
    segmentation = ann.get("segmentation", [])
    if not isinstance(segmentation, list) or not segmentation:
        skipped_annotations += 1
        continue

    if isinstance(segmentation[0], (int, float)):
        polygons = [segmentation]
    else:
        polygons = segmentation

    valid_coords = []
    for polygon in polygons:
        coords = np.asarray(polygon, dtype = float)

        if coords.size < 6 or coords.size % 2!= 0:
            continue
        if not np.isfinite(coords).all():
            continue

        valid_coords.append(coords.reshape(-1,2))

    if not valid_coords:
        skipped_annotations += 1
        continue

    coords = np.concatenate(valid_coords, axis = 0)
    
    x_min, y_min = coords.min(axis = 0)
    x_max, y_max = coords.max(axis = 0)

    width = max(0.0, x_max - x_min)
    height = max(0.0, y_max - y_min)

    bbox_area = width * height

    aspect_ratio = (
        max(width, height) / min(width, height)
        if min(width, height) > 0
        else np.nan
    )

    geometry_records.append({
        "image_id": ann["image_id"],
        "category_id": ann.get("category_id"),
        "bbox_width": width,
        "bbox_height": height,
        "bbox_area": bbox_area,
        "aspect_ratio": aspect_ratio
    })

geometry_df = pd.DataFrame(geometry_records)

print("Valid polygon annotations:", len(geometry_df))
print("Skipped annotations:", skipped_annotations)

display(
    geometry_df[
        ["bbox_width", "bbox_height",
         "bbox_area", "aspect_ratio"]
    ].describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    np.log1p(geometry_df["bbox_area"]),
    bins=50,
    edgecolor="black"
)
axes[0].set_title("Log Bounding-Box Area Distribution")
axes[0].set_xlabel("log(1 + bounding-box area)")

axes[1].hist(
    geometry_df["aspect_ratio"].dropna().clip(upper=30),
    bins=50,
    edgecolor="black"
)
axes[1].set_title("Bounding-Box Aspect Ratio")
axes[1].set_xlabel("Aspect ratio (clipped at 30)")

for ax in axes:
    ax.set_ylabel("Number of instances")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### 10. Temporal Distribution and Validation Leakage Assessment

Solar filament observations may exhibit temporal dependence because the same physical structures can persist across successive observations.

A conventional random train-validation split may therefore produce optimistic evaluation results if closely related solar observations appear in both subsets.

This analysis examines the chronological distribution of the dataset, observation frequency, repeated observation dates, and temporal coverage.

The findings will support a group-aware validation strategy that limits temporal overlap while retaining a representative range of filament densities and morphological characteristics.

A temporal grouping strategy reduces potential leakage but does not, by itself, guarantee independence between observations.

In [ ]:
image_summary["capture_datetime"] = pd.to_datetime(
    image_summary["date_captured"],
    errors="coerce"
)

image_summary["capture_date"] = (
    image_summary["capture_datetime"].dt.date
)

image_summary["year"] = (
    image_summary["capture_datetime"].dt.year
)

print("TEMPORAL DATA AUDIT")
print("=" * 60)

print("Earliest observation:",
      image_summary["capture_datetime"].min())

print("Latest observation:",
      image_summary["capture_datetime"].max())

print("Missing timestamps:",
      image_summary["capture_datetime"].isna().sum())

print("Unique observation dates:",
      image_summary["capture_date"].nunique())

print("\nIMAGES PER YEAR")
year_counts = (
    image_summary["year"]
    .value_counts()
    .sort_index()
)

display(year_counts.to_frame("image_count"))

print("\nMOST FREQUENT OBSERVATION DATES")
date_counts = (
    image_summary["capture_date"]
    .value_counts()
    .head(15)
)

display(date_counts.to_frame("image_count"))

print("\nREPEATED OBSERVATIONS")
print(
    "Dates containing multiple images:",
    (
        image_summary.groupby("capture_date")
        .size() > 1
    ).sum()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

year_counts.plot(
    kind="bar",
    ax=axes[0],
    edgecolor="black"
)

axes[0].set_title("Solar Observations by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of Images")

axes[1].scatter(
    image_summary["capture_datetime"],
    image_summary["instance_count"],
    s=12,
    alpha=0.5
)

axes[1].set_title("Filament Density Over Time")
axes[1].set_xlabel("Observation Date")
axes[1].set_ylabel("Annotated Instances")

for ax in axes:
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


### 12. Temporal-Block Training and Validation Partitioning

To reduce the risk of temporal leakage, observations are partitioned using their capture timestamps rather than independent random image sampling.

A grouped temporal split is constructed using 30-day intervals, with approximately 20% of groups assigned to validation. Training observations within three days of a validation observation are excluded.

The resulting partition is assessed for image-level uniqueness, temporal separation, filament-density distributions, and representation across observation years.

The validation partition will remain fixed for model comparison. Hyperparameter selection and feature development will rely on training-only experiments wherever possible.

This is a provisional validation design. A separate chronological robustness test will be conducted before selecting the final competition solution.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
SEED = 42
BLOCK_DAYS = 30
PURGE_DAYS = 3
VALIDATION_FRACTION = 0.20

split_df = image_summary.copy()

assert split_df["capture_datetime"].notna().all()
assert split_df["file_exists"].all()

reference_date = (
    split_df["capture_datetime"].min().normalize()
)

split_df["temporal_block"] = (
    (
        split_df["capture_datetime"] - reference_date
    ).dt.total_seconds() // (BLOCK_DAYS * 86400)
).astype(int)

splitter = GroupShuffleSplit(
    n_splits = 1,
    test_size = VALIDATION_FRACTION,
    random_state = SEED
)

train_idx, valid_idx = next(
    splitter.split(
        split_df, 
        groups = split_df["temporal_block"]
    )
)

train_df = split_df.iloc[train_idx].copy()
valid_df = split_df.iloc[valid_idx].copy()

train_times = train_df["capture_datetime"].to_numpy(
    dtype="datetime64[ns]"
)

valid_times = np.sort(
    valid_df["capture_datetime"].to_numpy(
        dtype="datetime64[ns]"
    )
)

positions = np.searchsorted(valid_times, train_times)

left_idx = np.clip(
    positions - 1, 0, len(valid_times) - 1
)

right_idx = np.clip(
    positions, 0, len(valid_times) - 1
)

nearest_gap = np.minimum(
    np.abs(train_times - valid_times[left_idx]),
    np.abs(train_times - valid_times[right_idx])
)

keep_train = nearest_gap > np.timedelta64(
    PURGE_DAYS, "D"
)

purged_count = int((~keep_train).sum())

train_df = train_df.loc[keep_train].copy()

assert len(train_df) > 0
assert len(valid_df) > 0

assert set(train_df["image_id"]).isdisjoint(
    set(valid_df["image_id"])
)

assert set(train_df["temporal_block"]).isdisjoint(
    set(valid_df["temporal_block"])
)

assert set(train_df["capture_date"]).isdisjoint(
    set(valid_df["capture_date"])
)

print("TRAIN-VALIDATION SUMMARY")
print("=" * 60)

print("Training images:", len(train_df))
print("Validation images:", len(valid_df))
print("Purged images:", purged_count)
print(
    "Validation fraction:",
    round(len(valid_df) / len(split_df), 4)
)

print("\nInstance-count comparison:")
display(pd.DataFrame({
    "Train": train_df["instance_count"].describe(),
    "Validation": valid_df["instance_count"].describe()
}))

print("\nYear distribution:")
display(
    pd.crosstab(
        pd.concat([
            train_df.assign(split="train"),
            valid_df.assign(split="validation")
        ])["year"],
        pd.concat([
            train_df.assign(split="train"),
            valid_df.assign(split="validation")
        ])["split"]
    )
)

split_manifest = pd.concat([
    train_df.assign(split="train"),
    valid_df.assign(split="validation")
], ignore_index=True)

split_manifest.to_csv(
    OUTPUT_ROOT / "temporal_split_manifest.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    train_df["instance_count"],
    bins=np.arange(0, 28) - 0.5,
    alpha=0.6,
    label="Training"
)

axes[0].hist(
    valid_df["instance_count"],
    bins=np.arange(0, 28) - 0.5,
    alpha=0.6,
    label="Validation"
)

axes[0].set_title("Filament Density Distribution")
axes[0].set_xlabel("Instances per image")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].scatter(
    train_df["capture_datetime"],
    train_df["instance_count"],
    s=12,
    alpha=0.5,
    label="Training"
)

axes[1].scatter(
    valid_df["capture_datetime"],
    valid_df["instance_count"],
    s=14,
    alpha=0.6,
    label="Validation"
)

axes[1].set_title("Temporal Separation")
axes[1].set_xlabel("Observation date")
axes[1].set_ylabel("Filament count")
axes[1].legend()

plt.tight_layout()
plt.show()

### 13. COCO Polygon Rasterization and Instance-Mask Generation

The dataset stores filament boundaries as polygon coordinates. These annotations must be converted into pixel-aligned masks before quantitative morphological analysis or supervised segmentation.

Two target representations are generated:

- Binary segmentation masks that identify foreground filament pixels.
- Instance segmentation masks that preserve the identity of individual annotated filaments.

Polygon validity, coordinate bounds, degenerate masks, and overlapping annotations are explicitly checked.

The rasterization convention will be held constant throughout experimentation. Before final competition evaluation, mask encoding and IoU matching will be compared with the organizer's self-evaluation implementation.

In [ ]:
from PIL import Image, ImageDraw
from collections import defaultdict

annotations_by_image = defaultdict(list)

for ann in annotations_data["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

image_lookup = {
    img["id"]: img
    for img in annotations_data["images"]
}

def get_polygons(annotation):
    segmentation = annotation.get("segmentation", [])

    if not isinstance(segmentation, list):
        raise TypeError(
            "Expected polygon segmentation, "
            "received another representation."
        )

    if not segmentation:
        return []

    if isinstance(segmentation[0], (int, float)):
        segmentation = [segmentation]

    valid = []

    for poly in segmentation:
        coords = np.asarray(poly, dtype=float)

        if coords.size < 6 or coords.size % 2 != 0:
            continue

        if not np.isfinite(coords).all():
            continue

        valid.append(coords.reshape(-1, 2))

    return valid

def rasterize_annotation(annotation, image_shape):
    height, width = image_shape

    canvas = Image.new(
        "L",
        (width, height),
        color=0
    )

    draw = ImageDraw.Draw(canvas)

    for polygon in get_polygons(annotation):
        vertices = [
            (float(x), float(y))
            for x, y in polygon
        ]

        draw.polygon(vertices, fill=1)

    return np.asarray(canvas, dtype=bool)

def load_solar_image(image_id):
    info = image_lookup[image_id]

    path = TRAIN_IMAGE_DIR / info["file_name"]

    with Image.open(path) as img:
        image = np.asarray(
            img.convert("L"),
            dtype=np.uint8
        )

    assert image.shape == (
        info["height"],
        info["width"]
    )

    return image

def load_instance_targets(image_id):
    info = image_lookup[image_id]

    shape = (
        info["height"],
        info["width"]
    )

    instance_mask = np.zeros(
        shape,
        dtype=np.int32
    )

    overlap_pixels = 0
    empty_annotations = 0

    for index, ann in enumerate(
        annotations_by_image[image_id], start=1
    ):
        mask = rasterize_annotation(ann, shape)

        if not mask.any():
            empty_annotations += 1
            continue

        overlap_pixels += int(
            np.count_nonzero(mask & (instance_mask > 0))
        )

        instance_mask[mask] = index

    binary_mask = instance_mask > 0

    return binary_mask, instance_mask, {
        "overlap_pixels": overlap_pixels,
        "empty_annotations": empty_annotations
    }


In [ ]:
sample_id = (
    train_df.sort_values(
        "instance_count",
        ascending=False
    ).iloc[0]["image_id"]
)

image = load_solar_image(sample_id)

binary_mask, instance_mask, mask_info = (
    load_instance_targets(sample_id)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original Observation")

axes[1].imshow(binary_mask, cmap="gray")
axes[1].set_title("Binary Filament Mask")

masked_instances = np.ma.masked_where(
    instance_mask == 0,
    instance_mask
)

axes[2].imshow(image, cmap="gray")
axes[2].imshow(
    masked_instances,
    cmap="nipy_spectral",
    alpha=0.65,
    interpolation="nearest"
)
axes[2].set_title("Instance Segmentation")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", image.shape)
print("Ground-truth annotations:",
      len(annotations_by_image[sample_id]))
print("Rasterized instance IDs:",
      np.count_nonzero(np.unique(instance_mask)))
print("Mask diagnostics:", mask_info)

### 14. Filament Morphology and Topological Characterization

This section derives geometric and topological descriptors from rasterized ground-truth filament masks.

Unlike bounding-box measurements, these features describe the actual segmented structures and provide insight into the intrinsic difficulty of the competition.

The analysis includes:

- Pixel-accurate area and perimeter.
- Solidity and eccentricity.
- Skeleton length and estimated filament width.
- Connected-component counts.
- Shape complexity and spatial extent.

The resulting instance-level table is intended for descriptive analysis and model-error diagnostics. Ground-truth-derived morphology will not be supplied as an inference-time model input.

In [ ]:
from skimage.measure import regionprops, label
from skimage.morphology import skeletonize
from scipy.ndimage import distance_transform_edt

def extract_morphology(mask):
    mask = np.asarray(mask,dtype = bool)

    if not mask.any():
        return None

    ys, xs = np.where(mask)

    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1

    crop = mask[y0:y1, x0:x1]

    props = regionprops(
        label(crop, connectivity = 2)
    )

    if not props: 
        return None

    component_count = len(props)

    skeleton = skeletonize(crop)
    distance = distance_transform_edt(crop)

    skeleton_widths = (
        2 * distance[skeleton]
        if skeleton.any()
        else np.array([])
    )

    area = int(crop.sum())

    perimeter = sum(p.perimeter for p in props)

    bbox_width = x1 - x0
    bbox_height = y1 - y0

    return {
        "area": area,
        "perimeter": perimeter,
        "bbox_width": bbox_width,
        "bbox_height": bbox_height,
        "extent": area / (
            bbox_width * bbox_height
        ),
        "skeleton_pixels": int(skeleton.sum()),
        "mean_width": (
            float(skeleton_widths.mean())
            if skeleton_widths.size else np.nan
        ),
        "median_width": (
            float(np.median(skeleton_widths))
            if skeleton_widths.size else np.nan
        ),
        "component_count": component_count,
        "shape_complexity": (
            perimeter ** 2 / (4 * np.pi * area)
            if area > 0 else np.nan
        )
    }

In [ ]:
morphology_records = []

sample_train_ids = train_df.sample(
    n=min(30, len(train_df)),
    random_state=SEED
)["image_id"]

for image_id in sample_train_ids:
    info = image_lookup[image_id]
    shape = (info["height"], info["width"])

    for ann in annotations_by_image[image_id]:
        mask = rasterize_annotation(ann, shape)

        features = extract_morphology(mask)

        if features is not None:
            morphology_records.append({
                "image_id": image_id,
                "annotation_id": ann["id"],
                "category_id": ann["category_id"],
                **features
            })

morphology_df = pd.DataFrame(
    morphology_records
)

print("Morphological instances analyzed:",
      len(morphology_df))

display(
    morphology_df.drop(
        columns=["image_id", "annotation_id"]
    ).describe()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, feature in zip(
    axes,
    ["area", "skeleton_pixels", "mean_width"]
):
    values = morphology_df[feature].dropna()

    ax.hist(
        np.log1p(values),
        bins=35,
        edgecolor="black"
    )

    ax.set_title(feature)
    ax.set_xlabel("log(1 + value)")

plt.tight_layout()
plt.show()

### 15. Image-Derived Feature Engineering

This section constructs a set of pixel-level descriptors designed to capture the visual characteristics of solar filaments.

The proposed feature families include grayscale intensity, local contrast, gradient structure, local variance, multi-scale dark-ridge response, structural coherence, and radial position.

These features will serve two purposes:

1. Support statistical investigation of relationships between solar-image structure and filament presence.
2. Provide interpretable predictors for classical machine learning and custom mathematical segmentation models.

All features are derived exclusively from observed image pixels. No ground-truth segmentation information is used during feature calculation.

In [ ]:
from scipy.ndimage import (
    gaussian_filter,
    gaussian_gradient_magnitude,
    maximum_filter1d
)
from skimage.filters import sato

def extract_pixel_features(image):
    img = image.astype(np.float32) / 255.0

    background = gaussian_filter(img, sigma=15)

    darkness = background - img

    mean_5 = gaussian_filter(img, sigma=5)
    second_5 = gaussian_filter(img**2, sigma=5)

    local_std_5 = np.sqrt(
        np.maximum(second_5 - mean_5**2, 0)
    )

    mean_15 = gaussian_filter(img, sigma=15)
    second_15 = gaussian_filter(img**2, sigma=15)

    local_std_15 = np.sqrt(
        np.maximum(second_15 - mean_15**2, 0)
    )

    gradient = gaussian_gradient_magnitude(
        img, sigma=2
    )

    ridge = sato(
        img,
        sigmas=(1, 2, 4, 8),
        black_ridges=True
    ).astype(np.float32)

    gx = gaussian_filter(
        img, sigma=1, order=(0, 1)
    )
    gy = gaussian_filter(
        img, sigma=1, order=(1, 0)
    )

    jxx = gaussian_filter(gx * gx, sigma=3)
    jyy = gaussian_filter(gy * gy, sigma=3)
    jxy = gaussian_filter(gx * gy, sigma=3)

    coherence = (
        np.sqrt(
            (jxx - jyy)**2 + 4 * jxy**2
        ) /
        (jxx + jyy + 1e-8)
    )

    h, w = img.shape

    yy, xx = np.ogrid[:h, :w]

    radial_distance = np.sqrt(
        ((xx - (w - 1)/2) / (w/2))**2 +
        ((yy - (h - 1)/2) / (h/2))**2
    ).astype(np.float32)

    contrast_ridge_interaction = (
        np.maximum(darkness, 0) * ridge
    )

    return {
        "intensity": img,
        "local_darkness": darkness,
        "local_std_5": local_std_5,
        "local_std_15": local_std_15,
        "gradient": gradient,
        "ridge_response": ridge,
        "coherence": coherence.astype(np.float32),
        "radial_distance": radial_distance,
        "contrast_ridge_interaction":
            contrast_ridge_interaction
    }


In [ ]:
sample_image = load_solar_image(sample_id)

feature_maps = extract_pixel_features(
    sample_image
)

fig, axes = plt.subplots(3, 3, figsize=(16, 15))

for ax, (feature_name, feature_map) in zip(
    axes.flat,
    feature_maps.items()
):
    ax.imshow(feature_map, cmap="gray")
    ax.set_title(feature_name)
    ax.axis("off")

plt.tight_layout()
plt.show()


### 16. Univariate Analysis of Engineered Image Features

Univariate analysis examines the distribution of each candidate predictor independently.

This analysis evaluates central tendency, dispersion, quantiles, skewness, extreme values, and missing or non-finite measurements.

The objective is to identify features that are numerically stable, sufficiently variable, and potentially informative for detecting filament structures.

Pixel samples will be drawn from training observations only. Foreground and background sampling will be documented to avoid confusing the exploratory sampling distribution with the true pixel-level class distribution.

#### 16.1 Objective

This section develops a reproducible feature-engineering pipeline for all observations assigned to the training partition of the MAGFiLO dataset.

The objective is to investigate the statistical properties of image-derived predictors while preserving temporal validation integrity and avoiding repeated, computationally expensive image processing.

#### 16.2 Feature Families

Nine candidate features are considered:

1. Grayscale intensity
2. Local darkness
3. Local standard deviation (σ = 5)
4. Local standard deviation (σ = 15)
5. Gradient magnitude
6. Multi-scale ridge response
7. Structure tensor coherence
8. Normalized radial distance
9. Contrast–ridge interaction

#### 16.3 Sampling Methodology

Each training image contributes up to 1,500 foreground pixels and 1,500 background pixels.

The balanced sampling design supports exploratory discrimination analysis but does not represent the natural foreground prevalence.

The complete image is processed at its original resolution before feature values are sampled.

#### 16.4 Computational Strategy

Feature extraction is executed incrementally, with one Parquet checkpoint per image.

Completed checkpoints are reused, preventing unnecessary recomputation within or across sessions when saved checkpoints are restored.

The extracted feature dataset will be reused for univariate, bivariate, multivariate, machine learning, and explainability experiments.

#### 16.5 Statistical Scope

Univariate statistics will quantify feature distributions, central tendency, dispersion, skewness, percentiles, and numerical stability.

Because sampled pixels from the same image are dependent, subsequent statistical inference will account for image-level and temporal grouping.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import hashlib
import json
import gc
import time
import shutil

SEED = 42

PIXELS_PER_CLASS = 1500

FEATURE_IMAGES = len(train_df)

FEATURE_VERSION = "v1"

FEATURE_NAMES = [
    "intensity",
    "local_darkness",
    "local_std_5",
    "local_std_15",
    "gradient",
    "ridge_response",
    "coherence",
    "radial_distance",
    "contrast_ridge_interaction"
]

feature_image_ids = (
    train_df["image_id"]
    .astype(str)
    .sort_values()
    .tolist()
)

config = {
    "version": FEATURE_VERSION,
    "seed": SEED,
    "pixels_per_class": PIXELS_PER_CLASS,
    "image_ids": feature_image_ids,
    "features": FEATURE_NAMES
}

config_hash = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode()
).hexdigest()[:12]

CACHE_ROOT = Path(
    "/kaggle/working/filament_feature_cache"
)

CACHE_DIR = CACHE_ROOT / config_hash

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with open(CACHE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("FULL TRAINING FEATURE EXTRACTION")
print("=" * 60)

print("Training images:", FEATURE_IMAGES)
print("Features:", len(FEATURE_NAMES))
print("Maximum sampled pixels per image:",
      PIXELS_PER_CLASS * 2)
print("Cache directory:", CACHE_DIR)
print("Cache version:", FEATURE_VERSION)


In [ ]:
import time

source_cache_dir = Path(
    "/kaggle/input/datasets/vasudevgrover/"
    "filament-feature-cache/filament_feature_cache"
) / CACHE_DIR.name

print("Source:", source_cache_dir, flush=True)
print("Destination:", CACHE_DIR, flush=True)

source_config_path = source_cache_dir / "config.json"

if not source_config_path.is_file():
    raise FileNotFoundError(
        f"Cache configuration not found: {source_config_path}\n"
        "Check that the filament-feature-cache dataset "
        "is attached and its path has not changed."
    )

with source_config_path.open("r") as file:
    source_config = json.load(file)

if source_config != config:
    raise ValueError(
        "The saved cache configuration does not match "
        "the current extraction configuration."
    )

CACHE_DIR.mkdir(parents=True, exist_ok=True)

restored = 0
already_present = 0
missing_source = []

started = time.perf_counter()

for position, image_id in enumerate(
    config["image_ids"],
    start=1
):
    image_key = hashlib.sha256(
        str(image_id).encode()
    ).hexdigest()[:20]

    filename = f"{image_key}.parquet"

    source_file = source_cache_dir / filename
    destination = CACHE_DIR / filename

    if destination.is_file():
        already_present += 1

    elif source_file.is_file():
        temporary = destination.with_suffix(".parquet.tmp")
        shutil.copy2(source_file, temporary)
        temporary.replace(destination)

        restored += 1

    else:
        missing_source.append(filename)

    if position % 100 == 0 or position == len(config["image_ids"]):
        print(
            f"Checked {position}/{len(config['image_ids'])} | "
            f"Restored: {restored} | "
            f"Already present: {already_present} | "
            f"Missing: {len(missing_source)} | "
            f"Elapsed: {time.perf_counter() - started:.1f}s",
            flush=True
        )

if missing_source:
    raise RuntimeError(
        f"{len(missing_source)} expected checkpoints are "
        "missing from both source and destination. "
        f"Examples: {missing_source[:5]}"
    )

print("\nCache restoration complete.", flush=True)

In [ ]:
from pathlib import Path
import hashlib
import shutil

input_root = Path("/kaggle/input/datasets")

source_dirs = [
    folder for folder in input_root.rglob("*")
    if folder.is_dir()
    and folder.name == CACHE_DIR.name
    and len(list(folder.glob("*.parquet"))) > 0
]

if len(source_dirs) != 1:
    raise RuntimeError(
        f"Expected one matching cache folder, found {len(source_dirs)}"
    )

source_dir = source_dirs[0]

print("Source cache:", source_dir)
print("Destination cache:", CACHE_DIR)

CACHE_DIR.mkdir(parents=True, exist_ok=True)

restored = 0

for image_id in feature_image_ids:
    key = hashlib.sha256(
        str(image_id).encode()
    ).hexdigest()[:20]

    filename = f"{key}.parquet"
    source = source_dir / filename
    destination = CACHE_DIR / filename

    if source.is_file() and not destination.is_file():
        shutil.copy2(source, destination)
        restored += 1

expected_paths = [
    CACHE_DIR / (
        hashlib.sha256(
            str(image_id).encode()
        ).hexdigest()[:20] + ".parquet"
    )
    for image_id in feature_image_ids
]

cached = sum(path.is_file() for path in expected_paths)

print("\nCACHE RESTORATION COMPLETE")
print("Files restored:", restored)
print("Cached images:", cached)
print("Expected images:", len(expected_paths))
print("Missing images:", len(expected_paths) - cached)

In [ ]:
def image_cache_path(image_id):

    image_key = hashlib.sha256(
        str(image_id).encode()
    ).hexdigest()[:20]

    return CACHE_DIR / f"{image_key}.parquet"


failed_images = []
newly_processed = 0
previously_cached = 0

start_time = time.perf_counter()

for position, image_id in enumerate(
    feature_image_ids, start=1
):

    output_path = image_cache_path(image_id)

    if output_path.exists():
        previously_cached += 1
        continue

    try:
        image = load_solar_image(image_id)

        target, _, _ = load_instance_targets(
            image_id
        )

        flat_target = target.ravel()

        foreground_idx = np.flatnonzero(
            flat_target
        )

        background_idx = np.flatnonzero(
            ~flat_target
        )

        image_seed = int(
            hashlib.sha256(
                f"{SEED}:{image_id}".encode()
            ).hexdigest()[:8],
            16
        )

        rng = np.random.default_rng(image_seed)

        selected_fg = rng.choice(
            foreground_idx,
            size=min(
                PIXELS_PER_CLASS,
                len(foreground_idx)
            ),
            replace=False
        )

        selected_bg = rng.choice(
            background_idx,
            size=min(
                PIXELS_PER_CLASS,
                len(background_idx)
            ),
            replace=False
        )

        selected_indices = np.concatenate([
            selected_fg,
            selected_bg
        ])

        features = extract_pixel_features(image)

        if set(features) != set(FEATURE_NAMES):
            raise ValueError(
                "Extracted feature schema mismatch."
            )

        frame = pd.DataFrame({
            name: features[name].ravel()[
                selected_indices
            ].astype(np.float32)
            for name in FEATURE_NAMES
        })

        frame["target"] = (
            flat_target[selected_indices]
            .astype(np.uint8)
        )

        frame["image_id"] = str(image_id)

        frame["foreground_fraction"] = (
            np.float32(target.mean())
        )

        if not np.isfinite(
            frame[FEATURE_NAMES].to_numpy()
        ).all():
            raise ValueError(
                "Non-finite feature values detected."
            )

        temporary_path = output_path.with_suffix(
            ".parquet.tmp"
        )

        frame.to_parquet(
            temporary_path,
            index=False,
            compression="zstd"
        )

        temporary_path.replace(output_path)

        newly_processed += 1

        del image, target, features, frame

        if newly_processed % 10 == 0:
            gc.collect()

        if (
            newly_processed % 5 == 0
            or position == len(feature_image_ids)
        ):
            print(
                f"Progress {position}/{FEATURE_IMAGES} | "
                f"New: {newly_processed} | "
                f"Cached: {previously_cached} | "
                f"Failed: {len(failed_images)}",
                flush=True
            )

    except Exception as error:

        failed_images.append({
            "image_id": image_id,
            "error": str(error)
        })

        print(
            f"Failed {image_id}: {error}",
            flush=True
        )

elapsed = time.perf_counter() - start_time

print("\nEXTRACTION SUMMARY")
print("=" * 60)

print("New images:", newly_processed)
print("Previously cached:", previously_cached)
print("Failed images:", len(failed_images))
print("Saved image files:",
      len(list(CACHE_DIR.glob("*.parquet"))))
print("Elapsed seconds:", round(elapsed, 2))

if failed_images:
    display(pd.DataFrame(failed_images))

In [ ]:
from pathlib import Path
import shutil
import time

source_cache_dir = Path(
    "/kaggle/input/datasets/vasudevgrover/"
    "filament-feature-cache/filament_feature_cache"
) / CACHE_DIR.name

cache_files = [
    image_cache_path(image_id)
    for image_id in feature_image_ids
]

print("Expected images:", len(cache_files))
print("Source cache:", source_cache_dir, flush=True)

frames = []
repaired_files = []
started = time.perf_counter()

required_columns = FEATURE_NAMES + [
    "target",
    "image_id",
    "foreground_fraction",
]


def read_checked_checkpoint(path, expected_image_id):
    if not path.is_file():
        raise FileNotFoundError(path)

    if path.stat().st_size == 0:
        raise ValueError(f"Empty checkpoint: {path.name}")

    frame = pd.read_parquet(path)

    missing = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{path.name}: missing columns {missing}"
        )

    if frame.empty:
        raise ValueError(f"{path.name}: no sampled pixels")

    if not frame["image_id"].astype(str).eq(
        str(expected_image_id)
    ).all():
        raise ValueError(
            f"{path.name}: image identity mismatch"
        )

    if not frame["target"].isin([0, 1]).all():
        raise ValueError(
            f"{path.name}: invalid target values"
        )

    if not np.isfinite(
        frame[FEATURE_NAMES].to_numpy()
    ).all():
        raise ValueError(
            f"{path.name}: non-finite feature values"
        )

    return frame


for position, (image_id, path) in enumerate(
    zip(feature_image_ids, cache_files),
    start=1
):
    try:
        frame = read_checked_checkpoint(path, image_id)

    except Exception as local_error:
        source = source_cache_dir / path.name

        print(
            f"Repairing {path.name}: {local_error}",
            flush=True
        )

        if source.resolve() == path.resolve():
            raise RuntimeError(
                "Source and destination must be different."
            ) from local_error

        temporary = path.with_suffix(".parquet.tmp")
        path.parent.mkdir(parents=True, exist_ok=True)

        try:
            shutil.copy2(source, temporary)

            # Validate the completed copy before replacing
            # the damaged or missing destination.
            frame = read_checked_checkpoint(
                temporary,
                image_id
            )

            temporary.replace(path)
            repaired_files.append(path.name)

        except Exception as source_error:
            temporary.unlink(missing_ok=True)

            raise RuntimeError(
                f"Could not recover checkpoint {path.name}. "
                f"The source file must be checked: {source}"
            ) from source_error

    frames.append(frame)

    if position % 100 == 0 or position == len(cache_files):
        print(
            f"Loaded {position}/{len(cache_files)} | "
            f"Repaired: {len(repaired_files)} | "
            f"Elapsed: {time.perf_counter() - started:.1f}s",
            flush=True
        )

pixel_df = pd.concat(
    frames,
    ignore_index=True
)

del frames

feature_columns = FEATURE_NAMES.copy()

prevalence_df = (
    pixel_df[["image_id", "foreground_fraction"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if prevalence_df["image_id"].duplicated().any():
    raise ValueError(
        "Inconsistent foreground fractions within an image."
    )

if pixel_df["image_id"].nunique() != FEATURE_IMAGES:
    raise ValueError(
        "Loaded image count does not match FEATURE_IMAGES."
    )

print("\nFEATURE DATASET READY")
print("=" * 60)
print("Repaired checkpoints:", len(repaired_files))
print("Total sampled pixels:", f"{len(pixel_df):,}")
print("Total images:", pixel_df["image_id"].nunique())
print("Feature count:", len(feature_columns))

print("\nForeground prevalence:")
display(prevalence_df["foreground_fraction"].describe())

print("\nFeature descriptive summary:")
display(pixel_df[feature_columns].describe().T)

#### 16.6 Univariate Feature Distribution Analysis

Univariate analysis is performed on the complete extracted training feature sample.

For each engineered feature, the following measurements are calculated:

- Mean and median
- Standard deviation
- Skewness
- First and ninety-ninth percentiles
- Minimum and maximum
- Non-finite observation count

Feature distributions are visualized using representative pixel samples to reduce plotting overhead.

Statistical summaries use the full extracted training feature dataset, whereas visualization uses a reproducible subset.

The analysis identifies candidate predictors with unusual distributional characteristics, low variability, extreme values, or numerical instability.

These descriptive findings will guide subsequent bivariate and multivariate investigations. Feature importance and statistical significance will be assessed separately.


In [ ]:
from scipy.stats import skew

univariate_records = []

for feature in feature_columns:

    x = pixel_df[feature].to_numpy(
        dtype=np.float64
    )

    finite_mask = np.isfinite(x)
    finite_x = x[finite_mask]

    if finite_x.size == 0:
        raise ValueError(
            f"No valid values found for {feature}"
        )

    univariate_records.append({
        "feature": feature,
        "count": len(x),
        "mean": finite_x.mean(),
        "median": np.median(finite_x),
        "std": finite_x.std(),
        "skewness": skew(finite_x),
        "min": finite_x.min(),
        "p01": np.percentile(finite_x, 1),
        "p25": np.percentile(finite_x, 25),
        "p75": np.percentile(finite_x, 75),
        "p99": np.percentile(finite_x, 99),
        "max": finite_x.max(),
        "nonfinite_count": int(
            (~finite_mask).sum()
        )
    })

univariate_df = pd.DataFrame(
    univariate_records
)

display(univariate_df.round(6))

univariate_df.to_csv(
    CACHE_ROOT / "univariate_statistics.csv",
    index=False
)

print("Univariate statistical analysis complete.")

In [ ]:
PLOT_SAMPLE_SIZE = 50000

plot_df = pixel_df.sample(
    n=min(PLOT_SAMPLE_SIZE, len(pixel_df)),
    random_state=SEED
)

fig, axes = plt.subplots(
    3, 3,
    figsize=(16, 12)
)

for ax, feature in zip(
    axes.flat,
    feature_columns
):

    values = plot_df[feature].to_numpy()
    values = values[np.isfinite(values)]

    ax.hist(
        values,
        bins=50,
        density=True,
        alpha=0.8
    )

    ax.set_title(feature)
    ax.set_xlabel("Feature value")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    CACHE_ROOT / "univariate_distributions.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
PLOT_PER_CLASS = 25000

fg_plot = pixel_df[
    pixel_df["target"] == 1
].sample(
    n=min(
        PLOT_PER_CLASS,
        (pixel_df["target"] == 1).sum()
    ),
    random_state=SEED
)

bg_plot = pixel_df[
    pixel_df["target"] == 0
].sample(
    n=min(
        PLOT_PER_CLASS,
        (pixel_df["target"] == 0).sum()
    ),
    random_state=SEED
)

fig, axes = plt.subplots(
    3, 3,
    figsize=(16, 12)
)

for ax, feature in zip(
    axes.flat,
    feature_columns
):

    foreground = fg_plot[feature].to_numpy()
    background = bg_plot[feature].to_numpy()

    ax.hist(
        background,
        bins=40,
        density=True,
        alpha=0.5,
        label="Background"
    )

    ax.hist(
        foreground,
        bins=40,
        density=True,
        alpha=0.5,
        label="Filament"
    )

    ax.set_title(feature)
    ax.grid(alpha=0.2)

axes[0, 0].legend()

plt.tight_layout()

plt.savefig(
    CACHE_ROOT / "foreground_background_distributions.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

## 17. Bivariate Statistical Analysis

### Objective

Investigate how each engineered image feature relates to filament
presence, including the strength, direction, shape, and consistency
of its association.

The analysis covers:

1. Linear association and standardized class separation.
2. Class-distribution differences and empirical cumulative distributions.
3. Feature-value response curves.
4. Single-feature detection tradeoffs.
5. Nonlinear dependence and estimator sensitivity.
6. Spatial variation in discrimination.
7. Within-image consistency and bootstrap uncertainty.
8. Temporal stability where capture metadata is available.

### Interpretation and sampling assumptions

The dataset contains cached pixels sampled separately from filament
and background regions within training images.

Consequently:

- The sampled filament fraction is not the natural full-image prevalence.
- Pooled statistics describe the cached sample.
- Pixels within an image are dependent.
- Pooled AUC may reflect both within-image discrimination and differences
  between images.
- Within-image AUC evaluates whether a feature separates the two classes
  inside the same observation.
- Statistical uncertainty is estimated using image-level summaries.
- Image-level bootstrap intervals remain conditional on treating image
  records as independent; temporal or duplicate-observation dependence
  requires the observation-group audit.
- Feature directions and thresholds are exploratory training-sample
  findings, not validated model performance.

No feature is removed solely because of a weak univariate relationship.
Interactions and complementary information are investigated in Step 18.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.feature_selection import mutual_info_classif

SEED = 42

required_variables = [
    "pixel_df",
    "feature_columns",
    "CACHE_ROOT",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Restore the feature cache before Step 17. "
        f"Missing variables: {missing_variables}"
    )

feature_columns = list(feature_columns)

if not feature_columns or len(set(feature_columns)) != len(feature_columns):
    raise ValueError("Feature names must be nonempty and unique.")

required_columns = feature_columns + [
    "target",
    "image_id",
    "foreground_fraction",
]

missing_columns = [
    column
    for column in required_columns
    if column not in pixel_df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

if pixel_df.empty:
    raise ValueError("The feature dataset is empty.")

if pixel_df["image_id"].isna().any():
    raise ValueError("Missing image identifiers.")

if not pixel_df["target"].isin([0, 1]).all():
    raise ValueError("Target must contain only 0 and 1.")

y_biv = pixel_df["target"].to_numpy(dtype=np.int8)

if np.unique(y_biv).size != 2:
    raise ValueError("Both target classes are required.")

for feature in feature_columns:
    values = pixel_df[feature].to_numpy(dtype=np.float64)

    if not np.isfinite(values).all():
        raise ValueError(
            f"{feature}: non-finite values. "
            "Investigate the cache before proceeding."
        )

prevalence_check = pixel_df[
    ["image_id", "foreground_fraction"]
].drop_duplicates()

if prevalence_check["image_id"].duplicated().any():
    raise ValueError("Inconsistent prevalence within an image.")

prevalence_values = prevalence_check[
    "foreground_fraction"
].to_numpy(dtype=float)

if (
    not np.isfinite(prevalence_values).all()
    or ((prevalence_values < 0) | (prevalence_values > 1)).any()
):
    raise ValueError("Invalid full-image foreground fractions.")

BIVARIATE_DIR = Path(CACHE_ROOT) / "step_17_bivariate"
BIVARIATE_DIR.mkdir(parents=True, exist_ok=True)

FG_COLOR = "#d95f02"
BG_COLOR = "#1f77b4"

feature_labels = {
    name: name.replace("_", " ").title()
    for name in feature_columns
}


def finish_bivariate_figure(fig, filename):
    fig.savefig(
        BIVARIATE_DIR / filename,
        dpi=170,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


def feature_grid():
    columns = 3
    rows = int(np.ceil(len(feature_columns) / columns))

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(17, 4 * rows),
        squeeze=False,
    )

    for ax in axes.flat[len(feature_columns):]:
        ax.set_visible(False)

    return fig, axes


print("STEP 17 INPUTS READY")
print("=" * 60)
print("Sampled pixels:", f"{len(pixel_df):,}")
print("Image records:", pixel_df["image_id"].nunique())
print("Features:", len(feature_columns))
print("Sampled filament fraction:", round(y_biv.mean(), 4))
print("Output directory:", BIVARIATE_DIR)

display(
    pixel_df.groupby("target")
    .size()
    .rename("sampled_pixels")
    .to_frame()
)

pd.DataFrame(
    {
        "feature": feature_columns,
        "label": [feature_labels[f] for f in feature_columns],
    }
).to_csv(
    BIVARIATE_DIR / "feature_manifest.csv",
    index=False,
)

### 17.1 Association Strength and Direction

For each feature, we calculate:

- Point-biserial correlation: linear association with the binary target.
- Standardized mean difference: difference between filament and
  background means divided by an equally weighted within-class
  reference standard deviation.
- Raw ROC-AUC: probability that a randomly selected filament sample
  has a higher feature value than a background sample, with half
  credit for ties.
- Rank-biserial association: 2 × raw AUC − 1.
- Separation AUC: max(AUC, 1 − AUC), allowing either ranking direction.

Positive signed measures indicate higher feature values for filament
pixels; negative measures indicate lower values.

Separation AUC chooses its direction on the current training sample.
It is an exploratory description and may be optimistic for weak features.

Constant features have undefined correlation and standardized mean
difference; their ranking AUC is 0.5.

In [ ]:
association_records = []

for feature in feature_columns:
    x = pixel_df[feature].to_numpy(dtype=np.float64)

    foreground = x[y_biv == 1]
    background = x[y_biv == 0]

    if min(len(foreground), len(background)) < 2:
        raise ValueError("At least two samples per class are required.")

    fg_mean = foreground.mean()
    bg_mean = background.mean()

    reference_sd = np.sqrt(
        (
            foreground.var(ddof=1)
            + background.var(ddof=1)
        ) / 2
    )

    smd = (
        (fg_mean - bg_mean) / reference_sd
        if reference_sd > 0
        else np.nan
    )

    # Pearson correlation with a binary target is point-biserial r.
    point_biserial = (
        np.corrcoef(x, y_biv)[0, 1]
        if np.ptp(x) > 0
        else np.nan
    )

    raw_auc = roc_auc_score(y_biv, x)

    association_records.append(
        {
            "feature": feature,
            "n_filament": len(foreground),
            "n_background": len(background),
            "filament_mean": fg_mean,
            "background_mean": bg_mean,
            "filament_median": np.median(foreground),
            "background_median": np.median(background),
            "filament_iqr": (
                np.quantile(foreground, 0.75)
                - np.quantile(foreground, 0.25)
            ),
            "background_iqr": (
                np.quantile(background, 0.75)
                - np.quantile(background, 0.25)
            ),
            "point_biserial_r": point_biserial,
            "standardized_mean_difference": smd,
            "raw_roc_auc": raw_auc,
            "rank_biserial": 2 * raw_auc - 1,
            "separation_auc": max(raw_auc, 1 - raw_auc),
            "orientation_sign": 1 if raw_auc >= 0.5 else -1,
            "ranking_direction": (
                "Higher"
                if raw_auc > 0.5
                else "Lower"
                if raw_auc < 0.5
                else "No preference"
            ),
        }
    )

bivariate_df = (
    pd.DataFrame(association_records)
    .sort_values("separation_auc", ascending=False)
    .reset_index(drop=True)
)

bivariate_df.insert(
    0,
    "rank",
    np.arange(1, len(bivariate_df) + 1),
)

bivariate_df.to_csv(
    BIVARIATE_DIR / "feature_target_associations.csv",
    index=False,
)

display(bivariate_df.round(4))

association_lookup = bivariate_df.set_index("feature")

# Freeze the pooled direction for subsequent diagnostic comparisons.
feature_direction = association_lookup[
    "orientation_sign"
].to_dict()

plot_table = bivariate_df.sort_values("separation_auc")
positions = np.arange(len(plot_table))
labels = [
    feature_labels[f]
    for f in plot_table["feature"]
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].barh(
    positions,
    plot_table["separation_auc"],
    color="#377eb8",
)
axes[0].axvline(0.5, color="black", linestyle="--")
axes[0].set_xlim(0.45, 1.02)
axes[0].set_title("Pooled single-feature separation")
axes[0].set_xlabel("Direction-independent AUC")

for index, value in enumerate(plot_table["separation_auc"]):
    axes[0].text(value + 0.006, index, f"{value:.3f}", va="center")

signed_metrics = [
    ("point_biserial_r", "Point-biserial correlation"),
    ("standardized_mean_difference", "Standardized mean difference"),
]

for ax, (metric, title) in zip(axes[1:], signed_metrics):
    values = plot_table[metric].to_numpy()

    ax.barh(
        positions,
        values,
        color=[
            FG_COLOR if value >= 0 else BG_COLOR
            for value in values
        ],
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Signed association")

for ax in axes:
    ax.set_yticks(positions)
    ax.set_yticklabels(labels)
    ax.grid(axis="x", alpha=0.2)

fig.suptitle(
    "Feature–target relationships in the cached training sample",
    fontsize=15,
)
fig.tight_layout()

finish_bivariate_figure(
    fig,
    "17_1_association_rankings.png",
)

heat_columns = [
    "point_biserial_r",
    "rank_biserial",
]

heat_data = association_lookup.loc[
    feature_columns, heat_columns
].to_numpy()

fig, ax = plt.subplots(figsize=(8, 7))

heat_image = ax.imshow(
    heat_data,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    aspect="auto",
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Point-biserial r", "Rank-biserial"])
ax.set_yticks(np.arange(len(feature_columns)))
ax.set_yticklabels([feature_labels[f] for f in feature_columns])

for row in range(heat_data.shape[0]):
    for column in range(heat_data.shape[1]):
        value = heat_data[row, column]
        ax.text(
            column,
            row,
            f"{value:.3f}",
            ha="center",
            va="center",
            color="white" if abs(value) > 0.6 else "black",
        )

fig.colorbar(heat_image, ax=ax, label="Signed association")
ax.set_title("Linear and rank-based association")
fig.tight_layout()

finish_bivariate_figure(
    fig,
    "17_1_association_heatmap.png",
)

### 17.2 Class-Distribution Separation

We examine differences between filament and background distributions
using three complementary views:

1. Empirical cumulative distribution functions (ECDFs), calculated
   from the complete cached sample.
2. Violin plots, calculated from a reproducible plotting subset.
3. Quantile comparisons, calculated from the complete cached sample.

An ECDF describes the fraction of observations at or below a feature
value. Separation between class ECDFs indicates distributional differences.

Quantile comparisons show whether differences are consistent across the
distribution or concentrated in particular tails.

Plotting samples are used only for visualization. Numerical association
statistics continue to use the complete cached sample.

Robust scaling in violin plots uses the pooled median and interquartile
range. It changes display units without changing class membership.

In [ ]:
PLOT_PER_CLASS = 25000

plot_classes = {
    target: pixel_df.loc[
        pixel_df["target"].eq(target),
        feature_columns,
    ].sample(
        n=min(
            PLOT_PER_CLASS,
            int(pixel_df["target"].eq(target).sum()),
        ),
        random_state=SEED,
    )
    for target in [0, 1]
}

# Exact ECDF heights at up to 1,200 display positions.
fig, axes = feature_grid()

for ax, feature in zip(axes.flat, feature_columns):
    for target, label_text, color in [
        (0, "Background", BG_COLOR),
        (1, "Filament", FG_COLOR),
    ]:
        sorted_values = np.sort(
            pixel_df.loc[
                pixel_df["target"].eq(target),
                feature,
            ].to_numpy(dtype=float)
        )

        indices = np.unique(
            np.linspace(
                0,
                len(sorted_values) - 1,
                min(1200, len(sorted_values)),
                dtype=int,
            )
        )

        display_x = sorted_values[indices]

        # searchsorted handles repeated values correctly.
        display_y = np.searchsorted(
            sorted_values,
            display_x,
            side="right",
        ) / len(sorted_values)

        ax.step(
            display_x,
            display_y,
            where="post",
            label=label_text,
            color=color,
        )

    ax.set_title(feature_labels[feature])
    ax.set_ylabel("Cumulative fraction")
    ax.set_xlabel("Feature value")
    ax.grid(alpha=0.2)

axes.flat[0].legend()
fig.tight_layout()

finish_bivariate_figure(fig, "17_2_class_ecdf.png")

# Violin plots with explicit robust display scaling.
fig, axes = feature_grid()

for ax, feature in zip(axes.flat, feature_columns):
    full_values = pixel_df[feature].to_numpy(dtype=float)
    q25, center, q75 = np.quantile(
        full_values,
        [0.25, 0.5, 0.75],
    )

    scale = q75 - q25
    if scale <= 0:
        scale = full_values.std()
    if scale <= 0:
        scale = 1.0

    transformed = [
        (
            plot_classes[target][feature].to_numpy(dtype=float)
            - center
        ) / scale
        for target in [0, 1]
    ]

    # Constant distributions cannot produce a kernel-density violin.
    if all(np.ptp(values) > 0 for values in transformed):
        parts = ax.violinplot(
            transformed,
            positions=[0, 1],
            showmedians=True,
            showextrema=False,
        )

        for body, color in zip(
            parts["bodies"],
            [BG_COLOR, FG_COLOR],
        ):
            body.set_facecolor(color)
            body.set_alpha(0.65)
    else:
        ax.boxplot(
            transformed,
            positions=[0, 1],
            showfliers=False,
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Background", "Filament"])
    ax.set_ylabel("(Value − pooled median) / display scale")
    ax.set_title(feature_labels[feature])
    ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
finish_bivariate_figure(fig, "17_2_class_violins.png")

quantile_records = []
quantile_levels = np.linspace(0.01, 0.99, 99)

fig, axes = feature_grid()

for ax, feature in zip(axes.flat, feature_columns):
    bg_quantiles = np.quantile(
        pixel_df.loc[pixel_df["target"].eq(0), feature],
        quantile_levels,
    )
    fg_quantiles = np.quantile(
        pixel_df.loc[pixel_df["target"].eq(1), feature],
        quantile_levels,
    )

    ax.scatter(
        bg_quantiles,
        fg_quantiles,
        c=quantile_levels,
        cmap="viridis",
        s=15,
    )

    lower = min(bg_quantiles.min(), fg_quantiles.min())
    upper = max(bg_quantiles.max(), fg_quantiles.max())

    ax.plot(
        [lower, upper],
        [lower, upper],
        color="black",
        linestyle="--",
        linewidth=1,
    )

    ax.set_title(feature_labels[feature])
    ax.set_xlabel("Background quantile")
    ax.set_ylabel("Filament quantile")
    ax.grid(alpha=0.2)

    quantile_records.extend(
        {
            "feature": feature,
            "quantile": q,
            "background_value": bg,
            "filament_value": fg,
        }
        for q, bg, fg in zip(
            quantile_levels,
            bg_quantiles,
            fg_quantiles,
        )
    )

fig.tight_layout()
finish_bivariate_figure(
    fig,
    "17_2_class_quantile_comparisons.png",
)

pd.DataFrame(quantile_records).to_csv(
    BIVARIATE_DIR / "class_quantiles.csv",
    index=False,
)

### 17.3 Feature-Value Response Curves

We partition each feature into pooled quantile bins and calculate the
fraction of cached samples labeled as filament within each bin.

These curves reveal:

- Monotonic associations.
- Threshold-like changes.
- Intermediate-value preferences.
- Saturation and nonmonotonic behavior.
- Bins with limited image support.

The vertical axis is the sampled filament fraction, not the probability
of a filament in a natural full-resolution image.

Repeated feature values can reduce the number of distinct quantile bins.
We retain the resulting bins rather than artificially splitting ties.

Image-support counts indicate how broadly each bin is represented.
No pixel-independence confidence intervals are attached to these curves.

In [ ]:
N_RESPONSE_BINS = 20

response_tables = []
fig, axes = feature_grid()

for ax, feature in zip(axes.flat, feature_columns):
    x = pixel_df[feature]

    if x.nunique() == 1:
        bin_codes = pd.Series(
            np.zeros(len(pixel_df), dtype=int),
            index=pixel_df.index,
        )
    else:
        bin_codes = pd.qcut(
            x,
            q=N_RESPONSE_BINS,
            labels=False,
            duplicates="drop",
        )

    temporary = pd.DataFrame(
        {
            "bin": bin_codes.to_numpy(),
            "value": x.to_numpy(),
            "target": y_biv,
            "image_id": pixel_df["image_id"].to_numpy(),
        }
    )

    summary = (
        temporary.groupby("bin", observed=True)
        .agg(
            feature_min=("value", "min"),
            feature_median=("value", "median"),
            feature_max=("value", "max"),
            sampled_filament_fraction=("target", "mean"),
            sampled_pixels=("target", "size"),
            represented_images=("image_id", "nunique"),
        )
        .reset_index()
    )

    summary.insert(0, "feature", feature)
    summary["quantile_bin_number"] = np.arange(
        1, len(summary) + 1
    )

    response_tables.append(summary)

    ax.plot(
        summary["quantile_bin_number"],
        summary["sampled_filament_fraction"],
        marker="o",
        markersize=4,
        color=FG_COLOR,
        label="Sampled filament fraction",
    )

    ax.axhline(
        y_biv.mean(),
        color="black",
        linestyle="--",
        linewidth=1,
        label="Overall sampled fraction",
    )

    support_axis = ax.twinx()
    support_axis.bar(
        summary["quantile_bin_number"],
        summary["represented_images"],
        color="gray",
        alpha=0.13,
    )
    support_axis.set_ylabel("Represented image records")

    ax.set_ylim(0, 1)
    ax.set_title(feature_labels[feature])
    ax.set_xlabel("Feature quantile bin: low → high")
    ax.set_ylabel("Sampled filament fraction")
    ax.grid(alpha=0.2)

axes.flat[0].legend(fontsize=8, loc="upper left")
fig.tight_layout()

finish_bivariate_figure(
    fig,
    "17_3_quantile_response_and_support.png",
)

response_df = pd.concat(response_tables, ignore_index=True)

response_df.to_csv(
    BIVARIATE_DIR / "quantile_response_statistics.csv",
    index=False,
)

display(response_df.head(20).round(4))

### 17.4 Single-Feature Detection Tradeoffs

Each feature is oriented using its pooled training-sample AUC direction.

We calculate:

- ROC curves.
- Background false-positive rate versus filament recall.
- Standardized-score threshold response curves.
- Maximum observed recall at selected false-positive-rate budgets.

Threshold scores are standardized using the pooled mean and standard
deviation, then oriented so larger scores indicate filament.

Operational thresholds are also saved in original feature units.

These results describe exploratory threshold rules. They are not final
segmentation thresholds, instance predictions, or PQ estimates.

Precision–recall curves are not interpreted as full-image performance
because class-based sampling changes the foreground prevalence.

In [ ]:
roc_cache = {}
operating_records = []

FPR_BUDGETS = [0.001, 0.01, 0.05, 0.10]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for feature in feature_columns:
    x = pixel_df[feature].to_numpy(dtype=float)
    direction = feature_direction[feature]

    center = x.mean()
    scale = x.std()

    if scale <= 0:
        scale = 1.0

    score = direction * (x - center) / scale

    fpr, tpr, thresholds = roc_curve(
        y_biv,
        score,
        drop_intermediate=True,
    )

    oriented_auc = roc_auc_score(y_biv, score)

    roc_cache[feature] = {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "center": center,
        "scale": scale,
        "direction": direction,
    }

    for ax in axes:
        ax.plot(
            fpr,
            tpr,
            linewidth=1.6,
            label=(
                f"{feature_labels[feature]} "
                f"({oriented_auc:.3f})"
            ),
        )

    for budget in FPR_BUDGETS:
        eligible = np.flatnonzero(fpr <= budget)
        best = eligible[np.argmax(tpr[eligible])]

        threshold_score = thresholds[best]

        original_threshold = (
            center + direction * threshold_score * scale
        )

        operating_records.append(
            {
                "feature": feature,
                "fpr_budget": budget,
                "achieved_fpr": fpr[best],
                "filament_recall": tpr[best],
                "standardized_threshold": threshold_score,
                "original_feature_threshold": original_threshold,
                "prediction_rule": (
                    "feature >= threshold"
                    if direction == 1
                    else "feature <= threshold"
                ),
            }
        )

axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_title("Pooled ROC curves")
axes[0].legend(fontsize=8, loc="lower right")

axes[1].set_xlim(0, 0.10)
axes[1].set_title("Low false-positive-rate region")

for ax in axes:
    ax.set_xlabel("Background false-positive rate")
    ax.set_ylabel("Filament recall")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)

fig.tight_layout()
finish_bivariate_figure(fig, "17_4_roc_curves.png")

fig, axes = feature_grid()

for ax, feature in zip(axes.flat, feature_columns):
    result = roc_cache[feature]
    thresholds = result["thresholds"]

    finite = np.flatnonzero(np.isfinite(thresholds))

    display_indices = finite[
        np.unique(
            np.linspace(
                0,
                len(finite) - 1,
                min(600, len(finite)),
                dtype=int,
            )
        )
    ]

    order = np.argsort(thresholds[display_indices])
    selected = display_indices[order]

    ax.plot(
        thresholds[selected],
        result["tpr"][selected],
        color=FG_COLOR,
        label="Filament recall",
    )
    ax.plot(
        thresholds[selected],
        result["fpr"][selected],
        color=BG_COLOR,
        label="Background FPR",
    )

    ax.set_title(feature_labels[feature])
    ax.set_xlabel("Oriented standardized threshold")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)

axes.flat[0].legend(fontsize=8)
fig.tight_layout()

finish_bivariate_figure(
    fig,
    "17_4_threshold_response.png",
)

operating_df = pd.DataFrame(operating_records)

operating_df.to_csv(
    BIVARIATE_DIR / "single_feature_operating_points.csv",
    index=False,
)

display(
    operating_df.pivot(
        index="feature",
        columns="fpr_budget",
        values="filament_recall",
    ).round(4)
)

### 17.5 Nonlinear Dependence and Estimator Sensitivity

Mutual information measures dependence beyond a strictly linear
relationship.

We estimate it on a reproducible subset of cached pixels, using several
nearest-neighbor settings to assess sensitivity.

Important limitations:

- Estimates depend on sample size, sampling design, ties, and estimator
  settings.
- The estimator treats engineered features as continuous.
- A fixed subset controls computational cost but does not represent
  independent pixel observations.
- The sensitivity range is not a confidence interval.
- Mutual information is descriptive and does not establish causal
  relevance or incremental predictive value.

We compare mutual information with absolute linear correlation and
single-feature AUC. Joint contributions are evaluated in Step 18.

In [ ]:
MI_SAMPLE_SIZE = min(50000, len(pixel_df))
MI_NEIGHBOR_SETTINGS = [3, 5, 10]

mi_sample = pixel_df.sample(
    n=MI_SAMPLE_SIZE,
    random_state=SEED,
)

mi_x = mi_sample[feature_columns].to_numpy(dtype=float)
mi_y = mi_sample["target"].to_numpy(dtype=np.int8)

# Standardize the MI input for numerical conditioning.
mi_center = mi_x.mean(axis=0)
mi_scale = mi_x.std(axis=0)
mi_scale[mi_scale == 0] = 1.0

mi_x = (mi_x - mi_center) / mi_scale

constant_mask = np.ptp(mi_x, axis=0) == 0
mi_records = []

for neighbors in MI_NEIGHBOR_SETTINGS:
    print(
        f"Estimating mutual information: k={neighbors}",
        flush=True,
    )

    estimates = mutual_info_classif(
        mi_x,
        mi_y,
        discrete_features=False,
        n_neighbors=neighbors,
        random_state=SEED,
    )

    estimates[constant_mask] = 0.0

    mi_records.extend(
        {
            "feature": feature,
            "neighbors": neighbors,
            "mutual_information_nats": value,
            "sampled_pixels": MI_SAMPLE_SIZE,
        }
        for feature, value in zip(
            feature_columns,
            estimates,
        )
    )

mi_df = pd.DataFrame(mi_records)

mi_summary = (
    mi_df.groupby("feature")
    .agg(
        mi_mean=("mutual_information_nats", "mean"),
        mi_min=("mutual_information_nats", "min"),
        mi_max=("mutual_information_nats", "max"),
    )
    .join(
        association_lookup[
            ["point_biserial_r", "separation_auc"]
        ]
    )
    .sort_values("mi_mean")
)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))

positions = np.arange(len(mi_summary))
means = mi_summary["mi_mean"].to_numpy()

axes[0].barh(
    positions,
    means,
    xerr=np.vstack(
        [
            means - mi_summary["mi_min"].to_numpy(),
            mi_summary["mi_max"].to_numpy() - means,
        ]
    ),
    color="#756bb1",
    capsize=3,
)

axes[0].set_yticks(positions)
axes[0].set_yticklabels(
    [feature_labels[f] for f in mi_summary.index]
)
axes[0].set_xlabel("Estimated mutual information (nats)")
axes[0].set_title("Mean and range across neighbor settings")

for feature, row in mi_summary.iterrows():
    axes[1].scatter(
        abs(row["point_biserial_r"]),
        row["mi_mean"],
        s=55,
    )
    axes[1].annotate(
        feature_labels[feature],
        (
            abs(row["point_biserial_r"]),
            row["mi_mean"],
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

axes[1].set_xlabel("Absolute point-biserial correlation")
axes[1].set_ylabel("Mean estimated mutual information")
axes[1].set_title("Linear association versus nonlinear dependence")

for ax in axes:
    ax.grid(alpha=0.2)

fig.tight_layout()
finish_bivariate_figure(fig, "17_5_mutual_information.png")

mi_df.to_csv(
    BIVARIATE_DIR / "mutual_information_sensitivity.csv",
    index=False,
)

mi_summary.to_csv(
    BIVARIATE_DIR / "mutual_information_summary.csv",
)

display(mi_summary.round(4))

### 17.6 Spatially Conditional Discrimination

We investigate whether feature discrimination changes with radial
position.

The existing radial-distance feature is based on image-center coordinates
normalized by half the image dimensions. It is not a fitted physical
solar-disk radius.

We therefore use image-centered radial bands and retain an outer band
that may contain limb and off-disk background.

For each feature and band, we calculate AUC using the direction fixed
from the pooled training analysis. We do not independently reverse weak
features inside each band.

Values below 0.5 indicate that the pooled ranking direction reverses
within that band.

For radial distance itself, conditioning on radial bands restricts its
range. Its conditional AUC must be interpreted accordingly.

AUC is undefined when a band contains only one target class.

In [ ]:
if "radial_distance" not in pixel_df.columns:
    raise ValueError("radial_distance is required.")

RADIAL_EDGES = [0, 0.35, 0.65, 0.90, np.inf]
RADIAL_LABELS = [
    "r < 0.35",
    "0.35–0.65",
    "0.65–0.90",
    "r ≥ 0.90",
]

radial_band = pd.cut(
    pixel_df["radial_distance"],
    bins=RADIAL_EDGES,
    labels=RADIAL_LABELS,
    include_lowest=True,
    right=False,
)

if radial_band.isna().any():
    raise ValueError("Some radial values fall outside the defined bands.")

spatial_records = []

for band in RADIAL_LABELS:
    selected = radial_band.eq(band).to_numpy()
    band_target = y_biv[selected]

    n_fg = int((band_target == 1).sum())
    n_bg = int((band_target == 0).sum())

    for feature in feature_columns:
        auc = np.nan

        if n_fg > 0 and n_bg > 0:
            score = (
                pixel_df.loc[selected, feature].to_numpy(dtype=float)
                * feature_direction[feature]
            )

            auc = roc_auc_score(band_target, score)

        spatial_records.append(
            {
                "radial_band": band,
                "feature": feature,
                "fixed_direction_auc": auc,
                "n_filament": n_fg,
                "n_background": n_bg,
                "represented_images": (
                    pixel_df.loc[selected, "image_id"].nunique()
                ),
            }
        )

spatial_df = pd.DataFrame(spatial_records)

spatial_matrix = spatial_df.pivot(
    index="feature",
    columns="radial_band",
    values="fixed_direction_auc",
).reindex(
    index=feature_columns,
    columns=RADIAL_LABELS,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(17, 7),
    gridspec_kw={"width_ratios": [1.6, 1]},
)

heat = axes[0].imshow(
    spatial_matrix.to_numpy(),
    cmap="RdBu",
    vmin=0,
    vmax=1,
    aspect="auto",
)

axes[0].set_xticks(np.arange(len(RADIAL_LABELS)))
axes[0].set_xticklabels(RADIAL_LABELS)
axes[0].set_yticks(np.arange(len(feature_columns)))
axes[0].set_yticklabels([feature_labels[f] for f in feature_columns])
axes[0].set_title("AUC by image-centered radial band")

for row in range(len(feature_columns)):
    for column in range(len(RADIAL_LABELS)):
        value = spatial_matrix.iloc[row, column]
        axes[0].text(
            column,
            row,
            f"{value:.3f}" if np.isfinite(value) else "N/A",
            ha="center",
            va="center",
            color=(
                "white"
                if np.isfinite(value) and abs(value - 0.5) > 0.3
                else "black"
            ),
        )

fig.colorbar(heat, ax=axes[0], label="Fixed-direction AUC")

support = (
    spatial_df.drop_duplicates("radial_band")
    .set_index("radial_band")
    .reindex(RADIAL_LABELS)
)

positions = np.arange(len(RADIAL_LABELS))

axes[1].bar(
    positions - 0.18,
    support["n_background"],
    width=0.36,
    label="Background",
    color=BG_COLOR,
)
axes[1].bar(
    positions + 0.18,
    support["n_filament"],
    width=0.36,
    label="Filament",
    color=FG_COLOR,
)

axes[1].set_xticks(positions)
axes[1].set_xticklabels(RADIAL_LABELS, rotation=25)
axes[1].set_ylabel("Cached sampled pixels")
axes[1].set_title("Class support within each band")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.2)

fig.tight_layout()
finish_bivariate_figure(fig, "17_6_radial_discrimination.png")

spatial_df.to_csv(
    BIVARIATE_DIR / "radial_discrimination.csv",
    index=False,
)

display(spatial_matrix.round(4))
display(support[["n_filament", "n_background", "represented_images"]])

### 17.7 Within-Image Consistency and Image-Level Uncertainty

Pooled discrimination can be influenced by differences between images.
We therefore calculate raw AUC separately inside each image record.

We then apply the same feature direction selected from the pooled sample.
The direction is never optimized separately for each image.

For each feature, we report:

- Mean and median within-image AUC.
- Interquartile range and lower/upper distribution quantiles.
- Fraction of eligible images with AUC above 0.5.
- Fraction with AUC below 0.5.
- Image-bootstrap interval for the mean within-image AUC.

Each eligible image has equal weight in the mean.

The bootstrap interval estimates uncertainty in the mean of the
within-image AUC summaries. It is not an interval for pooled pixel AUC,
nor a prediction interval for a future individual image.

The pooled feature direction is held fixed during bootstrap resampling.
The interval does not account for selecting that direction on the same
data.

Images with only one sampled target class are excluded from within-image
AUC and their count is reported.

Potential duplicate observations and temporal dependence must be resolved
before these image-level intervals are treated as final inferential
results.

In [ ]:
image_auc_records = []
image_groups = pixel_df.groupby("image_id", sort=False)
total_image_groups = image_groups.ngroups

for position, (image_id, frame) in enumerate(image_groups, start=1):
    image_target = frame["target"].to_numpy(dtype=np.int8)
    has_both_classes = np.unique(image_target).size == 2

    for feature in feature_columns:
        raw_auc = np.nan

        if has_both_classes:
            raw_auc = roc_auc_score(
                image_target,
                frame[feature].to_numpy(dtype=float),
            )

        direction = feature_direction[feature]

        fixed_auc = (
            raw_auc
            if direction == 1
            else 1 - raw_auc
        )

        image_auc_records.append(
            {
                "image_id": str(image_id),
                "feature": feature,
                "raw_auc": raw_auc,
                "fixed_direction_auc": fixed_auc,
                "n_filament": int((image_target == 1).sum()),
                "n_background": int((image_target == 0).sum()),
                "foreground_fraction": float(
                    frame["foreground_fraction"].iloc[0]
                ),
            }
        )

    if position % 100 == 0 or position == total_image_groups:
        print(
            f"Within-image analysis: {position}/{total_image_groups}",
            flush=True,
        )

image_auc_df = pd.DataFrame(image_auc_records)

BOOTSTRAP_REPLICATES = 2000
rng = np.random.default_rng(SEED)

stability_records = []

for feature in feature_columns:
    values = image_auc_df.loc[
        image_auc_df["feature"].eq(feature),
        "fixed_direction_auc",
    ].dropna().to_numpy()

    if len(values) == 0:
        raise ValueError(
            f"{feature}: no image records have both classes."
        )

    bootstrap_means = np.empty(BOOTSTRAP_REPLICATES)

    # Resampling image summaries avoids resampling millions of pixels.
    for replicate in range(BOOTSTRAP_REPLICATES):
        bootstrap_means[replicate] = rng.choice(
            values,
            size=len(values),
            replace=True,
        ).mean()

    ci_low, ci_high = np.quantile(
        bootstrap_means,
        [0.025, 0.975],
    )

    stability_records.append(
        {
            "feature": feature,
            "eligible_images": len(values),
            "excluded_images": total_image_groups - len(values),
            "mean_within_image_auc": values.mean(),
            "median_within_image_auc": np.median(values),
            "q05": np.quantile(values, 0.05),
            "q25": np.quantile(values, 0.25),
            "q75": np.quantile(values, 0.75),
            "q95": np.quantile(values, 0.95),
            "fraction_above_chance": (values > 0.5).mean(),
            "fraction_below_chance": (values < 0.5).mean(),
            "mean_auc_ci_low": ci_low,
            "mean_auc_ci_high": ci_high,
        }
    )

stability_df = (
    pd.DataFrame(stability_records)
    .sort_values("mean_within_image_auc")
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(19, 8))

ordered_features = stability_df["feature"].tolist()
positions = np.arange(len(ordered_features))

distributions = [
    image_auc_df.loc[
        image_auc_df["feature"].eq(feature),
        "fixed_direction_auc",
    ].dropna().to_numpy()
    for feature in ordered_features
]

axes[0].boxplot(
    distributions,
    vert=False,
    showfliers=False,
    positions=positions,
)

axes[0].set_yticks(positions)
axes[0].set_yticklabels(
    [feature_labels[f] for f in ordered_features]
)
axes[0].axvline(0.5, color="black", linestyle="--")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Within-image AUC")
axes[0].set_title("Distribution across image records")

means = stability_df["mean_within_image_auc"].to_numpy()

axes[1].errorbar(
    means,
    positions,
    xerr=np.vstack(
        [
            means - stability_df["mean_auc_ci_low"].to_numpy(),
            stability_df["mean_auc_ci_high"].to_numpy() - means,
        ]
    ),
    fmt="o",
    capsize=4,
    color="#238b45",
    label="Mean within-image AUC + 95% bootstrap interval",
)

pooled_values = association_lookup.loc[
    ordered_features,
    "separation_auc",
].to_numpy()

axes[1].scatter(
    pooled_values,
    positions,
    marker="x",
    s=60,
    color="#984ea3",
    label="Pooled separation AUC",
)

axes[1].set_yticks(positions)
axes[1].set_yticklabels(
    [feature_labels[f] for f in ordered_features]
)
axes[1].axvline(0.5, color="black", linestyle="--")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("AUC")
axes[1].set_title("Pooled versus average within-image discrimination")
axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
finish_bivariate_figure(fig, "17_7_image_stability.png")

image_auc_df.to_csv(
    BIVARIATE_DIR / "per_image_feature_auc.csv",
    index=False,
)

stability_df.to_csv(
    BIVARIATE_DIR / "within_image_auc_bootstrap.csv",
    index=False,
)

display(stability_df.round(4))

### 17.8 Temporal Stability of Feature Discrimination

Where image capture metadata is available, we summarize within-image AUC
over calendar periods.

The analysis uses the fixed pooled feature direction and summarizes
image-level results rather than pooling all pixels within a period.

We report both mean AUC and the number of eligible image records per
period. Sparse periods receive explicit support information.

These summaries describe variation over observed dates. They are not
out-of-time model validation and do not establish temporal independence.

If date metadata has not been restored in the current kernel, this
subsection records an explicit pending status rather than inferring dates
from undocumented image identifiers.

In [ ]:
date_candidates = []

# Metadata structures present in your notebook.
for variable_name in ["train_df", "image_summary"]:
    candidate = globals().get(variable_name)

    if isinstance(candidate, pd.DataFrame):
        id_column = (
            "image_id"
            if "image_id" in candidate.columns
            else "id"
            if "id" in candidate.columns
            else None
        )

        date_column = next(
            (
                name
                for name in [
                    "capture_datetime",
                    "date_captured",
                    "capture_date",
                ]
                if name in candidate.columns
            ),
            None,
        )

        if id_column and date_column:
            metadata = candidate[[id_column, date_column]].copy()
            metadata.columns = ["image_id", "capture_datetime"]
            date_candidates.append(metadata)

annotation_object = globals().get("annotations_data")

if isinstance(annotation_object, dict):
    annotation_images = pd.DataFrame(
        annotation_object.get("images", [])
    )

    if {"id", "date_captured"}.issubset(annotation_images.columns):
        metadata = annotation_images[
            ["id", "date_captured"]
        ].copy()
        metadata.columns = ["image_id", "capture_datetime"]
        date_candidates.append(metadata)

temporal_status = "pending_metadata"
temporal_summary = pd.DataFrame()

if not date_candidates:
    print(
        "Temporal analysis pending: capture metadata is not "
        "loaded in this kernel. Restore the annotation/date "
        "metadata and rerun this cell."
    )

else:
    capture_metadata = pd.concat(
        date_candidates,
        ignore_index=True,
    )

    capture_metadata["image_id"] = (
        capture_metadata["image_id"].astype(str)
    )

    capture_metadata["capture_datetime"] = pd.to_datetime(
        capture_metadata["capture_datetime"],
        errors="coerce",
        utc=True,
    )

    capture_metadata = (
        capture_metadata.dropna(subset=["capture_datetime"])
        .drop_duplicates()
    )

    relevant_ids = set(image_auc_df["image_id"])

    capture_metadata = capture_metadata[
        capture_metadata["image_id"].isin(relevant_ids)
    ]

    if capture_metadata["image_id"].duplicated().any():
        raise ValueError(
            "Conflicting capture timestamps for an image ID."
        )

    temporal_auc_df = image_auc_df.merge(
        capture_metadata,
        on="image_id",
        how="left",
        validate="many_to_one",
    )

    missing_date_images = temporal_auc_df.loc[
        temporal_auc_df["capture_datetime"].isna(),
        "image_id",
    ].nunique()

    print("Image records without valid dates:", missing_date_images)

    dated = temporal_auc_df.dropna(
        subset=["capture_datetime", "fixed_direction_auc"]
    ).copy()

    if dated.empty:
        print("Temporal analysis pending: no usable dated AUC records.")

    else:
        dated["quarter"] = (
            dated["capture_datetime"]
            .dt.tz_convert(None)
            .dt.to_period("Q")
            .astype(str)
        )

        temporal_summary = (
            dated.groupby(["quarter", "feature"])
            .agg(
                mean_auc=("fixed_direction_auc", "mean"),
                median_auc=("fixed_direction_auc", "median"),
                eligible_images=("image_id", "nunique"),
            )
            .reset_index()
        )

        quarter_order = sorted(
            temporal_summary["quarter"].unique()
        )

        mean_matrix = temporal_summary.pivot(
            index="feature",
            columns="quarter",
            values="mean_auc",
        ).reindex(
            index=feature_columns,
            columns=quarter_order,
        )

        support_matrix = temporal_summary.pivot(
            index="feature",
            columns="quarter",
            values="eligible_images",
        ).reindex(
            index=feature_columns,
            columns=quarter_order,
        )

        fig, axes = plt.subplots(
            2,
            1,
            figsize=(max(12, len(quarter_order) * 0.65), 11),
        )

        for ax, matrix, title, cmap, limits in [
            (
                axes[0],
                mean_matrix,
                "Mean within-image AUC by capture quarter",
                "RdBu",
                {"vmin": 0, "vmax": 1},
            ),
            (
                axes[1],
                support_matrix,
                "Eligible image-record support",
                "Blues",
                {},
            ),
        ]:
            heat = ax.imshow(
                matrix.to_numpy(),
                aspect="auto",
                cmap=cmap,
                **limits,
            )

            ax.set_xticks(np.arange(len(quarter_order)))
            ax.set_xticklabels(quarter_order, rotation=60)
            ax.set_yticks(np.arange(len(feature_columns)))
            ax.set_yticklabels(
                [feature_labels[f] for f in feature_columns]
            )
            ax.set_title(title)
            fig.colorbar(heat, ax=ax)

        fig.tight_layout()
        finish_bivariate_figure(
            fig,
            "17_8_temporal_feature_stability.png",
        )

        temporal_summary.to_csv(
            BIVARIATE_DIR / "temporal_auc_summary.csv",
            index=False,
        )

        temporal_status = (
            "complete"
            if missing_date_images == 0
            else "partial_missing_dates"
        )

        display(temporal_summary.head(20).round(4))

pd.DataFrame(
    [{"temporal_analysis_status": temporal_status}]
).to_csv(
    BIVARIATE_DIR / "temporal_analysis_status.csv",
    index=False,
)

### 17.9 Consolidated Evidence and Modeling Implications

The final table combines pooled association, nonlinear dependence, and
within-image stability.

Interpretation priorities:

- Strong pooled and within-image discrimination supports further
  supervised feature evaluation.
- A large difference between pooled and within-image AUC warrants
  investigation of image-level confounding and sampling composition.
- Variable or reversing within-image relationships suggest sensitivity
  to observation conditions.
- Nonlinear dependence with weak linear correlation motivates nonlinear
  models and response-curve inspection.
- Spatially varying relationships motivate region-specific diagnostics.
- Features with weak individual discrimination may still provide
  complementary information in multivariate models.

These analyses generate modeling hypotheses. They do not demonstrate
improved segmentation, instance reconstruction, or competition PQ.

Final conclusions will be written after reviewing the actual outputs.

In [ ]:
evidence_df = (
    bivariate_df.merge(
        stability_df,
        on="feature",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mi_summary[
            ["mi_mean", "mi_min", "mi_max"]
        ].reset_index(),
        on="feature",
        how="left",
        validate="one_to_one",
    )
)

evidence_df["pooled_minus_mean_within_image_auc"] = (
    evidence_df["separation_auc"]
    - evidence_df["mean_within_image_auc"]
)

evidence_df = evidence_df.sort_values(
    "mean_within_image_auc",
    ascending=False,
).reset_index(drop=True)

evidence_df.to_csv(
    BIVARIATE_DIR / "consolidated_bivariate_evidence.csv",
    index=False,
)

display_columns = [
    "feature",
    "ranking_direction",
    "point_biserial_r",
    "standardized_mean_difference",
    "separation_auc",
    "mean_within_image_auc",
    "mean_auc_ci_low",
    "mean_auc_ci_high",
    "fraction_below_chance",
    "pooled_minus_mean_within_image_auc",
    "mi_mean",
    "eligible_images",
]

display(evidence_df[display_columns].round(4))

print("STEP 17 COMPUTATIONS FINISHED")
print("=" * 60)
print("Temporal analysis status:", temporal_status)
print("Saved results:", BIVARIATE_DIR)
print(
    "\nInterpretation and final notebook verification "
    "remain pending until outputs are reviewed."
)

## 18. Multivariate Statistical Analysis

### 18.1 Analysis Design

Step 17 examined individual feature–target relationships. We now investigate
how the predictors behave together:

- Which features contain overlapping information?
- Which relationships differ between foreground and background?
- Do interactions add information beyond individual predictors?
- Can a smaller feature set retain useful discrimination?
- Does a nonlinear model exploit information missed by an additive model?

We reuse the existing nine-feature training cache without repeating extraction.

To control CPU and memory use while retaining the training images:

- Descriptive analyses use up to 200 pixels per class per image.
- Supervised comparisons use up to 80 pixels per class per image.
- Sampling is reproducible and performed without replacement.

The main validation partition remains untouched. Supervised comparisons use
three inner temporal-group folds constructed within the training partition,
with the existing purge interval applied to each fold.

These experiments measure discrimination in the class-stratified pixel sample.
They do not estimate natural-prevalence probabilities or instance-level PQ.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from threadpoolctl import threadpool_limits

S18_DIR = OUTPUT_ROOT / "step18_multivariate"
S18_DIR.mkdir(parents=True, exist_ok=True)

S18_CFG = {
    "seed": int(SEED),
    "analysis_per_class": 200,
    "model_per_class": 80,
    "folds": 3,
    "purge_days": int(PURGE_DAYS),
    "cmi_bins": 4,
    "cmi_null_repeats": 5,
    "bootstrap_repeats": 1000,
}

s18_features = list(feature_columns)

required = s18_features + ["image_id", "target"]

if not set(required).issubset(pixel_df.columns):
    raise ValueError("Run the feature-cache loading cells first.")

metadata_columns = [
    "image_id",
    "temporal_block",
    "capture_datetime",
]

if not set(metadata_columns).issubset(train_df.columns):
    raise ValueError("Training split metadata is incomplete.")

s18_meta = train_df[metadata_columns].copy()

s18_meta["image_id"] = s18_meta["image_id"].astype(str)
s18_meta["capture_datetime"] = pd.to_datetime(
    s18_meta["capture_datetime"]
)

if (
    s18_meta["image_id"].duplicated().any()
    or s18_meta.isna().any().any()
):
    raise ValueError("Duplicate or incomplete training metadata.")

cached_ids = set(pixel_df["image_id"].astype(str))

if cached_ids != set(s18_meta["image_id"]):
    raise ValueError(
        "Cache membership differs from the training manifest."
    )

if cached_ids & set(valid_df["image_id"].astype(str)):
    raise ValueError(
        "Validation images were found in the training cache."
    )

if not pixel_df["target"].isin([0, 1]).all():
    raise ValueError("Target must contain only zero and one.")

for feature in s18_features:
    if not np.isfinite(pixel_df[feature].to_numpy()).all():
        raise ValueError(f"Non-finite values in {feature}")


def s18_sample(frame, per_class, seed):
    """Reproducible sampling within each image and target class."""
    rng = np.random.default_rng(seed)
    positions = []

    for indices in frame.groupby(
        ["image_id", "target"],
        sort=True,
    ).indices.values():

        positions.append(
            rng.choice(
                indices,
                size=min(per_class, len(indices)),
                replace=False,
            )
        )

    return (
        frame.iloc[np.concatenate(positions)]
        .copy()
        .reset_index(drop=True)
    )


s18_base = pixel_df[required].copy()
s18_base["image_id"] = s18_base["image_id"].astype(str)

s18_analysis = s18_sample(
    s18_base,
    S18_CFG["analysis_per_class"],
    SEED,
)

s18_data = s18_sample(
    s18_base,
    S18_CFG["model_per_class"],
    SEED + 1,
)

s18_data = s18_data.merge(
    s18_meta,
    on="image_id",
    how="left",
    validate="many_to_one",
)

del s18_base

print("Analysis rows:", len(s18_analysis))
print("Modeling rows:", len(s18_data))
print("Training images:", len(s18_meta))
print("Features:", len(s18_features))
print("The external validation partition is untouched.")

### 18.2 Pooled, Within-Image and Class-Conditional Dependence

We examine three complementary views:

1. Pooled Pearson correlation measures linear dependence.
2. Pooled Spearman correlation measures monotonic dependence.
3. Within-image centered Pearson correlation removes each image's feature
   means before calculating pooled correlation.

The third view helps identify relationships driven by differences between
images rather than relationships within images. It is a pooled correlation
of centered observations, not the average of individual image correlations.

We also calculate separate Spearman matrices for filament and background
pixels. A pooled relationship can arise because both predictors distinguish
the classes, even when their dependence within each class is weaker.

High correlation identifies an ablation candidate. It does not automatically
justify deleting a predictor.

In [ ]:
def s18_heatmap(matrix, title, ax):
    image = ax.imshow(
        matrix,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(
        range(len(matrix.columns)),
        matrix.columns,
        rotation=90,
    )

    ax.set_yticks(
        range(len(matrix.index)),
        matrix.index,
    )

    for i in range(len(matrix)):
        for j in range(len(matrix.columns)):
            ax.text(
                j,
                i,
                f"{matrix.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=7,
            )

    ax.set_title(title)
    return image


s18_X = s18_analysis[s18_features]

s18_pearson = s18_X.corr(method="pearson")
s18_spearman = s18_X.corr(method="spearman")

s18_centered = (
    s18_X
    - s18_analysis.groupby("image_id")[s18_features]
    .transform("mean")
)

s18_within = s18_centered.corr()

fig, axes = plt.subplots(
    1,
    3,
    figsize=(23, 8),
    constrained_layout=True,
)

for matrix, title, ax in zip(
    [s18_pearson, s18_spearman, s18_within],
    [
        "Pooled Pearson",
        "Pooled Spearman",
        "Within-image centered Pearson",
    ],
    axes,
):
    image = s18_heatmap(matrix, title, ax)

fig.colorbar(image, ax=axes, shrink=0.65)

fig.savefig(
    S18_DIR / "correlation_overview.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7),
    constrained_layout=True,
)

for target_class, ax in enumerate(axes):
    matrix = (
        s18_analysis.loc[
            s18_analysis["target"] == target_class,
            s18_features,
        ]
        .corr(method="spearman")
    )

    s18_heatmap(
        matrix,
        ["Background Spearman", "Filament Spearman"][target_class],
        ax,
    )

    matrix.to_csv(
        S18_DIR / f"class_{target_class}_spearman.csv"
    )

fig.savefig(
    S18_DIR / "class_conditional_correlations.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

for name, matrix in [
    ("pearson", s18_pearson),
    ("spearman", s18_spearman),
    ("within_image", s18_within),
]:
    matrix.to_csv(
        S18_DIR / f"{name}_correlations.csv"
    )

s18_pairs = pd.DataFrame(
    [
        {
            "feature_a": a,
            "feature_b": b,
            "pearson": s18_pearson.loc[a, b],
            "spearman": s18_spearman.loc[a, b],
            "within_image": s18_within.loc[a, b],
        }
        for i, a in enumerate(s18_features)
        for b in s18_features[i + 1:]
    ]
)

s18_pairs["absolute_spearman"] = (
    s18_pairs["spearman"].abs()
)

s18_pairs = s18_pairs.sort_values(
    "absolute_spearman",
    ascending=False,
)

s18_pairs.to_csv(
    S18_DIR / "feature_pair_dependence.csv",
    index=False,
)

display(s18_pairs.head(15))

### 18.3 Multicollinearity and Principal Component Analysis

Variance Inflation Factor evaluates how closely each predictor can be
reconstructed linearly from the remaining predictors:

VIF = 1 / (1 − R²)

- Large VIF indicates linear redundancy.
- Infinite VIF indicates near-exact linear dependence.
- Undefined VIF indicates a constant predictor.

VIF is a linear-model diagnostic. A correlated predictor can still contribute
to nonlinear models.

PCA is performed after standardization so that feature units do not dominate
the decomposition. We inspect explained variance and principal-direction
coefficients.

PCA summarizes predictor variance, not target relevance. Retaining 95% of
variance does not imply retaining 95% of predictive information.

The descriptive PCA below uses training observations only. The later
cross-validation experiment fits a separate scaler and PCA inside each
training fold.

In [ ]:
s18_scaler = StandardScaler()

s18_Z = (
    s18_scaler.fit_transform(s18_X)
    .astype(np.float64)
)

s18_vif_rows = []

for j, feature in enumerate(s18_features):
    target = s18_Z[:, j]

    others = np.delete(
        s18_Z,
        j,
        axis=1,
    )

    design = np.column_stack(
        [
            np.ones(len(others)),
            others,
        ]
    )

    total = np.sum(
        (target - target.mean()) ** 2
    )

    if total <= 1e-12:
        vif = np.nan
        r_squared = np.nan

    else:
        coefficients = np.linalg.lstsq(
            design,
            target,
            rcond=None,
        )[0]

        residual = np.sum(
            (target - design @ coefficients) ** 2
        )

        r_squared = 1 - residual / total

        vif = (
            np.inf
            if residual / total <= 1e-12
            else total / residual
        )

    s18_vif_rows.append(
        {
            "feature": feature,
            "vif": vif,
            "r_squared": r_squared,
        }
    )

s18_vif = (
    pd.DataFrame(s18_vif_rows)
    .sort_values("vif", ascending=False)
)

s18_pca = PCA(
    svd_solver="full"
).fit(s18_Z)

s18_loadings = pd.DataFrame(
    s18_pca.components_.T,
    index=s18_features,
    columns=[
        f"PC{i + 1}"
        for i in range(len(s18_features))
    ],
)

s18_variance = pd.DataFrame(
    {
        "component": s18_loadings.columns,
        "explained_variance": (
            s18_pca.explained_variance_ratio_
        ),
        "cumulative_variance": np.cumsum(
            s18_pca.explained_variance_ratio_
        ),
    }
)

display(s18_vif)
display(s18_variance)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
    constrained_layout=True,
)

axes[0].plot(
    range(1, len(s18_features) + 1),
    s18_variance["cumulative_variance"],
    marker="o",
)

axes[0].axhline(
    0.95,
    color="red",
    linestyle="--",
    label="95% variance",
)

axes[0].set(
    xlabel="Number of principal components",
    ylabel="Cumulative explained variance",
    ylim=(0, 1.02),
)

axes[0].legend()

image = axes[1].imshow(
    s18_loadings,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    aspect="auto",
)

axes[1].set_xticks(
    range(len(s18_features)),
    s18_loadings.columns,
)

axes[1].set_yticks(
    range(len(s18_features)),
    s18_features,
)

axes[1].set_title(
    "PCA coefficients on standardized features"
)

fig.colorbar(image, ax=axes[1])

fig.savefig(
    S18_DIR / "vif_pca_diagnostics.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

s18_vif.to_csv(
    S18_DIR / "vif.csv",
    index=False,
)

s18_variance.to_csv(
    S18_DIR / "pca_variance.csv",
    index=False,
)

s18_loadings.to_csv(
    S18_DIR / "pca_coefficients.csv"
)

### 18.4 Joint Feature Response and Observation Support

We examine three physically motivated feature pairs:

- Local darkness and ridge response.
- Local variation and structural coherence.
- Radial position and local variation.

For each pair, we calculate a two-dimensional quantile-bin response surface.
Every response surface is accompanied by the number of distinct images
supporting each bin. Pixel counts are also saved.

The colors represent filament rates in our class-stratified sample.
They are not natural-prevalence probabilities.

These plots identify candidate relationships. Predictive interactions must
still be tested against models containing the corresponding main effects.
Low-support cells should not drive conclusions.

In [ ]:
def s18_bin(values, bins=4):
    """Quantile bins with duplicate edges removed."""
    edges = np.unique(
        np.quantile(
            values,
            np.linspace(0, 1, bins + 1),
        )
    )

    if len(edges) < 2:
        return np.zeros(
            len(values),
            dtype=int,
        )

    return np.searchsorted(
        edges[1:-1],
        values,
        side="right",
    )


s18_interaction_pairs = [
    ("local_darkness", "ridge_response"),
    ("local_std_15", "coherence"),
    ("radial_distance", "local_std_15"),
]

s18_surface_tables = []

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 10),
    constrained_layout=True,
)

for k, (feature_a, feature_b) in enumerate(
    s18_interaction_pairs
):
    temporary = pd.DataFrame(
        {
            "a_bin": s18_bin(
                s18_analysis[feature_a],
                6,
            ),
            "b_bin": s18_bin(
                s18_analysis[feature_b],
                6,
            ),
            "target": s18_analysis["target"],
            "image_id": s18_analysis["image_id"],
        }
    )

    table = (
        temporary.groupby(
            ["a_bin", "b_bin"],
            observed=True,
        )
        .agg(
            sampled_rate=("target", "mean"),
            pixels=("target", "size"),
            images=("image_id", "nunique"),
        )
        .reset_index()
    )

    table["feature_a"] = feature_a
    table["feature_b"] = feature_b

    s18_surface_tables.append(table)

    for row, value, title in [
        (0, "sampled_rate", "Sample filament rate"),
        (1, "images", "Distinct images"),
    ]:
        grid = table.pivot(
            index="b_bin",
            columns="a_bin",
            values=value,
        )

        color_limits = (
            {"vmin": 0, "vmax": 1}
            if row == 0
            else {}
        )

        image = axes[row, k].imshow(
            grid,
            origin="lower",
            aspect="auto",
            cmap="viridis",
            **color_limits,
        )

        axes[row, k].set(
            xlabel=f"{feature_a} quantile bin",
            ylabel=f"{feature_b} quantile bin",
            title=title,
        )

        fig.colorbar(
            image,
            ax=axes[row, k],
        )

fig.savefig(
    S18_DIR / "two_feature_sample_surfaces.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

pd.concat(
    s18_surface_tables,
    ignore_index=True,
).to_csv(
    S18_DIR / "interaction_bin_support.csv",
    index=False,
)

### 18.5 Conditional Information and Discretization Sensitivity

For each predictor, we select its most strongly Spearman-correlated companion.
We then estimate:

I(feature; target | image identity, binned companion)

This asks whether the discretized feature retains information about the target
within the same image and similar values of its correlated companion.

We repeat the calculation using three and four bins.

This is an empirical discrete conditional-information diagnostic, measured in
nats. It is not an estimate of continuous conditional mutual information.

Finite samples produce positive empirical information even under independence.
We therefore shuffle labels within image/conditioning-bin strata and report
the observed value alongside its shuffled reference.

Observed minus shuffled mean is a diagnostic, not an unbiased correction or
formal significance test. Spatial dependence is not represented by these
shuffles. No p-values are claimed.

Interpretation requires bin sensitivity checks and agreement with the grouped
predictive ablations.

In [ ]:
def s18_conditional_mi(x, y, strata):
    """Empirical discrete conditional MI in nats."""
    frame = pd.DataFrame(
        {
            "stratum": strata,
            "x": x,
            "y": y,
        }
    )

    counts = (
        frame.groupby(
            ["stratum", "x", "y"],
            observed=True,
        )
        .size()
        .rename("n")
        .reset_index()
    )

    counts["n_s"] = (
        counts.groupby("stratum")["n"]
        .transform("sum")
    )

    counts["n_sx"] = (
        counts.groupby(["stratum", "x"])["n"]
        .transform("sum")
    )

    counts["n_sy"] = (
        counts.groupby(["stratum", "y"])["n"]
        .transform("sum")
    )

    contribution = (
        counts["n"] / len(frame)
        * np.log(
            counts["n"] * counts["n_s"]
            / (
                counts["n_sx"]
                * counts["n_sy"]
            )
        )
    )

    return float(contribution.sum())


s18_cmi_rows = []

s18_rng = np.random.default_rng(
    SEED + 18
)

for feature in s18_features:
    candidates = (
        s18_spearman[feature]
        .drop(feature)
        .abs()
        .dropna()
    )

    if candidates.empty:
        continue

    conditioning_feature = candidates.idxmax()

    for bins in [3, S18_CFG["cmi_bins"]]:
        feature_bins = s18_bin(
            s18_analysis[feature],
            bins,
        )

        conditioning_bins = s18_bin(
            s18_analysis[conditioning_feature],
            bins,
        )

        strata = pd.factorize(
            pd.MultiIndex.from_arrays(
                [
                    s18_analysis["image_id"],
                    conditioning_bins,
                ]
            )
        )[0]

        labels = (
            s18_analysis["target"]
            .to_numpy()
        )

        observed = s18_conditional_mi(
            feature_bins,
            labels,
            strata,
        )

        locations = (
            pd.Series(np.arange(len(strata)))
            .groupby(strata)
            .indices
        )

        shuffled_values = []

        for repeat in range(
            S18_CFG["cmi_null_repeats"]
        ):
            shuffled_labels = labels.copy()

            for indices in locations.values():
                shuffled_labels[indices] = (
                    s18_rng.permutation(
                        labels[indices]
                    )
                )

            shuffled_values.append(
                s18_conditional_mi(
                    feature_bins,
                    shuffled_labels,
                    strata,
                )
            )

        shuffled_mean = np.mean(
            shuffled_values
        )

        s18_cmi_rows.append(
            {
                "feature": feature,
                "conditioned_on": conditioning_feature,
                "bins": bins,
                "cmi_nats": observed,
                "null_mean_nats": shuffled_mean,
                "excess_over_shuffle": (
                    observed - shuffled_mean
                ),
            }
        )

s18_cmi = pd.DataFrame(
    s18_cmi_rows
)

s18_cmi.to_csv(
    S18_DIR / "conditional_information_sensitivity.csv",
    index=False,
)

display(
    s18_cmi.sort_values(
        ["bins", "excess_over_shuffle"],
        ascending=[True, False],
    )
)

### 18.6 Prespecified Models and Purged Inner Folds

We compare ten diagnostic models:

1. Logistic regression with the eight main predictors.
2. Logistic regression with all nine reference predictors.
3. Logistic regression with the reference predictors plus two interactions.
4. Logistic regression after fold-fitted PCA retaining 95% variance.
5. Histogram gradient boosting with all nine reference predictors.
6. Boosting without the existing contrast–ridge interaction.
7. Boosting without radial distance.
8. Boosting with appearance predictors.
9. Boosting with ridge response and coherence.
10. Boosting with a prespecified compact four-feature set.

The additional interactions are:

- Local variation × coherence.
- Radial distance × local variation.

These are pointwise image-feature calculations and require no fitted parameters.

Each test fold contains complete temporal blocks from the training partition.
Training observations within the purge interval of any test-fold observation
are excluded.

This evaluates grouped interpolation across observed periods. It is not a
strictly chronological future holdout.

All model comparisons use identical folds. Scaling and PCA are fitted inside
each training fold. Boosting's internal early stopping is disabled so it does
not create an additional random validation partition.

In [ ]:
s18_data["contrast_coherence"] = (
    s18_data["local_std_15"]
    * s18_data["coherence"]
)

s18_data["radius_contrast"] = (
    s18_data["radial_distance"]
    * s18_data["local_std_15"]
)

s18_main = [
    feature
    for feature in s18_features
    if feature != "contrast_ridge_interaction"
]

s18_plus = s18_features + [
    "contrast_coherence",
    "radius_contrast",
]

s18_specs = {
    "LR_main": (
        "lr",
        s18_main,
    ),
    "LR_all9": (
        "lr",
        s18_features,
    ),
    "LR_interactions": (
        "lr",
        s18_plus,
    ),
    "LR_PCA95": (
        "pca",
        s18_features,
    ),
    "HGB_all9": (
        "hgb",
        s18_features,
    ),
    "HGB_no_interaction": (
        "hgb",
        s18_main,
    ),
    "HGB_no_radius": (
        "hgb",
        [
            feature
            for feature in s18_features
            if feature != "radial_distance"
        ],
    ),
    "HGB_appearance": (
        "hgb",
        [
            "intensity",
            "local_darkness",
            "local_std_5",
            "local_std_15",
            "gradient",
        ],
    ),
    "HGB_structure": (
        "hgb",
        [
            "ridge_response",
            "coherence",
        ],
    ),
    "HGB_compact": (
        "hgb",
        [
            "local_darkness",
            "ridge_response",
            "coherence",
            "local_std_5",
        ],
    ),
}


def s18_model(kind):
    if kind == "hgb":
        return HistGradientBoostingClassifier(
            max_iter=80,
            max_leaf_nodes=15,
            min_samples_leaf=50,
            l2_regularization=2.0,
            max_bins=63,
            early_stopping=False,
            random_state=SEED,
        )

    steps = [StandardScaler()]

    if kind == "pca":
        steps.append(
            PCA(
                n_components=0.95,
                svd_solver="full",
            )
        )

    return make_pipeline(
        *steps,
        LogisticRegression(
            C=1.0,
            max_iter=1500,
            random_state=SEED,
        ),
    )


s18_folds = []
s18_fold_manifest = []

group_cv = GroupKFold(
    n_splits=S18_CFG["folds"]
)

for fold, (train_images, test_images) in enumerate(
    group_cv.split(
        s18_meta,
        groups=s18_meta["temporal_block"],
    )
):
    training = s18_meta.iloc[
        train_images
    ].copy()

    testing = s18_meta.iloc[
        test_images
    ].copy()

    train_times = (
        training["capture_datetime"]
        .to_numpy(dtype="datetime64[ns]")
    )

    test_times = np.sort(
        testing["capture_datetime"]
        .to_numpy(dtype="datetime64[ns]")
    )

    positions = np.searchsorted(
        test_times,
        train_times,
    )

    left = np.clip(
        positions - 1,
        0,
        len(test_times) - 1,
    )

    right = np.clip(
        positions,
        0,
        len(test_times) - 1,
    )

    nearest_gap = np.minimum(
        np.abs(train_times - test_times[left]),
        np.abs(train_times - test_times[right]),
    )

    keep = nearest_gap > np.timedelta64(
        S18_CFG["purge_days"],
        "D",
    )

    used = training.loc[keep]

    train_indices = np.flatnonzero(
        s18_data["image_id"].isin(
            used["image_id"]
        )
    )

    test_indices = np.flatnonzero(
        s18_data["image_id"].isin(
            testing["image_id"]
        )
    )

    if (
        not len(train_indices)
        or s18_data.iloc[train_indices]["target"].nunique() != 2
    ):
        raise ValueError(
            f"Insufficient training data after purging in fold {fold}."
        )

    if (
        s18_data.iloc[test_indices]["target"].nunique()
        != 2
    ):
        raise ValueError(
            f"Both classes are needed in test fold {fold}."
        )

    assert set(
        used["temporal_block"]
    ).isdisjoint(
        set(testing["temporal_block"])
    )

    s18_folds.append(
        (train_indices, test_indices)
    )

    for image_ids, role in [
        (used["image_id"], "train"),
        (testing["image_id"], "test"),
        (
            training.loc[~keep, "image_id"],
            "purged",
        ),
    ]:
        s18_fold_manifest.extend(
            [
                {
                    "fold": fold,
                    "image_id": image_id,
                    "role": role,
                }
                for image_id in image_ids
            ]
        )

s18_fold_table = pd.DataFrame(
    s18_fold_manifest
)

s18_fold_table.to_csv(
    S18_DIR / "inner_fold_manifest.csv",
    index=False,
)

print(
    "Purged, training-only inner folds:",
    len(s18_folds),
)

display(
    s18_fold_table.groupby(
        ["fold", "role"]
    )
    .size()
    .unstack(fill_value=0)
)

### 18.7 Grouped Out-of-Fold Model Ablations

Every sampled pixel receives one prediction from a model that excluded its
image and temporal block.

We save:

- Fold-level sampled ROC-AUC, average precision and log loss.
- Per-image out-of-fold ROC-AUC.
- Standardized logistic coefficients.
- Sampled out-of-fold predictions.

Mean per-image AUC is the main diagnostic summary. This limits domination
by images contributing more sampled pixels.

Average precision and log loss refer to the artificial sampled class
distribution. They are not deployment precision, calibrated full-image
performance or segmentation PQ.

Standardized coefficients describe conditional associations within each
fitted model. Correlated predictors can make them unstable, and they should
not be interpreted as causal importance.

The feature sets are prespecified. We do not repeatedly search subsets using
these same out-of-fold results.

In [ ]:
s18_oof = {}

s18_scores = []
s18_image_scores = []
s18_coefficients = []

for name, (kind, columns) in s18_specs.items():
    prediction = np.full(
        len(s18_data),
        np.nan,
        dtype=np.float64,
    )

    for fold, (train_indices, test_indices) in enumerate(
        s18_folds
    ):
        estimator = s18_model(kind)

        with threadpool_limits(limits=2):
            estimator.fit(
                s18_data.iloc[train_indices][columns],
                s18_data.iloc[train_indices]["target"],
            )

            probabilities = estimator.predict_proba(
                s18_data.iloc[test_indices][columns]
            )[:, 1]

        prediction[test_indices] = probabilities

        fold_frame = s18_data.iloc[
            test_indices
        ][
            [
                "image_id",
                "temporal_block",
                "target",
            ]
        ].copy()

        fold_frame["p"] = probabilities

        for image_id, image in fold_frame.groupby(
            "image_id",
            sort=True,
        ):
            image_auc = (
                roc_auc_score(
                    image["target"],
                    image["p"],
                )
                if image["target"].nunique() == 2
                else np.nan
            )

            s18_image_scores.append(
                {
                    "model": name,
                    "fold": fold,
                    "image_id": image_id,
                    "temporal_block": int(
                        image["temporal_block"].iloc[0]
                    ),
                    "auc": image_auc,
                }
            )

        s18_scores.append(
            {
                "model": name,
                "fold": fold,
                "train_rows": len(train_indices),
                "test_rows": len(test_indices),
                "sample_auc": roc_auc_score(
                    fold_frame["target"],
                    probabilities,
                ),
                "sample_ap": average_precision_score(
                    fold_frame["target"],
                    probabilities,
                ),
                "sample_logloss": log_loss(
                    fold_frame["target"],
                    probabilities,
                    labels=[0, 1],
                ),
            }
        )

        if name in [
            "LR_main",
            "LR_interactions",
        ]:
            coefficients = (
                estimator.steps[-1][1]
                .coef_[0]
            )

            for feature, coefficient in zip(
                columns,
                coefficients,
            ):
                s18_coefficients.append(
                    {
                        "model": name,
                        "fold": fold,
                        "feature": feature,
                        "coefficient": coefficient,
                    }
                )

        print(
            name,
            "| fold",
            fold + 1,
            "complete",
            flush=True,
        )

    if not np.isfinite(prediction).all():
        raise ValueError(
            f"Incomplete out-of-fold predictions for {name}"
        )

    s18_oof[name] = prediction

s18_scores = pd.DataFrame(
    s18_scores
)

s18_image_scores = pd.DataFrame(
    s18_image_scores
)

s18_coefficients = pd.DataFrame(
    s18_coefficients
)

s18_scores.to_csv(
    S18_DIR / "grouped_fold_metrics.csv",
    index=False,
)

s18_image_scores.to_csv(
    S18_DIR / "per_image_oof_auc.csv",
    index=False,
)

s18_coefficients.to_csv(
    S18_DIR / "standardized_logistic_coefficients.csv",
    index=False,
)

s18_prediction_table = s18_data[
    [
        "image_id",
        "temporal_block",
        "target",
    ]
    + s18_features
].copy()

for name, prediction in s18_oof.items():
    s18_prediction_table[name] = prediction

s18_prediction_table.to_parquet(
    S18_DIR / "sample_oof_predictions.parquet",
    index=False,
)

s18_summary = (
    s18_image_scores.groupby("model")
    .agg(
        mean_image_auc=("auc", "mean"),
        median_image_auc=("auc", "median"),
        images_with_auc=("auc", "count"),
    )
    .sort_values(
        "mean_image_auc",
        ascending=False,
    )
)

display(s18_summary)

### 18.8 Paired Ablation Differences and Temporal Stability

We compare models on the same images and calculate per-image AUC differences.

The comparisons isolate:

- The original contrast–ridge interaction.
- The two additional interactions.
- PCA compression.
- Nonlinear versus additive modeling.
- Radial information.
- Appearance, structure and compact feature sets.

We bootstrap temporal blocks, keeping all images from a selected block
together. Each bootstrap statistic remains an image-weighted mean difference.

The intervals describe group variation conditional on the fitted
cross-validation procedure. They do not include complete retraining
uncertainty, repeated-seed variation or correction for multiple comparisons.

Interpret them together with fold-level results. A sampled-pixel gain is a
reason to test a candidate in dense-image segmentation, not proof of a PQ gain.

In [ ]:
s18_comparisons = [
    ("LR_all9", "LR_main"),
    ("LR_interactions", "LR_all9"),
    ("LR_PCA95", "LR_all9"),
    ("HGB_all9", "LR_interactions"),
    ("HGB_all9", "HGB_no_interaction"),
    ("HGB_all9", "HGB_no_radius"),
    ("HGB_all9", "HGB_appearance"),
    ("HGB_all9", "HGB_structure"),
    ("HGB_all9", "HGB_compact"),
]

s18_wide = s18_image_scores.pivot(
    index=[
        "image_id",
        "temporal_block",
    ],
    columns="model",
    values="auc",
)

s18_bootstrap_rows = []

rng = np.random.default_rng(
    SEED + 181
)

for candidate, baseline in s18_comparisons:
    delta = (
        (
            s18_wide[candidate]
            - s18_wide[baseline]
        )
        .dropna()
        .rename("delta")
        .reset_index()
    )

    grouped = (
        delta.groupby("temporal_block")["delta"]
        .agg(["sum", "count"])
    )

    if len(grouped) < 2:
        raise ValueError(
            "Not enough temporal groups for bootstrap."
        )

    draws = rng.integers(
        0,
        len(grouped),
        size=(
            S18_CFG["bootstrap_repeats"],
            len(grouped),
        ),
    )

    bootstrap_values = (
        grouped["sum"].to_numpy()[draws]
        .sum(axis=1)
        /
        grouped["count"].to_numpy()[draws]
        .sum(axis=1)
    )

    lower, upper = np.quantile(
        bootstrap_values,
        [0.025, 0.975],
    )

    s18_bootstrap_rows.append(
        {
            "candidate": candidate,
            "baseline": baseline,
            "mean_image_auc_delta": (
                delta["delta"].mean()
            ),
            "ci_low": lower,
            "ci_high": upper,
            "images": len(delta),
            "temporal_groups": len(grouped),
        }
    )

s18_deltas = pd.DataFrame(
    s18_bootstrap_rows
)

s18_deltas.to_csv(
    S18_DIR / "paired_temporal_bootstrap.csv",
    index=False,
)

display(s18_deltas)

fig, ax = plt.subplots(
    figsize=(11, 6)
)

for i, row in s18_deltas.iterrows():
    ax.plot(
        [row["ci_low"], row["ci_high"]],
        [i, i],
        color="steelblue",
    )

    ax.scatter(
        row["mean_image_auc_delta"],
        i,
        color="black",
    )

ax.axvline(
    0,
    color="red",
    linestyle="--",
)

ax.set_yticks(
    range(len(s18_deltas)),
    (
        s18_deltas["candidate"]
        + " minus "
        + s18_deltas["baseline"]
    ),
)

ax.set_xlabel(
    "Mean per-image sampled AUC difference\n"
    "with temporal-group bootstrap interval"
)

fig.tight_layout()

fig.savefig(
    S18_DIR / "paired_ablation_differences.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

### 18.9 Interpretation and Candidate Feature Sets

We retain the original nine predictors as the reference configuration.

Interpret the results through several sources of evidence:

- Correlation identifies dependence and possible redundancy.
- Within-image and class-conditional analysis checks pooled confounding.
- VIF describes linear multicollinearity.
- PCA describes predictor variance.
- Conditional information provides a discretized supporting diagnostic.
- Grouped ablations test incremental sampled predictive value.

A correlated feature should be removed only after a suitable predictive
comparison. Likewise, a strong interaction plot is insufficient without
an out-of-fold comparison against its main effects.

The appearance-only, structure-only and compact models are prespecified
candidate sets. Their sampled-pixel performance determines which deserve
later full-image experiments.

No features are automatically deleted, no main split is changed, and no
instance-level performance claim is made here.

The next modeling phase must evaluate promising candidates using dense
predictions, instance reconstruction and official-equivalent PQ.

In [ ]:
s18_groups = {
    "appearance": [
        "intensity",
        "local_darkness",
        "local_std_5",
        "local_std_15",
        "gradient",
    ],
    "structure": [
        "ridge_response",
        "coherence",
    ],
    "position": [
        "radial_distance",
    ],
    "reference_interaction": [
        "contrast_ridge_interaction",
    ],
}

s18_signal = s18_deltas.copy()

s18_signal["interval_above_zero"] = (
    s18_signal["ci_low"] > 0
)

s18_signal["interpretation"] = np.where(
    s18_signal["ci_low"] > 0,
    (
        "Consistent incremental sampled discrimination "
        "in this exploratory comparison"
    ),
    np.where(
        s18_signal["ci_high"] < 0,
        "Candidate performed worse in this comparison",
        (
            "Difference is not resolved "
            "by this exploratory interval"
        ),
    ),
)

display(
    s18_signal[
        [
            "candidate",
            "baseline",
            "mean_image_auc_delta",
            "interpretation",
        ]
    ]
)

import json

with open(
    S18_DIR / "step18_configuration.json",
    "w",
) as file:
    json.dump(
        {
            "configuration": S18_CFG,
            "features": s18_features,
            "feature_groups": s18_groups,
            "experiments": {
                name: {
                    "kind": kind,
                    "features": columns,
                }
                for name, (kind, columns)
                in s18_specs.items()
            },
            "caveats": [
                (
                    "Training-cache sample only; "
                    "external validation unused"
                ),
                (
                    "Pixel models here are diagnostic "
                    "and do not estimate PQ"
                ),
                (
                    "Conditional MI is discretized "
                    "and sample-dependent"
                ),
                (
                    "Bootstrap intervals condition on "
                    "the fitted cross-validation procedure"
                ),
                "No automatic feature deletion",
            ],
        },
        file,
        indent=2,
    )

s18_signal.to_csv(
    S18_DIR / "step18_interpretation_table.csv",
    index=False,
)

print("Step 18 outputs saved to:", S18_DIR)

print(
    "Retain the original nine predictors. "
    "Carry candidate sets forward to dense-image PQ tests."
)

## 19. Full-Population Filament Morphology and Geometry

### 19.1 Scope and Measurement Conventions

Step 14 examined a small training-image sample. We now measure every
annotation associated with the fixed training partition.

We investigate:

- Pixel-accurate area, perimeter and bounding geometry.
- Solidity, eccentricity, orientation and spatial extent.
- Disconnected components and holes.
- Weighted skeleton length and geodesic diameter.
- Endpoints, junction zones, branches and cycles.
- Local thickness proxies and centerline curvature.
- Overlapping annotations and neighboring-instance separation.
- Morphology strata for later model-error analysis.

Each annotation is rasterized independently using the current Step 13
rasterizer. We do not recover annotations from the integer instance-label
image, because overlapping annotations can overwrite one another there.

Multipart annotations retain their original annotation identity.

Measurements use native image-pixel coordinates, not physical solar units.
Rasterizer parity with the organizer remains a later audit requirement.

These label-derived measurements support EDA, auxiliary-target design and
failure analysis. They must not become inference-time model inputs.

In [ ]:
import json
import hashlib
import os
import types
import scipy
import skimage
import PIL

from scipy.ndimage import (
    distance_transform_edt,
    binary_erosion,
    gaussian_filter1d,
)
from scipy.sparse import coo_matrix, triu
from scipy.sparse.csgraph import (
    connected_components,
    dijkstra,
)
from scipy.spatial import cKDTree

from skimage.measure import (
    label,
    regionprops,
    perimeter_crofton,
    euler_number,
)
from skimage.morphology import skeletonize


S19_CFG = {
    "version": "s19_v1",
    "seed": int(SEED),
    "foreground_connectivity": 8,
    "diagonal_policy": (
        "omit diagonal if an orthogonal "
        "two-edge route exists"
    ),
    "exact_cyclic_max_nodes": 128,
    "cyclic_landmarks": 32,
    "curvature_sigma_px": 2.0,
    "curvature_min_length_px": 16.0,
    "rasterizer": (
        "current Step 13 independent polygon masks"
    ),
    "scipy_version": scipy.__version__,
    "skimage_version": skimage.__version__,
    "numpy_version": np.__version__,
    "pillow_version": PIL.__version__,
}

S19_ROOT = OUTPUT_ROOT / "step19_morphology"
S19_ROOT.mkdir(parents=True, exist_ok=True)

s19_meta = train_df[
    [
        "image_id",
        "temporal_block",
        "capture_datetime",
    ]
].copy()

s19_meta["image_id"] = (
    s19_meta["image_id"].astype(str)
)

if (
    s19_meta["image_id"].duplicated().any()
    or s19_meta.isna().any().any()
):
    raise ValueError(
        "Training metadata must be complete and unique."
    )

s19_native = {
    str(key): key
    for key in image_lookup
}

if len(s19_native) != len(image_lookup):
    raise ValueError(
        "String conversion creates ambiguous image IDs."
    )

if not set(
    s19_meta["image_id"]
).issubset(s19_native):
    raise ValueError(
        "Training images are missing from image_lookup."
    )

if set(
    s19_meta["image_id"]
) & set(
    valid_df["image_id"].astype(str)
):
    raise ValueError(
        "Training and validation image IDs overlap."
    )

s19_ids = sorted(
    s19_meta["image_id"]
)

s19_expected = sum(
    len(
        annotations_by_image[
            s19_native[image_id]
        ]
    )
    for image_id in s19_ids
)

print("Training images:", len(s19_ids))
print(
    "Training annotations to measure:",
    s19_expected,
)

print(
    "Each annotation is rasterized independently; "
    "IDs are never overwritten."
)

### 19.2 Weighted Skeleton Graphs and Branch Geometry

We convert each skeleton into a weighted graph:

- Orthogonal edges have length 1 pixel.
- Diagonal edges have length √2 pixels.
- A diagonal edge is omitted when an orthogonal two-edge route already
  connects the same pixels.

This diagonal policy avoids artificial triangle shortcuts at corners while
retaining diagonal-only connections.

Adjacent graph pixels with degree three or higher form junction zones.
Branches follow degree-two chains between endpoints or junction zones.
Degree-two components without endpoints are closed cycles.

We do not prune skeleton branches. Edges inside junction zones are accounted
for separately, and we verify that branch lengths plus internal junction
length equal total graph length.

Geodesic diameter means the maximum shortest-path distance:

- Tree components: exact using two sweeps.
- Cyclic components with at most 128 nodes: exact.
- Larger cyclic components: a deterministic landmark-based lower bound,
  explicitly marked as approximate.

For disconnected annotations, diameter is the largest finite component
diameter. We never connect components through background.

Curvature is estimated after arc-length resampling and Gaussian smoothing
at sigma 2 pixels. Branches shorter than 16 pixels are excluded, and six
samples are discarded at each end.

Missing curvature is not zero curvature. Graph cycle rank and mask hole
count are separate measurements.

In [ ]:
def s19_pixel_graph(skeleton):
    coords = np.argwhere(skeleton)

    index = np.full(
        skeleton.shape,
        -1,
        dtype=np.int32,
    )

    index[tuple(coords.T)] = np.arange(
        len(coords)
    )

    rows, cols, weights = [], [], []
    height, width = skeleton.shape

    for node, (row, col) in enumerate(coords):
        for dr, dc in [
            (0, 1),
            (1, -1),
            (1, 0),
            (1, 1),
        ]:
            rr = row + dr
            cc = col + dc

            if not (
                0 <= rr < height
                and 0 <= cc < width
            ):
                continue

            if not skeleton[rr, cc]:
                continue

            if dr and dc:
                if (
                    skeleton[row, cc]
                    or skeleton[rr, col]
                ):
                    continue

            neighbor = int(index[rr, cc])
            distance = float(np.hypot(dr, dc))

            rows.extend([node, neighbor])
            cols.extend([neighbor, node])
            weights.extend([distance, distance])

    graph = coo_matrix(
        (weights, (rows, cols)),
        shape=(len(coords), len(coords)),
    ).tocsr()

    graph.sort_indices()

    return coords, graph


def s19_branches(coords, graph):
    node_count = len(coords)
    degree = np.diff(graph.indptr)

    anchor = np.full(
        node_count,
        -1,
        dtype=int,
    )

    high_degree = np.flatnonzero(
        degree >= 3
    )

    zone_count = 0

    if len(high_degree):
        zone_count, zones = connected_components(
            graph[high_degree][:, high_degree],
            directed=False,
        )

        anchor[high_degree] = zones

    next_anchor = zone_count

    for node in np.flatnonzero(degree <= 1):
        anchor[node] = next_anchor
        next_anchor += 1

    neighbors = [
        graph.indices[
            graph.indptr[node]:
            graph.indptr[node + 1]
        ]
        for node in range(node_count)
    ]

    visited = set()
    paths = []

    def edge_key(a, b):
        return (
            min(int(a), int(b)),
            max(int(a), int(b)),
        )

    upper = triu(graph, k=1).tocoo()
    internal_length = 0.0

    for a, b, weight in zip(
        upper.row,
        upper.col,
        upper.data,
    ):
        if (
            anchor[a] >= 0
            and anchor[a] == anchor[b]
        ):
            visited.add(edge_key(a, b))
            internal_length += float(weight)

    def walk(start, neighbor):
        path = [
            int(start),
            int(neighbor),
        ]

        visited.add(
            edge_key(start, neighbor)
        )

        previous = int(start)
        current = int(neighbor)

        while anchor[current] < 0:
            options = [
                int(node)
                for node in neighbors[current]
                if node != previous
            ]

            if not options:
                break

            next_node = options[0]

            if edge_key(
                current,
                next_node,
            ) in visited:
                break

            visited.add(
                edge_key(current, next_node)
            )

            path.append(next_node)

            previous, current = (
                current,
                next_node,
            )

        return np.asarray(
            path,
            dtype=np.int32,
        )

    for node in np.flatnonzero(anchor >= 0):
        for neighbor in neighbors[node]:
            if edge_key(
                node,
                neighbor,
            ) not in visited:
                paths.append(
                    walk(node, int(neighbor))
                )

    # Remaining degree-two components are cycles.
    for a, b in zip(
        upper.row,
        upper.col,
    ):
        if edge_key(a, b) not in visited:
            paths.append(
                walk(int(a), int(b))
            )

    branch_length = sum(
        np.linalg.norm(
            np.diff(coords[path], axis=0),
            axis=1,
        ).sum()
        for path in paths
    )

    if not np.isclose(
        branch_length + internal_length,
        graph.sum() / 2,
    ):
        raise ValueError(
            "Skeleton branch decomposition lost graph edges."
        )

    return (
        paths,
        degree,
        int(zone_count),
        internal_length,
    )


def s19_diameter(coords, graph):
    component_count, component = (
        connected_components(
            graph,
            directed=False,
        )
    )

    best = 0.0
    best_chord = np.nan
    all_exact = True

    for component_id in range(component_count):
        nodes = np.flatnonzero(
            component == component_id
        )

        subgraph = graph[nodes][:, nodes]
        node_count = len(nodes)

        if node_count <= 1:
            continue

        edge_count = subgraph.nnz // 2

        if edge_count == node_count - 1:
            first_distances = dijkstra(
                subgraph,
                directed=False,
                indices=0,
            )

            a = int(
                np.argmax(first_distances)
            )

            distances = dijkstra(
                subgraph,
                directed=False,
                indices=a,
            )

            b = int(np.argmax(distances))
            distance = float(distances[b])

        else:
            exact = (
                node_count
                <= S19_CFG["exact_cyclic_max_nodes"]
            )

            all_exact &= exact

            if exact:
                sources = np.arange(node_count)

            else:
                sources = np.unique(
                    np.linspace(
                        0,
                        node_count - 1,
                        min(
                            node_count,
                            S19_CFG["cyclic_landmarks"],
                        ),
                        dtype=int,
                    )
                )

            distance = -1.0
            a, b = 0, 0

            # Process one source at a time to bound memory.
            for source in sources:
                distances = dijkstra(
                    subgraph,
                    directed=False,
                    indices=int(source),
                )

                target = int(
                    np.argmax(distances)
                )

                if distances[target] > distance:
                    distance = float(
                        distances[target]
                    )

                    a = int(source)
                    b = target

        if distance > best:
            best = distance

            best_chord = float(
                np.linalg.norm(
                    coords[nodes[a]]
                    - coords[nodes[b]]
                )
            )

    return (
        best,
        bool(all_exact),
        best_chord,
        int(component_count),
    )


def s19_curvature(points):
    arc = np.r_[
        0.0,
        np.cumsum(
            np.linalg.norm(
                np.diff(points, axis=0),
                axis=1,
            )
        ),
    ]

    if (
        arc[-1]
        < S19_CFG["curvature_min_length_px"]
    ):
        return np.array([])

    grid = np.arange(
        0,
        arc[-1] + 1e-9,
        1.0,
    )

    sigma = S19_CFG["curvature_sigma_px"]

    y = gaussian_filter1d(
        np.interp(
            grid,
            arc,
            points[:, 0],
        ),
        sigma,
    )

    x = gaussian_filter1d(
        np.interp(
            grid,
            arc,
            points[:, 1],
        ),
        sigma,
    )

    dx = np.gradient(x)
    dy = np.gradient(y)

    ddx = np.gradient(dx)
    ddy = np.gradient(dy)

    speed_squared = dx * dx + dy * dy

    curvature = (
        np.abs(dx * ddy - dy * ddx)
        / np.maximum(
            speed_squared,
            1e-12,
        ) ** 1.5
    )

    trim = int(
        np.ceil(3 * sigma)
    )

    valid = (
        (speed_squared > 0.05)
        & np.isfinite(curvature)
    )

    valid[:trim] = False
    valid[-trim:] = False

    return curvature[valid]

### 19.3 Independent Instance Measurements

Each nonempty mask is cropped and padded by one background pixel before
distance-transform, skeleton and perimeter calculations.

For border-touching annotations, this padding does not reveal the unobserved
continuation. We flag those instances as border-censored.

Shape measurements describe the annotation union. For multipart annotations,
union-level axes, eccentricity and solidity can reflect gaps between parts.

Thickness is measured as twice the Euclidean distance transform at skeleton
pixels. This is a digital width proxy:

- It is not an exact geometric diameter.
- A one-pixel-wide line has proxy width 2.
- Width quantiles are retained rather than relying on one average.

Neighbor separation is the minimum distance between foreground pixel centers:

- Overlapping masks have distance zero.
- Edge-adjacent foreground pixels have distance 1.
- Diagonally adjacent pixels have distance √2.

This is not a physical edge-to-edge gap.

Pairwise overlap sums may count a pixel multiple times when it overlaps
multiple neighboring annotations. They are not union-overlap areas.

In [ ]:
def s19_measure(mask):
    yy, xx = np.where(mask)

    if not len(yy):
        return None

    y0 = int(yy.min())
    y1 = int(yy.max() + 1)
    x0 = int(xx.min())
    x1 = int(xx.max() + 1)

    crop = mask[y0:y1, x0:x1].copy()
    padded = np.pad(crop, 1)

    component_labels = label(
        padded,
        connectivity=2,
    )

    component_sizes = np.bincount(
        component_labels.ravel()
    )[1:]

    # One label deliberately measures the annotation union.
    props = regionprops(
        padded.astype(np.uint8)
    )[0]

    skeleton = skeletonize(padded)

    distance = distance_transform_edt(
        padded
    )

    coords, graph = s19_pixel_graph(
        skeleton
    )

    if not len(coords):
        raise ValueError(
            "Nonempty mask produced an empty skeleton."
        )

    (
        paths,
        degree,
        junctions,
        internal_length,
    ) = s19_branches(coords, graph)

    (
        diameter,
        diameter_exact,
        chord,
        graph_components,
    ) = s19_diameter(coords, graph)

    widths = (
        2 * distance[tuple(coords.T)]
    )

    perimeter = float(
        perimeter_crofton(
            padded,
            directions=4,
        )
    )

    area = int(crop.sum())

    branch_records = []
    curvature_arrays = []

    for branch_id, path in enumerate(paths):
        points = coords[path]

        branch_length = float(
            np.linalg.norm(
                np.diff(points, axis=0),
                axis=1,
            ).sum()
        )

        curvature = s19_curvature(
            points
        )

        if curvature.size:
            curvature_arrays.append(
                curvature
            )

        branch_records.append(
            {
                "branch_id": branch_id,
                "length_px": branch_length,
                "closed": bool(
                    path[0] == path[-1]
                ),
                "curvature_points": int(
                    curvature.size
                ),
                "curvature_median": (
                    float(np.median(curvature))
                    if curvature.size
                    else np.nan
                ),
                "curvature_p90": (
                    float(
                        np.quantile(
                            curvature,
                            0.9,
                        )
                    )
                    if curvature.size
                    else np.nan
                ),
            }
        )

    all_curvature = (
        np.concatenate(curvature_arrays)
        if curvature_arrays
        else np.array([])
    )

    offset = np.array(
        [y0 - 1, x0 - 1]
    )

    global_coords = coords + offset

    upper = triu(
        graph,
        k=1,
    ).tocoo()

    flat_paths = (
        np.concatenate(paths)
        if paths
        else np.array(
            [],
            dtype=np.int32,
        )
    )

    path_offsets = np.r_[
        0,
        np.cumsum(
            [len(path) for path in paths]
        ),
    ]

    graph_assets = {
        "coords": global_coords.astype(
            np.int32
        ),
        "edges": np.column_stack(
            [
                upper.row,
                upper.col,
            ]
        ).astype(np.int32),
        "weights": upper.data,
        "branch_nodes": flat_paths,
        "branch_offsets": path_offsets,
        "width_proxy": widths.astype(
            np.float32
        ),
    }

    height, width = mask.shape

    centroid_y = float(
        props.centroid[0] + y0 - 1
    )

    centroid_x = float(
        props.centroid[1] + x0 - 1
    )

    features = {
        "area_px": area,
        "area_fraction": area / (height * width),
        "bbox_y0": y0,
        "bbox_y1": y1,
        "bbox_x0": x0,
        "bbox_x1": x1,
        "bbox_width_px": x1 - x0,
        "bbox_height_px": y1 - y0,
        "extent": area / crop.size,
        "perimeter_crofton_px": perimeter,
        "shape_complexity": (
            perimeter ** 2
            / (4 * np.pi * area)
        ),
        "solidity": float(props.solidity),
        "eccentricity": float(props.eccentricity),
        "major_axis_length_px": float(
            props.axis_major_length
        ),
        "minor_axis_length_px": float(
            props.axis_minor_length
        ),
        "orientation_deg_from_row_axis": float(
            np.degrees(props.orientation)
        ),
        "component_count": int(
            len(component_sizes)
        ),
        "largest_component_fraction": float(
            component_sizes.max() / area
        ),
        "hole_count": int(
            len(component_sizes)
            - euler_number(
                padded,
                connectivity=2,
            )
        ),
        "touches_image_border": bool(
            y0 == 0
            or x0 == 0
            or y1 == height
            or x1 == width
        ),
        "centroid_y": centroid_y,
        "centroid_x": centroid_x,
        "centroid_frame_radius": float(
            np.hypot(
                (
                    centroid_y
                    - (height - 1) / 2
                ) / (height / 2),
                (
                    centroid_x
                    - (width - 1) / 2
                ) / (width / 2),
            )
        ),
        "skeleton_pixels": int(len(coords)),
        "skeleton_length_px": float(
            graph.sum() / 2
        ),
        "geodesic_diameter_px": diameter,
        "diameter_is_exact": diameter_exact,
        "diameter_path_tortuosity": (
            diameter / chord
            if chord > 0
            else np.nan
        ),
        "skeleton_components": graph_components,
        "endpoints": int(
            (degree == 1).sum()
        ),
        "isolated_skeleton_pixels": int(
            (degree == 0).sum()
        ),
        "junction_zones": junctions,
        "graph_cycle_rank": int(
            graph.nnz // 2
            - len(coords)
            + graph_components
        ),
        "branch_count": len(paths),
        "junction_internal_length_px": internal_length,
        "width_proxy_p10": float(
            np.quantile(widths, 0.1)
        ),
        "width_proxy_median": float(
            np.median(widths)
        ),
        "width_proxy_p90": float(
            np.quantile(widths, 0.9)
        ),
        "width_proxy_mean": float(
            widths.mean()
        ),
        "curvature_points": int(
            all_curvature.size
        ),
        "curvature_median": (
            float(np.median(all_curvature))
            if all_curvature.size
            else np.nan
        ),
        "curvature_p90": (
            float(
                np.quantile(
                    all_curvature,
                    0.9,
                )
            )
            if all_curvature.size
            else np.nan
        ),
    }

    boundary = (
        crop
        & ~binary_erosion(
            crop,
            structure=np.ones((3, 3)),
            border_value=0,
        )
    )

    boundary_points = (
        np.argwhere(boundary)
        + np.array([y0, x0])
    )

    neighbor_data = {
        "crop": crop,
        "y0": y0,
        "y1": y1,
        "x0": x0,
        "x1": x1,
        "boundary": boundary_points,
        "tree": cKDTree(boundary_points),
    }

    return (
        features,
        branch_records,
        graph_assets,
        neighbor_data,
    )


def s19_overlap(a, b):
    y_start = max(a["y0"], b["y0"])
    y_end = min(a["y1"], b["y1"])

    x_start = max(a["x0"], b["x0"])
    x_end = min(a["x1"], b["x1"])

    if (
        y_start >= y_end
        or x_start >= x_end
    ):
        return 0

    a_region = a["crop"][
        y_start - a["y0"]:y_end - a["y0"],
        x_start - a["x0"]:x_end - a["x0"],
    ]

    b_region = b["crop"][
        y_start - b["y0"]:y_end - b["y0"],
        x_start - b["x0"]:x_end - b["x0"],
    ]

    return int(
        np.count_nonzero(
            a_region & b_region
        )
    )

### 19.4 Resumable Full-Population Extraction

We process one training image at a time.

For each image we save:

- An annotation-level measurement table.
- Branch measurements.
- Pairwise overlap and separation measurements.
- Compressed skeleton graph arrays.

The graph archive retains original-coordinate skeleton nodes, weighted edges,
branch paths and width proxies.

Source annotations, image dimensions, algorithm code, settings and relevant
library versions determine the cache identity.

The graph archive is written before the JSON completion checkpoint.
Completed images are reused when this cell is rerun.

Empty masks remain in the annotation ledger with an explicit status.
Unexpected exceptions are logged by image and prevent successful completion.

An annotation with no other nonempty annotation in its image has a missing
neighbor distance, not a distance of zero.

This extraction preserves the current rasterization convention and does not
modify the original masks or feature cache.

In [ ]:
def s19_code_object(code):
    constants = [
        (
            s19_code_object(value)
            if isinstance(value, types.CodeType)
            else repr(value)
        )
        for value in code.co_consts
    ]

    return {
        "bytecode": code.co_code.hex(),
        "names": code.co_names,
        "constants": constants,
    }


def s19_clean(value):
    """Convert missing/nonfinite values to JSON null."""
    if isinstance(value, dict):
        return {
            key: s19_clean(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            s19_clean(item)
            for item in value
        ]

    if isinstance(
        value,
        (float, np.floating),
    ):
        return (
            float(value)
            if np.isfinite(value)
            else None
        )

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.bool_):
        return bool(value)

    return value


def s19_process_image(
    image_id,
    graph_path,
):
    native_id = s19_native[image_id]
    info = image_lookup[native_id]

    shape = (
        int(info["height"]),
        int(info["width"]),
    )

    annotations = sorted(
        annotations_by_image[native_id],
        key=lambda annotation: str(
            annotation["id"]
        ),
    )

    records = []
    branch_rows = []
    assets = {}
    neighbors = []

    annotation_ids = [
        str(annotation["id"])
        for annotation in annotations
    ]

    if (
        len(set(annotation_ids))
        != len(annotation_ids)
    ):
        raise ValueError(
            "Duplicate annotation IDs within an image."
        )

    for position, annotation in enumerate(
        annotations
    ):
        mask = rasterize_annotation(
            annotation,
            shape,
        )

        measured = s19_measure(mask)

        row = {
            "image_id": image_id,
            "annotation_id": str(
                annotation["id"]
            ),
            "graph_key": f"a{position}",
        }

        if measured is None:
            row.update(
                status="empty_mask",
                area_px=0,
            )

            records.append(row)
            continue

        (
            features,
            branches,
            graph_arrays,
            neighbor_data,
        ) = measured

        row.update(
            status="ok",
            **features,
        )

        row.update(
            nearest_annotation_id=None,
            nearest_pixel_center_distance_px=np.nan,
            overlapping_annotation_count=0,
            pair_overlap_pixel_sum=0,
        )

        neighbors.append(
            (
                len(records),
                neighbor_data,
            )
        )

        for branch in branches:
            branch_rows.append(
                {
                    "image_id": image_id,
                    "annotation_id": str(
                        annotation["id"]
                    ),
                    **branch,
                }
            )

        for key, array in graph_arrays.items():
            assets[
                f"a{position}_{key}"
            ] = array

        records.append(row)

    pair_rows = []

    for position, (index_a, a) in enumerate(
        neighbors
    ):
        for index_b, b in neighbors[position + 1:]:
            overlap = s19_overlap(a, b)

            if overlap:
                separation = 0.0

            else:
                separation = float(
                    b["tree"].query(
                        a["boundary"],
                        k=1,
                    )[0].min()
                )

            pair_rows.append(
                {
                    "image_id": image_id,
                    "annotation_a": records[
                        index_a
                    ]["annotation_id"],
                    "annotation_b": records[
                        index_b
                    ]["annotation_id"],
                    "overlap_px": overlap,
                    "min_pixel_center_distance_px": separation,
                }
            )

            for left, right in [
                (index_a, index_b),
                (index_b, index_a),
            ]:
                row = records[left]

                current_distance = row[
                    "nearest_pixel_center_distance_px"
                ]

                if (
                    not np.isfinite(current_distance)
                    or separation < current_distance
                ):
                    row[
                        "nearest_pixel_center_distance_px"
                    ] = separation

                    row[
                        "nearest_annotation_id"
                    ] = records[right][
                        "annotation_id"
                    ]

                if overlap:
                    row[
                        "overlapping_annotation_count"
                    ] += 1

                    row[
                        "pair_overlap_pixel_sum"
                    ] += overlap

    temporary_graph = graph_path.with_suffix(
        ".npz.tmp"
    )

    with open(
        temporary_graph,
        "wb",
    ) as file:
        np.savez_compressed(
            file,
            **assets,
        )

    os.replace(
        temporary_graph,
        graph_path,
    )

    return {
        "image_id": image_id,
        "config_hash": s19_hash,
        "instances": records,
        "branches": branch_rows,
        "pairs": pair_rows,
    }


s19_source = []

for image_id in s19_ids:
    native_id = s19_native[image_id]
    info = image_lookup[native_id]

    s19_source.append(
        {
            "image_id": image_id,
            "height": int(info["height"]),
            "width": int(info["width"]),
            "annotations": sorted(
                annotations_by_image[native_id],
                key=lambda annotation: str(
                    annotation["id"]
                ),
            ),
        }
    )

source_hash = hashlib.sha256(
    json.dumps(
        s19_source,
        sort_keys=True,
    ).encode()
).hexdigest()

functions = [
    get_polygons,
    rasterize_annotation,
    s19_pixel_graph,
    s19_branches,
    s19_diameter,
    s19_curvature,
    s19_measure,
    s19_overlap,
    s19_process_image,
    s19_clean,
    s19_code_object,
]

code_hash = hashlib.sha256(
    json.dumps(
        [
            s19_code_object(
                function.__code__
            )
            for function in functions
        ],
        sort_keys=True,
    ).encode()
).hexdigest()

s19_config = {
    "settings": S19_CFG,
    "source_hash": source_hash,
    "code_hash": code_hash,
}

s19_hash = hashlib.sha256(
    json.dumps(
        s19_config,
        sort_keys=True,
    ).encode()
).hexdigest()[:16]

S19_CACHE = S19_ROOT / s19_hash
S19_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

(
    S19_CACHE / "config.json"
).write_text(
    json.dumps(
        s19_config,
        indent=2,
    )
)

s19_failures = []

for position, image_id in enumerate(
    s19_ids,
    start=1,
):
    key = hashlib.sha256(
        image_id.encode()
    ).hexdigest()[:20]

    checkpoint = (
        S19_CACHE / f"{key}.json"
    )

    graph_path = (
        S19_CACHE / f"{key}.npz"
    )

    try:
        reusable = False

        if (
            checkpoint.exists()
            and graph_path.exists()
        ):
            cached = json.loads(
                checkpoint.read_text()
            )

            reusable = (
                cached.get("image_id") == image_id
                and cached.get("config_hash") == s19_hash
                and len(cached["instances"])
                == len(
                    annotations_by_image[
                        s19_native[image_id]
                    ]
                )
            )

            if reusable:
                with np.load(
                    graph_path,
                    allow_pickle=False,
                ) as archive:
                    reusable = all(
                        (
                            f"{row['graph_key']}_coords"
                            in archive.files
                        )
                        for row in cached["instances"]
                        if row["status"] == "ok"
                    )

        if not reusable:
            result = s19_process_image(
                image_id,
                graph_path,
            )

            temporary = checkpoint.with_suffix(
                ".json.tmp"
            )

            temporary.write_text(
                json.dumps(
                    s19_clean(result),
                    allow_nan=False,
                )
            )

            os.replace(
                temporary,
                checkpoint,
            )

        if (
            position % 20 == 0
            or position == len(s19_ids)
        ):
            print(
                f"Images {position}/{len(s19_ids)}"
                f" | failures {len(s19_failures)}",
                flush=True,
            )

    except Exception as error:
        s19_failures.append(
            {
                "image_id": image_id,
                "error": repr(error),
            }
        )

pd.DataFrame(
    s19_failures,
    columns=[
        "image_id",
        "error",
    ],
).to_csv(
    S19_CACHE / "image_failures.csv",
    index=False,
)

if s19_failures:
    display(
        pd.DataFrame(s19_failures)
    )

    raise RuntimeError(
        "Some images failed. Resolve the logged errors "
        "and rerun this cell; completed images resume."
    )

print(
    "Population extraction complete. Cache:",
    S19_CACHE,
)

### 19.5 Population Accounting and Saved Tables

Every expected training annotation must appear exactly once as either:

- A successfully measured nonempty mask.
- An explicitly recorded empty mask.

Empty annotations remain in the master table but are excluded from morphology
distributions.

We retain:

- Whether geodesic diameter is exact or a lower bound.
- Whether the annotation touches the image border.
- Whether sufficient branch length exists to estimate curvature.
- Whether the annotation overlaps another annotation.
- Whether a neighboring annotation exists.

Undefined measurements remain missing values rather than fabricated zeros.

Temporal metadata is joined to the instance table so these descriptors can
later be connected to group-aware model errors.

In [ ]:
s19_payloads = []

for image_id in s19_ids:
    key = hashlib.sha256(
        image_id.encode()
    ).hexdigest()[:20]

    payload = json.loads(
        (
            S19_CACHE / f"{key}.json"
        ).read_text()
    )

    if (
        payload["image_id"] != image_id
        or payload["config_hash"] != s19_hash
    ):
        raise ValueError(
            "Checkpoint identity/configuration mismatch."
        )

    s19_payloads.append(payload)

s19_instances = pd.DataFrame(
    [
        row
        for payload in s19_payloads
        for row in payload["instances"]
    ]
)

s19_branches_df = pd.DataFrame(
    [
        row
        for payload in s19_payloads
        for row in payload["branches"]
    ]
)

s19_pairs_df = pd.DataFrame(
    [
        row
        for payload in s19_payloads
        for row in payload["pairs"]
    ]
)

if (
    len(s19_instances) != s19_expected
    or s19_instances.duplicated(
        [
            "image_id",
            "annotation_id",
        ]
    ).any()
):
    raise ValueError(
        "The annotation population was not "
        "accounted for uniquely."
    )

s19_instances = s19_instances.merge(
    s19_meta,
    on="image_id",
    how="left",
    validate="many_to_one",
)

s19_valid = s19_instances.loc[
    s19_instances["status"] == "ok"
].copy()

if s19_valid.empty:
    raise RuntimeError(
        "No valid instances were measured."
    )

for column in [
    "area_px",
    "skeleton_pixels",
    "width_proxy_median",
]:
    if not (
        s19_valid[column] > 0
    ).all():
        raise ValueError(
            f"Invalid {column}"
        )

if not (
    s19_valid["hole_count"] >= 0
).all():
    raise ValueError(
        "Invalid topology accounting."
    )

s19_instances.to_parquet(
    S19_CACHE / "all_training_instances.parquet",
    index=False,
)

s19_branches_df.to_csv(
    S19_CACHE / "all_training_branches.csv",
    index=False,
)

s19_pairs_df.to_csv(
    S19_CACHE / "within_image_instance_pairs.csv",
    index=False,
)

s19_population_summary = pd.Series(
    {
        "training_images": len(s19_ids),
        "annotations": len(s19_instances),
        "measured": len(s19_valid),
        "empty_masks": int(
            (
                s19_instances["status"]
                == "empty_mask"
            ).sum()
        ),
        "multipart": int(
            (
                s19_valid["component_count"] > 1
            ).sum()
        ),
        "border_touching": int(
            s19_valid[
                "touches_image_border"
            ].sum()
        ),
        "approximate_diameters": int(
            (
                ~s19_valid[
                    "diameter_is_exact"
                ].astype(bool)
            ).sum()
        ),
        "with_curvature": int(
            (
                s19_valid["curvature_points"] > 0
            ).sum()
        ),
    }
)

display(
    s19_population_summary.to_frame(
        "count"
    )
)

s19_population_summary.to_csv(
    S19_CACHE / "population_summary.csv"
)

### 19.6 Full-Population Distributions and Geometric Relationships

We summarize all valid training annotations rather than the previous
30-image sample.

The distributions cover:

- Area and perimeter.
- Width proxy and geodesic diameter.
- Total skeleton length.
- Curvature.
- Junction zones and disconnected components.
- Nearest-instance separation.

Log-one-plus displays reduce tail compression while retaining zeros such as
zero neighbor distance or zero junction count.

The joint plots help distinguish:

- Long, narrow structures from short, thick structures.
- Unbranched structures from complex skeleton networks.
- Isolated instances from crowded regions.

These are descriptive annotation statistics. We do not treat annotations
from the same image as independent observations for significance testing.

All length-like measurements remain pixel-scale quantities. Comparisons
across differently scaled acquisitions require caution.

In [ ]:
s19_plot_features = [
    "area_px",
    "perimeter_crofton_px",
    "width_proxy_median",
    "geodesic_diameter_px",
    "skeleton_length_px",
    "curvature_p90",
    "junction_zones",
    "component_count",
    "nearest_pixel_center_distance_px",
]

fig, axes = plt.subplots(
    3,
    3,
    figsize=(17, 13),
    constrained_layout=True,
)

for ax, feature in zip(
    axes.flat,
    s19_plot_features,
):
    values = pd.to_numeric(
        s19_valid[feature],
        errors="coerce",
    ).dropna()

    values = values[
        np.isfinite(values)
    ]

    ax.hist(
        np.log1p(values),
        bins=40,
        color="steelblue",
        edgecolor="white",
    )

    ax.set(
        title=feature,
        xlabel="log(1 + value)",
        ylabel="Instances",
    )

fig.savefig(
    S19_CACHE / "population_morphology_distributions.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

display(
    s19_valid[
        s19_plot_features
    ].describe(
        percentiles=[
            0.1,
            0.25,
            0.5,
            0.75,
            0.9,
            0.99,
        ]
    ).T
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5),
    constrained_layout=True,
)

relationships = [
    (
        "width_proxy_median",
        "geodesic_diameter_px",
    ),
    (
        "geodesic_diameter_px",
        "skeleton_length_px",
    ),
    (
        "nearest_pixel_center_distance_px",
        "width_proxy_median",
    ),
]

for ax, (x_feature, y_feature) in zip(
    axes,
    relationships,
):
    usable = s19_valid[
        [
            x_feature,
            y_feature,
        ]
    ].dropna()

    ax.hexbin(
        np.log1p(usable[x_feature]),
        np.log1p(usable[y_feature]),
        gridsize=35,
        mincnt=1,
        cmap="viridis",
    )

    ax.set(
        xlabel=f"log(1 + {x_feature})",
        ylabel=f"log(1 + {y_feature})",
    )

fig.savefig(
    S19_CACHE / "geometry_relationships.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

### 19.7 Morphology Strata for Later Failure Analysis

We define training-derived flags for:

- Low width proxy.
- Large geodesic diameter.
- High curvature.
- Small nearest-instance separation.
- Multipart annotations.
- Border-censored annotations.
- Overlapping annotations.

These are candidate difficulty strata, not demonstrated model-failure labels.
Their actual difficulty will be assessed after predictions are available.

Quantile thresholds use eligible valid training annotations only.
Ties can make flagged proportions differ from 25%.

Missing curvature and missing neighbor distance remain unknown rather than
being classified as easy.

Approximate diameter measurements retain their exactness flag. When analyzing
length-dependent model failures, inspect those cases separately.

We save the thresholds for reuse and do not recompute them on validation
or test labels.

In [ ]:
s19_rules = {
    "thin_proxy": (
        "width_proxy_median",
        0.25,
        "low",
    ),
    "long_geodesic": (
        "geodesic_diameter_px",
        0.75,
        "high",
    ),
    "high_curvature": (
        "curvature_p90",
        0.75,
        "high",
    ),
    "crowded": (
        "nearest_pixel_center_distance_px",
        0.25,
        "low",
    ),
}

s19_thresholds = {}

for flag, (
    feature,
    quantile,
    direction,
) in s19_rules.items():
    threshold = float(
        s19_valid[feature].quantile(
            quantile
        )
    )

    s19_thresholds[flag] = {
        "feature": feature,
        "quantile": quantile,
        "threshold": threshold,
        "direction": direction,
    }

    values = pd.to_numeric(
        s19_valid[feature],
        errors="coerce",
    )

    s19_valid[flag] = pd.Series(
        pd.NA,
        index=s19_valid.index,
        dtype="boolean",
    )

    eligible = (
        values.notna()
        & np.isfinite(values)
    )

    if direction == "low":
        condition = (
            values[eligible] <= threshold
        )
    else:
        condition = (
            values[eligible] >= threshold
        )

    s19_valid.loc[
        eligible,
        flag,
    ] = condition

s19_valid["multipart"] = (
    s19_valid["component_count"] > 1
)

s19_valid["border_censored"] = (
    s19_valid["touches_image_border"]
    .astype(bool)
)

s19_valid["overlapping_labels"] = (
    s19_valid[
        "overlapping_annotation_count"
    ] > 0
)

s19_flags = list(s19_rules) + [
    "multipart",
    "border_censored",
    "overlapping_labels",
]

s19_flag_summary = pd.DataFrame(
    [
        {
            "stratum": flag,
            "eligible": int(
                s19_valid[flag]
                .notna()
                .sum()
            ),
            "instances": int(
                s19_valid[flag]
                .fillna(False)
                .sum()
            ),
        }
        for flag in s19_flags
    ]
)

display(s19_flag_summary)

s19_valid.to_parquet(
    S19_CACHE / "morphology_with_difficulty_flags.parquet",
    index=False,
)

s19_flag_summary.to_csv(
    S19_CACHE / "difficulty_stratum_counts.csv",
    index=False,
)

(
    S19_CACHE
    / "training_derived_difficulty_thresholds.json"
).write_text(
    json.dumps(
        s19_clean(s19_thresholds),
        indent=2,
        allow_nan=False,
    )
)

### 19.8 Native-Resolution Morphology Gallery

We inspect deterministic extreme examples for:

- Width.
- Geodesic diameter.
- Curvature.
- Crowding.
- Component count.
- Junction-zone count.

The left panel shows the independent annotation on the image.
The right panel shows skeleton pixels colored by the EDT width proxy.

These examples check measurement behavior. They are not representative
performance examples or evidence that a particular morphology is difficult.

Inspect whether:

- Skeleton branches correspond to visible structure.
- Multipart annotations retain their original identity.
- Border-censored objects are recognizable.
- Width colors agree with visible thickness.
- Extreme values reflect meaningful geometry or annotation artifacts.

The saved graph arrays and measurement tables remain linked by image ID,
annotation ID and graph key.

This step does not overwrite the original annotations, feature cache or split.

In [ ]:
s19_examples = []

example_specs = [
    (
        "Thin",
        "width_proxy_median",
        True,
    ),
    (
        "Long",
        "geodesic_diameter_px",
        False,
    ),
    (
        "Curved",
        "curvature_p90",
        False,
    ),
    (
        "Crowded",
        "nearest_pixel_center_distance_px",
        True,
    ),
    (
        "Multipart",
        "component_count",
        False,
    ),
    (
        "Branched",
        "junction_zones",
        False,
    ),
]

for title, feature, ascending in example_specs:
    candidates = (
        s19_valid.dropna(
            subset=[feature]
        )
        .sort_values(
            [
                feature,
                "image_id",
                "annotation_id",
            ],
            ascending=[
                ascending,
                True,
                True,
            ],
        )
    )

    if len(candidates):
        s19_examples.append(
            (
                title,
                candidates.iloc[0],
            )
        )

fig, axes = plt.subplots(
    len(s19_examples),
    2,
    figsize=(
        14,
        5 * len(s19_examples),
    ),
    squeeze=False,
)

for row_number, (
    title,
    row,
) in enumerate(s19_examples):
    native_id = s19_native[
        row["image_id"]
    ]

    image = load_solar_image(
        native_id
    )

    annotation = next(
        annotation
        for annotation in annotations_by_image[
            native_id
        ]
        if str(annotation["id"])
        == row["annotation_id"]
    )

    mask = rasterize_annotation(
        annotation,
        image.shape,
    )

    margin = 20

    y0 = max(
        0,
        int(row["bbox_y0"]) - margin,
    )

    y1 = min(
        image.shape[0],
        int(row["bbox_y1"]) + margin,
    )

    x0 = max(
        0,
        int(row["bbox_x0"]) - margin,
    )

    x1 = min(
        image.shape[1],
        int(row["bbox_x1"]) + margin,
    )

    key = hashlib.sha256(
        row["image_id"].encode()
    ).hexdigest()[:20]

    with np.load(
        S19_CACHE / f"{key}.npz",
        allow_pickle=False,
    ) as archive:
        coords = archive[
            f"{row['graph_key']}_coords"
        ]

        widths = archive[
            f"{row['graph_key']}_width_proxy"
        ]

    for ax in axes[row_number]:
        ax.imshow(
            image[y0:y1, x0:x1],
            cmap="gray",
        )

        ax.axis("off")

    overlay = np.ma.masked_where(
        ~mask[y0:y1, x0:x1],
        mask[y0:y1, x0:x1],
    )

    axes[row_number, 0].imshow(
        overlay,
        cmap="autumn",
        alpha=0.45,
        vmin=0,
        vmax=1,
    )

    axes[row_number, 0].set_title(
        f"{title}: image {row['image_id']}, "
        f"annotation {row['annotation_id']}"
    )

    scatter = axes[row_number, 1].scatter(
        coords[:, 1] - x0,
        coords[:, 0] - y0,
        c=widths,
        s=4,
        cmap="plasma",
    )

    fig.colorbar(
        scatter,
        ax=axes[row_number, 1],
        label="2 × EDT width proxy in pixels",
    )

    axes[row_number, 1].set_title(
        f"Diameter {row['geodesic_diameter_px']:.1f}px"
        f" | components {int(row['component_count'])}"
        f" | junction zones {int(row['junction_zones'])}"
    )

fig.tight_layout()

fig.savefig(
    S19_CACHE / "native_resolution_geometry_gallery.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

print(
    "Step 19 complete:",
    S19_CACHE,
)

print(
    "Geometry and difficulty flags are "
    "label-derived diagnostics, not inference inputs."
)

## 20. Data Split and Annotation Audit

### 20.1 Purpose and audit contract

This section establishes whether our data, annotations, temporal
partition and feature cache are trustworthy enough for model development.

The audit covers:

1. Image and annotation identity, uniqueness and referential integrity.
2. Physical image readability, dimensions, native representation,
   grayscale conversion and orientation metadata.
3. Independent timestamp parsing and temporal-block reconstruction.
4. Exact reconstruction of the existing grouped and purged split.
5. Polygon structure, coordinate bounds and decoded-mask integrity.
6. Independent instance masks, overlapping annotations and information
   lost by conversion to a single integer-label image.
7. Exact duplicate images and candidate near-duplicate observations.
8. Reviewed observation clusters and cross-partition dependence.
9. Sensitivity to longer temporal purges and duplicate-cluster exclusion.
10. Feature-cache membership, schema, targets, sampling and numerical
    reproducibility.
11. Training–validation distribution differences.
12. Fitting provenance, unresolved findings and reproducible audit outputs.

### Audit principles

- Preserve the current split while investigating it.
- Preserve original image and annotation IDs.
- Keep independent instance masks authoritative where annotations overlap.
- Treat similarity thresholds as screening rules, not proof of duplication.
- Separate automated failures, review findings and unverified provenance.
- Do not interpret an absence of detected duplicates as proof that all
  observations are physically independent.
- Do not claim official mask-decoding or evaluator parity in this step.
  That remains Step 21.
- Do not use validation statistics to repeatedly redesign an easier split.

In [ ]:
import hashlib
import inspect
import itertools
import json
import math
import platform
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import gaussian_filter, shift as ndi_shift
from scipy.fft import dctn
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
from sklearn.model_selection import GroupShuffleSplit
from skimage.measure import label
from skimage.metrics import structural_similarity
from skimage.registration import phase_cross_correlation
from IPython.display import display

_required = [
    "annotations_data", "ANNOTATION_PATH", "TRAIN_IMAGE_DIR",
    "OUTPUT_ROOT", "split_df", "train_df", "valid_df",
    "rasterize_annotation", "load_solar_image",
    "load_instance_targets", "extract_pixel_features",
    "CACHE_DIR", "FEATURE_NAMES", "image_cache_path",
    "SEED", "BLOCK_DAYS", "PURGE_DAYS",
    "VALIDATION_FRACTION", "PIXELS_PER_CLASS",
]

_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run the earlier notebook cells first. Missing: "
        + ", ".join(_missing)
    )

AUDIT_CONFIG = {
    "audit_version": "step20_v1",
    "seed": int(SEED),
    "block_days": int(BLOCK_DAYS),
    "purge_days": int(PURGE_DAYS),
    "validation_fraction": float(VALIDATION_FRACTION),
    "thumbnail_size": 128,
    "descriptor_size": 32,
    "candidate_detail_cosine": 0.93,
    "candidate_phash_distance": 6,
    "candidate_temporal_days": 7,
    "maximum_registration_shift_pixels": 12,
    "feature_replay_images": 5,
    "feature_replay_rtol": 1e-4,
    "feature_replay_atol": 2e-6,
}

def a20_jsonable(value):
    if isinstance(value, dict):
        return {
            str(k): a20_jsonable(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [a20_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (Path, pd.Timestamp)):
        return str(value)
    return value

def a20_digest(value):
    payload = json.dumps(
        a20_jsonable(value),
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode()
    return hashlib.sha256(payload).hexdigest()

def a20_file_hash(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

def a20_array_hash(array):
    array = np.ascontiguousarray(array)
    header = json.dumps({
        "shape": list(array.shape),
        "dtype": array.dtype.str,
    }, sort_keys=True).encode()
    hasher = hashlib.sha256(header)
    hasher.update(array.tobytes())
    return hasher.hexdigest()

def a20_source(function):
    try:
        source = inspect.getsource(function)
        return {
            "available": True,
            "sha256": hashlib.sha256(source.encode()).hexdigest(),
            "source": source,
        }
    except (OSError, TypeError):
        return {
            "available": False,
            "sha256": None,
            "source": None,
        }

audit_findings = []

def a20_issue(severity, check, entity, detail):
    audit_findings.append({
        "severity": severity,
        "check": check,
        "entity": str(entity),
        "detail": str(detail),
    })

def a20_check(condition, check, detail, severity="BLOCKER"):
    if not bool(condition):
        a20_issue(severity, check, "dataset", detail)

def a20_save(frame, name):
    frame.to_csv(AUDIT_DIR / name, index=False)

_source_registry = {
    name: a20_source(globals()[name])
    for name in [
        "rasterize_annotation",
        "load_solar_image",
        "load_instance_targets",
        "extract_pixel_features",
        "image_cache_path",
    ]
}

_split_signature = {
    "train_ids": sorted(train_df["image_id"].astype(str).tolist()),
    "validation_ids": sorted(valid_df["image_id"].astype(str).tolist()),
}

AUDIT_RUN_ID = a20_digest({
    "annotation_sha256": a20_file_hash(ANNOTATION_PATH),
    "split": _split_signature,
    "config": AUDIT_CONFIG,
    "sources": {
        k: v["sha256"] for k, v in _source_registry.items()
    },
})[:16]

AUDIT_DIR = (
    Path(OUTPUT_ROOT) / "step20_audit" / AUDIT_RUN_ID
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

with open(AUDIT_DIR / "audit_configuration.json", "w") as handle:
    json.dump(AUDIT_CONFIG, handle, indent=2)

with open(AUDIT_DIR / "current_function_sources.json", "w") as handle:
    json.dump(_source_registry, handle, indent=2)

print("Audit run:", AUDIT_RUN_ID)
print("Output:", AUDIT_DIR)
print("Current split:", {
    "train": len(train_df),
    "validation": len(valid_df),
})
print(
    "Current function fingerprints describe this session; "
    "they do not prove which code created historical caches."
)

### 20.2 Identity, schema and manifest reconciliation

The annotation JSON is the source of image and annotation identity.
The current train and validation tables define the partition being audited.

We independently check:

- Missing and duplicate image, annotation and category IDs.
- IDs that collide after string conversion.
- Missing filenames and duplicate filename references.
- Annotations referencing nonexistent images or categories.
- Partition records referencing nonexistent images.
- Repeated or shared image IDs across partitions.
- Images excluded from both partitions.
- Missing or ambiguous physical files.

Excluded images are retained in the audit manifest. They are not
automatically assumed to be correctly purged; that is verified later.

In [ ]:
_a20_images = annotations_data["images"]
_a20_annotations = annotations_data["annotations"]
_a20_categories = annotations_data.get("categories", [])

def a20_validate_ids(records, kind):
    values = [record.get("id") for record in records]

    missing = [
        i for i, value in enumerate(values)
        if value is None or str(value).strip() == ""
    ]

    counts = Counter(str(value) for value in values)
    duplicates = {
        key: count for key, count in counts.items()
        if count > 1
    }

    if missing:
        a20_issue(
            "BLOCKER", f"{kind}_missing_ids",
            kind, f"Record positions: {missing[:30]}"
        )

    if duplicates:
        a20_issue(
            "BLOCKER", f"{kind}_duplicate_or_colliding_ids",
            kind, json.dumps(duplicates)
        )

    return not missing and not duplicates

_identity_ok = all([
    a20_validate_ids(_a20_images, "image"),
    a20_validate_ids(_a20_annotations, "annotation"),
    a20_validate_ids(_a20_categories, "category"),
])

if not _identity_ok:
    a20_save(pd.DataFrame(audit_findings), "integrity_checks.csv")
    raise RuntimeError(
        "Identity failures must be resolved before dictionary lookup "
        "can safely represent these records. See integrity_checks.csv."
    )

a20_image_lookup = {
    str(record["id"]): record for record in _a20_images
}

a20_raw_image_ids = {
    str(record["id"]): record["id"] for record in _a20_images
}

a20_category_ids = {
    str(record["id"]) for record in _a20_categories
}

a20_annotations_by_image = defaultdict(list)

for ann in _a20_annotations:
    image_id = str(ann.get("image_id"))

    if image_id not in a20_image_lookup:
        a20_issue(
            "BLOCKER", "orphan_annotation",
            ann["id"], f"Unknown image_id={image_id}"
        )
    else:
        a20_annotations_by_image[image_id].append(ann)

    if str(ann.get("category_id")) not in a20_category_ids:
        a20_issue(
            "BLOCKER", "unknown_category",
            ann["id"], ann.get("category_id")
        )

a20_train_ids = set(train_df["image_id"].astype(str))
a20_valid_ids = set(valid_df["image_id"].astype(str))
a20_known_ids = set(a20_image_lookup)

for name, frame in [
    ("train", train_df),
    ("validation", valid_df),
    ("split_df", split_df),
]:
    a20_check(
        not frame["image_id"].astype(str).duplicated().any(),
        f"{name}_unique_membership",
        f"{name} has repeated image IDs."
    )
    a20_check(
        set(frame["image_id"].astype(str)) <= a20_known_ids,
        f"{name}_known_image_ids",
        f"{name} contains IDs outside the annotation JSON."
    )

a20_check(
    a20_train_ids.isdisjoint(a20_valid_ids),
    "train_validation_identity_separation",
    "At least one image ID belongs to both partitions."
)

_manifest_records = []

for image_id, info in a20_image_lookup.items():
    filename = info.get("file_name")
    path = (
        Path(TRAIN_IMAGE_DIR) / str(filename)
        if filename is not None else None
    )

    if image_id in a20_train_ids:
        partition = "train"
    elif image_id in a20_valid_ids:
        partition = "validation"
    else:
        partition = "excluded"

    _manifest_records.append({
        "image_id": image_id,
        "file_name": filename,
        "path": str(path) if path else None,
        "split": partition,
        "declared_height": info.get("height"),
        "declared_width": info.get("width"),
        "date_captured": info.get("date_captured"),
        "annotation_count": len(
            a20_annotations_by_image[image_id]
        ),
        "file_exists": bool(path and path.is_file()),
    })

audit_manifest = pd.DataFrame(_manifest_records)

for _, row in audit_manifest.iterrows():
    if not row["file_exists"]:
        a20_issue(
            "BLOCKER", "missing_image",
            row["image_id"], row["path"]
        )
    if not isinstance(row["file_name"], str) or not row["file_name"]:
        a20_issue(
            "BLOCKER", "invalid_filename",
            row["image_id"], row["file_name"]
        )

_duplicate_filenames = audit_manifest[
    audit_manifest["file_name"].duplicated(keep=False)
]

for filename, group in _duplicate_filenames.groupby("file_name"):
    a20_issue(
        "REVIEW", "multiple_ids_same_filename",
        filename, group["image_id"].tolist()
    )

_expected_paths = {
    str(Path(path).resolve())
    for path in audit_manifest["path"].dropna()
}

_unreferenced = [
    str(path)
    for path in Path(TRAIN_IMAGE_DIR).rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
        and str(path.resolve()) not in _expected_paths
    )
]

a20_save(
    pd.DataFrame({"unreferenced_image_path": _unreferenced}),
    "unreferenced_training_files.csv"
)

if _unreferenced:
    a20_issue(
        "REVIEW", "unreferenced_training_files",
        "training_directory", len(_unreferenced)
    )

a20_save(audit_manifest, "audit_image_manifest.csv")

display(
    audit_manifest.groupby("split").agg(
        images=("image_id", "size"),
        annotations=("annotation_count", "sum"),
        missing_files=("file_exists", lambda x: int((~x).sum())),
    )
)

print("Unreferenced physical training files:", len(_unreferenced))

### 20.3 Physical image and preprocessing audit

Images are inspected in their native representation before grayscale
conversion. This is necessary because an 8-bit grayscale loader can hide
higher bit depth, color information or orientation metadata.

For each image we record:

- Encoded-file SHA-256.
- Native decoded-array SHA-256.
- Grayscale decoded-array SHA-256.
- Native mode, dtype, dimensions and channel count.
- EXIF orientation.
- Intensity quantiles, standard deviation and saturation fractions.
- Agreement between the audited grayscale decode and our current loader.

Exact file duplicates and exact decoded duplicates are different:
two differently encoded files can contain identical model inputs.

Small thumbnails are retained for duplicate screening. Their central
interior is used so that the common solar-disk boundary contributes less
to similarity. These thumbnails are audit descriptors, not model inputs.

In [ ]:
A20_THUMB_SIZE = AUDIT_CONFIG["thumbnail_size"]

a20_thumbnails = {}
_image_audit_records = []

for position, row in enumerate(
    audit_manifest.itertuples(index=False), start=1
):
    result = {
        "image_id": row.image_id,
        "split": row.split,
        "readable": False,
    }

    try:
        path = Path(row.path)
        result["file_sha256"] = a20_file_hash(path)

        with Image.open(path) as image:
            image.load()
            native = np.array(image)
            gray = np.array(image.convert("L"), dtype=np.uint8)

            result.update({
                "native_mode": image.mode,
                "native_dtype": str(native.dtype),
                "native_channels": (
                    native.shape[2] if native.ndim == 3 else 1
                ),
                "exif_orientation": int(image.getexif().get(274, 1)),
                "height": int(gray.shape[0]),
                "width": int(gray.shape[1]),
                "native_sha256": a20_array_hash(native),
                "gray_sha256": a20_array_hash(gray),
            })

        expected_shape = (
            int(row.declared_height),
            int(row.declared_width),
        )

        if gray.shape != expected_shape:
            a20_issue(
                "BLOCKER", "image_dimension_mismatch",
                row.image_id,
                f"Decoded={gray.shape}; declared={expected_shape}"
            )

        if result["exif_orientation"] != 1:
            a20_issue(
                "REVIEW", "nonidentity_exif_orientation",
                row.image_id, result["exif_orientation"]
            )

        if native.dtype != np.uint8:
            a20_issue(
                "REVIEW", "native_to_uint8_conversion",
                row.image_id,
                f"{native.dtype} is converted by current grayscale loader."
            )

        if native.ndim == 3:
            rgb = native[..., :3].astype(np.float64)
            channel_disagreement = (
                float(np.max(np.ptp(rgb, axis=2)))
                if rgb.shape[2] == 3 else np.nan
            )
            result["maximum_rgb_channel_difference"] = (
                channel_disagreement
            )
            if channel_disagreement > 0:
                a20_issue(
                    "REVIEW", "color_to_grayscale_conversion",
                    row.image_id,
                    "Native channels differ; verify intended conversion."
                )

        loader_output = load_solar_image(
            a20_raw_image_ids[row.image_id]
        )
        loader_equal = np.array_equal(gray, loader_output)

        result["current_loader_equal"] = loader_equal
        if not loader_equal:
            a20_issue(
                "BLOCKER", "loader_decode_disagreement",
                row.image_id,
                "Current loader differs from independently decoded grayscale."
            )

        values = gray.astype(np.float32)
        q = np.quantile(values, [0, .01, .05, .5, .95, .99, 1])

        result.update({
            "readable": True,
            "minimum": float(q[0]),
            "p01": float(q[1]),
            "p05": float(q[2]),
            "median": float(q[3]),
            "p95": float(q[4]),
            "p99": float(q[5]),
            "maximum": float(q[6]),
            "mean": float(values.mean()),
            "std": float(values.std()),
            "zero_fraction": float((gray == 0).mean()),
            "full_scale_fraction": float((gray == 255).mean()),
        })

        if result["std"] == 0:
            a20_issue(
                "BLOCKER", "constant_image",
                row.image_id, "No grayscale variation."
            )

        thumb = np.asarray(
            Image.fromarray(gray).resize(
                (A20_THUMB_SIZE, A20_THUMB_SIZE),
                Image.Resampling.BILINEAR,
            ),
            dtype=np.float32,
        ) / 255.0

        a20_thumbnails[row.image_id] = thumb

    except Exception as error:
        result["error"] = repr(error)
        a20_issue(
            "BLOCKER", "image_read_or_loader_failure",
            row.image_id, repr(error)
        )

    _image_audit_records.append(result)

    if position % 25 == 0 or position == len(audit_manifest):
        a20_save(
            pd.DataFrame(_image_audit_records),
            "physical_image_audit.csv"
        )
        print(f"Images audited: {position}/{len(audit_manifest)}")

image_audit_df = pd.DataFrame(_image_audit_records)

a20_save(image_audit_df, "physical_image_audit.csv")

display(
    image_audit_df.groupby("split")[
        ["readable", "std", "zero_fraction", "full_scale_fraction"]
    ].agg(["count", "mean"])
)

print("Readable images:", int(image_audit_df["readable"].sum()))
print("Thumbnails available:", len(a20_thumbnails))

### 20.4 Independent temporal audit and exact split reconstruction

We verify the existing partition using the actual implementation:

- Temporal groups are measured from the earliest normalized timestamp.
- Groups span BLOCK_DAYS.
- GroupShuffleSplit chooses validation groups.
- Training images are retained only when their nearest validation
  observation is strictly farther than PURGE_DAYS.

Consequently, a gap exactly equal to the purge duration must be excluded.

We also compare annotation timestamps with timestamps encoded in filenames.
A date-only metadata value and a full filename timestamp have different
precision; their time-of-day difference is reported rather than
automatically interpreted as corruption.

The existing split is a grouped development split, not a strictly
out-of-time validation set.

In [ ]:
def a20_parse_metadata_time(value):
    if value is None or str(value).strip() == "":
        return pd.NaT
    return pd.to_datetime(value, errors="coerce", utc=True)

def a20_filename_time(filename):
    match = re.match(r"^(\d{14})", Path(str(filename)).name)
    if not match:
        return pd.NaT
    return pd.to_datetime(
        match.group(1),
        format="%Y%m%d%H%M%S",
        errors="coerce",
        utc=True,
    )

audit_manifest["metadata_time"] = pd.to_datetime(
    [
        a20_parse_metadata_time(value)
        for value in audit_manifest["date_captured"]
    ],
    utc=True,
)

audit_manifest["filename_time"] = pd.to_datetime(
    [
        a20_filename_time(value)
        for value in audit_manifest["file_name"]
    ],
    utc=True,
)

audit_manifest["timestamp_difference_seconds"] = (
    audit_manifest["metadata_time"]
    - audit_manifest["filename_time"]
).dt.total_seconds()

for row in audit_manifest.itertuples(index=False):
    if pd.isna(row.metadata_time):
        a20_issue(
            "BLOCKER", "unparseable_metadata_timestamp",
            row.image_id, row.date_captured
        )
    elif pd.notna(row.filename_time):
        if row.metadata_time.normalize() != row.filename_time.normalize():
            a20_issue(
                "REVIEW", "timestamp_calendar_date_disagreement",
                row.image_id,
                f"Metadata={row.metadata_time}; filename={row.filename_time}"
            )

a20_save(audit_manifest, "audit_image_manifest.csv")

if audit_manifest["metadata_time"].isna().any():
    raise RuntimeError(
        "Resolve unparseable timestamps before reconstructing the split."
    )

_a20_reference = audit_manifest["metadata_time"].min().normalize()

audit_manifest["recomputed_temporal_block"] = (
    (
        audit_manifest["metadata_time"] - _a20_reference
    ).dt.total_seconds()
    // (AUDIT_CONFIG["block_days"] * 86400)
).astype(np.int64)

_current_split_times = split_df[
    ["image_id", "capture_datetime", "temporal_block"]
].copy()

_current_split_times["image_id"] = (
    _current_split_times["image_id"].astype(str)
)
_current_split_times["capture_datetime"] = pd.to_datetime(
    _current_split_times["capture_datetime"], utc=True
)

_time_comparison = audit_manifest.merge(
    _current_split_times,
    on="image_id",
    how="left",
    validate="one_to_one",
)

a20_check(
    set(_current_split_times["image_id"]) == a20_known_ids,
    "split_population_matches_annotation_population",
    "split_df does not contain exactly the JSON image population."
)

a20_check(
    (
        _time_comparison["metadata_time"]
        == _time_comparison["capture_datetime"]
    ).all(),
    "split_timestamp_consistency",
    "Current split timestamps differ from independently parsed metadata."
)

a20_check(
    (
        _time_comparison["recomputed_temporal_block"]
        == _time_comparison["temporal_block"]
    ).all(),
    "temporal_block_consistency",
    "Stored temporal blocks differ from independent reconstruction."
)

# Reproduce the original row ordering; GroupShuffleSplit operates
# on the supplied population and its group labels.
_rebuild = split_df.copy()
_rebuild["image_id"] = _rebuild["image_id"].astype(str)

_time_by_id = audit_manifest.set_index("image_id")["metadata_time"]
_block_by_id = audit_manifest.set_index(
    "image_id"
)["recomputed_temporal_block"]

_rebuild["audit_time"] = _rebuild["image_id"].map(_time_by_id)
_rebuild["audit_block"] = _rebuild["image_id"].map(_block_by_id)

_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=AUDIT_CONFIG["validation_fraction"],
    random_state=AUDIT_CONFIG["seed"],
)

_rebuilt_train_idx, _rebuilt_valid_idx = next(
    _splitter.split(_rebuild, groups=_rebuild["audit_block"])
)

_rebuilt_train = _rebuild.iloc[_rebuilt_train_idx].copy()
_rebuilt_valid = _rebuild.iloc[_rebuilt_valid_idx].copy()

def a20_nearest_gaps(query_times, reference_times):
    query = pd.DatetimeIndex(query_times).asi8
    reference = np.sort(pd.DatetimeIndex(reference_times).asi8)

    if len(reference) == 0:
        raise ValueError("Reference partition is empty.")

    positions = np.searchsorted(reference, query)
    left = np.clip(positions - 1, 0, len(reference) - 1)
    right = np.clip(positions, 0, len(reference) - 1)

    nearest = np.minimum(
        np.abs(query - reference[left]),
        np.abs(query - reference[right]),
    )

    return nearest / (86400 * 1e9)

_rebuilt_train["nearest_validation_gap_days"] = a20_nearest_gaps(
    _rebuilt_train["audit_time"],
    _rebuilt_valid["audit_time"],
)

_expected_train_ids = set(
    _rebuilt_train.loc[
        _rebuilt_train["nearest_validation_gap_days"]
        > AUDIT_CONFIG["purge_days"],
        "image_id",
    ]
)

_expected_valid_ids = set(_rebuilt_valid["image_id"])
_expected_excluded_ids = (
    a20_known_ids - _expected_train_ids - _expected_valid_ids
)

a20_check(
    a20_train_ids == _expected_train_ids,
    "exact_train_split_reproduction",
    {
        "unexpected_train": sorted(a20_train_ids - _expected_train_ids),
        "missing_train": sorted(_expected_train_ids - a20_train_ids),
    }
)

a20_check(
    a20_valid_ids == _expected_valid_ids,
    "exact_validation_split_reproduction",
    {
        "unexpected_validation": sorted(
            a20_valid_ids - _expected_valid_ids
        ),
        "missing_validation": sorted(
            _expected_valid_ids - a20_valid_ids
        ),
    }
)

_a20_train_rows = audit_manifest[
    audit_manifest["split"] == "train"
].copy()
_a20_valid_rows = audit_manifest[
    audit_manifest["split"] == "validation"
].copy()

_a20_train_rows["nearest_validation_gap_days"] = a20_nearest_gaps(
    _a20_train_rows["metadata_time"],
    _a20_valid_rows["metadata_time"],
)

a20_check(
    (
        _a20_train_rows["nearest_validation_gap_days"]
        > AUDIT_CONFIG["purge_days"]
    ).all(),
    "actual_purge_distance",
    "A retained training image violates the strict purge boundary."
)

a20_check(
    set(_a20_train_rows["recomputed_temporal_block"]).isdisjoint(
        set(_a20_valid_rows["recomputed_temporal_block"])
    ),
    "actual_temporal_group_separation",
    "A temporal block crosses training and validation."
)

a20_save(
    _a20_train_rows[
        ["image_id", "metadata_time", "nearest_validation_gap_days"]
    ],
    "nearest_cross_split_temporal_distances.csv"
)

audit_manifest["exclusion_reason"] = np.where(
    audit_manifest["image_id"].isin(_expected_excluded_ids),
    "Expected exclusion under reconstructed purge",
    "",
)

a20_save(
    audit_manifest[audit_manifest["split"] == "excluded"],
    "excluded_images.csv"
)
a20_save(audit_manifest, "audit_image_manifest.csv")

print("Expected train:", len(_expected_train_ids))
print("Expected validation:", len(_expected_valid_ids))
print("Expected excluded:", len(_expected_excluded_ids))
print(
    "Minimum retained train–validation gap:",
    _a20_train_rows["nearest_validation_gap_days"].min(),
    "days"
)

display(
    _a20_train_rows.nsmallest(
        15, "nearest_validation_gap_days"
    )[
        ["image_id", "metadata_time", "nearest_validation_gap_days"]
    ]
)

### 20.5 Annotation, decoding and instance-overlap audit

The existing polygon parser can skip malformed polygon parts.
This audit validates every part explicitly before calling the current
rasterizer, so that silent partial annotation loss becomes visible.

For each annotation we record:

- Representation and polygon-part count.
- Finite coordinates and minimum distinct-vertex count.
- Coordinate excursions outside the image boundary.
- Polygon validity where Shapely is available.
- Rasterized area and connected-component count.
- Border contact and disagreement with declared area.
- Exact duplicated masks within the same image.
- Area retained after conversion to the current integer-label image.

For each image we distinguish:

- Unique pixels covered by two or more annotations.
- Excess coverage: the number of additional mask memberships.
- Independent instance identity.
- The foreground union.
- The current last-write-wins integer-label representation.

Multipart or disconnected annotations are not automatically errors.
Overlapping masks are not automatically annotation defects.
However, an integer-label image cannot retain both instance memberships
at an overlapping pixel.

Optional comparison with pycocotools identifies decoder differences.
It does not establish organizer parity; Step 21 remains authoritative.

In [ ]:
try:
    from shapely.geometry import Polygon as ShapelyPolygon
    A20_SHAPELY_AVAILABLE = True
except ImportError:
    A20_SHAPELY_AVAILABLE = False

try:
    from pycocotools import mask as coco_mask
    A20_COCO_AVAILABLE = True
except ImportError:
    A20_COCO_AVAILABLE = False

print("Shapely polygon-validity checks:", A20_SHAPELY_AVAILABLE)
print("Optional COCO decoding comparison:", A20_COCO_AVAILABLE)

def a20_strict_polygon_parts(annotation):
    segmentation = annotation.get("segmentation")

    if not isinstance(segmentation, list):
        raise TypeError(
            "Current notebook decoder supports polygons only; "
            f"received {type(segmentation).__name__}."
        )

    if not segmentation:
        raise ValueError("Empty segmentation list.")

    if isinstance(segmentation[0], (int, float)):
        segmentation = [segmentation]

    parts = []

    for part_number, polygon in enumerate(segmentation):
        values = np.asarray(polygon, dtype=np.float64)

        if values.ndim != 1:
            raise ValueError(
                f"Polygon part {part_number} is not a flat coordinate list."
            )
        if values.size < 6 or values.size % 2:
            raise ValueError(
                f"Polygon part {part_number} has invalid coordinate count."
            )
        if not np.isfinite(values).all():
            raise ValueError(
                f"Polygon part {part_number} has nonfinite coordinates."
            )

        points = values.reshape(-1, 2)

        if len(np.unique(points, axis=0)) < 3:
            raise ValueError(
                f"Polygon part {part_number} has fewer than 3 distinct vertices."
            )

        parts.append(points)

    return parts

_annotation_audit_records = []
_mask_image_records = []
_overlap_pair_records = []

for position, image_row in enumerate(
    audit_manifest.itertuples(index=False), start=1
):
    image_id = image_row.image_id
    raw_id = a20_raw_image_ids[image_id]
    shape = (
        int(image_row.declared_height),
        int(image_row.declared_width),
    )
    h, w = shape

    coverage = np.zeros(shape, dtype=np.uint32)
    expected_labels = np.zeros(shape, dtype=np.int32)

    # Store cropped independent masks rather than a full image per instance.
    instances = []
    signatures = {}

    for instance_index, ann in enumerate(
        a20_annotations_by_image[image_id], start=1
    ):
        record = {
            "image_id": image_id,
            "annotation_id": str(ann["id"]),
            "split": image_row.split,
            "instance_index": instance_index,
            "valid_decode": False,
        }

        try:
            parts = a20_strict_polygon_parts(ann)

            outside_vertices = sum(
                int(
                    (
                        (p[:, 0] < 0) | (p[:, 0] > w)
                        | (p[:, 1] < 0) | (p[:, 1] > h)
                    ).sum()
                )
                for p in parts
            )

            record["polygon_parts"] = len(parts)
            record["outside_vertex_count"] = outside_vertices

            if outside_vertices:
                a20_issue(
                    "REVIEW", "polygon_outside_image",
                    ann["id"],
                    f"{outside_vertices} vertices outside [0,W] × [0,H]."
                )

            if A20_SHAPELY_AVAILABLE:
                invalid_parts = [
                    k for k, p in enumerate(parts)
                    if not ShapelyPolygon(p).is_valid
                    or ShapelyPolygon(p).area <= 0
                ]
                record["invalid_geometric_parts"] = len(invalid_parts)

                if invalid_parts:
                    a20_issue(
                        "REVIEW", "invalid_polygon_geometry",
                        ann["id"], invalid_parts
                    )

            mask = np.asarray(
                rasterize_annotation(ann, shape)
            )

            if mask.shape != shape:
                raise ValueError(
                    f"Mask shape {mask.shape} differs from {shape}."
                )
            if not np.isin(mask, [0, 1]).all():
                raise ValueError("Decoded mask is not binary.")

            mask = mask.astype(bool)
            area = int(mask.sum())

            record["decoded_area"] = area
            record["valid_decode"] = True

            if area == 0:
                a20_issue(
                    "BLOCKER", "empty_decoded_annotation",
                    ann["id"],
                    "Annotation has no pixels under the current decoder."
                )
                _annotation_audit_records.append(record)
                continue

            ys, xs = np.nonzero(mask)
            y0, y1 = int(ys.min()), int(ys.max()) + 1
            x0, x1 = int(xs.min()), int(xs.max()) + 1
            crop = mask[y0:y1, x0:x1].copy()

            component_count = int(
                label(crop, connectivity=2).max()
            )

            record.update({
                "component_count_8": component_count,
                "border_contact": bool(
                    mask[0].any() or mask[-1].any()
                    or mask[:, 0].any() or mask[:, -1].any()
                ),
                "x0": x0, "y0": y0, "x1": x1, "y1": y1,
            })

            declared_area = ann.get("area")
            if isinstance(declared_area, (int, float)):
                record["declared_area"] = float(declared_area)
                record["area_relative_difference"] = (
                    abs(area - declared_area)
                    / max(abs(declared_area), 1.0)
                )

            # Canonical signature includes position and cropped mask.
            signature = a20_digest({
                "bounds": [y0, y1, x0, x1],
                "mask_hash": a20_array_hash(crop),
            })

            if signature in signatures:
                a20_issue(
                    "REVIEW", "identical_instance_masks",
                    ann["id"],
                    f"Same decoded mask as annotation {signatures[signature]}"
                )
            else:
                signatures[signature] = str(ann["id"])

            if A20_COCO_AVAILABLE:
                polygons = [
                    p.reshape(-1).tolist() for p in parts
                ]
                rles = coco_mask.frPyObjects(polygons, h, w)
                reference_mask = coco_mask.decode(
                    coco_mask.merge(rles)
                ).astype(bool)

                intersection = int((mask & reference_mask).sum())
                union = int((mask | reference_mask).sum())

                record["pillow_coco_iou"] = (
                    intersection / union if union else 1.0
                )
                record["pillow_coco_disagreement_pixels"] = int(
                    np.count_nonzero(mask != reference_mask)
                )

            coverage[mask] += 1
            expected_labels[mask] = instance_index

            instances.append({
                "annotation_id": str(ann["id"]),
                "instance_index": instance_index,
                "bounds": (y0, y1, x0, x1),
                "crop": crop,
                "record": record,
            })

        except Exception as error:
            record["error"] = repr(error)
            a20_issue(
                "BLOCKER", "annotation_decode_or_schema_failure",
                ann["id"], repr(error)
            )

        _annotation_audit_records.append(record)

    for left, right in itertools.combinations(instances, 2):
        ay0, ay1, ax0, ax1 = left["bounds"]
        by0, by1, bx0, bx1 = right["bounds"]

        y0, y1 = max(ay0, by0), min(ay1, by1)
        x0, x1 = max(ax0, bx0), min(ax1, bx1)

        if y1 <= y0 or x1 <= x0:
            continue

        a = left["crop"][
            y0-ay0:y1-ay0, x0-ax0:x1-ax0
        ]
        b = right["crop"][
            y0-by0:y1-by0, x0-bx0:x1-bx0
        ]

        overlap = int((a & b).sum())
        if overlap:
            _overlap_pair_records.append({
                "image_id": image_id,
                "annotation_a": left["annotation_id"],
                "annotation_b": right["annotation_id"],
                "intersection_pixels": overlap,
                "fraction_of_a": (
                    overlap / left["record"]["decoded_area"]
                ),
                "fraction_of_b": (
                    overlap / right["record"]["decoded_area"]
                ),
            })

    for instance in instances:
        y0, y1, x0, x1 = instance["bounds"]
        retained = int(
            (
                (
                    expected_labels[y0:y1, x0:x1]
                    == instance["instance_index"]
                )
                & instance["crop"]
            ).sum()
        )

        record = instance["record"]
        record["area_retained_in_integer_labels"] = retained
        record["area_lost_to_overwrite"] = (
            record["decoded_area"] - retained
        )

    union_mask = coverage > 0

    image_result = {
        "image_id": image_id,
        "split": image_row.split,
        "annotation_count": len(
            a20_annotations_by_image[image_id]
        ),
        "successfully_decoded_nonempty_instances": len(instances),
        "foreground_pixels": int(union_mask.sum()),
        "foreground_fraction": float(union_mask.mean()),
        "unique_overlap_pixels": int((coverage >= 2).sum()),
        "excess_mask_memberships": int(
            np.maximum(
                coverage.astype(np.int64) - 1, 0
            ).sum()
        ),
        "maximum_instance_multiplicity": int(coverage.max()),
    }

    try:
        current_binary, current_labels, current_info = (
            load_instance_targets(raw_id)
        )

        image_result["binary_union_equal"] = np.array_equal(
            current_binary, union_mask
        )
        image_result["integer_labels_equal"] = np.array_equal(
            current_labels, expected_labels
        )

        if not image_result["binary_union_equal"]:
            a20_issue(
                "BLOCKER", "binary_union_disagreement",
                image_id, "Independent union differs from target loader."
            )

        if not image_result["integer_labels_equal"]:
            a20_issue(
                "BLOCKER", "integer_target_disagreement",
                image_id, "Current target differs from reconstructed policy."
            )

    except Exception as error:
        a20_issue(
            "BLOCKER", "target_loader_failure",
            image_id, repr(error)
        )

    _mask_image_records.append(image_result)

    if position % 25 == 0 or position == len(audit_manifest):
        a20_save(
            pd.DataFrame(_annotation_audit_records),
            "annotation_audit.csv"
        )
        a20_save(
            pd.DataFrame(_mask_image_records),
            "image_mask_audit.csv"
        )
        print(
            f"Annotation audit: {position}/{len(audit_manifest)} images"
        )

annotation_audit_df = pd.DataFrame(_annotation_audit_records)
image_mask_audit_df = pd.DataFrame(_mask_image_records)
overlap_pairs_df = pd.DataFrame(
    _overlap_pair_records,
    columns=[
        "image_id", "annotation_a", "annotation_b",
        "intersection_pixels", "fraction_of_a", "fraction_of_b",
    ],
)

a20_save(annotation_audit_df, "annotation_audit.csv")
a20_save(image_mask_audit_df, "image_mask_audit.csv")
a20_save(overlap_pairs_df, "overlapping_annotation_pairs.csv")

_overlap_images = image_mask_audit_df[
    image_mask_audit_df["unique_overlap_pixels"] > 0
]

if len(_overlap_images):
    a20_issue(
        "REVIEW", "independent_instance_storage_required",
        "target_representation",
        f"{len(_overlap_images)} images contain overlapping annotations. "
        "The current integer map loses some instance memberships."
    )

if A20_COCO_AVAILABLE:
    _different = annotation_audit_df.get(
        "pillow_coco_disagreement_pixels", pd.Series(dtype=float)
    ).fillna(0).gt(0)

    if _different.any():
        a20_issue(
            "REVIEW", "decoder_convention_difference",
            "rasterization",
            f"{int(_different.sum())} annotations differ between "
            "current Pillow and optional COCO rasterization. "
            "Resolve the authoritative convention in Step 21."
        )

display(
    image_mask_audit_df.groupby("split")[
        [
            "foreground_fraction",
            "unique_overlap_pixels",
            "excess_mask_memberships",
        ]
    ].agg(["mean", "max"])
)

display(
    annotation_audit_df.sort_values(
        "area_lost_to_overwrite", ascending=False
    ).head(15)
)

### 20.6 Exact duplicate observations

We compare three notions of identity:

1. Identical encoded files.
2. Identical native decoded arrays.
3. Identical grayscale inputs supplied to the model.

A cross-partition duplicate of the model input is a blocking finding.

Duplicates within training or validation are retained for review because
they can alter sampling exposure or evaluation weighting.

All duplicate groups retain their member image IDs and partitions.
The audit does not silently deduplicate the dataset.

In [ ]:
_exact_group_records = []
_exact_edge_records = []

for hash_column, identity_type in [
    ("file_sha256", "encoded_file"),
    ("native_sha256", "native_pixels"),
    ("gray_sha256", "model_grayscale_pixels"),
]:
    available = image_audit_df.dropna(subset=[hash_column])

    for digest, group in available.groupby(hash_column):
        if len(group) < 2:
            continue

        members = sorted(group["image_id"].astype(str))
        partitions = set(group["split"])
        cross_split = {"train", "validation"} <= partitions

        group_id = a20_digest({
            "identity_type": identity_type,
            "digest": digest,
        })[:16]

        for image_id in members:
            _exact_group_records.append({
                "group_id": group_id,
                "identity_type": identity_type,
                "content_sha256": digest,
                "image_id": image_id,
                "split": audit_manifest.set_index(
                    "image_id"
                ).at[image_id, "split"],
                "cross_train_validation": cross_split,
            })

        if identity_type == "model_grayscale_pixels":
            for image_a, image_b in itertools.combinations(members, 2):
                _exact_edge_records.append({
                    "image_a": image_a,
                    "image_b": image_b,
                    "edge_type": "exact_grayscale_duplicate",
                })

        a20_issue(
            "BLOCKER" if cross_split else "REVIEW",
            f"exact_duplicate_{identity_type}",
            group_id,
            f"Members={members}; partitions={sorted(partitions)}"
        )

exact_duplicate_groups_df = pd.DataFrame(
    _exact_group_records,
    columns=[
        "group_id", "identity_type", "content_sha256",
        "image_id", "split", "cross_train_validation",
    ],
)

exact_duplicate_edges_df = pd.DataFrame(
    _exact_edge_records,
    columns=["image_a", "image_b", "edge_type"],
).drop_duplicates()

a20_save(
    exact_duplicate_groups_df,
    "exact_duplicate_groups.csv"
)
a20_save(
    exact_duplicate_edges_df,
    "exact_duplicate_edges.csv"
)

display(exact_duplicate_groups_df.head(30))
print("Exact grayscale duplicate edges:", len(exact_duplicate_edges_df))

### 20.7 Near-duplicate and temporally correlated observation screening

Solar images share a strong disk structure. Similarity of raw images can
therefore be high even when their filament configurations differ.

We screen image pairs using complementary signals:

- Central-interior high-pass image descriptors.
- Perceptual hashes.
- Temporal proximity.

All image pairs are screened at descriptor resolution.
Candidate pairs are then compared at a higher thumbnail resolution,
including translation registration and high-pass correlation.

The candidate set is the union of these screens. No candidates are
silently discarded using an arbitrary top-k limit.

Limitations:

- Registration estimates translation only.
- Scale, rotation, disk-center errors and solar evolution can remain.
- Thumbnail similarity can miss small, faint or localized structures.
- A high score does not prove that two observations are duplicates.
- A low score does not prove physical independence.

Cross-partition candidates require image-based review. A finding of
shared persistent structures can justify observation grouping even when
the images are not pixel duplicates.

In [ ]:
_a20_descriptor_ids = sorted(a20_thumbnails)
_a20_n = len(_a20_descriptor_ids)
_a20_s = AUDIT_CONFIG["thumbnail_size"]

_y, _x = np.mgrid[:_a20_s, :_a20_s]
_a20_radius = np.sqrt(
    ((_x - (_a20_s - 1) / 2) / (_a20_s / 2)) ** 2
    + ((_y - (_a20_s - 1) / 2) / (_a20_s / 2)) ** 2
)

_a20_interior = _a20_radius < 0.78
_a20_window = (
    np.outer(np.hanning(_a20_s), np.hanning(_a20_s))
    * _a20_interior
).astype(np.float32)

def a20_detail_image(image):
    return (
        gaussian_filter(image, 0.7)
        - gaussian_filter(image, 4.0)
    ).astype(np.float32)

def a20_unit_vector(image):
    vector = image.ravel().astype(np.float32)
    vector -= vector.mean()
    norm = float(np.linalg.norm(vector))
    return vector / norm if norm > 1e-12 else np.zeros_like(vector)

_descriptors = []
_phashes = []

for image_id in _a20_descriptor_ids:
    thumb = a20_thumbnails[image_id]
    detail = a20_detail_image(thumb) * _a20_window

    small = np.asarray(
        Image.fromarray(detail).resize(
            (
                AUDIT_CONFIG["descriptor_size"],
                AUDIT_CONFIG["descriptor_size"],
            ),
            Image.Resampling.BILINEAR,
        ),
        dtype=np.float32,
    )

    _descriptors.append(a20_unit_vector(small))

    central = thumb[
        _a20_s // 4:3 * _a20_s // 4,
        _a20_s // 4:3 * _a20_s // 4,
    ]
    hash_image = np.asarray(
        Image.fromarray(central).resize(
            (32, 32), Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )

    coefficients = dctn(hash_image, norm="ortho")[:8, :8].ravel()
    threshold = np.median(coefficients[1:])
    bits = coefficients > threshold
    bits[0] = False
    _phashes.append(bits)

_a20_descriptors = np.stack(_descriptors)
_a20_phashes = np.stack(_phashes)

_a20_meta = audit_manifest.set_index("image_id")
_a20_times = pd.DatetimeIndex(
    _a20_meta.loc[_a20_descriptor_ids, "metadata_time"]
).asi8

_candidate_records = []

# Batched matrix multiplication avoids Python-level dot products
# for every possible image pair.
for start in range(0, _a20_n, 128):
    end = min(start + 128, _a20_n)
    similarities = (
        _a20_descriptors[start:end] @ _a20_descriptors.T
    )

    for local_i, i in enumerate(range(start, end)):
        if i + 1 >= _a20_n:
            continue

        js = np.arange(i + 1, _a20_n)
        cosine = similarities[local_i, js]

        hamming = np.count_nonzero(
            _a20_phashes[js] != _a20_phashes[i], axis=1
        )

        gaps = (
            np.abs(_a20_times[js] - _a20_times[i])
            / (86400 * 1e9)
        )

        selected = (
            (cosine >= AUDIT_CONFIG["candidate_detail_cosine"])
            | (hamming <= AUDIT_CONFIG["candidate_phash_distance"])
            | (gaps <= AUDIT_CONFIG["candidate_temporal_days"])
        )

        for j, similarity, distance, gap in zip(
            js[selected],
            cosine[selected],
            hamming[selected],
            gaps[selected],
        ):
            image_a = _a20_descriptor_ids[i]
            image_b = _a20_descriptor_ids[j]

            split_a = _a20_meta.at[image_a, "split"]
            split_b = _a20_meta.at[image_b, "split"]

            _candidate_records.append({
                "image_a": image_a,
                "image_b": image_b,
                "split_a": split_a,
                "split_b": split_b,
                "cross_train_validation": (
                    {split_a, split_b} == {"train", "validation"}
                ),
                "time_gap_days": float(gap),
                "detail_descriptor_cosine": float(similarity),
                "phash_hamming": int(distance),
            })

near_candidate_df = pd.DataFrame(
    _candidate_records,
    columns=[
        "image_a", "image_b", "split_a", "split_b",
        "cross_train_validation", "time_gap_days",
        "detail_descriptor_cosine", "phash_hamming",
    ],
)

a20_save(near_candidate_df, "near_duplicate_candidates.csv")

print("Images screened:", _a20_n)
print("Total possible pairs:", _a20_n * (_a20_n - 1) // 2)
print("Candidate pairs:", len(near_candidate_df))
print(
    "Cross-partition candidates:",
    int(near_candidate_df["cross_train_validation"].sum())
)

display(
    near_candidate_df.sort_values(
        ["cross_train_validation", "detail_descriptor_cosine"],
        ascending=[False, False],
    ).head(20)
)

In [ ]:
def a20_ncc(left, right, valid):
    a = left[valid].astype(np.float64)
    b = right[valid].astype(np.float64)

    if len(a) < 100:
        return np.nan

    a -= a.mean()
    b -= b.mean()

    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    return (
        float(np.dot(a, b) / denominator)
        if denominator > 1e-12 else np.nan
    )

_pair_diagnostics = []

for position, pair in enumerate(
    near_candidate_df.itertuples(index=False), start=1
):
    result = pair._asdict()
    result["registration_ok"] = False

    image_a = a20_thumbnails[pair.image_a]
    image_b = a20_thumbnails[pair.image_b]

    detail_a = a20_detail_image(image_a)
    detail_b = a20_detail_image(image_b)

    result["unregistered_detail_ncc"] = a20_ncc(
        detail_a, detail_b, _a20_interior
    )

    try:
        estimated_shift, registration_error, _ = (
            phase_cross_correlation(
                detail_a * _a20_window,
                detail_b * _a20_window,
                upsample_factor=5,
            )
        )

        result["shift_y"] = float(estimated_shift[0])
        result["shift_x"] = float(estimated_shift[1])

        allowed = (
            np.isfinite(estimated_shift).all()
            and np.max(np.abs(estimated_shift))
            <= AUDIT_CONFIG["maximum_registration_shift_pixels"]
        )

        if allowed:
            aligned_b = ndi_shift(
                image_b,
                estimated_shift,
                order=1,
                mode="constant",
                cval=0,
                prefilter=False,
            )

            support = ndi_shift(
                np.ones_like(image_b),
                estimated_shift,
                order=0,
                mode="constant",
                cval=0,
                prefilter=False,
            ) > 0.5

            valid = _a20_interior & support
            aligned_detail_b = a20_detail_image(aligned_b)

            result["registered_detail_ncc"] = a20_ncc(
                detail_a, aligned_detail_b, valid
            )
            result["registered_raw_ncc"] = a20_ncc(
                image_a, aligned_b, valid
            )
            result["registered_mean_absolute_difference"] = float(
                np.abs(image_a[valid] - aligned_b[valid]).mean()
            )

            _, ssim_map = structural_similarity(
                image_a,
                aligned_b,
                data_range=1.0,
                full=True,
            )
            result["registered_interior_ssim"] = float(
                ssim_map[valid].mean()
            )
            result["registration_ok"] = True

    except Exception as error:
        result["registration_error"] = repr(error)

    # Content-bound key prevents reusing decisions for changed image bytes.
    result["pair_id"] = a20_digest({
        "a": pair.image_a,
        "b": pair.image_b,
        "a_hash": image_audit_df.set_index(
            "image_id"
        ).at[pair.image_a, "gray_sha256"],
        "b_hash": image_audit_df.set_index(
            "image_id"
        ).at[pair.image_b, "gray_sha256"],
    })[:24]

    _pair_diagnostics.append(result)

    if position % 50 == 0 or position == len(near_candidate_df):
        a20_save(
            pd.DataFrame(_pair_diagnostics),
            "near_duplicate_pair_diagnostics.csv"
        )
        print(
            f"Detailed pair checks: {position}/{len(near_candidate_df)}"
        )

near_pair_df = pd.DataFrame(
    _pair_diagnostics,
    columns=list(near_candidate_df.columns) + [
        "registration_ok",
        "unregistered_detail_ncc",
        "shift_y", "shift_x",
        "registered_detail_ncc",
        "registered_raw_ncc",
        "registered_mean_absolute_difference",
        "registered_interior_ssim",
        "registration_error",
        "pair_id",
    ],
)

a20_save(
    near_pair_df,
    "near_duplicate_pair_diagnostics.csv"
)

display(
    near_pair_df.sort_values(
        ["cross_train_validation", "registered_detail_ncc"],
        ascending=[False, False],
        na_position="last",
    ).head(25)
)

### 20.8 Visual review and persistent pair decisions

Every cross-training/validation candidate is placed in a review queue.

Allowed decisions:

- duplicate: the observations reproduce substantially the same input.
- correlated: visible persistent structures justify keeping the pair
  within one observation group for validation.
- distinct: the screening signal does not indicate material shared
  observation content for the declared validation objective.
- uncertain: available evidence is insufficient.

Each completed decision requires a written reason.

Review uses the images and their acquisition information. It must not
use model performance to decide which observations should be removed.

The initial gallery shows the highest-ranked candidates only.
Use the gallery function to inspect any remaining pair at native
resolution. A screening score alone is not a completed review.

In [ ]:
PAIR_REVIEW_PATH = AUDIT_DIR / "pair_review.csv"

_review_base = near_pair_df[
    near_pair_df["cross_train_validation"]
].copy()

_review_base = _review_base.sort_values(
    "registered_detail_ncc",
    ascending=False,
    na_position="last",
)

_review_base["decision"] = ""
_review_base["reason"] = ""

if PAIR_REVIEW_PATH.exists():
    _old_review = pd.read_csv(
        PAIR_REVIEW_PATH,
        dtype={"pair_id": str},
        keep_default_na=False,
    )

    if _old_review["pair_id"].duplicated().any():
        raise ValueError("pair_review.csv has repeated pair_id values.")

    _old_decisions = _old_review.set_index("pair_id")

    for column in ["decision", "reason"]:
        _review_base[column] = (
            _review_base["pair_id"].map(_old_decisions[column])
            .fillna("")
        )

pair_review_df = _review_base
a20_save(pair_review_df, "pair_review.csv")

def a20_show_pair(image_a, image_b, native=False):
    if native:
        a = load_solar_image(a20_raw_image_ids[str(image_a)])
        b = load_solar_image(a20_raw_image_ids[str(image_b)])
        a = a.astype(np.float32) / 255.0
        b = b.astype(np.float32) / 255.0
    else:
        a = a20_thumbnails[str(image_a)]
        b = a20_thumbnails[str(image_b)]

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))

    for axis, image, image_id in zip(
        axes, [a, b], [str(image_a), str(image_b)]
    ):
        interior_values = image[
            image.shape[0] // 4:3 * image.shape[0] // 4,
            image.shape[1] // 4:3 * image.shape[1] // 4,
        ]

        low, high = np.quantile(interior_values, [.01, .99])

        axis.imshow(
            image, cmap="gray",
            vmin=float(low),
            vmax=float(max(high, low + 1e-6)),
        )
        axis.set_title(
            f"{image_id}\n"
            f"{_a20_meta.at[image_id, 'split']} | "
            f"{_a20_meta.at[image_id, 'metadata_time']}"
        )
        axis.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

for pair in pair_review_df.head(6).itertuples(index=False):
    print(
        "Pair:", pair.pair_id,
        "| gap:", round(pair.time_gap_days, 4),
        "| detail NCC:", pair.registered_detail_ncc,
    )
    a20_show_pair(pair.image_a, pair.image_b)

display(
    pair_review_df[
        [
            "pair_id", "image_a", "image_b", "time_gap_days",
            "registered_detail_ncc", "decision", "reason",
        ]
    ].head(30)
)

print("Review queue saved:", PAIR_REVIEW_PATH)
print(
    "For full-resolution review use:\n"
    "a20_show_pair('IMAGE_ID_A', 'IMAGE_ID_B', native=True)"
)

In [ ]:
def a20_record_pair_decision(pair_id, decision, reason):
    allowed = {"duplicate", "correlated", "distinct", "uncertain"}

    if decision not in allowed:
        raise ValueError(f"Decision must be one of {sorted(allowed)}")

    if not str(reason).strip():
        raise ValueError("A nonempty reason is required.")

    review = pd.read_csv(
        PAIR_REVIEW_PATH,
        dtype={"pair_id": str},
        keep_default_na=False,
    )

    selected = review["pair_id"].eq(str(pair_id))

    if selected.sum() != 1:
        raise ValueError("pair_id must identify exactly one review row.")

    review.loc[selected, "decision"] = decision
    review.loc[selected, "reason"] = str(reason).strip()
    review.to_csv(PAIR_REVIEW_PATH, index=False)

    print("Recorded:", pair_id, decision)

### 20.9 Dependence clusters and split-sensitivity analysis

Confirmed exact grayscale duplicates and reviewed correlated or duplicate
pairs define edges in an observation graph.

Connected components identify observation clusters. Transitive grouping
is conservative: a cluster can contain observations connected through
intermediate observations, even when its endpoints differ.

Any cluster containing both training and validation images is a blocking
finding for the current validation protocol.

We additionally quantify:

- Training-image retention under longer temporal purge durations.
- Further exclusions required to remove training members connected to
  validation through confirmed dependence edges.

These are diagnostic alternative protocols. They do not replace the
current split or authorize choosing whichever protocol scores highest.

In [ ]:
pair_review_df = pd.read_csv(
    PAIR_REVIEW_PATH,
    dtype={"pair_id": str},
    keep_default_na=False,
)

_allowed_decisions = {
    "", "duplicate", "correlated", "distinct", "uncertain"
}

_invalid_decisions = (
    ~pair_review_df["decision"].isin(_allowed_decisions)
)
if _invalid_decisions.any():
    raise ValueError("pair_review.csv contains invalid decision labels.")

_completed_without_reason = (
    pair_review_df["decision"].isin(
        ["duplicate", "correlated", "distinct"]
    )
    & pair_review_df["reason"].str.strip().eq("")
)
if _completed_without_reason.any():
    raise ValueError("Completed pair decisions require written reasons.")

_a20_parent = {image_id: image_id for image_id in a20_known_ids}

def a20_find(image_id):
    while _a20_parent[image_id] != image_id:
        _a20_parent[image_id] = _a20_parent[_a20_parent[image_id]]
        image_id = _a20_parent[image_id]
    return image_id

def a20_union(left, right):
    a, b = a20_find(left), a20_find(right)
    if a != b:
        _a20_parent[max(a, b)] = min(a, b)

for edge in exact_duplicate_edges_df.itertuples(index=False):
    a20_union(edge.image_a, edge.image_b)

_confirmed_pairs = pair_review_df[
    pair_review_df["decision"].isin(["duplicate", "correlated"])
]

for pair in _confirmed_pairs.itertuples(index=False):
    a20_union(str(pair.image_a), str(pair.image_b))

cluster_manifest_df = audit_manifest[
    ["image_id", "split", "metadata_time"]
].copy()

cluster_manifest_df["cluster_id"] = (
    cluster_manifest_df["image_id"].map(a20_find)
)

_cluster_summary = cluster_manifest_df.groupby("cluster_id").agg(
    image_count=("image_id", "size"),
    partitions=("split", lambda x: "|".join(sorted(set(x)))),
    first_observation=("metadata_time", "min"),
    last_observation=("metadata_time", "max"),
)

_cross_clusters = _cluster_summary[
    _cluster_summary["partitions"].str.contains("train")
    & _cluster_summary["partitions"].str.contains("validation")
]

for cluster_id, row in _cross_clusters.iterrows():
    a20_issue(
        "BLOCKER", "confirmed_dependence_crosses_partition",
        cluster_id,
        f"{row['image_count']} observations; {row['partitions']}"
    )

_validation_clusters = set(
    cluster_manifest_df.loc[
        cluster_manifest_df["split"] == "validation",
        "cluster_id",
    ]
)

_train_cluster_map = cluster_manifest_df.set_index(
    "image_id"
)["cluster_id"]

_sensitivity_records = []

for purge_days in sorted(set(
    [0, 1, 3, 7, 14, 27, AUDIT_CONFIG["purge_days"]]
)):
    temporal_keep = (
        _rebuilt_train["nearest_validation_gap_days"] > purge_days
    )

    retained = _rebuilt_train.loc[temporal_keep, "image_id"]

    connected_to_validation = retained.map(
        _train_cluster_map
    ).isin(_validation_clusters)

    _sensitivity_records.append({
        "purge_days": purge_days,
        "train_before_purge": len(_rebuilt_train),
        "train_after_temporal_purge": len(retained),
        "removed_by_temporal_purge": int((~temporal_keep).sum()),
        "additional_confirmed_cluster_exclusions": int(
            connected_to_validation.sum()
        ),
        "train_after_temporal_and_cluster_exclusion": int(
            (~connected_to_validation).sum()
        ),
        "validation_images": len(_rebuilt_valid),
    })

purge_sensitivity_df = pd.DataFrame(_sensitivity_records)

a20_save(cluster_manifest_df, "observation_cluster_manifest.csv")
a20_save(
    _cluster_summary.reset_index(),
    "observation_cluster_summary.csv"
)
a20_save(purge_sensitivity_df, "purge_sensitivity.csv")

display(purge_sensitivity_df)

_pending_pairs = pair_review_df[
    ~pair_review_df["decision"].isin(
        ["duplicate", "correlated", "distinct"]
    )
]

print("Confirmed cross-partition clusters:", len(_cross_clusters))
print("Unresolved cross-partition candidate pairs:", len(_pending_pairs))
print("The existing split has not been modified.")

### 20.10 Feature-cache integrity and sampling reconstruction

Cache membership must match the retained training partition exactly.

For every expected image checkpoint we verify:

- File existence and readability.
- Expected feature and metadata columns.
- A single correct image ID.
- Finite feature values.
- Binary targets.
- Foreground prevalence consistent with the full decoded foreground union.
- Foreground and background sample counts consistent with the original
  sampling policy.
- Content hashes for future provenance.

The existing cache does not save sampled pixel indices. We therefore
reconstruct the original deterministic sampling order from the original
seed and image ID.

On a deterministic subset of training images, we replay the complete
feature extraction and compare sampled features and targets with the
stored checkpoint.

Subset replay is evidence of consistency for inspected images; it does
not certify every cached feature value. Historical extraction-code
provenance cannot be reconstructed solely from a configuration file.

In [ ]:
_a20_expected_cache_ids = sorted(a20_train_ids)
_a20_expected_cache_paths = {
    image_id: Path(image_cache_path(image_id))
    for image_id in _a20_expected_cache_ids
}

a20_check(
    len(set(_a20_expected_cache_paths.values()))
    == len(_a20_expected_cache_paths),
    "cache_filename_collision",
    "Distinct image IDs map to the same checkpoint path."
)

_cache_configuration_path = Path(CACHE_DIR) / "config.json"
_cache_configuration = None

if _cache_configuration_path.is_file():
    with open(_cache_configuration_path) as handle:
        _cache_configuration = json.load(handle)

    a20_check(
        set(map(str, _cache_configuration.get("image_ids", [])))
        == a20_train_ids,
        "cache_config_training_membership",
        "Cache configuration IDs differ from retained training IDs."
    )
    a20_check(
        _cache_configuration.get("features") == list(FEATURE_NAMES),
        "cache_config_feature_schema",
        "Cache configuration feature list differs from current FEATURE_NAMES."
    )
    a20_check(
        _cache_configuration.get("seed") == int(SEED),
        "cache_config_seed",
        "Cache configuration seed differs from current seed."
    )
    a20_check(
        _cache_configuration.get("pixels_per_class")
        == int(PIXELS_PER_CLASS),
        "cache_config_sampling",
        "Cache sampling count differs from current sampling policy."
    )
else:
    a20_issue(
        "BLOCKER", "cache_config_missing",
        CACHE_DIR, "config.json is unavailable."
    )

_mask_summary_by_id = image_mask_audit_df.set_index("image_id")
_expected_columns = (
    list(FEATURE_NAMES)
    + ["target", "image_id", "foreground_fraction"]
)

_cache_audit_records = []

for position, image_id in enumerate(
    _a20_expected_cache_ids, start=1
):
    path = _a20_expected_cache_paths[image_id]
    result = {
        "image_id": image_id,
        "path": str(path),
        "exists": path.is_file(),
        "readable": False,
    }

    try:
        frame = pd.read_parquet(path)
        result["readable"] = True
        result["rows"] = len(frame)
        result["sha256"] = a20_file_hash(path)

        if set(frame.columns) != set(_expected_columns):
            raise ValueError(
                f"Unexpected cache schema: {list(frame.columns)}"
            )

        if (
            frame["image_id"].isna().any()
            or set(frame["image_id"].astype(str)) != {image_id}
        ):
            raise ValueError("Cache image_id does not match checkpoint.")

        if not frame["target"].isin([0, 1]).all():
            raise ValueError("Targets contain missing or nonbinary values.")

        features = frame[FEATURE_NAMES].to_numpy(
            dtype=np.float64
        )
        if not np.isfinite(features).all():
            raise ValueError("Features contain nonfinite values.")

        prevalence = frame["foreground_fraction"].to_numpy(
            dtype=np.float64
        )
        if (
            not np.isfinite(prevalence).all()
            or not ((prevalence >= 0) & (prevalence <= 1)).all()
        ):
            raise ValueError("Invalid foreground_fraction values.")

        if not np.allclose(
            prevalence, prevalence[0], rtol=0, atol=1e-8
        ):
            raise ValueError("Foreground prevalence varies within one image.")

        mask_info = _mask_summary_by_id.loc[image_id]
        foreground_pixels = int(mask_info["foreground_pixels"])

        info = a20_image_lookup[image_id]
        total_pixels = int(info["height"]) * int(info["width"])

        expected_foreground = min(
            int(PIXELS_PER_CLASS), foreground_pixels
        )
        expected_background = min(
            int(PIXELS_PER_CLASS),
            total_pixels - foreground_pixels,
        )

        actual_foreground = int(frame["target"].eq(1).sum())
        actual_background = int(frame["target"].eq(0).sum())

        if (
            actual_foreground != expected_foreground
            or actual_background != expected_background
        ):
            raise ValueError(
                "Sample counts differ from original policy: "
                f"actual=({actual_foreground},{actual_background}), "
                f"expected=({expected_foreground},{expected_background})"
            )

        if not np.allclose(
            prevalence,
            float(mask_info["foreground_fraction"]),
            rtol=0,
            atol=1e-7,
        ):
            raise ValueError(
                "Cached foreground prevalence differs from decoded union."
            )

        result.update({
            "foreground_rows": actual_foreground,
            "background_rows": actual_background,
            "foreground_fraction": float(prevalence[0]),
            "schema_and_sampling_ok": True,
        })

    except Exception as error:
        result["schema_and_sampling_ok"] = False
        result["error"] = repr(error)
        a20_issue(
            "BLOCKER", "cache_integrity_failure",
            image_id, repr(error)
        )

    _cache_audit_records.append(result)

    if position % 100 == 0:
        print(
            f"Cache files audited: {position}/{len(_a20_expected_cache_ids)}"
        )

cache_audit_df = pd.DataFrame(_cache_audit_records)

_expected_path_set = {
    path.resolve() for path in _a20_expected_cache_paths.values()
}
_extra_cache_files = [
    str(path)
    for path in Path(CACHE_DIR).glob("*.parquet")
    if path.resolve() not in _expected_path_set
]

if _extra_cache_files:
    a20_issue(
        "REVIEW", "unexpected_cache_checkpoints",
        CACHE_DIR,
        f"{len(_extra_cache_files)} extra Parquet files. "
        "Ensure downstream loaders use the explicit training manifest."
    )

if "pixel_df" in globals():
    a20_check(
        set(pixel_df["image_id"].astype(str)) == a20_train_ids,
        "loaded_pixel_dataframe_membership",
        "pixel_df does not contain exactly the retained training images."
    )

a20_save(cache_audit_df, "feature_cache_audit.csv")
a20_save(
    pd.DataFrame({"unexpected_path": _extra_cache_files}),
    "unexpected_cache_checkpoints.csv"
)

print("Expected cache files:", len(_a20_expected_cache_ids))
print(
    "Files passing schema and sampling checks:",
    int(cache_audit_df["schema_and_sampling_ok"].sum())
)
print("Extra cache files:", len(_extra_cache_files))

display(
    cache_audit_df[
        ~cache_audit_df["schema_and_sampling_ok"]
    ].head(20)
)

In [ ]:
_replay_population = (
    audit_manifest[audit_manifest["split"] == "train"]
    .merge(
        image_mask_audit_df[
            ["image_id", "foreground_fraction"]
        ],
        on="image_id",
        validate="one_to_one",
    )
    .sort_values(["foreground_fraction", "image_id"])
)

_replay_count = min(
    AUDIT_CONFIG["feature_replay_images"],
    len(_replay_population),
)

_replay_positions = np.unique(
    np.linspace(
        0, len(_replay_population) - 1,
        _replay_count, dtype=int
    )
)

_replay_ids = _replay_population.iloc[
    _replay_positions
]["image_id"].tolist()

_feature_replay_records = []

for image_id in _replay_ids:
    print("Replaying:", image_id, flush=True)

    try:
        raw_id = a20_raw_image_ids[image_id]
        image = load_solar_image(raw_id)
        target, _, _ = load_instance_targets(raw_id)
        flat_target = np.asarray(target, dtype=bool).ravel()

        foreground_idx = np.flatnonzero(flat_target)
        background_idx = np.flatnonzero(~flat_target)

        image_seed = int(
            hashlib.sha256(
                f"{SEED}:{image_id}".encode()
            ).hexdigest()[:8],
            16,
        )

        rng = np.random.default_rng(image_seed)

        selected_foreground = rng.choice(
            foreground_idx,
            size=min(PIXELS_PER_CLASS, len(foreground_idx)),
            replace=False,
        )
        selected_background = rng.choice(
            background_idx,
            size=min(PIXELS_PER_CLASS, len(background_idx)),
            replace=False,
        )

        selected_indices = np.concatenate([
            selected_foreground,
            selected_background,
        ])

        stored = pd.read_parquet(
            _a20_expected_cache_paths[image_id]
        )

        expected_targets = (
            flat_target[selected_indices].astype(np.uint8)
        )

        if not np.array_equal(
            stored["target"].to_numpy(), expected_targets
        ):
            raise ValueError(
                "Cached target order differs from deterministic replay."
            )

        maps = extract_pixel_features(image)

        if set(maps) != set(FEATURE_NAMES):
            raise ValueError("Replayed feature schema differs.")

        for feature in FEATURE_NAMES:
            expected = (
                np.asarray(maps[feature]).ravel()[selected_indices]
                .astype(np.float32)
            )
            observed = stored[feature].to_numpy(
                dtype=np.float32
            )

            if expected.shape != observed.shape:
                raise ValueError(
                    f"{feature}: replay and cache shapes differ."
                )

            passed = bool(
                np.isfinite(expected).all()
                and np.isfinite(observed).all()
                and np.allclose(
                    observed, expected,
                    rtol=AUDIT_CONFIG["feature_replay_rtol"],
                    atol=AUDIT_CONFIG["feature_replay_atol"],
                )
            )

            _feature_replay_records.append({
                "image_id": image_id,
                "feature": feature,
                "passed": passed,
                "maximum_absolute_difference": float(
                    np.max(np.abs(observed - expected))
                ),
                "sampled_indices_sha256": a20_array_hash(
                    selected_indices
                ),
            })

            if not passed:
                a20_issue(
                    "BLOCKER", "feature_replay_mismatch",
                    f"{image_id}:{feature}",
                    "Current extraction does not reproduce cached samples."
                )

        del image, target, maps, stored

    except Exception as error:
        a20_issue(
            "BLOCKER", "feature_replay_failure",
            image_id, repr(error)
        )
        _feature_replay_records.append({
            "image_id": image_id,
            "feature": "__image_replay__",
            "passed": False,
            "error": repr(error),
        })

feature_replay_df = pd.DataFrame(_feature_replay_records)
a20_save(feature_replay_df, "feature_replay_audit.csv")

display(feature_replay_df)

print(
    "Replay covers the listed images only. "
    "Historical cache code provenance remains a separate review item."
)

### 20.11 Training–validation distribution diagnostics

Partition independence and partition similarity are different questions.

We compare image-level distributions of:

- Instance counts.
- Foreground prevalence.
- Image brightness and contrast.
- Saturation fractions.
- Acquisition year and filename station token.

Continuous comparisons include:

- Medians and interquartile ranges.
- Standardized mean differences.
- Wasserstein distance normalized by training IQR.
- Jensen–Shannon distance using training-derived histogram boundaries.

These are descriptive diagnostics, not pixel-level significance tests.

A distribution difference is not automatically a reason to change the
split. It may represent the generalization difficulty the model must solve.

No normalization, feature selection or model component is fitted using
these validation summaries.

In [ ]:
distribution_df = (
    audit_manifest[
        [
            "image_id", "split", "file_name",
            "metadata_time", "annotation_count",
        ]
    ]
    .merge(
        image_audit_df[
            [
                "image_id", "mean", "std",
                "zero_fraction", "full_scale_fraction",
            ]
        ],
        on="image_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        image_mask_audit_df[
            ["image_id", "foreground_fraction"]
        ],
        on="image_id",
        how="left",
        validate="one_to_one",
    )
)

distribution_df["year"] = distribution_df["metadata_time"].dt.year

# Preserve this as an unverified filename token.
# Do not claim it is a verified observatory mapping.
distribution_df["filename_station_token"] = (
    distribution_df["file_name"]
    .str.extract(r"^\d{14}([^.]*)", expand=False)
    .fillna("unparsed")
)

_distribution_records = []

for feature in [
    "annotation_count", "foreground_fraction",
    "mean", "std", "zero_fraction", "full_scale_fraction",
]:
    train_values = distribution_df.loc[
        distribution_df["split"] == "train", feature
    ].dropna().to_numpy(dtype=float)

    valid_values = distribution_df.loc[
        distribution_df["split"] == "validation", feature
    ].dropna().to_numpy(dtype=float)

    if len(train_values) < 2 or len(valid_values) < 2:
        continue

    reference_sd = np.sqrt(
        (
            train_values.var(ddof=1)
            + valid_values.var(ddof=1)
        ) / 2
    )

    q25, q75 = np.quantile(train_values, [.25, .75])
    training_iqr = q75 - q25

    inner_edges = np.unique(
        np.quantile(train_values, np.linspace(0, 1, 11))
    )
    bins = np.concatenate([
        [-np.inf],
        inner_edges,
        [np.inf],
    ])

    train_hist = np.histogram(train_values, bins=bins)[0]
    valid_hist = np.histogram(valid_values, bins=bins)[0]

    train_prob = train_hist / train_hist.sum()
    valid_prob = valid_hist / valid_hist.sum()

    _distribution_records.append({
        "feature": feature,
        "train_n": len(train_values),
        "validation_n": len(valid_values),
        "train_median": np.median(train_values),
        "validation_median": np.median(valid_values),
        "train_iqr": training_iqr,
        "validation_iqr": (
            np.quantile(valid_values, .75)
            - np.quantile(valid_values, .25)
        ),
        "standardized_mean_difference_validation_minus_train": (
            (valid_values.mean() - train_values.mean()) / reference_sd
            if reference_sd > 0 else np.nan
        ),
        "wasserstein_distance": wasserstein_distance(
            train_values, valid_values
        ),
        "wasserstein_divided_by_train_iqr": (
            wasserstein_distance(train_values, valid_values)
            / training_iqr
            if training_iqr > 0 else np.nan
        ),
        "jensen_shannon_distance_base2": float(
            jensenshannon(train_prob, valid_prob, base=2)
        ),
    })

distribution_comparison_df = pd.DataFrame(
    _distribution_records
)

a20_save(
    distribution_comparison_df,
    "train_validation_distribution_comparison.csv"
)

display(distribution_comparison_df.round(5))

for category in ["year", "filename_station_token"]:
    table = pd.crosstab(
        distribution_df[category],
        distribution_df["split"],
    )
    a20_save(
        table.reset_index(),
        f"partition_distribution_{category}.csv"
    )
    display(table)

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

for axis, feature in zip(
    axes.flat,
    [
        "annotation_count", "foreground_fraction",
        "mean", "std", "zero_fraction", "full_scale_fraction",
    ],
):
    for partition, color in [
        ("train", "tab:blue"),
        ("validation", "tab:orange"),
    ]:
        values = np.sort(
            distribution_df.loc[
                distribution_df["split"] == partition,
                feature,
            ].dropna().to_numpy()
        )

        if len(values):
            axis.step(
                values,
                np.arange(1, len(values) + 1) / len(values),
                where="post",
                label=partition,
                color=color,
            )

    axis.set_title(feature)
    axis.set_ylabel("Empirical cumulative fraction")
    axis.grid(alpha=.25)

axes[0, 0].legend()
plt.tight_layout()
plt.savefig(
    AUDIT_DIR / "partition_distribution_ecdfs.png",
    dpi=160, bbox_inches="tight",
)
plt.show()
plt.close(fig)

### 20.12 Fitting and analysis provenance

Dataset membership checks cannot establish how a previously fitted
scaler, PCA, feature selector or model was trained.

This review records evidence for:

1. All Step 18 transformations and feature-selection decisions being
   fitted inside the appropriate training folds.
2. No validation labels or ground-truth morphology entering inference
   predictors.
3. Current cache extraction provenance and the scope of replay evidence.
4. Image orientation and preprocessing conventions.
5. Step 19 morphology coverage, exclusions and training-only derivation.
6. Preservation of independent instances where annotations overlap.

A review item must contain concrete evidence, such as cell references,
fit-image manifests, fold memberships, saved configurations or an
explicitly documented implementation policy.

A completed checkbox without evidence is insufficient.

Step 21 official decoding and metric parity remain a separate gate.

In [ ]:
PROVENANCE_REVIEW_PATH = AUDIT_DIR / "provenance_review.csv"

_provenance_items = [
    {
        "item": "step18_fold_local_fitting",
        "requirement": (
            "Scalers, PCA, feature selection and supervised comparisons "
            "fit only on the training side of each inner grouped fold."
        ),
    },
    {
        "item": "no_ground_truth_in_inference_features",
        "requirement": (
            "Label-derived morphology, masks and auxiliary targets are "
            "not required as inputs to test-time prediction."
        ),
    },
    {
        "item": "historical_cache_provenance",
        "requirement": (
            "Document which code/data created the cache and what replay "
            "does and does not verify. Resolve detected mismatches."
        ),
    },
    {
        "item": "orientation_and_preprocessing",
        "requirement": (
            "Verify image orientation, grayscale conversion and native "
            "representation against aligned annotation overlays."
        ),
    },
    {
        "item": "step19_population_coverage",
        "requirement": (
            "Every allowed training annotation appears in the Step 19 "
            "morphology output or has an explicit exclusion reason; "
            "no validation-derived cutoffs are fitted."
        ),
    },
    {
        "item": "independent_instance_representation",
        "requirement": (
            "Document independent-mask preservation and the treatment "
            "of overlaps in training, reconstruction and evaluation."
        ),
    },
]

provenance_review_df = pd.DataFrame(_provenance_items)
provenance_review_df["status"] = "pending"
provenance_review_df["evidence"] = ""

if PROVENANCE_REVIEW_PATH.exists():
    old = pd.read_csv(
        PROVENANCE_REVIEW_PATH,
        keep_default_na=False,
    ).set_index("item")

    for column in ["status", "evidence"]:
        mapped = provenance_review_df["item"].map(old[column])
        provenance_review_df[column] = mapped.fillna(
            provenance_review_df[column]
        )

a20_save(provenance_review_df, "provenance_review.csv")

def a20_record_provenance(item, status, evidence):
    if status not in {"verified", "pending", "failed"}:
        raise ValueError(
            "Status must be verified, pending or failed."
        )

    if status in {"verified", "failed"} and not str(evidence).strip():
        raise ValueError("Concrete evidence is required.")

    frame = pd.read_csv(
        PROVENANCE_REVIEW_PATH,
        keep_default_na=False,
    )

    selected = frame["item"].eq(item)

    if selected.sum() != 1:
        raise ValueError("Unknown or repeated provenance item.")

    frame.loc[selected, "status"] = status
    frame.loc[selected, "evidence"] = str(evidence).strip()
    frame.to_csv(PROVENANCE_REVIEW_PATH, index=False)

    print("Recorded:", item, status)

display(provenance_review_df)

### 20.13 Audit decision and frozen outputs

The audit distinguishes three outcomes:

- BLOCKED:
  At least one automated blocking failure, failed provenance item or
  confirmed cross-partition dependence remains.

- REVIEW_REQUIRED:
  Automated blocking checks have no detected failure, but candidate-pair,
  provenance or other review findings remain unresolved.

- AUTOMATED_CHECKS_PASSED:
  Implemented automated checks passed and required reviews have been
  documented. This is not proof of physical independence and does not
  replace Step 21 official decoding and metric parity.

Review findings are acknowledged individually with evidence.
Acknowledgement is appropriate only after investigation; it must not be
used to dismiss an unresolved defect.

The audit saves the source annotation fingerprint, split membership,
physical image hashes, checkpoint hashes, configuration, environment,
findings and decisions.

The current split is preserved. Any correction creates a deliberately
versioned replacement protocol and invalidates comparisons that mix
the old and new versions.

In [ ]:
# This cell can be rerun after recording pair/provenance decisions.
# Rebuild the cluster cell first if pair decisions have changed.

pair_review_df = pd.read_csv(
    PAIR_REVIEW_PATH,
    dtype={"pair_id": str},
    keep_default_na=False,
)

provenance_review_df = pd.read_csv(
    PROVENANCE_REVIEW_PATH,
    keep_default_na=False,
)

# Construct fresh findings rather than appending gate results repeatedly.
final_findings_df = pd.DataFrame(
    audit_findings,
    columns=["severity", "check", "entity", "detail"],
).drop_duplicates()

final_findings_df["finding_id"] = [
    a20_digest({
        "severity": row.severity,
        "check": row.check,
        "entity": row.entity,
        "detail": row.detail,
    })[:20]
    for row in final_findings_df.itertuples(index=False)
]

FINDING_REVIEW_PATH = AUDIT_DIR / "finding_review.csv"

review_findings = final_findings_df[
    final_findings_df["severity"] == "REVIEW"
].copy()

review_findings["review_status"] = "pending"
review_findings["evidence"] = ""

if FINDING_REVIEW_PATH.exists():
    previous = pd.read_csv(
        FINDING_REVIEW_PATH,
        dtype={"finding_id": str},
        keep_default_na=False,
    ).set_index("finding_id")

    if previous.index.duplicated().any():
        raise ValueError("Repeated finding_id in finding_review.csv.")

    for column in ["review_status", "evidence"]:
        review_findings[column] = (
            review_findings["finding_id"]
            .map(previous[column])
            .fillna(review_findings[column])
        )

a20_save(review_findings, "finding_review.csv")
a20_save(final_findings_df, "integrity_checks.csv")

def a20_record_finding_review(finding_id, status, evidence):
    if status not in {"resolved", "documented", "pending"}:
        raise ValueError(
            "Status must be resolved, documented or pending."
        )
    if status != "pending" and not str(evidence).strip():
        raise ValueError("Evidence is required.")

    frame = pd.read_csv(
        FINDING_REVIEW_PATH,
        dtype={"finding_id": str},
        keep_default_na=False,
    )
    selected = frame["finding_id"].eq(str(finding_id))

    if selected.sum() != 1:
        raise ValueError("Unknown or repeated finding ID.")

    frame.loc[selected, "review_status"] = status
    frame.loc[selected, "evidence"] = str(evidence).strip()
    frame.to_csv(FINDING_REVIEW_PATH, index=False)

    print("Recorded:", finding_id, status)

_pending_pairs = pair_review_df[
    ~pair_review_df["decision"].isin(
        ["duplicate", "correlated", "distinct"]
    )
    | pair_review_df["reason"].str.strip().eq("")
]

_pending_provenance = provenance_review_df[
    (provenance_review_df["status"] != "verified")
    | provenance_review_df["evidence"].str.strip().eq("")
]

_failed_provenance = provenance_review_df[
    provenance_review_df["status"] == "failed"
]

_pending_findings = review_findings[
    ~review_findings["review_status"].isin(
        ["resolved", "documented"]
    )
    | review_findings["evidence"].str.strip().eq("")
]

_blockers = final_findings_df[
    final_findings_df["severity"] == "BLOCKER"
]

# Re-evaluate confirmed clusters from current review decisions.
# This avoids relying on stale cluster findings after a review change.
_current_parent = {image_id: image_id for image_id in a20_known_ids}

def _current_find(value):
    while _current_parent[value] != value:
        _current_parent[value] = _current_parent[_current_parent[value]]
        value = _current_parent[value]
    return value

def _current_union(left, right):
    a, b = _current_find(left), _current_find(right)
    if a != b:
        _current_parent[max(a, b)] = min(a, b)

for edge in exact_duplicate_edges_df.itertuples(index=False):
    _current_union(str(edge.image_a), str(edge.image_b))

for pair in pair_review_df[
    pair_review_df["decision"].isin(["duplicate", "correlated"])
].itertuples(index=False):
    _current_union(str(pair.image_a), str(pair.image_b))

_current_clusters = audit_manifest[["image_id", "split"]].copy()
_current_clusters["cluster_id"] = (
    _current_clusters["image_id"].map(_current_find)
)

_current_cross_clusters = (
    _current_clusters.groupby("cluster_id")["split"]
    .agg(lambda values: {"train", "validation"} <= set(values))
)

_current_cross_cluster_count = int(_current_cross_clusters.sum())

# Cluster findings are recomputed above; retain other automated blockers.
_noncluster_blockers = _blockers[
    _blockers["check"] != "confirmed_dependence_crosses_partition"
]

if (
    len(_noncluster_blockers)
    or len(_failed_provenance)
    or _current_cross_cluster_count
):
    audit_status = "BLOCKED"
elif (
    len(_pending_pairs)
    or len(_pending_provenance)
    or len(_pending_findings)
):
    audit_status = "REVIEW_REQUIRED"
else:
    audit_status = "AUTOMATED_CHECKS_PASSED"

summary = {
    "audit_run_id": AUDIT_RUN_ID,
    "status": audit_status,
    "images": len(audit_manifest),
    "annotations": len(_a20_annotations),
    "training_images": len(a20_train_ids),
    "validation_images": len(a20_valid_ids),
    "excluded_images": int(
        audit_manifest["split"].eq("excluded").sum()
    ),
    "automated_noncluster_blocking_findings": len(
        _noncluster_blockers
    ),
    "confirmed_cross_partition_clusters": (
        _current_cross_cluster_count
    ),
    "unresolved_cross_partition_candidate_pairs": len(
        _pending_pairs
    ),
    "pending_or_failed_provenance_items": len(
        _pending_provenance
    ),
    "unresolved_review_findings": len(_pending_findings),
    "feature_replay_images": len(_replay_ids),
    "official_decoder_and_metric_parity": (
        "Not established here; required in Step 21."
    ),
    "split_modified_by_audit": False,
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

with open(AUDIT_DIR / "audit_summary.json", "w") as handle:
    json.dump(a20_jsonable(summary), handle, indent=2)

with open(AUDIT_DIR / "frozen_input_fingerprints.json", "w") as handle:
    json.dump({
        "annotation_sha256": a20_file_hash(ANNOTATION_PATH),
        "train_ids": sorted(a20_train_ids),
        "validation_ids": sorted(a20_valid_ids),
        "image_hash_table": "physical_image_audit.csv",
        "cache_hash_table": "feature_cache_audit.csv",
        "function_source_table": "current_function_sources.json",
    }, handle, indent=2)

_report_lines = [
    "# Step 20 audit",
    "",
    f"Status: {audit_status}",
    f"Run: {AUDIT_RUN_ID}",
    "",
    "## Counts",
    json.dumps(a20_jsonable(summary), indent=2),
    "",
    "## Interpretation",
    "The audit preserves the existing split.",
    "Screening similarity is not a physical-independence guarantee.",
    "Historical provenance requires documented evidence.",
    "Official decoding and metric parity remain Step 21.",
]

(AUDIT_DIR / "audit_report.md").write_text(
    "\n".join(_report_lines),
    encoding="utf-8",
)

print(json.dumps(a20_jsonable(summary), indent=2))

print("\nAUTOMATED BLOCKING FINDINGS")
display(_noncluster_blockers)

print("\nUNRESOLVED PAIR REVIEWS")
display(_pending_pairs.head(30))

print("\nPROVENANCE ITEMS")
display(provenance_review_df)

print("\nUNRESOLVED REVIEW FINDINGS")
display(_pending_findings)

print("\nAudit outputs:", AUDIT_DIR)

if audit_status == "BLOCKED":
    print(
        "\nResolve blocking findings before relying on comparative "
        "training results. Corrections require rerunning the affected "
        "audit cells; a review acknowledgement cannot waive a blocker."
    )
elif audit_status == "REVIEW_REQUIRED":
    print(
        "\nAutomated checks found no remaining blocking failure, "
        "but the displayed evidence reviews are incomplete."
    )
else:
    print(
        "\nImplemented checks and required reviews are complete. "
        "Proceed to Step 21: official decoding and evaluator parity."
    )

### 20.13 Visual Audit Diagnostics

These figures complement the audit tables by showing:

- The distribution of automated findings.
- Image-quality outliers.
- Temporal partitioning and purge boundaries.
- Training–validation composition.
- Annotation overlaps and instance information lost through overwriting.
- Differences between available polygon decoders.
- Candidate duplicate observations before and after registration.
- Confirmed observation clusters.
- Numerical differences during feature-cache replay.

Flagged examples are selected using audit measurements, not model scores.
Display normalization is used for inspection only and does not modify
the images supplied to the model.

Visual similarity, apparent overlay alignment and clean plots do not
replace numerical checks or documented review.

In [ ]:
A20_FIGURE_DIR = AUDIT_DIR / "figures"
A20_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def a20_finish_figure(fig, filename):
    fig.savefig(
        A20_FIGURE_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
    plt.close(fig)

def a20_display_limits(image):
    h, w = image.shape
    interior = image[h // 4:3 * h // 4, w // 4:3 * w // 4]

    low, high = np.quantile(interior, [0.01, 0.99])
    return float(low), float(max(high, low + 1e-6))

# -------------------------------------------------------------
# A. Findings dashboard
# -------------------------------------------------------------
_viz_findings = pd.DataFrame(
    audit_findings,
    columns=["severity", "check", "entity", "detail"],
).drop_duplicates()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

severity_order = ["BLOCKER", "REVIEW"]
severity_counts = (
    _viz_findings["severity"]
    .value_counts()
    .reindex(severity_order, fill_value=0)
)

bars = axes[0].bar(
    severity_order,
    severity_counts.values,
    color=["#c62828", "#ef9b20"],
)
axes[0].bar_label(bars, padding=4)
axes[0].set_title("Recorded automated findings")
axes[0].set_ylabel("Number of distinct findings")
axes[0].set_ylim(0, max(1, severity_counts.max()) * 1.2)
axes[0].grid(axis="y", alpha=0.2)

top_checks = (
    _viz_findings["check"].value_counts().head(12).sort_values()
)

if len(top_checks):
    axes[1].barh(
        top_checks.index,
        top_checks.values,
        color="#496b8a",
    )
    axes[1].set_xlabel("Number of findings")
else:
    axes[1].text(
        0.5, 0.5, "No findings recorded",
        ha="center", va="center",
        transform=axes[1].transAxes,
    )

axes[1].set_title("Most frequent finding types")
axes[1].grid(axis="x", alpha=0.2)

fig.suptitle(
    "Step 20 audit findings — counts are not a severity-weighted score",
    fontsize=14,
)
fig.tight_layout()
a20_finish_figure(fig, "01_audit_findings_dashboard.png")

# -------------------------------------------------------------
# B. Deterministic image-quality outlier gallery
# -------------------------------------------------------------
_quality = image_audit_df[
    image_audit_df["readable"].fillna(False)
].copy()

_outlier_examples = []
_seen = set()

for metric, ascending, description in [
    ("std", True, "Lowest full-image contrast"),
    ("std", False, "Highest full-image contrast"),
    ("mean", True, "Lowest full-image mean"),
    ("mean", False, "Highest full-image mean"),
    ("zero_fraction", False, "Highest zero fraction"),
    ("full_scale_fraction", False, "Highest 255 fraction"),
]:
    ranked = _quality.sort_values(
        [metric, "image_id"],
        ascending=[ascending, True],
    )

    selected = ranked[~ranked["image_id"].isin(_seen)].head(1)

    if len(selected):
        row = selected.iloc[0]
        _seen.add(row["image_id"])
        _outlier_examples.append((row, metric, description))

if _outlier_examples:
    fig, axes = plt.subplots(
        2, 3, figsize=(16, 11), squeeze=False
    )

    for axis in axes.flat:
        axis.axis("off")

    for axis, (row, metric, description) in zip(
        axes.flat, _outlier_examples
    ):
        image_id = str(row["image_id"])
        image = load_solar_image(a20_raw_image_ids[image_id])

        # Fixed scale preserves brightness differences across examples.
        axis.imshow(image, cmap="gray", vmin=0, vmax=255)
        axis.set_title(
            f"{description}\n"
            f"{image_id} | {row['split']}\n"
            f"{metric}={row[metric]:.5g}",
            fontsize=10,
        )

    fig.suptitle(
        "Image-quality extremes — fixed grayscale scale [0, 255]\n"
        "Full-image measurements include the off-disk background",
        fontsize=14,
    )
    fig.tight_layout()
    a20_finish_figure(fig, "02_image_quality_outliers.png")
else:
    print("No readable images available for the outlier gallery.")

In [ ]:
_partition_colors = {
    "train": "#2878b5",
    "validation": "#e58b25",
    "excluded": "#888888",
}

# -------------------------------------------------------------
# A. Temporal partition timeline with temporal-block boundaries
# -------------------------------------------------------------
fig, axes = plt.subplots(
    2, 1, figsize=(17, 9),
    gridspec_kw={"height_ratios": [1, 1.25]},
)

for y, partition in enumerate(["train", "validation", "excluded"]):
    rows = audit_manifest[
        audit_manifest["split"] == partition
    ]

    axes[0].scatter(
        rows["metadata_time"],
        np.full(len(rows), y),
        s=12,
        alpha=0.65,
        color=_partition_colors[partition],
        label=f"{partition} ({len(rows)})",
    )

block_boundaries = pd.date_range(
    start=_a20_reference,
    end=audit_manifest["metadata_time"].max(),
    freq=f"{AUDIT_CONFIG['block_days']}D",
)

# Full-period plot: light boundaries avoid obscuring observations.
for boundary in block_boundaries:
    axes[0].axvline(
        boundary, color="black", alpha=0.04, linewidth=0.5
    )

axes[0].set_yticks(
    [0, 1, 2], ["Training", "Validation", "Excluded"]
)
axes[0].set_title(
    "Actual temporal partition — faint lines show block boundaries"
)
axes[0].legend(loc="upper left", ncol=3)
axes[0].grid(axis="y", alpha=0.2)

for partition in ["train", "validation", "excluded"]:
    rows = audit_manifest[
        audit_manifest["split"] == partition
    ]
    axes[1].scatter(
        rows["metadata_time"],
        rows["annotation_count"],
        s=12,
        alpha=0.55,
        color=_partition_colors[partition],
        label=partition,
    )

axes[1].set_title("Annotation density through time")
axes[1].set_xlabel("Metadata timestamp")
axes[1].set_ylabel("Annotations per image")
axes[1].grid(alpha=0.2)

fig.tight_layout()
a20_finish_figure(fig, "03_temporal_split_timeline.png")

# -------------------------------------------------------------
# B. Nearest cross-partition distances and purge sensitivity
# -------------------------------------------------------------
gaps = (
    _a20_train_rows["nearest_validation_gap_days"]
    .dropna()
    .to_numpy(dtype=float)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if len(gaps):
    # Log scale keeps both short and long distances visible.
    axes[0].hist(
        gaps, bins=35, color="#2878b5",
        edgecolor="white", alpha=0.85,
    )
    axes[0].set_xscale("symlog", linthresh=1)

    sorted_gaps = np.sort(gaps)
    axes[1].step(
        sorted_gaps,
        np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps),
        where="post",
        color="#2878b5",
    )
    axes[1].set_xscale("symlog", linthresh=1)

for axis in axes[:2]:
    axis.axvline(
        AUDIT_CONFIG["purge_days"],
        color="#c62828",
        linestyle="--",
        label=(
            f"Strict purge boundary: "
            f"{AUDIT_CONFIG['purge_days']} days"
        ),
    )
    axis.set_xlabel("Nearest validation observation gap, days")
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)

axes[0].set_title("Retained training-image distances")
axes[0].set_ylabel("Images")
axes[1].set_title("Distance empirical cumulative distribution")
axes[1].set_ylabel("Cumulative fraction of training images")

axes[2].plot(
    purge_sensitivity_df["purge_days"],
    purge_sensitivity_df["train_after_temporal_purge"],
    marker="o",
    label="Temporal purge",
    color="#2878b5",
)
axes[2].plot(
    purge_sensitivity_df["purge_days"],
    purge_sensitivity_df[
        "train_after_temporal_and_cluster_exclusion"
    ],
    marker="s",
    label="Temporal + confirmed clusters",
    color="#a64d79",
)
axes[2].axvline(
    AUDIT_CONFIG["purge_days"],
    color="#c62828",
    linestyle="--",
    alpha=0.7,
)
axes[2].set_title("Protocol sensitivity — diagnostic alternatives")
axes[2].set_xlabel("Purge duration, days")
axes[2].set_ylabel("Training images retained")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.2)

fig.tight_layout()
a20_finish_figure(fig, "04_purge_distance_and_sensitivity.png")

# -------------------------------------------------------------
# C. Partition composition: proportions, not unequal raw totals
# -------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for axis, category in zip(
    axes, ["year", "filename_station_token"]
):
    retained = distribution_df[
        distribution_df["split"].isin(["train", "validation"])
    ]

    counts = pd.crosstab(
        retained[category],
        retained["split"],
    ).reindex(columns=["train", "validation"], fill_value=0)

    proportions = counts.div(
        counts.sum(axis=0).replace(0, np.nan),
        axis=1,
    )

    proportions.plot.bar(
        ax=axis,
        color=[
            _partition_colors["train"],
            _partition_colors["validation"],
        ],
        width=0.8,
    )

    axis.set_title(
        f"Partition composition by {category}"
        + (
            " — filename tokens are not verified station mappings"
            if category == "filename_station_token" else ""
        )
    )
    axis.set_ylabel("Fraction within partition")
    axis.set_xlabel(category)
    axis.grid(axis="y", alpha=0.2)

fig.tight_layout()
a20_finish_figure(fig, "05_partition_composition.png")

In [ ]:
def a20_annotation_panels(image_id, annotation_id=None):
    """
    Display a native-resolution crop around the selected annotation.

    With annotation_id=None, select the annotation losing the most
    pixels under the current integer-label overwrite policy.
    """
    image_id = str(image_id)
    raw_id = a20_raw_image_ids[image_id]

    image = load_solar_image(raw_id)
    shape = image.shape

    annotations_here = a20_annotations_by_image[image_id]

    if not annotations_here:
        print("No annotations:", image_id)
        return

    if annotation_id is None:
        candidates = annotation_audit_df[
            annotation_audit_df["image_id"].astype(str).eq(image_id)
        ].copy()

        if "area_lost_to_overwrite" in candidates:
            candidates = candidates.sort_values(
                "area_lost_to_overwrite",
                ascending=False,
                na_position="last",
            )

        annotation_id = (
            str(candidates.iloc[0]["annotation_id"])
            if len(candidates)
            else str(annotations_here[0]["id"])
        )

    matches = [
        (index, ann)
        for index, ann in enumerate(annotations_here, start=1)
        if str(ann["id"]) == str(annotation_id)
    ]

    if len(matches) != 1:
        raise ValueError("Annotation ID is not unique within this image.")

    instance_index, selected_ann = matches[0]

    coverage = np.zeros(shape, dtype=np.uint32)

    for ann in annotations_here:
        try:
            decoded = np.asarray(
                rasterize_annotation(ann, shape), dtype=bool
            )
            coverage[decoded] += 1
        except Exception:
            # A failure is recorded by the numerical annotation audit.
            continue

    selected_mask = np.asarray(
        rasterize_annotation(selected_ann, shape), dtype=bool
    )

    _, current_labels, _ = load_instance_targets(raw_id)

    lost_pixels = (
        selected_mask & (current_labels != instance_index)
    )

    if selected_mask.any():
        ys, xs = np.nonzero(selected_mask)
        padding = 25

        y0 = max(0, int(ys.min()) - padding)
        y1 = min(shape[0], int(ys.max()) + padding + 1)
        x0 = max(0, int(xs.min()) - padding)
        x1 = min(shape[1], int(xs.max()) + padding + 1)
    else:
        y0, y1, x0, x1 = 0, shape[0], 0, shape[1]

    crop_slice = np.s_[y0:y1, x0:x1]
    cropped_image = image[crop_slice]
    low, high = a20_display_limits(image)

    fig, axes = plt.subplots(2, 3, figsize=(17, 11))

    for axis in axes.flat:
        axis.imshow(
            cropped_image,
            cmap="gray",
            vmin=low,
            vmax=high,
            extent=(x0, x1, y1, y0),
        )
        axis.set_xlim(x0, x1)
        axis.set_ylim(y1, y0)
        axis.set_xlabel("Original x coordinate")
        axis.set_ylabel("Original y coordinate")

    axes[0, 0].set_title("Image and source polygon vertices")

    try:
        parts = a20_strict_polygon_parts(selected_ann)
        for polygon in parts:
            closed = np.vstack([polygon, polygon[0]])
            axes[0, 0].plot(
                closed[:, 0], closed[:, 1],
                color="#00e5ff", linewidth=1,
            )
    except Exception as error:
        axes[0, 0].text(
            0.02, 0.02, str(error)[:150],
            transform=axes[0, 0].transAxes,
            color="red", fontsize=8,
        )

    overlays = [
        (
            axes[0, 1], selected_mask[crop_slice],
            "Independent selected-instance mask",
            "autumn",
        ),
        (
            axes[0, 2], coverage[crop_slice],
            "Independent annotation coverage multiplicity",
            "viridis",
        ),
        (
            axes[1, 0], current_labels[crop_slice],
            "Current integer labels: last annotation wins",
            "nipy_spectral",
        ),
        (
            axes[1, 1], lost_pixels[crop_slice],
            f"Selected-instance pixels overwritten: {int(lost_pixels.sum())}",
            "autumn",
        ),
    ]

    for axis, overlay, title, cmap in overlays:
        masked = np.ma.masked_where(overlay == 0, overlay)

        artist = axis.imshow(
            masked,
            cmap=cmap,
            alpha=0.75,
            interpolation="nearest",
            extent=(x0, x1, y1, y0),
        )
        axis.set_title(title, fontsize=10)

        if title.startswith("Independent annotation coverage"):
            fig.colorbar(
                artist, ax=axis, fraction=0.046, pad=0.04,
                label="Mask memberships",
            )

    if A20_COCO_AVAILABLE:
        try:
            parts = a20_strict_polygon_parts(selected_ann)
            rles = coco_mask.frPyObjects(
                [p.reshape(-1).tolist() for p in parts],
                shape[0], shape[1],
            )
            reference = coco_mask.decode(
                coco_mask.merge(rles)
            ).astype(bool)

            # 1: current only; 2: COCO only.
            disagreement = (
                (selected_mask & ~reference).astype(np.uint8)
                + 2 * (reference & ~selected_mask).astype(np.uint8)
            )

            from matplotlib.colors import ListedColormap, BoundaryNorm

            decoder_cmap = ListedColormap(
                ["#ff304f", "#00d9ff"]
            )
            decoder_norm = BoundaryNorm([0.5, 1.5, 2.5], 2)

            artist = axes[1, 2].imshow(
                np.ma.masked_where(
                    disagreement[crop_slice] == 0,
                    disagreement[crop_slice],
                ),
                cmap=decoder_cmap,
                norm=decoder_norm,
                interpolation="nearest",
                alpha=0.95,
                extent=(x0, x1, y1, y0),
            )
            colorbar = fig.colorbar(
                artist, ax=axes[1, 2],
                ticks=[1, 2], fraction=0.046, pad=0.04,
            )
            colorbar.ax.set_yticklabels(
                ["Current only", "COCO only"]
            )
            axes[1, 2].set_title(
                "Decoder differences — Step 21 determines authority",
                fontsize=10,
            )
        except Exception as error:
            axes[1, 2].set_title("Decoder comparison unavailable")
            axes[1, 2].text(
                .02, .02, str(error)[:150],
                transform=axes[1, 2].transAxes,
                color="red", fontsize=8,
            )
    else:
        axes[1, 2].set_title(
            "COCO comparison unavailable: package not installed"
        )

    fig.suptitle(
        f"Annotation audit | image={image_id} | "
        f"annotation={annotation_id}\n"
        "Cropped display retains original pixel coordinates",
        fontsize=13,
    )
    fig.tight_layout()

    safe_key = a20_digest({
        "image": image_id,
        "annotation": str(annotation_id),
    })[:12]

    a20_finish_figure(
        fig, f"06_annotation_panels_{safe_key}.png"
    )


# A. Automatically inspect the largest overwrite cases.
_lost_cases = annotation_audit_df[
    annotation_audit_df["valid_decode"].fillna(False)
].copy()

_lost_cases = _lost_cases.sort_values(
    "area_lost_to_overwrite",
    ascending=False,
    na_position="last",
)

_selected_annotation_pairs = []

for row in _lost_cases[
    _lost_cases["area_lost_to_overwrite"].fillna(0) > 0
].head(3).itertuples(index=False):
    _selected_annotation_pairs.append(
        (str(row.image_id), str(row.annotation_id))
    )

# B. Include the largest decoder discrepancy when available.
if (
    A20_COCO_AVAILABLE
    and "pillow_coco_disagreement_pixels" in annotation_audit_df
):
    decoder_cases = annotation_audit_df.sort_values(
        "pillow_coco_disagreement_pixels",
        ascending=False,
        na_position="last",
    )

    for row in decoder_cases[
        decoder_cases["pillow_coco_disagreement_pixels"].fillna(0) > 0
    ].head(1).itertuples(index=False):
        _selected_annotation_pairs.append(
            (str(row.image_id), str(row.annotation_id))
        )

# C. If there are no flagged cases, show a deterministic reference.
if not _selected_annotation_pairs and len(_lost_cases):
    row = _lost_cases.iloc[0]
    _selected_annotation_pairs.append(
        (str(row["image_id"]), str(row["annotation_id"]))
    )

for image_id, annotation_id in dict.fromkeys(
    _selected_annotation_pairs
):
    a20_annotation_panels(image_id, annotation_id)

print(
    "Inspect any additional annotation with:\n"
    "a20_annotation_panels('IMAGE_ID', 'ANNOTATION_ID')"
)

In [ ]:
_polygon_checks = {
    "polygon_outside_image",
    "invalid_polygon_geometry",
    "annotation_decode_or_schema_failure",
    "empty_decoded_annotation",
}

_flagged_annotation_ids = (
    _viz_findings.loc[
        _viz_findings["check"].isin(_polygon_checks),
        "entity",
    ]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

_annotation_lookup_viz = {
    str(ann["id"]): ann for ann in _a20_annotations
}

for annotation_id in _flagged_annotation_ids[:6]:
    ann = _annotation_lookup_viz.get(annotation_id)

    if ann is None:
        continue

    image_id = str(ann.get("image_id"))
    if image_id not in a20_raw_image_ids:
        continue

    try:
        image = load_solar_image(a20_raw_image_ids[image_id])
    except Exception as error:
        print("Cannot display", image_id, repr(error))
        continue

    h, w = image.shape
    low, high = a20_display_limits(image)

    fig, axis = plt.subplots(figsize=(9, 9))
    axis.imshow(
        image, cmap="gray", vmin=low, vmax=high,
        extent=(0, w, h, 0),
    )

    from matplotlib.patches import Rectangle

    axis.add_patch(
        Rectangle(
            (0, 0), w, h,
            fill=False, edgecolor="yellow", linewidth=2,
        )
    )

    segmentation = ann.get("segmentation")
    point_sets = []

    if isinstance(segmentation, list) and segmentation:
        parts = (
            [segmentation]
            if isinstance(segmentation[0], (int, float))
            else segmentation
        )

        for part in parts:
            try:
                values = np.asarray(part, dtype=float)
                if (
                    values.ndim == 1
                    and values.size >= 2
                    and values.size % 2 == 0
                ):
                    points = values.reshape(-1, 2)
                    finite_points = points[
                        np.isfinite(points).all(axis=1)
                    ]

                    if len(finite_points):
                        point_sets.append(finite_points)
                        axis.plot(
                            finite_points[:, 0],
                            finite_points[:, 1],
                            "-o",
                            color="#ff304f",
                            markersize=2,
                            linewidth=1,
                        )
            except (TypeError, ValueError):
                continue

    if point_sets:
        all_points = np.concatenate(point_sets)
        xmin = min(0, float(all_points[:, 0].min()))
        xmax = max(w, float(all_points[:, 0].max()))
        ymin = min(0, float(all_points[:, 1].min()))
        ymax = max(h, float(all_points[:, 1].max()))

        xpad = max(5, (xmax - xmin) * 0.02)
        ypad = max(5, (ymax - ymin) * 0.02)

        axis.set_xlim(xmin - xpad, xmax + xpad)
        axis.set_ylim(ymax + ypad, ymin - ypad)

    reasons = _viz_findings.loc[
        _viz_findings["entity"].astype(str).eq(annotation_id)
        & _viz_findings["check"].isin(_polygon_checks),
        "check",
    ].tolist()

    axis.set_title(
        f"Flagged annotation {annotation_id} | image {image_id}\n"
        f"{', '.join(reasons)}\n"
        "Yellow: image boundary | Red: finite supplied coordinates",
        fontsize=10,
    )
    axis.set_xlabel("Original x coordinate")
    axis.set_ylabel("Original y coordinate")

    fig.tight_layout()
    a20_finish_figure(
        fig,
        f"07_polygon_issue_{a20_digest(annotation_id)[:12]}.png",
    )

if not _flagged_annotation_ids:
    print("No flagged polygon-coordinate cases were recorded.")

print(
    "Gallery shows up to six flagged annotations. "
    "The complete population remains in annotation_audit.csv."
)

In [ ]:
def a20_duplicate_diagnostic_panels(pair_id):
    matches = near_pair_df[
        near_pair_df["pair_id"].astype(str).eq(str(pair_id))
    ]

    if len(matches) != 1:
        raise ValueError("pair_id must identify exactly one pair.")

    pair = matches.iloc[0]

    image_a = a20_thumbnails[str(pair["image_a"])]
    image_b = a20_thumbnails[str(pair["image_b"])]

    detail_a = a20_detail_image(image_a)
    detail_b = a20_detail_image(image_b)

    registered = bool(pair["registration_ok"])

    if registered:
        estimated_shift = np.array(
            [pair["shift_y"], pair["shift_x"]],
            dtype=float,
        )

        aligned_b = ndi_shift(
            image_b, estimated_shift,
            order=1, mode="constant", cval=0,
            prefilter=False,
        )
        support = ndi_shift(
            np.ones_like(image_b), estimated_shift,
            order=0, mode="constant", cval=0,
            prefilter=False,
        ) > 0.5

        valid = _a20_interior & support
        aligned_detail = a20_detail_image(aligned_b)
    else:
        aligned_b = image_b
        aligned_detail = detail_b
        valid = _a20_interior

    # Shared raw display scale supports fair visual comparison.
    shared_values = np.concatenate([
        image_a[_a20_interior],
        image_b[_a20_interior],
    ])
    low, high = np.quantile(shared_values, [0.01, 0.99])
    high = max(float(high), float(low) + 1e-6)

    raw_difference = np.ma.masked_where(
        ~valid, np.abs(image_a - aligned_b)
    )
    detail_difference = np.ma.masked_where(
        ~valid, detail_a - aligned_detail
    )

    raw_limit = max(
        float(np.quantile(raw_difference.compressed(), .99)),
        1e-6,
    )
    detail_limit = max(
        float(np.quantile(
            np.abs(detail_difference.compressed()), .99
        )),
        1e-6,
    )

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    for axis in axes.flat:
        axis.axis("off")

    axes[0, 0].imshow(
        image_a, cmap="gray", vmin=low, vmax=high
    )
    axes[0, 0].set_title(
        f"A: {pair['image_a']}\n{pair['split_a']}"
    )

    axes[0, 1].imshow(
        image_b, cmap="gray", vmin=low, vmax=high
    )
    axes[0, 1].set_title(
        f"B: {pair['image_b']}\n{pair['split_b']}"
    )

    axes[0, 2].imshow(
        aligned_b, cmap="gray", vmin=low, vmax=high
    )
    axes[0, 2].set_title(
        (
            f"B after translation: "
            f"dy={pair['shift_y']:.2f}, dx={pair['shift_x']:.2f}"
        )
        if registered else
        "Registration unavailable/rejected — B remains unaligned"
    )

    raw_artist = axes[1, 0].imshow(
        raw_difference, cmap="magma",
        vmin=0, vmax=raw_limit,
    )
    axes[1, 0].set_title("Absolute raw difference in valid interior")
    fig.colorbar(
        raw_artist, ax=axes[1, 0],
        fraction=.046, pad=.04,
    )

    detail_artist = axes[1, 1].imshow(
        detail_difference, cmap="coolwarm",
        vmin=-detail_limit, vmax=detail_limit,
    )
    axes[1, 1].set_title("Signed high-pass difference")
    fig.colorbar(
        detail_artist, ax=axes[1, 1],
        fraction=.046, pad=.04,
    )

    # Red/cyan disagreement view after per-image interior
    # contrast normalization; diagnostic display only.
    def normalized_view(image):
        values = image[valid]
        qlow, qhigh = np.quantile(values, [.01, .99])
        return np.clip(
            (image - qlow) / max(qhigh - qlow, 1e-6),
            0, 1,
        )

    a_view = normalized_view(image_a)
    b_view = normalized_view(aligned_b)

    comparison_rgb = np.stack(
        [a_view, b_view, b_view], axis=-1
    )
    comparison_rgb[~valid] = 0

    axes[1, 2].imshow(comparison_rgb)
    axes[1, 2].set_title(
        "Red: A | Cyan: B | Gray: similar local appearance"
    )

    fig.suptitle(
        f"Pair {pair_id} | gap={pair['time_gap_days']:.3f} days\n"
        f"Detail NCC={pair.get('registered_detail_ncc', np.nan):.4f} | "
        f"Interior SSIM="
        f"{pair.get('registered_interior_ssim', np.nan):.4f}\n"
        "Thumbnail diagnostics support review; they do not decide it",
        fontsize=12,
    )

    fig.tight_layout()
    a20_finish_figure(
        fig,
        f"08_duplicate_diagnostics_{pair_id}.png",
    )


_top_duplicate_cases = near_pair_df.sort_values(
    ["cross_train_validation", "registered_detail_ncc"],
    ascending=[False, False],
    na_position="last",
).head(4)

for pair_id in _top_duplicate_cases["pair_id"]:
    a20_duplicate_diagnostic_panels(pair_id)

if near_pair_df.empty:
    print("No candidate duplicate pairs were identified.")

print(
    "Inspect another pair using:\n"
    "a20_duplicate_diagnostic_panels('REAL_PAIR_ID')\n"
    "Use a20_show_pair(..., native=True) for full-resolution review."
)

In [ ]:
# -------------------------------------------------------------
# A. Confirmed observation clusters
# Uses the cluster manifest generated by Cell 20.
# Rerun Cell 20 after changing pair-review decisions.
# -------------------------------------------------------------
_cluster_sizes_viz = (
    cluster_manifest_df.groupby("cluster_id").size()
)
_multi_clusters_viz = _cluster_sizes_viz[
    _cluster_sizes_viz > 1
].index

_cluster_flags_viz = (
    cluster_manifest_df.groupby("cluster_id")["split"]
    .agg(lambda values: {"train", "validation"} <= set(values))
)

_ranked_clusters_viz = pd.DataFrame({
    "size": _cluster_sizes_viz,
    "cross_partition": _cluster_flags_viz,
}).loc[_multi_clusters_viz].sort_values(
    ["cross_partition", "size"],
    ascending=[False, False],
)

_selected_clusters_viz = (
    _ranked_clusters_viz.head(12).index.tolist()
)

if _selected_clusters_viz:
    fig, axis = plt.subplots(
        figsize=(16, max(4, len(_selected_clusters_viz) * 0.5))
    )

    tick_labels = []

    for y, cluster_id in enumerate(_selected_clusters_viz):
        members = cluster_manifest_df[
            cluster_manifest_df["cluster_id"] == cluster_id
        ].sort_values("metadata_time")

        axis.plot(
            members["metadata_time"],
            np.full(len(members), y),
            color="#aaaaaa",
            linewidth=2,
            zorder=1,
        )

        for partition, color in _partition_colors.items():
            rows = members[members["split"] == partition]
            axis.scatter(
                rows["metadata_time"],
                np.full(len(rows), y),
                color=color,
                s=40,
                zorder=2,
            )

        crosses = bool(
            _ranked_clusters_viz.at[
                cluster_id, "cross_partition"
            ]
        )
        tick_labels.append(
            f"{cluster_id} ({len(members)})"
            + (" — CROSS-SPLIT" if crosses else "")
        )

    axis.set_yticks(
        range(len(_selected_clusters_viz)),
        tick_labels,
    )
    axis.set_xlabel("Observation timestamp")
    axis.set_title(
        "Confirmed observation clusters\n"
        "Blue: training | Orange: validation | Gray: excluded"
    )
    axis.grid(axis="x", alpha=.2)

    fig.tight_layout()
    a20_finish_figure(fig, "09_confirmed_cluster_timeline.png")
else:
    print(
        "No multi-image confirmed clusters to display. "
        "Unreviewed candidate pairs are not confirmed clusters."
    )

# -------------------------------------------------------------
# B. Feature replay: numerical differences and tolerance failures
# -------------------------------------------------------------
_replay_viz = feature_replay_df[
    feature_replay_df["feature"].isin(FEATURE_NAMES)
].copy()

if len(_replay_viz):
    difference_table = _replay_viz.pivot(
        index="image_id",
        columns="feature",
        values="maximum_absolute_difference",
    ).reindex(columns=FEATURE_NAMES)

    passed_table = _replay_viz.pivot(
        index="image_id",
        columns="feature",
        values="passed",
    ).reindex(
        index=difference_table.index,
        columns=FEATURE_NAMES,
    )

    differences = difference_table.to_numpy(dtype=float)

    # Log display distinguishes small discrepancies.
    # Values at or below 1e-12 share the display floor.
    log_differences = np.log10(
        np.maximum(differences, 1e-12)
    )
    log_differences = np.ma.masked_invalid(log_differences)

    failures = np.where(
        passed_table.isna().to_numpy(),
        np.nan,
        (~passed_table.fillna(False).astype(bool)).to_numpy(
            dtype=float
        ),
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(18, max(4, len(difference_table) * 0.65)),
    )

    left = axes[0].imshow(
        log_differences,
        aspect="auto",
        cmap="viridis",
        interpolation="nearest",
    )
    fig.colorbar(
        left, ax=axes[0],
        label="log10 maximum absolute difference; floor = 1e-12",
        fraction=.035, pad=.03,
    )
    axes[0].set_title("Cache replay numerical differences")

    from matplotlib.colors import ListedColormap

    right = axes[1].imshow(
        np.ma.masked_invalid(failures),
        aspect="auto",
        cmap=ListedColormap(["#a8d5ba", "#c62828"]),
        vmin=0, vmax=1,
        interpolation="nearest",
    )
    bar = fig.colorbar(
        right, ax=axes[1],
        ticks=[0, 1], fraction=.035, pad=.03,
    )
    bar.ax.set_yticklabels(["Passed", "Failed"])

    axes[1].set_title(
        "Actual np.allclose decisions\n"
        "Uses configured absolute and relative tolerances"
    )

    for axis in axes:
        axis.set_xticks(
            range(len(FEATURE_NAMES)),
            FEATURE_NAMES,
            rotation=65,
            ha="right",
        )
        axis.set_yticks(
            range(len(difference_table)),
            difference_table.index,
        )
        axis.set_ylabel("Replayed training image")

    for i in range(len(difference_table)):
        for j in range(len(FEATURE_NAMES)):
            value = differences[i, j]
            if np.isfinite(value):
                axes[0].text(
                    j, i, f"{value:.1e}",
                    ha="center", va="center",
                    fontsize=7,
                    color=(
                        "white"
                        if log_differences[i, j] < -5
                        else "black"
                    ),
                )

    fig.tight_layout()
    a20_finish_figure(fig, "10_feature_cache_replay_heatmaps.png")
else:
    print(
        "No feature-level replay results to plot. "
        "Inspect feature_replay_audit.csv for image-level failures."
    )

print("Visual audit figures saved:", A20_FIGURE_DIR)

## 21. Official Metric Implementation and Parity Testing

### 21.1 Evaluation contract

The organizer's self-evaluation notebook evaluates predictions against
each annotator–image record separately.

Ground-truth identifiers:
    <annotator_id>-<physical_image_id>_<instance_suffix>

Prediction identifiers:
    <physical_image_id>_<instance_suffix>

The same prediction set is evaluated against each available annotation
record for its physical image.

Published version 6 behavior:

- Mask representation: compressed COCO RLE.
- Default decoded image dimensions: 2048 × 2048.
- Qualifying relation: IoU strictly greater than 0.5.
- Every qualifying GT–prediction pair contributes to the TP-IoU list.
- A prediction is an FP when it has no qualifying GT relation.
- A GT instance is an FN when it has no qualifying prediction relation.
- Counts and qualifying IoUs are pooled across annotator–image records.
- The final score is not the mean of per-image PQ values.
- A zero score denominator produces PQ = 0.
- Structural relation plots use IoU > 0, not the PQ threshold.

The published dataframe adapter iterates over entries represented in
the ground-truth dataframe. Its behavior on empty tables or prediction
IDs outside those entries must be characterized rather than silently
assumed to represent a complete submission validator.

### Scope of parity

We separately verify:

1. The pinned reference implementation.
2. Independent overlap calculations.
3. PQ accounting and aggregation.
4. RLE serialization.
5. Identifier routing and repeated annotator evaluation.
6. Real-image mask tests.
7. Visual matching and structural diagnostics.

The self-evaluation notebook accepts RLE masks. It does not establish
which polygon-to-mask rasterization convention generated the official
ground-truth RLEs.

Therefore, parity with this scorer does not by itself certify that our
current Pillow polygon rasterization matches the hidden evaluator.

In [ ]:
import ast
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from pycocotools import mask as mask_util
except ImportError as error:
    raise ImportError(
        "Step 21 requires torch and pycocotools. "
        "Install them in the Kaggle environment, or attach compatible "
        "offline wheels if Internet is disabled. "
        "Do not substitute a different RLE implementation silently."
    ) from error

S21_DIR = Path(OUTPUT_ROOT) / "step21_metric_parity"
S21_DIR.mkdir(parents=True, exist_ok=True)

S21_FIG_DIR = S21_DIR / "figures"
S21_FIG_DIR.mkdir(exist_ok=True)

S21_ATOL = 2e-7

S21_REFERENCE = {
    "author": "Azim Ahmadzadeh",
    "notebook": "Self_Evaluation_Notebook",
    "version": 6,
    "url": (
        "https://www.kaggle.com/code/"
        "azimahmadzadeh/self-evaluation-notebook"
    ),
    "retrieval_endpoint": (
        "https://www.kaggle.com/api/v1/kernels/pull/"
        "azimahmadzadeh/self-evaluation-notebook"
    ),
    "license": "Apache-2.0, as displayed on the notebook page",
    "parity_atol": S21_ATOL,
}

def s21_save(frame, filename):
    frame.to_csv(S21_DIR / filename, index=False)

def s21_finish(fig, filename):
    fig.savefig(
        S21_FIG_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
    plt.close(fig)

print("PyTorch:", torch.__version__)
print("Reference notebook version:", S21_REFERENCE["version"])
print("Results:", S21_DIR)

### 21.2 Pinned official reference

The following functions originate from the organizer's version 6
self-evaluation notebook.

Only docstrings, comments and formatting have been removed.
The reference executes in a separate namespace so that its generic
function names do not overwrite our notebook functions.

Syntax fingerprints are calculated after removing function docstrings.
They preserve executable expressions, function signatures and annotations.

We do not modify the official scorer to impose a preferred matching
algorithm or alternative empty-case convention.

In [ ]:
S21_HELPER_SOURCE = r'''
def get_overlap_matrices(gt_layers: torch.Tensor, pred_layers: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    n_gt, height, width = gt_layers.shape
    n_pred = pred_layers.shape[0]
    gt_flat = gt_layers.reshape(n_gt, height * width)
    pred_flat = pred_layers.reshape(n_pred, height * width)
    intersection = torch.matmul(gt_flat, pred_flat.t())
    gt_areas = gt_flat.sum(dim=1).view(-1, 1)
    pred_areas = pred_flat.sum(dim=1).view(1, -1)
    union = gt_areas + pred_areas - intersection
    iou_matrix = torch.where(union == 0, torch.tensor(0.0, device=gt_layers.device), intersection / union)
    dice_matrix = torch.where(union == 0, torch.tensor(0.0, device=gt_layers.device), 2 * intersection / (gt_areas + pred_areas))
    return (iou_matrix, dice_matrix)

def fp_count_hit(hit_matrix: torch.Tensor) -> int:
    gt_matches_per_pred = hit_matrix.sum(dim=0)
    return (gt_matches_per_pred == 0).sum().item()

def fn_count_hit(hit_matrix: torch.Tensor) -> int:
    pred_matches_per_gt = hit_matrix.sum(dim=1)
    return (pred_matches_per_gt == 0).sum().item()

def rles_to_layers(rles: list[str], height: int=2048, width: int=2048) -> np.ndarray:
    if not rles:
        return np.zeros((0, height, width), dtype=np.float32)
    rle_dicts = [{'size': [height, width], 'counts': rle} for rle in rles]
    masks = mask_util.decode(rle_dicts)
    return masks.transpose(2, 0, 1).astype(np.float32)

def process_entry(annotator_image: str, gt_annotator_image_ids: pd.Series, pred_image_ids: pd.Series, gt_df: pd.DataFrame, pred_df: pd.DataFrame) -> tuple[torch.Tensor, torch.Tensor, int, int]:
    image_id = annotator_image.split('-', maxsplit=1)[1]
    gt_rles = gt_df.loc[gt_annotator_image_ids == annotator_image, 'segmentation_rle'].tolist()
    gt_layers = torch.from_numpy(rles_to_layers(gt_rles))
    pred_rles = pred_df.loc[pred_image_ids == image_id, 'segmentation_rle'].tolist()
    pred_layers = torch.from_numpy(rles_to_layers(pred_rles))
    n_gt = len(gt_rles)
    n_pred = len(pred_rles)
    iou_matrix, dice_matrix = get_overlap_matrices(gt_layers, pred_layers)
    return (iou_matrix, dice_matrix, n_gt, n_pred)

def get_overlap_df(gt_df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    gt_annotator_image_ids = gt_df['filament_id'].str.split('_', n=1).str[0]
    pred_image_ids = pred_df['filament_id'].str.split('_', n=1).str[0]
    annotator_images = gt_annotator_image_ids.unique()
    overlap_df = pd.DataFrame({'annotator_image': annotator_images})
    results = overlap_df['annotator_image'].apply(process_entry, gt_annotator_image_ids=gt_annotator_image_ids, pred_image_ids=pred_image_ids, gt_df=gt_df, pred_df=pred_df)
    overlap_df['iou_matrix'], overlap_df['dice_matrix'], overlap_df['n_gt'], overlap_df['n_pred'] = zip(*results)
    return overlap_df
'''

S21_SCORE_SOURCE = r'''
def get_pq_score(overlap_df: pd.DataFrame) -> float:
    iou_threshold: float = 0.5
    tp_iou_scores: list[float] = []
    fp_count = 0
    fn_count = 0
    for row in overlap_df.itertuples(index=False):
        iou_matrix = row.iou_matrix
        n_gt = row.n_gt
        n_pred = row.n_pred
        if n_gt == 0:
            fp_count += n_pred
            continue
        if n_pred == 0:
            fn_count += n_gt
            continue
        hit_matrix = iou_matrix > iou_threshold
        tp_iou_scores.extend(iou_matrix[hit_matrix].tolist())
        fp_count += fp_count_hit(hit_matrix)
        fn_count += fn_count_hit(hit_matrix)
    tp_count = len(tp_iou_scores)
    denominator = tp_count + 0.5 * fp_count + 0.5 * fn_count
    if denominator > 0:
        pq_score = sum(tp_iou_scores) / denominator
    else:
        pq_score = 0.0
    return pq_score
'''

def s21_normalized_source_hash(source):
    tree = ast.parse(source)

    # Remove function docstrings, preserving executable code.
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if (
                node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Constant)
                and isinstance(node.body[0].value.value, str)
            ):
                node.body.pop(0)

    # Explicit serialization avoids version-dependent ast.dump()
    # formatting, including omission of empty lists in Python 3.13.
    def stable_dump(node):
        if isinstance(node, ast.AST):
            fields = []

            for name, value in ast.iter_fields(node):
                if value is None:
                    continue

                fields.append(
                    f"{name}={stable_dump(value)}"
                )

            return (
                f"{type(node).__name__}("
                + ", ".join(fields)
                + ")"
            )

        if isinstance(node, list):
            return (
                "["
                + ", ".join(stable_dump(value) for value in node)
                + "]"
            )

        return repr(node)

    normalized = stable_dump(tree)

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

_expected_hashes = {
    "helpers": (
        "668e2ba74696127c4dadbaed16657be638"
        "6278c6313e6307b73b406053f20b02"
    ),
    "score": (
        "19d3621572b8f7a733d75ca13d21ab6df"
        "b85c865a1c391b22f3ea788b292868c"
    ),
}

_actual_hashes = {
    "helpers": s21_normalized_source_hash(S21_HELPER_SOURCE),
    "score": s21_normalized_source_hash(S21_SCORE_SOURCE),
}

assert _actual_hashes == _expected_hashes, (
    "The pasted reference differs from the inspected official source."
)

s21_reference_namespace = {
    "torch": torch,
    "np": np,
    "pd": pd,
    "mask_util": mask_util,
}

exec(S21_HELPER_SOURCE, s21_reference_namespace)
exec(S21_SCORE_SOURCE, s21_reference_namespace)

S21_REFERENCE["normalized_source_hashes"] = _actual_hashes

with open(S21_DIR / "reference_provenance.json", "w") as handle:
    json.dump(S21_REFERENCE, handle, indent=2)

(S21_DIR / "official_reference_compact.py").write_text(
    "# Reference: Azim Ahmadzadeh, Self_Evaluation_Notebook, v6\n"
    "# Original notebook license: Apache-2.0\n"
    "# Docstrings/comments removed; executable syntax verified.\n\n"
    + S21_HELPER_SOURCE + "\n" + S21_SCORE_SOURCE,
    encoding="utf-8",
)

print("Pinned official reference syntax verified.")

### 21.3 Independent overlap and accounting implementation

Our independent implementation uses COCO RLE intersection/union
operations rather than the reference's dense PyTorch matrix product.

Mask stacks retain independent instances, including overlaps.
No conversion to a single integer-label image is performed.

The implementation returns:

- Pairwise IoU and Dice matrices.
- Qualifying-pair count and summed IoU.
- Unmatched prediction and GT counts.
- Per-instance relation degrees.
- Foreground-union Dice as a separate diagnostic.

Two distinct relation thresholds are retained:

- IoU > 0.5: official PQ accounting.
- IoU > 0: structural contact diagnostics.

Nonzero contact does not establish a successful instance match.

In [ ]:
def s21_validate_stack(stack):
    array = np.asarray(stack)

    if array.ndim != 3:
        raise ValueError("Expected mask stack with shape (N,H,W).")

    if not np.isin(array, [0, 1]).all():
        raise ValueError("Mask stack must contain only binary values.")

    return array.astype(bool, copy=False)

def s21_encode(mask):
    encoded = mask_util.encode(
        np.asfortranarray(np.asarray(mask, dtype=np.uint8))
    )
    return {
        "size": list(encoded["size"]),
        "counts": encoded["counts"].decode("ascii"),
    }

def s21_decode(rle):
    return mask_util.decode({
        "size": list(rle["size"]),
        "counts": rle["counts"].encode("ascii"),
    }).astype(bool)

def s21_overlap(gt, pred):
    gt = s21_validate_stack(gt)
    pred = s21_validate_stack(pred)

    if gt.shape[1:] != pred.shape[1:]:
        raise ValueError("GT and prediction spatial dimensions differ.")

    ng, npred = len(gt), len(pred)

    if ng == 0 or npred == 0:
        zeros = np.zeros((ng, npred), dtype=np.float64)
        return zeros, zeros.copy()

    gt_rles = [s21_encode(mask) for mask in gt]
    pred_rles = [s21_encode(mask) for mask in pred]

    # COCO API returns (detections, GT); transpose to (GT, predictions).
    iou = np.asarray(
        mask_util.iou(pred_rles, gt_rles, [0] * ng),
        dtype=np.float64,
    ).T

    dice = np.divide(
        2 * iou,
        1 + iou,
        out=np.zeros_like(iou),
        where=(1 + iou) != 0,
    )

    return iou, dice

def s21_account(iou):
    iou = np.asarray(iou, dtype=np.float64)

    if iou.ndim != 2:
        raise ValueError("IoU matrix must have shape (GT,predictions).")

    if not np.isfinite(iou).all():
        raise ValueError("IoU matrix contains nonfinite values.")

    if ((iou < 0) | (iou > 1)).any():
        raise ValueError("IoUs must lie in [0,1].")

    hits = iou > 0.5
    tp_pairs = int(hits.sum())
    fp = int((hits.sum(axis=0) == 0).sum())
    fn = int((hits.sum(axis=1) == 0).sum())
    sum_iou = float(iou[hits].sum())

    denominator = tp_pairs + 0.5 * fp + 0.5 * fn

    return {
        "n_gt": iou.shape[0],
        "n_pred": iou.shape[1],
        "tp_pairs": tp_pairs,
        "fp": fp,
        "fn": fn,
        "sum_iou": sum_iou,
        "denominator": denominator,
        "pq": sum_iou / denominator if denominator else 0.0,
        "sq": sum_iou / tp_pairs if tp_pairs else np.nan,
        "rq": tp_pairs / denominator if denominator else 0.0,
    }

def s21_pooled(accounts):
    summed = {
        key: sum(float(row[key]) for row in accounts)
        for key in ["tp_pairs", "fp", "fn", "sum_iou"]
    }

    denominator = (
        summed["tp_pairs"]
        + 0.5 * summed["fp"]
        + 0.5 * summed["fn"]
    )

    summed["denominator"] = denominator
    summed["pq"] = (
        summed["sum_iou"] / denominator if denominator else 0.0
    )

    return summed

def s21_reference_row(gt, pred, name):
    iou, dice = s21_reference_namespace["get_overlap_matrices"](
        torch.from_numpy(
            np.ascontiguousarray(gt, dtype=np.float32)
        ),
        torch.from_numpy(
            np.ascontiguousarray(pred, dtype=np.float32)
        ),
    )
    return {
        "annotator_image": name,
        "iou_matrix": iou,
        "dice_matrix": dice,
        "n_gt": len(gt),
        "n_pred": len(pred),
    }

def s21_relation_ledger(iou, case_name):
    iou = np.asarray(iou)
    records = []

    for side, axis in [("ground_truth", 1), ("prediction", 0)]:
        for index, (contact_degree, pq_degree) in enumerate(zip(
            (iou > 0).sum(axis=axis),
            (iou > 0.5).sum(axis=axis),
        )):
            records.append({
                "case": case_name,
                "side": side,
                "instance_index": index,
                "nonzero_overlap_degree": int(contact_degree),
                "pq_relation_degree": int(pq_degree),
            })

    return records

def s21_union_dice(gt, pred):
    shape = gt.shape[1:]

    gt_union = (
        gt.any(axis=0) if len(gt) else np.zeros(shape, dtype=bool)
    )
    pred_union = (
        pred.any(axis=0) if len(pred) else np.zeros(shape, dtype=bool)
    )

    denominator = int(gt_union.sum()) + int(pred_union.sum())

    # Diagnostic convention only; official PQ still returns zero
    # for an explicitly represented empty–empty overlap record.
    return (
        2 * int((gt_union & pred_union).sum()) / denominator
        if denominator else 1.0
    )

print("Independent scorer ready.")

### 21.4 Synthetic fixtures and expected outcomes

Fixtures isolate evaluator behavior from model quality.

The tests cover:

- Perfect predictions.
- Missing and false-positive instances.
- Empty sides and explicitly represented empty–empty records.
- IoU below, exactly at and above 0.5.
- Equal and unequal splits and merges.
- Duplicate predictions.
- Overlapping ground-truth instances.
- Disconnected parts belonging to one instance.
- Boundary shifts.
- Zero-area mask layers.
- Instance-order invariance.
- Pooled versus image-macro aggregation.

Some fixtures intentionally violate sensible production-output
constraints. Their purpose is to expose scorer semantics, not recommend
those predictions.

In particular, the reference can credit repeated qualifying relations.
We preserve this in parity tests and separately avoid duplicate or
otherwise invalid production outputs.

In [ ]:
S21_SHAPE = (64, 64)

def s21_mask(indices):
    mask = np.zeros(S21_SHAPE, dtype=bool)
    mask.ravel()[np.asarray(list(indices), dtype=int)] = True
    return mask

def s21_stack(masks):
    return (
        np.stack(masks).astype(bool)
        if len(masks)
        else np.zeros((0, *S21_SHAPE), dtype=bool)
    )

_a = s21_mask(range(100))
_b = s21_mask(range(200, 300))
_other = s21_mask(range(400, 500))
_half_a = s21_mask(range(50))
_half_b = s21_mask(range(50, 100))
_sixty = s21_mask(range(60))
_forty = s21_mask(range(60, 100))
_zero = np.zeros(S21_SHAPE, dtype=bool)

_multipart = np.zeros(S21_SHAPE, dtype=bool)
_multipart[8:12, 8:12] = True
_multipart[35:39, 35:39] = True

_rectangle = np.zeros(S21_SHAPE, dtype=bool)
_rectangle[15:25, 15:25] = True
_shifted = np.zeros_like(_rectangle)
_shifted[15:25, 16:26] = True

s21_fixtures = {}

def s21_add(name, gt, pred, expected_pq):
    s21_fixtures[name] = {
        "gt": s21_stack(gt),
        "pred": s21_stack(pred),
        "expected_pq": expected_pq,
    }

s21_add("perfect", [_a, _b], [_a, _b], 1.0)
s21_add("all_missing", [_a, _b], [], 0.0)
s21_add("false_positive_only", [], [_a], 0.0)
s21_add("empty_both_explicit_record", [], [], 0.0)

s21_add("iou_below_half", [_a], [s21_mask(range(49))], 0.0)
s21_add("iou_exactly_half", [_a], [s21_mask(range(50))], 0.0)
s21_add("iou_above_half", [_a], [s21_mask(range(51))], 0.51)

s21_add("equal_split", [_a], [_half_a, _half_b], 0.0)
s21_add("unequal_split", [_a], [_sixty, _forty], 0.4)

s21_add("equal_merge", [_half_a, _half_b], [_a], 0.0)
s21_add("unequal_merge", [_sixty, _forty], [_a], 0.4)

s21_add("duplicate_prediction_reference_behavior", [_a], [_a, _a], 1.0)
s21_add("overlapping_gt_reference_behavior", [_a, _a], [_a], 1.0)

s21_add("perfect_plus_false_positive", [_a], [_a, _other], 2 / 3)
s21_add("perfect_plus_missing_gt", [_a, _b], [_a], 2 / 3)
s21_add("multipart_one_instance", [_multipart], [_multipart], 1.0)
s21_add("boundary_shift", [_rectangle], [_shifted], 90 / 110)
s21_add("zero_area_layers", [_zero], [_zero], 0.0)

s21_test_records = []
s21_reference_rows = []
s21_accounts = []
s21_ledger_records = []

for name, fixture in s21_fixtures.items():
    gt, pred = fixture["gt"], fixture["pred"]

    local_iou, local_dice = s21_overlap(gt, pred)
    reference_row = s21_reference_row(gt, pred, name)

    ref_iou = reference_row["iou_matrix"].numpy()
    ref_dice = reference_row["dice_matrix"].numpy()

    official_pq = s21_reference_namespace["get_pq_score"](
        pd.DataFrame([reference_row])
    )

    account = s21_account(local_iou)

    iou_ok = np.allclose(
        local_iou, ref_iou, rtol=0, atol=S21_ATOL
    )
    dice_ok = np.allclose(
        local_dice, ref_dice, rtol=0, atol=S21_ATOL
    )
    score_ok = np.isclose(
        account["pq"], official_pq, rtol=0, atol=S21_ATOL
    )
    expectation_ok = np.isclose(
        official_pq, fixture["expected_pq"],
        rtol=0, atol=S21_ATOL,
    )

    # Reverse both instance orders; score must remain invariant.
    reversed_iou, _ = s21_overlap(gt[::-1], pred[::-1])
    order_ok = np.isclose(
        s21_account(reversed_iou)["pq"],
        account["pq"],
        rtol=0, atol=S21_ATOL,
    )

    s21_test_records.append({
        "case": name,
        **account,
        "official_pq": official_pq,
        "expected_pq": fixture["expected_pq"],
        "union_dice_diagnostic": s21_union_dice(gt, pred),
        "iou_parity": bool(iou_ok),
        "dice_parity": bool(dice_ok),
        "pq_parity": bool(score_ok),
        "expected_behavior": bool(expectation_ok),
        "instance_order_invariance": bool(order_ok),
        "passed": bool(
            iou_ok and dice_ok and score_ok
            and expectation_ok and order_ok
        ),
    })

    s21_reference_rows.append(reference_row)
    s21_accounts.append(account)
    s21_ledger_records.extend(
        s21_relation_ledger(local_iou, name)
    )

s21_tests_df = pd.DataFrame(s21_test_records)
s21_ledger_df = pd.DataFrame(s21_ledger_records)

s21_save(s21_tests_df, "synthetic_parity_tests.csv")
s21_save(s21_ledger_df, "synthetic_relation_ledger.csv")

_synthetic_official_pooled = (
    s21_reference_namespace["get_pq_score"](
        pd.DataFrame(s21_reference_rows)
    )
)
_synthetic_local_pooled = s21_pooled(s21_accounts)["pq"]

assert np.isclose(
    _synthetic_official_pooled,
    _synthetic_local_pooled,
    rtol=0, atol=S21_ATOL,
), "Pooled aggregation parity failed."

display(s21_tests_df)

print("Official pooled synthetic PQ:", _synthetic_official_pooled)
print("Local pooled synthetic PQ:", _synthetic_local_pooled)
print(
    "Image-macro synthetic PQ:",
    s21_tests_df["local_pq"].mean()
    if "local_pq" in s21_tests_df
    else s21_tests_df["pq"].mean(),
)

assert s21_tests_df["passed"].all(), (
    "At least one synthetic parity test failed."
)

### 21.5 Synthetic matching visualizations

For each selected fixture, show:

- Ground-truth instance memberships.
- Predicted instance memberships.
- Pairwise IoU.
- Official PQ and foreground-union Dice.

Independent instances are drawn as separate contours.
They are not merged into an integer-label mask for visualization.

Highlighted IoU-matrix entries indicate qualifying official relations.
Multiple qualifying entries in a row or column remain visible.

In [ ]:
def s21_draw_instances(axis, masks, title):
    union = (
        masks.any(axis=0)
        if len(masks)
        else np.zeros(masks.shape[1:], dtype=bool)
    )

    axis.imshow(union, cmap="gray", vmin=0, vmax=1)

    for index, mask in enumerate(masks):
        if mask.any():
            color = plt.cm.tab10(index % 10)

            # Padding exposes contours for masks touching image edges.
            padded = np.pad(mask.astype(float), 1)
            axis.contour(
                padded,
                levels=[0.5],
                colors=[color],
                linewidths=1.3,
                extent=(-1.5, mask.shape[1] + .5,
                        mask.shape[0] + .5, -1.5),
            )

            ys, xs = np.nonzero(mask)
            axis.text(
                float(xs.mean()), float(ys.mean()),
                str(index), color=color,
                fontsize=10, fontweight="bold",
            )

    axis.set_title(title)
    axis.axis("off")

def s21_show_case(name):
    fixture = s21_fixtures[name]
    gt, pred = fixture["gt"], fixture["pred"]
    iou, _ = s21_overlap(gt, pred)
    account = s21_account(iou)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    s21_draw_instances(axes[0], gt, f"GT: {len(gt)} instances")
    s21_draw_instances(axes[1], pred, f"Prediction: {len(pred)} instances")

    if iou.size:
        artist = axes[2].imshow(
            iou, cmap="viridis", vmin=0, vmax=1,
            aspect="auto",
        )

        for i in range(iou.shape[0]):
            for j in range(iou.shape[1]):
                axes[2].text(
                    j, i,
                    f"{iou[i, j]:.3f}"
                    + ("\nPQ hit" if iou[i, j] > .5 else ""),
                    ha="center", va="center",
                    color="white" if iou[i, j] < .65 else "black",
                    fontsize=9,
                )

        axes[2].set_xticks(range(len(pred)))
        axes[2].set_yticks(range(len(gt)))
        fig.colorbar(artist, ax=axes[2], fraction=.046, pad=.04)
    else:
        axes[2].text(
            .5, .5, f"Empty matrix: {iou.shape}",
            ha="center", va="center",
            transform=axes[2].transAxes,
        )

    axes[2].set_title("IoU matrix — strict > 0.5")
    axes[2].set_xlabel("Prediction index")
    axes[2].set_ylabel("GT index")

    fig.suptitle(
        f"{name} | PQ={account['pq']:.4f} | "
        f"union Dice={s21_union_dice(gt, pred):.4f}\n"
        f"TP pairs={account['tp_pairs']}, "
        f"FP={account['fp']}, FN={account['fn']}",
        fontsize=12,
    )

    fig.tight_layout()
    s21_finish(fig, f"fixture_{name}.png")

for name in [
    "perfect",
    "iou_exactly_half",
    "iou_above_half",
    "equal_split",
    "equal_merge",
    "duplicate_prediction_reference_behavior",
    "multipart_one_instance",
    "boundary_shift",
]:
    s21_show_case(name)

comparison = s21_tests_df.set_index("case")[
    ["official_pq", "union_dice_diagnostic"]
]

fig, axis = plt.subplots(figsize=(15, 6))
comparison.plot.bar(ax=axis, width=.8)

axis.set_ylim(0, 1.08)
axis.set_ylabel("Score")
axis.set_title(
    "Instance PQ versus foreground-union Dice\n"
    "A perfect foreground union can conceal failed instance reconstruction"
)
axis.grid(axis="y", alpha=.2)
axis.tick_params(axis="x", labelrotation=70)

fig.tight_layout()
s21_finish(fig, "synthetic_pq_vs_union_dice.png")

### 21.6 Physical-image identity and annotation routing

JSON image IDs identify annotator–image entries, not necessarily distinct
physical observations.

We independently derive:

- Annotation-entry ID from JSON image ID.
- Annotator ID from the prefix before the first hyphen.
- Physical-image ID from the remaining suffix.
- Physical-image filename stem.

The physical-image suffix must agree with the filename stem.

Repeated annotation entries for one physical image are retained as
separate evaluation records.

A physical image appearing in both training and validation is a blocking
partition failure.

Repeated physical files within one partition are legitimate when they
represent separate annotators. They should be distinguished from
accidental duplicated observations in the Step 20 review.

In [ ]:
s21_train_entries = set(train_df["image_id"].astype(str))
s21_valid_entries = set(valid_df["image_id"].astype(str))

_s21_registry_records = []

for info in annotations_data["images"]:
    entry_id = str(info["id"])

    if "-" not in entry_id:
        raise ValueError(
            f"Cannot parse annotator–image ID: {entry_id}"
        )

    annotator_id, physical_id = entry_id.split("-", 1)
    filename_stem = Path(info["file_name"]).stem

    if physical_id != filename_stem:
        raise ValueError(
            f"Identifier/filename mismatch: "
            f"{entry_id} versus {info['file_name']}"
        )

    if "_" in physical_id:
        raise ValueError(
            "Physical image ID contains an underscore, which conflicts "
            "with the published scorer's first-underscore parser."
        )

    partition = (
        "train" if entry_id in s21_train_entries
        else "validation" if entry_id in s21_valid_entries
        else "excluded"
    )

    _s21_registry_records.append({
        "entry_id": entry_id,
        "annotator_id": annotator_id,
        "physical_image_id": physical_id,
        "file_name": info["file_name"],
        "height": int(info["height"]),
        "width": int(info["width"]),
        "split": partition,
    })

s21_registry_df = pd.DataFrame(_s21_registry_records)

s21_physical_summary = (
    s21_registry_df.groupby("physical_image_id")
    .agg(
        annotation_entries=("entry_id", "size"),
        annotators=("annotator_id", "nunique"),
        partitions=("split", lambda x: "|".join(sorted(set(x)))),
        filenames=("file_name", "nunique"),
    )
    .reset_index()
)

_cross_partition_physical = s21_physical_summary[
    s21_physical_summary["partitions"].str.contains("train")
    & s21_physical_summary["partitions"].str.contains("validation")
]

s21_save(s21_registry_df, "annotator_physical_image_registry.csv")
s21_save(s21_physical_summary, "physical_image_summary.csv")

print("Annotation entries:", len(s21_registry_df))
print("Physical images:", len(s21_physical_summary))
print(
    "Physical images with multiple annotation entries:",
    int((s21_physical_summary["annotation_entries"] > 1).sum()),
)
print(
    "Physical images crossing train/validation:",
    len(_cross_partition_physical),
)

display(s21_physical_summary.head(15))

assert _cross_partition_physical.empty, (
    "A physical image crosses training and validation."
)

fig, axis = plt.subplots(figsize=(9, 4))

counts = (
    s21_physical_summary["annotation_entries"]
    .value_counts()
    .sort_index()
)

axis.bar(counts.index.astype(str), counts.values, color="#496b8a")
axis.set_xlabel("Annotation entries per physical image")
axis.set_ylabel("Physical images")
axis.set_title("Annotation multiplicity — not accidental file duplication")
axis.grid(axis="y", alpha=.2)

fig.tight_layout()
s21_finish(fig, "annotation_entries_per_physical_image.png")

### 21.7 Real-image RLE and dataframe-adapter parity

Select up to three training physical images, prioritizing those with
multiple annotation entries.

For each physical image:

1. Decode each annotation independently using the current notebook
   polygon rasterizer.
2. Serialize each mask as compressed COCO RLE.
3. Verify exact RLE round-trip reconstruction.
4. Use the first annotator's masks as controlled test predictions.
5. Evaluate those predictions against every annotation entry for that
   physical image using both implementations.
6. Compare overlap matrices and pooled PQ.

These are evaluator fixtures, not model predictions or validation results.

Predictions copied from one annotator need not be perfect against other
annotators. That behavior is expected and is made visible.

Only 2048 × 2048 cases enter the unmodified published dataframe adapter
because its decoder defaults to those dimensions.

This experiment preserves the current polygon convention. It does not
claim that this convention produced the official ground-truth RLEs.

In [ ]:
_s21_ann_by_entry = {}
for info in annotations_data["images"]:
    _s21_ann_by_entry[str(info["id"])] = []

for ann in annotations_data["annotations"]:
    _s21_ann_by_entry[str(ann["image_id"])].append(ann)

_s21_info_by_entry = {
    str(info["id"]): info
    for info in annotations_data["images"]
}

_training_physical = set(
    s21_registry_df.loc[
        s21_registry_df["split"] == "train",
        "physical_image_id",
    ]
)

_eligible = s21_physical_summary[
    s21_physical_summary["physical_image_id"].isin(
        _training_physical
    )
].sort_values(
    ["annotation_entries", "physical_image_id"],
    ascending=[False, True],
)

s21_real_physical_ids = (
    _eligible.head(3)["physical_image_id"].tolist()
)

_s21_gt_rows = []
_s21_pred_rows = []
_s21_roundtrip_records = []
_s21_real_stacks = {}

for physical_id in s21_real_physical_ids:
    entries = (
        s21_registry_df[
            s21_registry_df["physical_image_id"] == physical_id
        ]
        .sort_values("entry_id")
    )

    if not (
        entries["height"].eq(2048).all()
        and entries["width"].eq(2048).all()
    ):
        raise ValueError(
            f"{physical_id}: official adapter assumes 2048×2048."
        )

    masks_by_entry = {}

    for entry in entries.itertuples(index=False):
        shape = (entry.height, entry.width)
        masks = []

        for index, ann in enumerate(
            _s21_ann_by_entry[entry.entry_id]
        ):
            # Explicitly retain the project's existing polygon convention.
            mask = np.asarray(
                rasterize_annotation(ann, shape),
                dtype=bool,
            )

            if not mask.any():
                raise ValueError(
                    f"Empty decoded production annotation: {ann['id']}"
                )

            rle = s21_encode(mask)
            decoded = s21_decode(rle)
            roundtrip_ok = np.array_equal(mask, decoded)

            _s21_roundtrip_records.append({
                "entry_id": entry.entry_id,
                "annotation_id": str(ann["id"]),
                "roundtrip_ok": roundtrip_ok,
                "area": int(mask.sum()),
            })

            if not roundtrip_ok:
                raise AssertionError("RLE round-trip mismatch.")

            _s21_gt_rows.append({
                "filament_id": f"{entry.entry_id}_{index}",
                "segmentation_rle": rle["counts"],
            })

            masks.append(mask)

        masks_by_entry[entry.entry_id] = (
            np.stack(masks)
            if masks
            else np.zeros((0, *shape), dtype=bool)
        )

    first_entry = entries.iloc[0]["entry_id"]
    controlled_predictions = masks_by_entry[first_entry]

    for index, mask in enumerate(controlled_predictions):
        _s21_pred_rows.append({
            "filament_id": f"{physical_id}_{index}",
            "segmentation_rle": s21_encode(mask)["counts"],
        })

    for entry_id, gt_masks in masks_by_entry.items():
        _s21_real_stacks[entry_id] = {
            "gt": gt_masks,
            "pred": controlled_predictions,
            "physical_image_id": physical_id,
        }

s21_gt_fixture_df = pd.DataFrame(
    _s21_gt_rows,
    columns=["filament_id", "segmentation_rle"],
)

s21_pred_fixture_df = pd.DataFrame(
    _s21_pred_rows,
    columns=["filament_id", "segmentation_rle"],
)

assert len(s21_gt_fixture_df), "No real GT fixtures were constructed."

s21_official_real_overlap = (
    s21_reference_namespace["get_overlap_df"](
        s21_gt_fixture_df,
        s21_pred_fixture_df,
    )
)

s21_real_records = []
s21_real_accounts = []

for row in s21_official_real_overlap.itertuples(index=False):
    fixture = _s21_real_stacks[row.annotator_image]

    local_iou, local_dice = s21_overlap(
        fixture["gt"], fixture["pred"]
    )
    account = s21_account(local_iou)

    iou_error = (
        float(np.abs(local_iou - row.iou_matrix.numpy()).max())
        if local_iou.size else 0.0
    )
    dice_error = (
        float(np.abs(local_dice - row.dice_matrix.numpy()).max())
        if local_dice.size else 0.0
    )

    parity_ok = (
        iou_error <= S21_ATOL
        and dice_error <= S21_ATOL
    )

    s21_real_records.append({
        "entry_id": row.annotator_image,
        "physical_image_id": fixture["physical_image_id"],
        **account,
        "maximum_iou_error": iou_error,
        "maximum_dice_error": dice_error,
        "passed": parity_ok,
    })
    s21_real_accounts.append(account)

    safe_name = hashlib.sha256(
        row.annotator_image.encode()
    ).hexdigest()[:12]

    np.savez_compressed(
        S21_DIR / f"real_overlap_{safe_name}.npz",
        iou=local_iou,
        dice=local_dice,
    )

s21_real_df = pd.DataFrame(s21_real_records)
s21_roundtrip_df = pd.DataFrame(_s21_roundtrip_records)

s21_official_real_pq = (
    s21_reference_namespace["get_pq_score"](
        s21_official_real_overlap
    )
)
s21_local_real_pq = s21_pooled(s21_real_accounts)["pq"]

s21_real_pooled_ok = bool(np.isclose(
    s21_official_real_pq,
    s21_local_real_pq,
    rtol=0,
    atol=S21_ATOL,
))

s21_save(s21_real_df, "real_image_parity.csv")
s21_save(s21_roundtrip_df, "rle_roundtrip_tests.csv")

display(s21_real_df)

print("Official real-fixture pooled PQ:", s21_official_real_pq)
print("Local real-fixture pooled PQ:", s21_local_real_pq)
print("These are controlled evaluator fixtures, not model results.")

assert s21_real_df["passed"].all()
assert s21_roundtrip_df["roundtrip_ok"].all()
assert s21_real_pooled_ok

### 21.8 Adapter boundary behavior

The published self-evaluation adapter is not a complete input validator.

We characterize:

- A missing prediction set.
- An empty ground-truth dataframe.
- Predictions whose physical-image ID is absent from ground truth.
- Serialization of compressed COCO RLE through CSV.

These observations describe the pinned self-evaluation implementation.
They do not establish that the leaderboard accepts invalid identifiers,
overlapping predictions or other malformed submissions.

Production code must explicitly validate its image universe and output
format, rather than relying on whether the reference happens to ignore
an input row.

In [ ]:
s21_adapter_observations = []

# A. No predictions for real GT entries.
_empty_prediction_table = pd.DataFrame({
    "filament_id": pd.Series(dtype="object"),
    "segmentation_rle": pd.Series(dtype="object"),
})

_missing_overlap = s21_reference_namespace["get_overlap_df"](
    s21_gt_fixture_df,
    _empty_prediction_table,
)
_missing_score = s21_reference_namespace["get_pq_score"](
    _missing_overlap
)

assert _missing_score == 0.0

s21_adapter_observations.append({
    "case": "no_predictions_for_represented_gt",
    "behavior": "returns_score",
    "value": _missing_score,
})

# B. Empty GT table: capture the actual adapter response.
_empty_gt_table = pd.DataFrame({
    "filament_id": pd.Series(dtype="object"),
    "segmentation_rle": pd.Series(dtype="object"),
})

try:
    result = s21_reference_namespace["get_overlap_df"](
        _empty_gt_table,
        _empty_prediction_table,
    )
    behavior = f"returned dataframe with {len(result)} rows"
except Exception as error:
    behavior = f"raises {type(error).__name__}: {error}"

s21_adapter_observations.append({
    "case": "empty_ground_truth_dataframe",
    "behavior": behavior,
    "value": np.nan,
})

# C. Unknown physical-image ID.
_unknown_row = s21_pred_fixture_df.iloc[[0]].copy()
_unknown_row["filament_id"] = "UNKNOWNPHYSICALIMAGE_0"

_with_unknown = pd.concat(
    [s21_pred_fixture_df, _unknown_row],
    ignore_index=True,
)

_unknown_overlap = s21_reference_namespace["get_overlap_df"](
    s21_gt_fixture_df,
    _with_unknown,
)
_unknown_score = s21_reference_namespace["get_pq_score"](
    _unknown_overlap
)

s21_adapter_observations.append({
    "case": "prediction_for_unrepresented_image",
    "behavior": (
        "same_score_as_without_unknown_row"
        if np.isclose(
            _unknown_score, s21_official_real_pq,
            rtol=0, atol=S21_ATOL,
        )
        else "score_changed"
    ),
    "value": _unknown_score,
})

# D. CSV round trip must preserve compressed RLE text exactly.
_csv_path = S21_DIR / "controlled_prediction_fixture.csv"
s21_pred_fixture_df.to_csv(_csv_path, index=False)

_reloaded_csv = pd.read_csv(
    _csv_path,
    dtype=str,
    keep_default_na=False,
)

s21_csv_roundtrip_ok = bool(
    _reloaded_csv.equals(
        s21_pred_fixture_df.astype(str).reset_index(drop=True)
    )
)

assert s21_csv_roundtrip_ok, "CSV serialization changed IDs or RLE strings."

s21_adapter_behavior_df = pd.DataFrame(s21_adapter_observations)
s21_save(
    s21_adapter_behavior_df,
    "official_adapter_boundary_behavior.csv"
)

display(s21_adapter_behavior_df)

print(
    "Unknown prediction IDs must be rejected by our production "
    "input validator even if this reference adapter ignores them."
)

### 21.9 Real-image diagnostic visualizations

Show:

- Annotator-specific PQ for controlled predictions.
- Pooled PQ versus the mean of per-entry scores.
- Real overlap matrices.
- Qualifying-pair IoU and nonzero-contact IoU distributions.
- Nonzero-contact degrees on the GT and prediction sides.

Interpretation:

- Ground-truth contact degree > 1 indicates multiple predicted masks
  contacting a GT instance; it is a fragmentation warning.
- Prediction contact degree > 1 indicates contact with multiple GT
  instances; it is an over-merging warning.
- Small incidental contacts can also create these warnings.
- Contact degree is not the same as a qualifying PQ relation.
- Different annotators can produce different scores for the same
  physical-image prediction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

positions = np.arange(len(s21_real_df))

axes[0].bar(
    positions,
    s21_real_df["pq"],
    color="#496b8a",
)
axes[0].set_xticks(
    positions,
    s21_real_df["entry_id"],
    rotation=65,
    ha="right",
)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Per-entry PQ")
axes[0].set_title(
    "Same controlled predictions, separate annotation entries"
)
axes[0].grid(axis="y", alpha=.2)

labels = ["Official pooled", "Local pooled", "Entry-macro diagnostic"]
values = [
    s21_official_real_pq,
    s21_local_real_pq,
    s21_real_df["pq"].mean(),
]

bars = axes[1].bar(
    labels, values,
    color=["#2878b5", "#3a9d70", "#e58b25"],
)
axes[1].bar_label(bars, fmt="%.5f", padding=3)
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Aggregation conventions")
axes[1].set_ylabel("PQ")
axes[1].tick_params(axis="x", labelrotation=20)
axes[1].grid(axis="y", alpha=.2)

fig.tight_layout()
s21_finish(fig, "real_entry_scores_and_aggregation.png")

# Real IoU matrices.
for row in s21_official_real_overlap.head(4).itertuples(index=False):
    iou = row.iou_matrix.numpy()

    fig, axis = plt.subplots(figsize=(8, 6))

    if iou.size:
        artist = axis.imshow(
            iou, cmap="viridis", vmin=0, vmax=1,
            aspect="auto",
        )

        hit_y, hit_x = np.nonzero(iou > .5)
        axis.scatter(
            hit_x, hit_y,
            marker="s", s=75,
            facecolors="none", edgecolors="red",
            linewidths=1.3,
            label="IoU > 0.5",
        )

        fig.colorbar(artist, ax=axis, label="IoU")
        axis.legend(loc="upper right")
    else:
        axis.text(
            .5, .5, f"Empty matrix: {iou.shape}",
            ha="center", va="center",
            transform=axis.transAxes,
        )

    axis.set_title(
        f"Real fixture: {row.annotator_image}\n"
        "Rows: this annotator's GT | Columns: shared predictions"
    )
    axis.set_xlabel("Prediction instance")
    axis.set_ylabel("GT instance")

    fig.tight_layout()

    safe_name = hashlib.sha256(
        row.annotator_image.encode()
    ).hexdigest()[:12]
    s21_finish(fig, f"real_iou_matrix_{safe_name}.png")

# Pairwise-score distributions.
_all_real_matrices = [
    row.iou_matrix.numpy()
    for row in s21_official_real_overlap.itertuples(index=False)
]

_all_pair_values = np.concatenate([
    matrix.ravel() for matrix in _all_real_matrices
]) if _all_real_matrices else np.array([])

_contact_values = _all_pair_values[_all_pair_values > 0]
_qualifying_values = _all_pair_values[_all_pair_values > .5]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(
    _contact_values, bins=np.linspace(0, 1, 31),
    color="#496b8a", edgecolor="white",
)
axes[0].axvline(.5, color="red", linestyle="--")
axes[0].set_title("All nonzero-contact pair IoUs")
axes[0].set_xlabel("IoU")
axes[0].set_ylabel("Pairs")

axes[1].hist(
    _qualifying_values, bins=np.linspace(.5, 1, 21),
    color="#3a9d70", edgecolor="white",
)
axes[1].set_title("Qualifying official pair IoUs")
axes[1].set_xlabel("IoU strictly above 0.5")
axes[1].set_ylabel("Pairs")

for axis in axes:
    axis.grid(axis="y", alpha=.2)

fig.tight_layout()
s21_finish(fig, "real_pair_iou_distributions.png")

_gt_degrees = []
_pred_degrees = []

for matrix in _all_real_matrices:
    _gt_degrees.extend((matrix > 0).sum(axis=1).tolist())
    _pred_degrees.extend((matrix > 0).sum(axis=0).tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for axis, values, title in [
    (
        axes[0], _gt_degrees,
        "Predictions contacting each GT instance",
    ),
    (
        axes[1], _pred_degrees,
        "GT instances contacted by each prediction",
    ),
]:
    counts = pd.Series(values, dtype=int).value_counts().sort_index()

    axis.bar(
        counts.index,
        counts.values,
        color=[
            "#c62828" if degree == 0
            else "#3a9d70" if degree == 1
            else "#e58b25"
            for degree in counts.index
        ],
    )
    axis.set_xticks(counts.index)
    axis.set_title(title + "\nContact threshold: IoU > 0")
    axis.set_xlabel("Contact degree")
    axis.set_ylabel("Instances across annotation entries")
    axis.grid(axis="y", alpha=.2)

fig.tight_layout()
s21_finish(fig, "real_structural_contact_degrees.png")

### 21.10 Parity report and remaining limitations

Passing this section establishes agreement with the pinned version 6
self-evaluation functions for the tested cases.

It does not establish:

- Acceptance of a final Kaggle submission.
- Equivalence to undisclosed leaderboard input checks.
- Official polygon-to-mask rasterization parity.
- Generalization performance.
- Completion of unresolved Step 20 integrity findings.

The selected real images are controlled scorer tests.
Their scores must not be reported as model-validation performance.

Before comparative training:

- Resolve blocking Step 20 findings.
- Preserve physical-image partition separation.
- Document the polygon-decoding convention.
- Keep independent annotation entries separate during evaluation.
- Use pooled official accounting for model selection.
- Retain auxiliary morphology and contact diagnostics separately.

In [ ]:
s21_summary = {
    "reference": S21_REFERENCE,
    "synthetic_cases": len(s21_tests_df),
    "synthetic_cases_passed": int(s21_tests_df["passed"].sum()),
    "synthetic_pooled_parity": bool(np.isclose(
        _synthetic_official_pooled,
        _synthetic_local_pooled,
        rtol=0, atol=S21_ATOL,
    )),
    "real_physical_images_tested": len(s21_real_physical_ids),
    "real_annotation_entries_tested": len(s21_real_df),
    "real_overlap_parity": bool(s21_real_df["passed"].all()),
    "real_pooled_parity": s21_real_pooled_ok,
    "rle_roundtrips_passed": bool(
        s21_roundtrip_df["roundtrip_ok"].all()
    ),
    "csv_roundtrip_passed": s21_csv_roundtrip_ok,
    "cross_partition_physical_images": len(
        _cross_partition_physical
    ),
    "polygon_rasterization_official_parity": (
        "UNRESOLVED: the published scorer consumes RLE masks."
    ),
    "hidden_submission_validation_parity": (
        "NOT ESTABLISHED by this self-evaluation notebook."
    ),
    "model_performance_measured": False,
}

_required_checks = [
    s21_tests_df["passed"].all(),
    s21_summary["synthetic_pooled_parity"],
    s21_summary["real_overlap_parity"],
    s21_summary["real_pooled_parity"],
    s21_summary["rle_roundtrips_passed"],
    s21_summary["csv_roundtrip_passed"],
    s21_summary["cross_partition_physical_images"] == 0,
]

s21_summary["status"] = (
    "REFERENCE_PARITY_PASSED_WITH_DOCUMENTED_LIMITATIONS"
    if all(_required_checks)
    else "PARITY_FAILED"
)

with open(S21_DIR / "parity_summary.json", "w") as handle:
    json.dump(s21_summary, handle, indent=2)

(S21_DIR / "parity_report.md").write_text(
    "# Step 21 metric parity report\n\n"
    f"Status: {s21_summary['status']}\n\n"
    "Reference: Azim Ahmadzadeh, Self_Evaluation_Notebook, v6.\n\n"
    "Synthetic and real-mask tests compare independent RLE-based "
    "overlap calculations against the pinned dense PyTorch reference.\n\n"
    "Physical-image routing and repeated annotator evaluation are "
    "preserved. All qualifying IoU>0.5 pairs are counted as in the "
    "published reference.\n\n"
    "These are scorer fixtures, not model-validation results.\n\n"
    "Official polygon-rasterization and hidden submission-validation "
    "parity remain unestablished. Step 20 blockers remain independent.\n\n"
    "```json\n"
    + json.dumps(s21_summary, indent=2)
    + "\n```\n",
    encoding="utf-8",
)

print(json.dumps(s21_summary, indent=2))
print("\nSaved reports:", S21_DIR)
print("Saved figures:", S21_FIG_DIR)

assert all(_required_checks), (
    "Step 21 parity failed. Inspect the saved test tables."
)

## 22. Topological and Connectivity Analysis

### 22.1 Objectives and interpretation

Geometry describes an object's size and shape. Topology describes its
connectivity and holes.

For a two-dimensional binary mask:

- β0 is the number of foreground connected components.
- β1 is the number of enclosed background holes.
- Euler characteristic χ = β0 − β1.

We measure both digital-connectivity conventions:

- Foreground 8-connectivity, background 4-connectivity.
- Foreground 4-connectivity, background 8-connectivity.

A difference between these conventions identifies diagonal-sensitive
structure. It does not automatically identify an annotation error.

This section separates three questions:

1. What topology does each independently annotated instance have?
2. Which distinct instances become connected in the foreground union?
3. Which structures in an image-derived score map persist across
   thresholds?

Important constraints:

- An annotation may legitimately contain disconnected parts or holes.
- Separate annotators are never combined into one ground-truth union.
- Skeleton pixel degree is a local diagnostic, not a validated count
  of anatomical branches or collapsed graph junctions.
- Erosion, dilation and hole filling are diagnostic perturbations.
  They do not modify the original training targets.
- Ridge scores are image-derived responses, not calibrated probabilities.
- Persistence is initially a diagnostic, not a segmentation algorithm.
- Any proposed connectivity intervention must later be assessed using
  the official instance-level evaluator.

In [ ]:
import hashlib
import json
import sys
import types
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy
import skimage
from scipy import ndimage as ndi
from skimage.measure import euler_number
from skimage.morphology import skeletonize
from skimage.filters import sato
from skimage.transform import resize

from IPython.display import display

_s22_required = [
    "annotations_data",
    "train_df",
    "valid_df",
    "OUTPUT_ROOT",
    "TRAIN_IMAGE_DIR",
    "get_polygons",
    "rasterize_annotation",
    "load_solar_image",
]

_s22_missing = [
    name for name in _s22_required
    if name not in globals()
]

if _s22_missing:
    raise RuntimeError(
        "Run the earlier setup cells first. Missing: "
        + ", ".join(_s22_missing)
    )

try:
    import gudhi
    S22_GUDHI_AVAILABLE = True
except ImportError:
    gudhi = None
    S22_GUDHI_AVAILABLE = False

S22_CFG = {
    "version": "step22_v1",
    "foreground_connectivity": 8,
    "background_connectivity": 4,
    "perturbation_radius_px": 1,
    "ridge_sigmas": [1, 2, 4, 8],
    "ridge_normalization_positive_percentile": 99.0,
    "roi_padding_native_px": 24,
    "maximum_rois": 6,
    "maximum_rois_per_physical_image": 2,
    "maximum_persistence_side": 192,
    "thresholds": np.linspace(0, 1, 41).tolist(),
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "skimage_version": skimage.__version__,
    "python_version": sys.version.split()[0],
    "gudhi_version": (
        getattr(gudhi, "__version__", None)
        if S22_GUDHI_AVAILABLE else None
    ),
}

S22_ROOT = Path(OUTPUT_ROOT) / "step22_topology"
S22_ROOT.mkdir(parents=True, exist_ok=True)

s22_raw_ids = {
    str(info["id"]): info["id"]
    for info in annotations_data["images"]
}

s22_info = {
    str(info["id"]): info
    for info in annotations_data["images"]
}

s22_annotations = defaultdict(list)

for annotation in annotations_data["annotations"]:
    s22_annotations[str(annotation["image_id"])].append(annotation)

s22_train_ids = sorted(
    train_df["image_id"].astype(str).tolist()
)

if len(s22_train_ids) != len(set(s22_train_ids)):
    raise ValueError("Training annotation-entry IDs are repeated.")

if set(s22_train_ids) & set(valid_df["image_id"].astype(str)):
    raise ValueError("Training and validation annotation entries overlap.")

def s22_digest(value):
    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode()
    ).hexdigest()

def s22_code_record(code):
    return {
        "bytecode": code.co_code.hex(),
        "names": list(code.co_names),
        "constants": [
            s22_code_record(value)
            if isinstance(value, types.CodeType)
            else repr(value)
            for value in code.co_consts
        ],
    }

def s22_file_hash(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(block)

    return digest.hexdigest()

def s22_finish(fig, filename):
    fig.savefig(
        S22_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
    plt.close(fig)

print("Training annotation entries:", len(s22_train_ids))
print(
    "Training annotations:",
    sum(len(s22_annotations[key]) for key in s22_train_ids)
)
print("GUDHI available:", S22_GUDHI_AVAILABLE)
print(
    "No additional package is required for the exact H0 analysis. "
    "Cubical H1 persistence is supplementary when GUDHI is available."
)

### 22.2 Binary topology, skeleton continuity and perturbations

Each mask is cropped to its foreground extent and padded with background.
This preserves bounded-mask topology while reducing computation.

We calculate:

- Components and holes under both connectivity conventions.
- Hole area under the 8-foreground / 4-background convention.
- Largest-component fraction.
- Skeleton component count and skeleton pixel count.
- Raw endpoint and high-degree skeleton-pixel counts.
- Components and area after a one-pixel erosion.
- Components and area after a one-pixel dilation.

Interpretation:

- Erosion can expose narrow connections, but can also remove entire
  thin components. We record disappearance separately.
- Dilation can join separate components. It does not establish that
  the components should belong to the same predicted instance.
- A high-degree skeleton pixel is not equivalent to a graph junction
  after junction-zone collapsing.
- Hole filling is measured only to describe enclosed background area.

In [ ]:
S22_FG4 = ndi.generate_binary_structure(2, 1)
S22_FG8 = ndi.generate_binary_structure(2, 2)
S22_PIXEL_NEIGHBORS = np.ones((3, 3), dtype=np.uint8)
S22_PIXEL_NEIGHBORS[1, 1] = 0

def s22_betti(mask, connectivity=8):
    mask = np.asarray(mask, dtype=bool)

    if mask.ndim != 2:
        raise ValueError("Expected a two-dimensional binary mask.")

    # Explicit exterior background for finite-image topology.
    padded = np.pad(mask, 1, constant_values=False)

    structure = S22_FG8 if connectivity == 8 else S22_FG4

    _, components = ndi.label(
        padded,
        structure=structure,
    )

    chi = int(
        euler_number(
            padded,
            connectivity=2 if connectivity == 8 else 1,
        )
    )

    holes = int(components) - chi

    if holes < 0:
        raise AssertionError("Invalid component/Euler accounting.")

    return {
        "components": int(components),
        "holes": holes,
        "euler": chi,
    }

def s22_topology(mask):
    mask = np.asarray(mask, dtype=bool)

    if mask.ndim != 2:
        raise ValueError("Expected a two-dimensional binary mask.")

    area = int(mask.sum())

    if area:
        ys, xs = np.nonzero(mask)

        crop = mask[
            int(ys.min()):int(ys.max()) + 1,
            int(xs.min()):int(xs.max()) + 1,
        ]
    else:
        crop = np.zeros((1, 1), dtype=bool)

    padded = np.pad(crop, 1, constant_values=False)

    topology8 = s22_betti(crop, 8)
    topology4 = s22_betti(crop, 4)

    component_labels, _ = ndi.label(
        padded,
        structure=S22_FG8,
    )
    component_areas = np.bincount(component_labels.ravel())[1:]

    skeleton = skeletonize(padded)

    _, skeleton_components = ndi.label(
        skeleton,
        structure=S22_FG8,
    )

    degree = ndi.convolve(
        skeleton.astype(np.uint8),
        S22_PIXEL_NEIGHBORS,
        mode="constant",
        cval=0,
    )

    eroded = ndi.binary_erosion(
        padded,
        structure=S22_FG8,
        iterations=1,
        border_value=0,
    )

    dilated = ndi.binary_dilation(
        padded,
        structure=S22_FG8,
        iterations=1,
        border_value=0,
    )

    _, eroded_components = ndi.label(
        eroded, structure=S22_FG8
    )
    _, dilated_components = ndi.label(
        dilated, structure=S22_FG8
    )

    filled = ndi.binary_fill_holes(
        padded,
        structure=S22_FG4,
    )

    return {
        "area_px": area,
        "components8": topology8["components"],
        "holes8": topology8["holes"],
        "euler8": topology8["euler"],
        "components4": topology4["components"],
        "holes4": topology4["holes"],
        "euler4": topology4["euler"],
        "hole_area_px": int((filled & ~padded).sum()),
        "largest_component_fraction": (
            float(component_areas.max() / area)
            if area and len(component_areas) else np.nan
        ),
        "skeleton_pixels": int(skeleton.sum()),
        "skeleton_components8": int(skeleton_components),
        "raw_skeleton_endpoint_pixels": int(
            (skeleton & (degree == 1)).sum()
        ),
        "raw_skeleton_high_degree_pixels": int(
            (skeleton & (degree >= 3)).sum()
        ),
        "eroded_area_px": int(eroded.sum()),
        "eroded_components8": int(eroded_components),
        "vanishes_after_erosion": bool(area and not eroded.any()),
        "dilated_area_px": int(dilated.sum()),
        "dilated_components8": int(dilated_components),
    }

# -------------------------------------------------------------
# Synthetic binary-topology tests
# -------------------------------------------------------------
_s22_test_masks = {}

_solid = np.zeros((32, 32), dtype=bool)
_solid[5:25, 5:25] = True
_s22_test_masks["solid"] = (_solid, 1, 0)

_ring = _solid.copy()
_ring[10:20, 10:20] = False
_s22_test_masks["ring"] = (_ring, 1, 1)

_multipart = np.zeros((32, 32), dtype=bool)
_multipart[3:8, 3:8] = True
_multipart[20:25, 20:25] = True
_s22_test_masks["multipart"] = (_multipart, 2, 0)

_diagonal = np.eye(2, dtype=bool)
_s22_test_masks["diagonal"] = (_diagonal, 1, 0)

_s22_topology_tests = []

for name, (mask, expected_components, expected_holes) in (
    _s22_test_masks.items()
):
    result = s22_betti(mask, 8)

    passed = (
        result["components"] == expected_components
        and result["holes"] == expected_holes
        and result["euler"]
        == expected_components - expected_holes
    )

    _s22_topology_tests.append({
        "case": name,
        **result,
        "passed": passed,
    })

assert s22_betti(_diagonal, 4)["components"] == 2
assert all(row["passed"] for row in _s22_topology_tests)

display(pd.DataFrame(_s22_topology_tests))
print("Binary-topology correctness checks passed.")

### 22.3 Resumable full-training-population measurement

We process every training annotation independently.

For each annotator–image entry we also construct its foreground union
and record which union components contain pixels from multiple
independently annotated instances.

This distinguishes:

- An annotation that is internally disconnected.
- Multiple annotations that contact or overlap in their union.
- Multiple annotators describing the same physical image.

Only the first two are connectivity measurements here. Separate
annotators remain separate records.

One checkpoint is written per annotation entry. Checkpoints are reused
only when their configuration and annotation-source signatures match.

In [ ]:
_s22_function_fingerprints = {
    function.__name__: s22_code_record(function.__code__)
    for function in [
        get_polygons,
        rasterize_annotation,
        s22_betti,
        s22_topology,
    ]
}

S22_RUN_HASH = s22_digest({
    "configuration": S22_CFG,
    "functions": _s22_function_fingerprints,
})[:16]

S22_DIR = S22_ROOT / S22_RUN_HASH
S22_DIR.mkdir(parents=True, exist_ok=True)

S22_CHECKPOINT_DIR = S22_DIR / "entry_checkpoints"
S22_CHECKPOINT_DIR.mkdir(exist_ok=True)

with open(S22_DIR / "configuration.json", "w") as handle:
    json.dump(S22_CFG, handle, indent=2)

s22_payloads = []
s22_failures = []
s22_reused = 0

for position, entry_id in enumerate(s22_train_ids, start=1):
    info = s22_info[entry_id]
    annotations_here = sorted(
        s22_annotations[entry_id],
        key=lambda annotation: str(annotation["id"]),
    )

    source_signature = s22_digest({
        "entry_id": entry_id,
        "height": int(info["height"]),
        "width": int(info["width"]),
        "annotations": annotations_here,
    })

    checkpoint_key = hashlib.sha256(
        entry_id.encode()
    ).hexdigest()[:20]

    checkpoint = (
        S22_CHECKPOINT_DIR / f"{checkpoint_key}.json"
    )

    if checkpoint.is_file():
        try:
            payload = json.loads(checkpoint.read_text())

            reusable = (
                payload.get("run_hash") == S22_RUN_HASH
                and payload.get("source_signature") == source_signature
                and payload.get("status") == "complete"
                and len(payload["instances"]) == len(annotations_here)
            )

            if reusable:
                s22_payloads.append(payload)
                s22_reused += 1
                continue

        except (OSError, ValueError, KeyError):
            pass

    shape = (int(info["height"]), int(info["width"]))
    coverage = np.zeros(shape, dtype=np.uint32)

    instance_records = []
    cropped_masks = []
    entry_errors = []

    for annotation in annotations_here:
        annotation_id = str(annotation["id"])

        try:
            mask = np.asarray(
                rasterize_annotation(annotation, shape),
                dtype=bool,
            )

            if mask.shape != shape:
                raise ValueError("Decoded mask dimensions disagree.")

            measurements = s22_topology(mask)

            if mask.any():
                ys, xs = np.nonzero(mask)

                y0, y1 = int(ys.min()), int(ys.max()) + 1
                x0, x1 = int(xs.min()), int(xs.max()) + 1

                cropped_masks.append({
                    "annotation_id": annotation_id,
                    "bounds": (y0, y1, x0, x1),
                    "mask": mask[y0:y1, x0:x1].copy(),
                })

                bounds = {
                    "y0": y0, "y1": y1,
                    "x0": x0, "x1": x1,
                }

                coverage[mask] += 1

                touches_border = bool(
                    mask[0].any() or mask[-1].any()
                    or mask[:, 0].any() or mask[:, -1].any()
                )
            else:
                bounds = {
                    "y0": None, "y1": None,
                    "x0": None, "x1": None,
                }
                touches_border = False

            record = {
                "entry_id": entry_id,
                "physical_image_id": Path(info["file_name"]).stem,
                "annotation_id": annotation_id,
                "status": "ok" if mask.any() else "empty",
                "touches_image_border": touches_border,
                **bounds,
                **measurements,
            }

            instance_records.append(record)

        except Exception as error:
            entry_errors.append({
                "entry_id": entry_id,
                "annotation_id": annotation_id,
                "error": repr(error),
            })

        finally:
            if "mask" in locals():
                del mask

    union = coverage > 0

    union_labels, union_component_count = ndi.label(
        union,
        structure=S22_FG8,
    )

    union_topology = s22_betti(union, 8)

    memberships = defaultdict(set)

    for item in cropped_masks:
        y0, y1, x0, x1 = item["bounds"]

        component_ids = np.unique(
            union_labels[y0:y1, x0:x1][item["mask"]]
        )

        for component_id in component_ids:
            if component_id > 0:
                memberships[int(component_id)].add(
                    item["annotation_id"]
                )

    shared_components = {
        key: members
        for key, members in memberships.items()
        if len(members) > 1
    }

    shared_annotation_ids = set().union(
        *shared_components.values()
    ) if shared_components else set()

    membership_records = [
        {
            "entry_id": entry_id,
            "union_component": int(component_id),
            "annotation_id": annotation_id,
            "annotations_in_union_component": len(members),
        }
        for component_id, members in memberships.items()
        for annotation_id in sorted(members)
    ]

    payload = {
        "run_hash": S22_RUN_HASH,
        "source_signature": source_signature,
        "status": "failed" if entry_errors else "complete",
        "entry_id": entry_id,
        "instances": instance_records,
        "memberships": membership_records,
        "image_summary": {
            "entry_id": entry_id,
            "physical_image_id": Path(info["file_name"]).stem,
            "annotation_count": len(annotations_here),
            "foreground_fraction": float(union.mean()),
            "union_components8": int(union_component_count),
            "union_holes8": union_topology["holes"],
            "union_euler8": union_topology["euler"],
            "unique_overlap_pixels": int((coverage > 1).sum()),
            "shared_union_components": len(shared_components),
            "annotations_in_shared_components": len(
                shared_annotation_ids
            ),
        },
        "errors": entry_errors,
    }

    temporary = checkpoint.with_suffix(".json.tmp")

    # NaN measurements are converted to null for strict JSON.
    def s22_clean_json(value):
        if isinstance(value, dict):
            return {
                key: s22_clean_json(item)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [s22_clean_json(item) for item in value]
        if isinstance(value, float) and not np.isfinite(value):
            return None
        return value

    temporary.write_text(
        json.dumps(
            s22_clean_json(payload),
            allow_nan=False,
        )
    )
    temporary.replace(checkpoint)

    s22_payloads.append(payload)
    s22_failures.extend(entry_errors)

    if position % 20 == 0 or position == len(s22_train_ids):
        print(
            f"Entries {position}/{len(s22_train_ids)} | "
            f"reused {s22_reused} | "
            f"errors {len(s22_failures)}",
            flush=True,
        )

s22_instances_df = pd.DataFrame([
    record
    for payload in s22_payloads
    for record in payload["instances"]
])

s22_images_df = pd.DataFrame([
    payload["image_summary"]
    for payload in s22_payloads
])

s22_memberships_df = pd.DataFrame([
    record
    for payload in s22_payloads
    for record in payload["memberships"]
])

s22_instances_df.to_csv(
    S22_DIR / "training_instance_topology.csv",
    index=False,
)
s22_images_df.to_csv(
    S22_DIR / "training_union_topology.csv",
    index=False,
)
s22_memberships_df.to_csv(
    S22_DIR / "union_component_instance_memberships.csv",
    index=False,
)
pd.DataFrame(
    s22_failures,
    columns=["entry_id", "annotation_id", "error"],
).to_csv(
    S22_DIR / "extraction_failures.csv",
    index=False,
)

expected_keys = {
    (entry_id, str(annotation["id"]))
    for entry_id in s22_train_ids
    for annotation in s22_annotations[entry_id]
}

measured_keys = set(zip(
    s22_instances_df["entry_id"].astype(str),
    s22_instances_df["annotation_id"].astype(str),
))

assert not s22_failures, (
    "Resolve extraction_failures.csv and rerun this cell. "
    "Completed entries will resume."
)
assert measured_keys == expected_keys
assert not s22_instances_df.duplicated(
    ["entry_id", "annotation_id"]
).any()

s22_valid_df = s22_instances_df[
    s22_instances_df["status"] == "ok"
].copy()

s22_valid_df["diagonal_sensitive"] = (
    s22_valid_df["components4"]
    != s22_valid_df["components8"]
)

s22_valid_df["erosion_component_increase"] = (
    s22_valid_df["eroded_components8"]
    - s22_valid_df["components8"]
)

s22_valid_df["skeleton_component_difference"] = (
    s22_valid_df["skeleton_components8"]
    - s22_valid_df["components8"]
)

print("\nPopulation accounting")
print("Expected annotations:", len(expected_keys))
print("Measured records:", len(s22_instances_df))
print("Nonempty masks:", len(s22_valid_df))
print("Reused entry checkpoints:", s22_reused)
print("Output:", S22_DIR)

display(
    s22_valid_df[
        [
            "components8", "holes8", "components4",
            "largest_component_fraction",
            "erosion_component_increase",
        ]
    ].describe()
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for axis, column, title in [
    (
        axes[0, 0], "components8",
        "Connected components per annotation: foreground 8",
    ),
    (
        axes[0, 1], "holes8",
        "Holes per annotation: foreground 8 / background 4",
    ),
]:
    counts = (
        s22_valid_df[column]
        .value_counts()
        .sort_index()
    )

    axis.bar(
        counts.index.astype(str),
        counts.values,
        color="#496b8a",
    )
    axis.set_title(title)
    axis.set_xlabel("Count")
    axis.set_ylabel("Annotations")
    axis.tick_params(axis="x", labelrotation=60)

axes[0, 2].hist(
    s22_valid_df["largest_component_fraction"].dropna(),
    bins=np.linspace(0, 1, 31),
    color="#3a9d70",
    edgecolor="white",
)
axes[0, 2].set_title("Fraction of area in the largest component")
axes[0, 2].set_xlabel("Largest-component area / instance area")
axes[0, 2].set_ylabel("Annotations")

_categories = pd.Series({
    "Disconnected": int(
        (s22_valid_df["components8"] > 1).sum()
    ),
    "Contains holes": int(
        (s22_valid_df["holes8"] > 0).sum()
    ),
    "Diagonal-sensitive": int(
        s22_valid_df["diagonal_sensitive"].sum()
    ),
    "Erosion increases components": int(
        (s22_valid_df["erosion_component_increase"] > 0).sum()
    ),
    "Vanishes under erosion": int(
        s22_valid_df["vanishes_after_erosion"].sum()
    ),
})

axes[1, 0].barh(
    _categories.index,
    _categories.values,
    color="#e58b25",
)
axes[1, 0].set_title(
    "Topology flags — categories can overlap"
)
axes[1, 0].set_xlabel("Annotations")

_union_counts = (
    s22_images_df["shared_union_components"]
    .value_counts()
    .sort_index()
)
axes[1, 1].bar(
    _union_counts.index.astype(str),
    _union_counts.values,
    color="#a64d79",
)
axes[1, 1].set_title(
    "Union components containing multiple annotations"
)
axes[1, 1].set_xlabel("Shared components per annotator–image entry")
axes[1, 1].set_ylabel("Annotation entries")

axes[1, 2].scatter(
    s22_images_df["annotation_count"],
    s22_images_df["annotations_in_shared_components"],
    alpha=.45,
    s=20,
    color="#496b8a",
)
axes[1, 2].set_title(
    "Annotation density versus shared connectivity"
)
axes[1, 2].set_xlabel("Annotations in entry")
axes[1, 2].set_ylabel("Annotations in shared union components")

for axis in axes.flat:
    axis.grid(alpha=.2)

fig.tight_layout()
s22_finish(fig, "01_population_topology.png")

s22_flag_summary = _categories.rename(
    "annotation_count"
).rename_axis("flag").reset_index()

s22_flag_summary.to_csv(
    S22_DIR / "topology_flag_summary.csv",
    index=False,
)

display(s22_flag_summary)

In [ ]:
s22_annotation_lookup = {
    (str(annotation["image_id"]), str(annotation["id"])): annotation
    for annotation in annotations_data["annotations"]
}

_gallery_candidates = []

for column, condition in [
    ("components8", s22_valid_df["components8"] > 1),
    ("holes8", s22_valid_df["holes8"] > 0),
    (
        "erosion_component_increase",
        s22_valid_df["erosion_component_increase"] > 0,
    ),
    ("area_px", s22_valid_df["status"].eq("ok")),
]:
    candidates = s22_valid_df[condition].sort_values(
        [column, "entry_id", "annotation_id"],
        ascending=[False, True, True],
    )

    for row in candidates.head(1).itertuples(index=False):
        _gallery_candidates.append(
            (str(row.entry_id), str(row.annotation_id))
        )

for entry_id, annotation_id in dict.fromkeys(_gallery_candidates):
    info = s22_info[entry_id]
    annotation = s22_annotation_lookup[
        (entry_id, annotation_id)
    ]

    image = load_solar_image(s22_raw_ids[entry_id])

    mask = np.asarray(
        rasterize_annotation(annotation, image.shape),
        dtype=bool,
    )

    ys, xs = np.nonzero(mask)
    padding = 20

    y0 = max(0, int(ys.min()) - padding)
    y1 = min(image.shape[0], int(ys.max()) + padding + 1)
    x0 = max(0, int(xs.min()) - padding)
    x1 = min(image.shape[1], int(xs.max()) + padding + 1)

    crop_image = image[y0:y1, x0:x1]
    crop_mask = mask[y0:y1, x0:x1]

    labels, _ = ndi.label(
        crop_mask,
        structure=S22_FG8,
    )
    filled = ndi.binary_fill_holes(
        crop_mask,
        structure=S22_FG4,
    )
    holes = filled & ~crop_mask
    skeleton = skeletonize(crop_mask)

    measurement = s22_topology(mask)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    axes[0].imshow(crop_image, cmap="gray", vmin=0, vmax=255)
    axes[0].contour(
        crop_mask, levels=[.5],
        colors=["cyan"], linewidths=1,
    )
    axes[0].set_title("Image and selected annotation")

    axes[1].imshow(
        np.ma.masked_where(labels == 0, labels),
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[1].set_title(
        f"Independent components: {measurement['components8']}"
    )

    axes[2].imshow(crop_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(
        np.ma.masked_where(~holes, holes),
        cmap="autumn",
        interpolation="nearest",
    )
    axes[2].set_title(
        f"Enclosed holes: {measurement['holes8']}"
    )

    axes[3].imshow(crop_mask, cmap="gray", vmin=0, vmax=1)
    axes[3].imshow(
        np.ma.masked_where(~skeleton, skeleton),
        cmap="autumn",
        interpolation="nearest",
    )
    axes[3].set_title(
        "Skeleton\n"
        f"components={measurement['skeleton_components8']}"
    )

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        f"Entry {entry_id} | annotation {annotation_id}\n"
        "Native-resolution crop; original target remains unchanged",
        fontsize=12,
    )
    fig.tight_layout()

    key = s22_digest({
        "entry": entry_id,
        "annotation": annotation_id,
    })[:12]

    s22_finish(fig, f"02_topology_gallery_{key}.png")

### 22.4 Exact H0 persistence on a superlevel filtration

For an image-derived score map S, the foreground at threshold τ is:

    F(τ) = {pixels p: S(p) ≥ τ}

As τ decreases:

- Components appear at local maxima.
- Components grow.
- Components merge when connecting pixels enter the foreground.

We compute exact zero-dimensional persistence for the pixel-neighbor
graph using a descending union–find algorithm and the elder rule.

A component's finite lifetime is:

    birth_score − merge_score

Equal-valued plateaus are handled with a deterministic tie order.
Zero-lifetime intervals are omitted because they do not survive any
threshold interval.

The final surviving component is essential. It is recorded separately;
its death and lifetime are not assigned artificial finite values.

We validate the intervals by reconstructing component counts at selected
thresholds and comparing them with direct connected-component labeling.

H0 on the pixel graph describes connectivity only. We do not compute
graph-cycle counts and call them image holes.

Where GUDHI is available, a separate closed-pixel-square cubical
filtration provides H0 and H1 persistence.

In [ ]:
def s22_h0_persistence(score, connectivity=8):
    score = np.asarray(score, dtype=np.float64)

    if (
        score.ndim != 2
        or not score.size
        or not np.isfinite(score).all()
    ):
        raise ValueError("Expected a finite, nonempty 2D score map.")

    if connectivity not in {4, 8}:
        raise ValueError("Connectivity must be 4 or 8.")

    height, width = score.shape
    flat = score.ravel()
    count = flat.size

    parent = np.arange(count, dtype=np.int64)
    birth = flat.copy()
    active = np.zeros(count, dtype=bool)

    offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    if connectivity == 8:
        offsets += [
            (-1, -1), (-1, 1),
            (1, -1), (1, 1),
        ]

    records = []

    def find(pixel):
        while parent[pixel] != pixel:
            parent[pixel] = parent[parent[pixel]]
            pixel = parent[pixel]
        return pixel

    for pixel in np.argsort(-flat, kind="stable"):
        pixel = int(pixel)
        active[pixel] = True
        row, column = divmod(pixel, width)
        level = float(flat[pixel])

        for dy, dx in offsets:
            yy, xx = row + dy, column + dx

            if not (
                0 <= yy < height
                and 0 <= xx < width
            ):
                continue

            neighbor = yy * width + xx

            if not active[neighbor]:
                continue

            a, b = find(pixel), find(neighbor)

            if a == b:
                continue

            if (
                birth[a] > birth[b]
                or (birth[a] == birth[b] and a < b)
            ):
                older, younger = a, b
            else:
                older, younger = b, a

            lifetime = float(birth[younger] - level)

            if lifetime > 0:
                by, bx = divmod(younger, width)

                records.append({
                    "dimension": 0,
                    "birth": float(birth[younger]),
                    "death": level,
                    "lifetime": lifetime,
                    "essential": False,
                    "birth_y": int(by),
                    "birth_x": int(bx),
                })

            parent[younger] = older

    roots = {
        find(int(pixel))
        for pixel in np.flatnonzero(active)
    }

    for root in sorted(roots):
        by, bx = divmod(root, width)

        records.append({
            "dimension": 0,
            "birth": float(birth[root]),
            "death": np.nan,
            "lifetime": np.nan,
            "essential": True,
            "birth_y": int(by),
            "birth_x": int(bx),
        })

    return pd.DataFrame(
        records,
        columns=[
            "dimension", "birth", "death", "lifetime",
            "essential", "birth_y", "birth_x",
        ],
    )

def s22_interval_component_count(intervals, threshold):
    alive = (
        intervals["birth"].ge(threshold)
        & (
            intervals["essential"]
            | intervals["death"].lt(threshold)
        )
    )
    return int(alive.sum())

def s22_cubical_persistence(score):
    if not S22_GUDHI_AVAILABLE:
        return pd.DataFrame(
            columns=[
                "dimension", "birth", "death",
                "lifetime", "essential",
            ]
        )

    # Sublevel filtration of -S equals a superlevel filtration of S.
    # Using -S avoids a needless 1-S subtraction.
    complex_ = gudhi.CubicalComplex(
        top_dimensional_cells=-np.asarray(score, dtype=float)
    )

    complex_.compute_persistence(
        homology_coeff_field=2,
        min_persistence=0.0,
    )

    records = []

    for dimension in [0, 1]:
        intervals = complex_.persistence_intervals_in_dimension(
            dimension
        )

        for birth_filtration, death_filtration in intervals:
            essential = bool(np.isinf(death_filtration))
            birth_score = float(-birth_filtration)
            death_score = (
                np.nan if essential
                else float(-death_filtration)
            )

            records.append({
                "dimension": dimension,
                "birth": birth_score,
                "death": death_score,
                "lifetime": (
                    np.nan if essential
                    else birth_score - death_score
                ),
                "essential": essential,
            })

    return pd.DataFrame(
        records,
        columns=[
            "dimension", "birth", "death",
            "lifetime", "essential",
        ],
    )

# -------------------------------------------------------------
# Validate plateau handling and exact-threshold conventions.
# -------------------------------------------------------------
s22_persistence_tests = []
_rng22 = np.random.default_rng(42)

for connectivity in [4, 8]:
    structure = (
        S22_FG4 if connectivity == 4 else S22_FG8
    )

    for shape in [(1, 1), (2, 2), (7, 9)]:
        for repeat in range(8):
            test_score = (
                _rng22.integers(0, 5, size=shape) / 4
            )

            intervals = s22_h0_persistence(
                test_score,
                connectivity=connectivity,
            )

            for threshold in [
                0, .125, .25, .5, .75, 1, 1.01
            ]:
                _, direct_count = ndi.label(
                    test_score >= threshold,
                    structure=structure,
                )

                interval_count = s22_interval_component_count(
                    intervals,
                    threshold,
                )

                s22_persistence_tests.append({
                    "connectivity": connectivity,
                    "shape": str(shape),
                    "repeat": repeat,
                    "threshold": threshold,
                    "direct_components": int(direct_count),
                    "interval_components": interval_count,
                    "passed": interval_count == direct_count,
                })

s22_persistence_tests_df = pd.DataFrame(
    s22_persistence_tests
)

assert s22_persistence_tests_df["passed"].all()

s22_persistence_tests_df.to_csv(
    S22_DIR / "h0_correctness_tests.csv",
    index=False,
)

if S22_GUDHI_AVAILABLE:
    ring_persistence = s22_cubical_persistence(
        _ring.astype(float)
    )

    ring_h1 = ring_persistence[
        ring_persistence["dimension"] == 1
    ]

    alive_holes = (
        ring_h1["birth"].ge(.5)
        & (
            ring_h1["essential"]
            | ring_h1["death"].lt(.5)
        )
    )

    assert int(alive_holes.sum()) == 1

print(
    "Exact H0 threshold-reconstruction checks passed:",
    len(s22_persistence_tests_df),
)
print(
    "Cubical H1:",
    "available and ring-tested"
    if S22_GUDHI_AVAILABLE
    else "not run; GUDHI is not installed",
)

### 22.5 Image-derived ridge filtrations on selected training regions

We select representative training annotations using topology flags,
then calculate a full-resolution Sato ridge map from the source image.

Processing rules:

- Image-derived maps are computed without ground-truth inputs.
- Labels select diagnostic regions and provide reference overlays only.
- Full-image ridge computation precedes cropping, avoiding artificial
  filter boundaries around individual annotations.
- Ridge amplitudes are divided by the image's positive-response
  99th percentile and clipped to [0,1].
- This normalization supports inspection; it is not probability calibration.
- Comparisons of absolute score lifetimes across images must account
  for image-specific normalization.
- Persistence regions are bounded to a maximum side length.
- The native annotation crop and resized annotation topology are recorded
  separately because resizing can erase thin structures or holes.
- Essential persistence intervals are excluded from finite-lifetime
  summaries.

In [ ]:
s22_roi_rows = []
_s22_selected_keys = set()
_s22_physical_counts = defaultdict(int)

_selection_specs = [
    (
        "disconnected",
        s22_valid_df[s22_valid_df["components8"] > 1],
        "components8",
    ),
    (
        "holes",
        s22_valid_df[s22_valid_df["holes8"] > 0],
        "holes8",
    ),
    (
        "erosion_sensitive",
        s22_valid_df[
            s22_valid_df["erosion_component_increase"] > 0
        ],
        "erosion_component_increase",
    ),
    (
        "large",
        s22_valid_df,
        "area_px",
    ),
]

for selection_reason, candidates, sort_column in _selection_specs:
    candidates = candidates.sort_values(
        [sort_column, "entry_id", "annotation_id"],
        ascending=[False, True, True],
    )

    for row in candidates.itertuples(index=False):
        key = (str(row.entry_id), str(row.annotation_id))
        physical_id = str(row.physical_image_id)

        if key in _s22_selected_keys:
            continue

        if (
            _s22_physical_counts[physical_id]
            >= S22_CFG["maximum_rois_per_physical_image"]
        ):
            continue

        s22_roi_rows.append({
            **row._asdict(),
            "selection_reason": selection_reason,
        })
        _s22_selected_keys.add(key)
        _s22_physical_counts[physical_id] += 1

        # One representative from each priority category first.
        break

# Fill remaining slots with deterministic large-instance examples.
for row in s22_valid_df.sort_values(
    ["area_px", "entry_id", "annotation_id"],
    ascending=[False, True, True],
).itertuples(index=False):
    if len(s22_roi_rows) >= S22_CFG["maximum_rois"]:
        break

    key = (str(row.entry_id), str(row.annotation_id))
    physical_id = str(row.physical_image_id)

    if (
        key in _s22_selected_keys
        or _s22_physical_counts[physical_id]
        >= S22_CFG["maximum_rois_per_physical_image"]
    ):
        continue

    s22_roi_rows.append({
        **row._asdict(),
        "selection_reason": "additional_large_example",
    })
    _s22_selected_keys.add(key)
    _s22_physical_counts[physical_id] += 1

if not s22_roi_rows:
    raise RuntimeError("No valid training regions are available.")

S22_RIDGE_DIR = S22_DIR / "ridge_cache"
S22_RIDGE_DIR.mkdir(exist_ok=True)

s22_roi_manifest = []
s22_curve_records = []
s22_persistence_summary = []

_ph_code_hash = s22_digest({
    "h0": s22_code_record(s22_h0_persistence.__code__),
    "cubical": s22_code_record(s22_cubical_persistence.__code__),
})

_roi_selection = pd.DataFrame(s22_roi_rows)

# Group by physical image so its full ridge map is loaded only once.
for physical_id, regions in _roi_selection.groupby(
    "physical_image_id",
    sort=True,
):
    first_entry = str(regions.iloc[0]["entry_id"])
    info = s22_info[first_entry]
    image_path = Path(TRAIN_IMAGE_DIR) / info["file_name"]

    ridge_key = s22_digest({
        "image_sha256": s22_file_hash(image_path),
        "sigmas": S22_CFG["ridge_sigmas"],
        "black_ridges": True,
        "input_scaling": "uint8_divided_by_255",
        "skimage": skimage.__version__,
    })[:20]

    ridge_path = S22_RIDGE_DIR / f"{ridge_key}.npy"

    image = load_solar_image(
        s22_raw_ids[first_entry]
    )

    if ridge_path.is_file():
        ridge = np.load(ridge_path, mmap_mode="r")

        if (
            ridge.shape != image.shape
            or ridge.dtype != np.float32
        ):
            raise ValueError("Invalid cached ridge-map shape/dtype.")
    else:
        print(
            "Computing full-image ridge map:",
            physical_id,
            flush=True,
        )

        ridge = sato(
            image.astype(np.float32) / 255.0,
            sigmas=tuple(S22_CFG["ridge_sigmas"]),
            black_ridges=True,
        ).astype(np.float32)

        if not np.isfinite(ridge).all():
            raise ValueError("Nonfinite ridge responses.")

        temporary = ridge_path.with_suffix(".npy.tmp")

        with open(temporary, "wb") as handle:
            np.save(handle, ridge)

        temporary.replace(ridge_path)

    positive = np.asarray(ridge)[np.asarray(ridge) > 0]

    normalization_scale = (
        float(np.percentile(
            positive,
            S22_CFG["ridge_normalization_positive_percentile"],
        ))
        if positive.size else 0.0
    )

    del positive

    for region in regions.itertuples(index=False):
        entry_id = str(region.entry_id)
        annotation_id = str(region.annotation_id)

        padding = S22_CFG["roi_padding_native_px"]

        y0 = max(0, int(region.y0) - padding)
        y1 = min(image.shape[0], int(region.y1) + padding)
        x0 = max(0, int(region.x0) - padding)
        x1 = min(image.shape[1], int(region.x1) + padding)

        raw_score = np.asarray(
            ridge[y0:y1, x0:x1],
            dtype=np.float64,
        )

        if normalization_scale > 0:
            native_score = np.clip(
                raw_score / normalization_scale,
                0, 1,
            )
        else:
            native_score = np.zeros_like(raw_score)

        annotation = s22_annotation_lookup[
            (entry_id, annotation_id)
        ]

        native_mask = np.asarray(
            rasterize_annotation(annotation, image.shape),
            dtype=bool,
        )[y0:y1, x0:x1]

        native_image = image[y0:y1, x0:x1]

        scale = min(
            1.0,
            S22_CFG["maximum_persistence_side"]
            / max(native_score.shape),
        )

        analysis_shape = tuple(
            max(1, int(round(size * scale)))
            for size in native_score.shape
        )

        if analysis_shape != native_score.shape:
            score = resize(
                native_score,
                analysis_shape,
                order=1,
                preserve_range=True,
                anti_aliasing=True,
            ).astype(np.float64)

            analysis_mask = resize(
                native_mask.astype(np.uint8),
                analysis_shape,
                order=0,
                preserve_range=True,
                anti_aliasing=False,
            ) > .5
        else:
            score = native_score.copy()
            analysis_mask = native_mask.copy()

        score = np.clip(score, 0, 1)

        roi_id = s22_digest({
            "entry": entry_id,
            "annotation": annotation_id,
            "bounds": [y0, y1, x0, x1],
            "ridge_key": ridge_key,
            "analysis_shape": list(analysis_shape),
            "configuration": S22_CFG,
            "persistence_code": _ph_code_hash,
        })[:20]

        h0 = s22_h0_persistence(score, connectivity=8)
        cubical = s22_cubical_persistence(score)

        h0.to_csv(
            S22_DIR / f"{roi_id}_h0_intervals.csv",
            index=False,
        )
        cubical.to_csv(
            S22_DIR / f"{roi_id}_cubical_intervals.csv",
            index=False,
        )

        np.savez_compressed(
            S22_DIR / f"{roi_id}_region.npz",
            score=score,
            analysis_mask=analysis_mask,
            native_mask=native_mask,
            native_image=native_image,
            native_score=native_score.astype(np.float32),
        )

        native_topology = s22_betti(native_mask, 8)
        resized_topology = s22_betti(analysis_mask, 8)

        s22_roi_manifest.append({
            "roi_id": roi_id,
            "entry_id": entry_id,
            "physical_image_id": physical_id,
            "annotation_id": annotation_id,
            "selection_reason": region.selection_reason,
            "y0": y0, "y1": y1,
            "x0": x0, "x1": x1,
            "native_height": native_score.shape[0],
            "native_width": native_score.shape[1],
            "analysis_height": score.shape[0],
            "analysis_width": score.shape[1],
            "normalization_scale": normalization_scale,
            "native_components8": native_topology["components"],
            "analysis_components8": resized_topology["components"],
            "native_holes8": native_topology["holes"],
            "analysis_holes8": resized_topology["holes"],
            "gudhi_computed": S22_GUDHI_AVAILABLE,
        })

        for threshold in S22_CFG["thresholds"]:
            foreground = score >= threshold

            topology8 = s22_betti(foreground, 8)
            topology4 = s22_betti(foreground, 4)

            reconstructed_components = (
                s22_interval_component_count(h0, threshold)
            )

            assert (
                reconstructed_components
                == topology8["components"]
            ), "H0 interval/direct component-count disagreement."

            if S22_GUDHI_AVAILABLE:
                cubical_h0 = cubical[
                    cubical["dimension"] == 0
                ]
                cubical_h1 = cubical[
                    cubical["dimension"] == 1
                ]

                b0 = s22_interval_component_count(
                    cubical_h0, threshold
                )
                b1 = s22_interval_component_count(
                    cubical_h1, threshold
                )

                assert b0 == topology8["components"]
                assert b1 == topology8["holes"]

            s22_curve_records.append({
                "roi_id": roi_id,
                "threshold": threshold,
                "components8": topology8["components"],
                "holes8": topology8["holes"],
                "euler8": topology8["euler"],
                "components4": topology4["components"],
                "holes4": topology4["holes"],
                "foreground_fraction": float(foreground.mean()),
                "h0_reconstructed_components": reconstructed_components,
            })

        finite_h0 = h0.loc[
            ~h0["essential"], "lifetime"
        ].to_numpy(dtype=float)

        h1 = cubical[
            cubical["dimension"] == 1
        ]
        finite_h1 = h1.loc[
            ~h1["essential"], "lifetime"
        ].to_numpy(dtype=float)

        s22_persistence_summary.append({
            "roi_id": roi_id,
            "finite_h0_intervals": len(finite_h0),
            "essential_h0_intervals": int(
                h0["essential"].sum()
            ),
            "h0_lifetime_median": (
                float(np.median(finite_h0))
                if len(finite_h0) else np.nan
            ),
            "h0_lifetime_p90": (
                float(np.quantile(finite_h0, .9))
                if len(finite_h0) else np.nan
            ),
            "h0_lifetime_maximum": (
                float(finite_h0.max())
                if len(finite_h0) else np.nan
            ),
            "finite_h1_intervals": (
                len(finite_h1)
                if S22_GUDHI_AVAILABLE else np.nan
            ),
            "h1_lifetime_maximum": (
                float(finite_h1.max())
                if len(finite_h1) else np.nan
            ),
        })

        print(
            "Region finished:",
            roi_id,
            "| shape:", score.shape,
            "| finite H0 intervals:", len(finite_h0),
            flush=True,
        )

    del image, ridge

s22_roi_manifest_df = pd.DataFrame(s22_roi_manifest)
s22_curves_df = pd.DataFrame(s22_curve_records)
s22_persistence_summary_df = pd.DataFrame(
    s22_persistence_summary
)

s22_roi_manifest_df.to_csv(
    S22_DIR / "persistence_region_manifest.csv",
    index=False,
)
s22_curves_df.to_csv(
    S22_DIR / "threshold_topology_curves.csv",
    index=False,
)
s22_persistence_summary_df.to_csv(
    S22_DIR / "persistence_summary.csv",
    index=False,
)

display(s22_roi_manifest_df)
display(s22_persistence_summary_df)

In [ ]:
for region in s22_roi_manifest_df.itertuples(index=False):
    roi_id = region.roi_id

    with np.load(
        S22_DIR / f"{roi_id}_region.npz",
        allow_pickle=False,
    ) as saved:
        native_image = saved["native_image"].copy()
        native_mask = saved["native_mask"].copy()
        native_score = saved["native_score"].copy()
        score = saved["score"].copy()

    h0 = pd.read_csv(
        S22_DIR / f"{roi_id}_h0_intervals.csv"
    )
    curves = s22_curves_df[
        s22_curves_df["roi_id"] == roi_id
    ]

    finite = h0[~h0["essential"]].copy()
    essential = h0[h0["essential"]].copy()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0, 0].imshow(
        native_image,
        cmap="gray",
        vmin=0,
        vmax=255,
    )
    if native_mask.any() and not native_mask.all():
        axes[0, 0].contour(
            native_mask,
            levels=[.5],
            colors=["cyan"],
            linewidths=1,
        )
    axes[0, 0].set_title("Native image and reference annotation")
    axes[0, 0].axis("off")

    artist = axes[0, 1].imshow(
        native_score,
        cmap="magma",
        vmin=0,
        vmax=1,
    )
    axes[0, 1].set_title(
        "Full-image-derived ridge response\n"
        "Image-specific normalization; not a probability"
    )
    axes[0, 1].axis("off")
    fig.colorbar(
        artist, ax=axes[0, 1],
        fraction=.046, pad=.04,
    )

    axes[0, 2].plot(
        curves["threshold"],
        curves["components8"],
        label="β0: foreground 8",
    )
    axes[0, 2].plot(
        curves["threshold"],
        curves["holes8"],
        label="β1: foreground 8 / background 4",
    )
    axes[0, 2].plot(
        curves["threshold"],
        curves["components4"],
        linestyle="--",
        alpha=.7,
        label="β0: foreground 4",
    )
    axes[0, 2].set_title("Threshold topology curves")
    axes[0, 2].set_xlabel("Superlevel threshold")
    axes[0, 2].set_ylabel("Count")
    axes[0, 2].legend(fontsize=8)
    axes[0, 2].grid(alpha=.2)

    if len(finite):
        axes[1, 0].scatter(
            finite["birth"],
            finite["death"],
            s=12,
            alpha=.5,
            color="#496b8a",
        )

    axes[1, 0].plot(
        [0, 1], [0, 1],
        linestyle="--",
        color="gray",
    )
    axes[1, 0].set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="Birth score",
        ylabel="Merge/death score",
        title=(
            "Finite H0 persistence diagram\n"
            f"Essential intervals excluded: {len(essential)}"
        ),
    )
    axes[1, 0].grid(alpha=.2)

    if len(finite):
        longest = finite.nlargest(20, "lifetime").reset_index(
            drop=True
        )

        for index, interval in longest.iterrows():
            axes[1, 1].plot(
                [interval["death"], interval["birth"]],
                [index, index],
                linewidth=3,
                color="#3a9d70",
            )

        axes[1, 1].set_yticks(
            range(len(longest)),
            range(1, len(longest) + 1),
        )
    else:
        axes[1, 1].text(
            .5, .5, "No positive finite lifetimes",
            ha="center", va="center",
            transform=axes[1, 1].transAxes,
        )

    axes[1, 1].set_title("Longest 20 finite H0 intervals")
    axes[1, 1].set_xlabel("Score interval: merge → birth")
    axes[1, 1].set_ylabel("Lifetime rank")
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].grid(alpha=.2)

    if S22_GUDHI_AVAILABLE:
        cubical = pd.read_csv(
            S22_DIR / f"{roi_id}_cubical_intervals.csv"
        )
        finite_h1 = cubical[
            (cubical["dimension"] == 1)
            & ~cubical["essential"]
        ]

        axes[1, 2].scatter(
            finite_h1["birth"],
            finite_h1["death"],
            s=14,
            alpha=.6,
            color="#a64d79",
        )
        axes[1, 2].plot(
            [0, 1], [0, 1],
            color="gray", linestyle="--",
        )
        axes[1, 2].set(
            xlabel="Hole birth score",
            ylabel="Hole death score",
            xlim=(0, 1),
            ylim=(0, 1),
            title="Finite cubical H1 persistence",
        )
        axes[1, 2].grid(alpha=.2)
    else:
        axes[1, 2].plot(
            curves["threshold"],
            curves["foreground_fraction"],
            color="#e58b25",
        )
        axes[1, 2].set(
            xlabel="Superlevel threshold",
            ylabel="Foreground fraction",
            title=(
                "Foreground occupancy\n"
                "Formal H1 persistence not run: GUDHI unavailable"
            ),
        )
        axes[1, 2].grid(alpha=.2)

    fig.suptitle(
        f"{region.entry_id} | annotation {region.annotation_id}\n"
        f"Reason: {region.selection_reason} | "
        f"analysis grid: {region.analysis_height}×{region.analysis_width}",
        fontsize=12,
    )

    fig.tight_layout()
    s22_finish(
        fig,
        f"03_persistence_diagnostics_{roi_id}.png",
    )

In [ ]:
# Show threshold effects for the first selected region.
_first_roi = s22_roi_manifest_df.iloc[0]
_roi_id = _first_roi["roi_id"]

with np.load(
    S22_DIR / f"{_roi_id}_region.npz",
    allow_pickle=False,
) as saved:
    _score = saved["score"].copy()
    _reference = saved["analysis_mask"].copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0, 0].imshow(_score, cmap="magma", vmin=0, vmax=1)
axes[0, 0].set_title("Analysis-grid ridge response")

for axis, threshold in zip(
    list(axes.flat)[1:],
    [.10, .25, .40, .60, .80],
):
    foreground = _score >= threshold
    component_labels, count = ndi.label(
        foreground,
        structure=S22_FG8,
    )

    axis.imshow(
        np.ma.masked_where(
            component_labels == 0,
            component_labels,
        ),
        cmap="nipy_spectral",
        interpolation="nearest",
    )

    if _reference.any() and not _reference.all():
        axis.contour(
            _reference,
            levels=[.5],
            colors=["white"],
            linewidths=.7,
        )

    topology = s22_betti(foreground, 8)

    axis.set_title(
        f"Threshold ≥ {threshold:.2f}\n"
        f"components={count}, holes={topology['holes']}"
    )

for axis in axes.flat:
    axis.axis("off")

fig.suptitle(
    "Threshold sensitivity — colors identify components at each threshold\n"
    "Colors do not track component identity between panels",
    fontsize=12,
)
fig.tight_layout()
s22_finish(fig, "04_threshold_mask_gallery.png")

_resolution_changes = s22_roi_manifest_df[
    (
        s22_roi_manifest_df["native_components8"]
        != s22_roi_manifest_df["analysis_components8"]
    )
    | (
        s22_roi_manifest_df["native_holes8"]
        != s22_roi_manifest_df["analysis_holes8"]
    )
]

print(
    "Selected reference masks whose topology changed under resizing:",
    len(_resolution_changes),
)

display(
    s22_roi_manifest_df[
        [
            "roi_id",
            "native_height",
            "native_width",
            "analysis_height",
            "analysis_width",
            "native_components8",
            "analysis_components8",
            "native_holes8",
            "analysis_holes8",
        ]
    ]
)

print(
    "Persistence results describe the recorded analysis grid. "
    "They must not be presented as native-resolution topology when "
    "downsampling changes the reference structure."
)

### 22.6 Interpretation and intervention hypotheses

This step does not select a connectivity correction solely because its
visualization looks cleaner.

Candidate hypotheses for later experiments:

1. Erosion-sensitive, thin connections:
   Investigate resolution and continuity objectives. Measure whether
   improved continuity increases correct instance matches or false merges.

2. Multiple annotations in one union component:
   Investigate boundary, affinity or instance-separation evidence.
   Do not use foreground connectivity alone to assign instance identity.

3. Legitimate disconnected annotations:
   Preserve the possibility of one predicted instance containing
   multiple disconnected parts. Connected components alone may be an
   insufficient reconstruction method.

4. Short-lived image-derived components:
   Investigate persistence as a candidate confidence feature.
   Do not discard them automatically: faint real filaments can also
   have short normalized lifetimes.

5. Long-lived unwanted structures:
   Persistence measures stability, not semantic correctness.
   Sunspots, artifacts and background structures can persist.

6. Holes:
   Investigate annotation conventions and image evidence before
   proposing hole filling or a topology penalty.

Any intervention must be compared against a simpler baseline using
fixed partitions, full-image reconstruction, the pinned official scorer,
and saved per-instance errors.

Topology extracted from ground truth remains a diagnostic or auxiliary
training target. It is not an inference-time input.

In [ ]:
s22_summary = {
    "step": 22,
    "run_hash": S22_RUN_HASH,
    "training_annotation_entries": len(s22_train_ids),
    "expected_annotations": len(expected_keys),
    "measured_annotations": len(s22_instances_df),
    "nonempty_annotations": len(s22_valid_df),
    "empty_annotations": int(
        s22_instances_df["status"].eq("empty").sum()
    ),
    "extraction_errors": len(s22_failures),
    "disconnected_annotations": int(
        s22_valid_df["components8"].gt(1).sum()
    ),
    "annotations_with_holes": int(
        s22_valid_df["holes8"].gt(0).sum()
    ),
    "diagonal_sensitive_annotations": int(
        s22_valid_df["diagonal_sensitive"].sum()
    ),
    "entries_with_shared_union_components": int(
        s22_images_df["shared_union_components"].gt(0).sum()
    ),
    "h0_correctness_checks": len(s22_persistence_tests_df),
    "h0_correctness_checks_passed": bool(
        s22_persistence_tests_df["passed"].all()
    ),
    "persistence_regions": len(s22_roi_manifest_df),
    "reference_regions_changed_by_resizing": len(
        _resolution_changes
    ),
    "formal_cubical_h1_status": (
        "COMPUTED"
        if S22_GUDHI_AVAILABLE
        else "NOT_RUN_GUDHI_UNAVAILABLE"
    ),
    "predictive_improvement_demonstrated": False,
    "original_targets_modified": False,
    "validation_used_for_fitting": False,
}

with open(
    S22_DIR / "step22_summary.json", "w"
) as handle:
    json.dump(s22_summary, handle, indent=2)

print(json.dumps(s22_summary, indent=2))
print("\nStep 22 outputs:", S22_DIR)

assert measured_keys == expected_keys
assert not s22_failures
assert s22_persistence_tests_df["passed"].all()

print(
    "\nCore topology and H0 diagnostics completed. "
    "Interpret the saved outputs before promoting a connectivity rule."
)

## 23. Differential Geometry and Skeleton Graphs

### 23.1 Scope and conventions

A skeleton graph represents centerline pixels as nodes and neighboring
pixel connections as weighted edges.

We retain Step 19's graph-construction convention:

- Orthogonal steps have length 1 pixel.
- Diagonal steps have length √2 pixels.
- Redundant diagonal connections follow the existing Step 19 policy.

We distinguish:

- Total graph-edge length.
- Ordered path length.
- Endpoint count.
- Junction-pixel count.
- Connected junction zones.
- Graph cycle rank.

These quantities are not interchangeable.

Coordinates:

- Arrays and graph coordinates use row/column order.
- Saved geometric profiles use x/y order in original image coordinates.
- Lengths are measured in native image pixels.
- Curvature is measured in inverse pixels.
- Axial orientation is represented modulo π.

An axial orientation describes an unoriented line. An endpoint's outward
direction is different: it points away from the observed fragment and
is useful for testing whether two endpoints face one another.

Ground-truth geometry is used for diagnostics and controlled training
experiments. Future inference must derive these descriptors from
predicted masks and observed images.

No link is automatically accepted or added to our segmentation here.

In [ ]:
import hashlib
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from scipy.sparse import issparse, triu
from scipy.sparse.csgraph import connected_components, laplacian

from skimage.morphology import skeletonize

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

from IPython.display import display

_s23_required = [
    "annotations_data", "train_df", "valid_df",
    "OUTPUT_ROOT", "rasterize_annotation",
    "load_solar_image", "s19_pixel_graph",
]

_s23_missing = [
    name for name in _s23_required
    if name not in globals()
]

if _s23_missing:
    raise RuntimeError(
        "Run the earlier notebook setup and Step 19 graph-definition "
        "cells first. Missing: " + ", ".join(_s23_missing)
    )

S23_CFG = {
    "version": "step23_v1",
    "resample_spacing_px": 2.0,
    "smoothing_sigma_px": 3.0,
    "minimum_profile_length_px": 18.0,
    "endpoint_window_px": 16.0,
    "minimum_endpoint_support_px": 8.0,
    "maximum_link_gap_px": 64.0,
    "link_samples": 33,
    "link_band_half_width_px": 2.0,
    "pilot_entries": 80,
    "maximum_positive_cuts_per_entry": 3,
    "cut_radii_px": [4, 8],
    "folds": 3,
    "purge_days": int(PURGE_DAYS),
    "seed": int(SEED),
    "spectrum_examples": 8,
    "maximum_spectrum_nodes": 256,
}

S23_DIR = Path(OUTPUT_ROOT) / "step23_geometry"
S23_DIR.mkdir(parents=True, exist_ok=True)

S23_GRAPH_DIR = S23_DIR / "graphs"
S23_GRAPH_DIR.mkdir(exist_ok=True)

def s23_digest(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode()
    ).hexdigest()

def s23_finish(fig, filename):
    fig.savefig(
        S23_DIR / filename,
        dpi=160,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
    plt.close(fig)

s23_info = {
    str(info["id"]): info
    for info in annotations_data["images"]
}

s23_raw_ids = {
    str(info["id"]): info["id"]
    for info in annotations_data["images"]
}

s23_annotations = defaultdict(list)

for annotation in annotations_data["annotations"]:
    s23_annotations[str(annotation["image_id"])].append(annotation)

s23_annotation_lookup = {
    (str(annotation["image_id"]), str(annotation["id"])): annotation
    for annotation in annotations_data["annotations"]
}

s23_meta = train_df[
    ["image_id", "temporal_block", "capture_datetime"]
].copy()

s23_meta["image_id"] = s23_meta["image_id"].astype(str)
s23_meta["capture_datetime"] = pd.to_datetime(
    s23_meta["capture_datetime"]
)

assert not s23_meta["image_id"].duplicated().any()
assert not s23_meta.isna().any().any()
assert set(s23_meta["image_id"]).isdisjoint(
    set(valid_df["image_id"].astype(str))
)

with open(S23_DIR / "configuration.json", "w") as handle:
    json.dump(S23_CFG, handle, indent=2)

print("Training annotation entries:", len(s23_meta))
print("Output:", S23_DIR)

### 23.2 Ordered graph paths and differential geometry

We decompose graph edges into maximal paths whose internal nodes have
degree two.

Paths terminate at endpoints or junction pixels. Pure cycles are traced
separately. Edges internal to a connected junction zone are recorded
separately rather than called independent branches.

For each sufficiently long path:

1. Calculate cumulative weighted arc length.
2. Resample at a fixed physical spacing.
3. Smooth x and y using a sigma expressed in native pixels.
4. Estimate tangent and curvature.
5. Exclude unstable open-path boundary samples.

For an endpoint, trace a short inward chain without crossing a junction.
Fit a local principal direction to that chain and orient it outward.

Reliability includes:

- Available inward support length.
- Directional anisotropy.
- Whether the minimum support requirement is met.

Short or ambiguous endpoints remain recorded. They are not assigned
invented confident directions.

In [ ]:
def s23_graph(mask):
    """Reuse Step 19's graph builder; translate to native coordinates."""
    mask = np.asarray(mask, dtype=bool)

    if not mask.any():
        raise ValueError("Cannot construct a graph for an empty mask.")

    yy, xx = np.nonzero(mask)
    y0, y1 = int(yy.min()), int(yy.max()) + 1
    x0, x1 = int(xx.min()), int(xx.max()) + 1

    crop = np.pad(mask[y0:y1, x0:x1], 1)
    skeleton = skeletonize(crop)

    result = s19_pixel_graph(skeleton)

    # Locate the coordinate array and sparse graph, then validate them.
    if not isinstance(result, (tuple, list)):
        raise TypeError(
            "Unexpected Step 19 graph return format."
        )

    graphs = [
        item for item in result
        if issparse(item)
    ]
    coordinate_arrays = [
        item for item in result
        if isinstance(item, np.ndarray)
        and item.ndim == 2
        and item.shape[1] == 2
    ]

    if len(graphs) != 1 or len(coordinate_arrays) != 1:
        raise ValueError(
            "Cannot unambiguously identify Step 19 graph coordinates "
            "and sparse adjacency."
        )

    graph = graphs[0].tocsr().astype(float)
    coords = coordinate_arrays[0].astype(float)

    if graph.shape != (len(coords), len(coords)):
        raise ValueError("Graph/coordinate dimensions disagree.")

    if (graph - graph.T).nnz:
        raise ValueError("Expected a symmetric graph.")

    if graph.nnz and (
        not np.isfinite(graph.data).all()
        or (graph.data <= 0).any()
    ):
        raise ValueError("Graph edge lengths must be positive and finite.")

    edges = triu(graph, k=1).tocoo()

    if edges.nnz:
        expected_lengths = np.linalg.norm(
            coords[edges.row] - coords[edges.col],
            axis=1,
        )
        if not np.allclose(
            edges.data, expected_lengths,
            rtol=0, atol=1e-7,
        ):
            raise ValueError(
                "Step 19 graph weights are not pixel step lengths."
            )

    coords += np.array([y0 - 1, x0 - 1])

    return coords, graph

def s23_paths(graph):
    """Trace all non-junction-zone edges exactly once."""
    graph = graph.tocsr()
    degree = np.diff(graph.indptr)
    junction_nodes = np.flatnonzero(degree >= 3)

    zone = np.full(graph.shape[0], -1, dtype=int)

    if len(junction_nodes):
        zone_count, zone_labels = connected_components(
            graph[junction_nodes][:, junction_nodes],
            directed=False,
        )
        zone[junction_nodes] = zone_labels
    else:
        zone_count = 0

    def neighbors(node):
        return graph.indices[
            graph.indptr[node]:graph.indptr[node + 1]
        ]

    def edge_key(a, b):
        return (min(int(a), int(b)), max(int(a), int(b)))

    visited = set()
    paths = []
    internal_junction_length = 0.0

    edges = triu(graph, k=1).tocoo()

    for a, b, length in zip(
        edges.row, edges.col, edges.data
    ):
        if zone[a] >= 0 and zone[a] == zone[b]:
            visited.add(edge_key(a, b))
            internal_junction_length += float(length)

    def walk(start, following):
        path = [int(start), int(following)]
        visited.add(edge_key(start, following))
        previous, current = int(start), int(following)

        while degree[current] == 2:
            next_nodes = [
                int(node)
                for node in neighbors(current)
                if int(node) != previous
            ]

            if not next_nodes:
                break

            following = next_nodes[0]
            key = edge_key(current, following)

            if key in visited:
                break

            visited.add(key)
            path.append(following)
            previous, current = current, following

            if current == start:
                break

        return np.asarray(path, dtype=int)

    for start in np.flatnonzero(degree != 2):
        for following in neighbors(start):
            if edge_key(start, following) not in visited:
                paths.append(walk(start, following))

    # Remaining edges belong to cycles not reached from a critical node.
    for a, b in zip(edges.row, edges.col):
        if edge_key(a, b) not in visited:
            paths.append(walk(a, b))

    traced_length = sum(
        float(graph[path[:-1], path[1:]].sum())
        for path in paths
    )

    if not np.isclose(
        traced_length + internal_junction_length,
        float(edges.data.sum()),
        rtol=1e-8, atol=1e-6,
    ):
        raise AssertionError("Path tracing lost or duplicated graph edges.")

    return paths, zone, int(zone_count), internal_junction_length

def s23_profile(points_yx):
    points = np.asarray(points_yx, dtype=float)[:, ::-1]

    if len(points) < 2:
        return None

    arc = np.r_[
        0.0,
        np.cumsum(
            np.linalg.norm(np.diff(points, axis=0), axis=1)
        ),
    ]

    if arc[-1] < S23_CFG["minimum_profile_length_px"]:
        return None

    closed = bool(np.array_equal(points[0], points[-1]))
    spacing = S23_CFG["resample_spacing_px"]

    grid = np.arange(0, arc[-1], spacing)

    if len(grid) < 5:
        return None

    xy = np.column_stack([
        np.interp(grid, arc, points[:, 0]),
        np.interp(grid, arc, points[:, 1]),
    ])

    sigma_samples = (
        S23_CFG["smoothing_sigma_px"] / spacing
    )

    smooth = ndi.gaussian_filter1d(
        xy,
        sigma=sigma_samples,
        axis=0,
        mode="wrap" if closed else "nearest",
    )

    if closed:
        first = (
            np.roll(smooth, -1, axis=0)
            - np.roll(smooth, 1, axis=0)
        ) / (2 * spacing)
        second = (
            np.roll(smooth, -1, axis=0)
            - 2 * smooth
            + np.roll(smooth, 1, axis=0)
        ) / spacing**2
    else:
        first = np.gradient(smooth, spacing, axis=0)
        second = np.gradient(first, spacing, axis=0)

    speed2 = np.sum(first**2, axis=1)

    curvature = np.abs(
        first[:, 0] * second[:, 1]
        - first[:, 1] * second[:, 0]
    ) / np.maximum(speed2, 1e-12)**1.5

    valid = np.isfinite(curvature) & (speed2 > .05)

    if not closed:
        trim = int(np.ceil(3 * sigma_samples))
        valid[:trim] = False
        valid[-trim:] = False

    tangent = first / np.maximum(
        np.linalg.norm(first, axis=1, keepdims=True),
        1e-12,
    )

    theta = np.mod(
        np.arctan2(tangent[:, 1], tangent[:, 0]),
        np.pi,
    )

    return {
        "arc": grid,
        "xy": smooth,
        "tangent": tangent,
        "theta_axial": theta,
        "curvature": curvature,
        "valid": valid,
        "closed": closed,
    }

def s23_endpoints(coords, graph):
    degree = np.diff(graph.indptr)
    records = []

    for endpoint in np.flatnonzero(degree == 1):
        chain = [int(endpoint)]
        previous = -1
        current = int(endpoint)
        support = 0.0

        while support < S23_CFG["endpoint_window_px"]:
            neighbors = graph.indices[
                graph.indptr[current]:graph.indptr[current + 1]
            ]
            choices = [
                int(node) for node in neighbors
                if int(node) != previous
            ]

            if len(choices) != 1:
                break

            following = choices[0]
            support += float(graph[current, following])
            chain.append(following)
            previous, current = current, following

            if degree[current] != 2:
                break

        points_xy = coords[chain][:, ::-1]

        direction = np.array([np.nan, np.nan])
        anisotropy = np.nan

        if len(points_xy) >= 3:
            centered = points_xy - points_xy.mean(axis=0)
            covariance = centered.T @ centered / len(centered)
            values, vectors = np.linalg.eigh(covariance)
            direction = vectors[:, -1]

            outward_reference = (
                points_xy[0] - points_xy[1:].mean(axis=0)
            )

            if np.dot(direction, outward_reference) < 0:
                direction = -direction

            anisotropy = float(
                (values[-1] - values[0])
                / max(values.sum(), 1e-12)
            )

        reliable = bool(
            support >= S23_CFG["minimum_endpoint_support_px"]
            and np.isfinite(direction).all()
        )

        records.append({
            "node": int(endpoint),
            "x": float(coords[endpoint, 1]),
            "y": float(coords[endpoint, 0]),
            "outward_x": float(direction[0]),
            "outward_y": float(direction[1]),
            "support_px": support,
            "anisotropy": anisotropy,
            "direction_reliable": reliable,
        })

    return records

In [ ]:
# Basic geometry checks before the population run.
_horizontal = np.zeros((32, 32), dtype=bool)
_horizontal[16, 4:25] = True

_coords, _graph = s23_graph(_horizontal)
_paths, _, _, _internal = s23_paths(_graph)
_endpoints = s23_endpoints(_coords, _graph)

assert len(_endpoints) == 2
assert np.isclose(triu(_graph, k=1).sum(), 20.0)

_outward_vectors = np.array([
    [row["outward_x"], row["outward_y"]]
    for row in _endpoints
])

assert np.isfinite(_outward_vectors).all()
assert np.dot(_outward_vectors[0], _outward_vectors[1]) < -.99

print("Straight-line length and endpoint-direction checks passed.")

s23_instance_rows = []
s23_endpoint_rows = []
s23_branch_rows = []

for position, entry_id in enumerate(
    sorted(s23_meta["image_id"]), start=1
):
    info = s23_info[entry_id]
    shape = (int(info["height"]), int(info["width"]))

    for annotation in s23_annotations[entry_id]:
        annotation_id = str(annotation["id"])

        mask = np.asarray(
            rasterize_annotation(annotation, shape),
            dtype=bool,
        )

        if not mask.any():
            s23_instance_rows.append({
                "entry_id": entry_id,
                "annotation_id": annotation_id,
                "status": "empty",
            })
            continue

        coords, graph = s23_graph(mask)
        paths, zones, zone_count, internal_length = s23_paths(graph)
        endpoints = s23_endpoints(coords, graph)

        component_count, _ = connected_components(
            graph, directed=False
        )

        edges = triu(graph, k=1).tocoo()
        degree = np.diff(graph.indptr)

        graph_key = s23_digest({
            "entry_id": entry_id,
            "annotation_id": annotation_id,
            "configuration": S23_CFG,
        })[:20]

        archive = {
            "coords_yx": coords,
            "graph_indptr": graph.indptr,
            "graph_indices": graph.indices,
            "graph_data": graph.data,
            "graph_shape": np.asarray(graph.shape),
        }

        valid_curvatures = []

        for branch_id, path in enumerate(paths):
            branch_length = float(
                graph[path[:-1], path[1:]].sum()
            )

            profile = s23_profile(coords[path])
            chord = float(
                np.linalg.norm(coords[path[-1]] - coords[path[0]])
            )

            profile_count = 0
            curvature_p90 = np.nan

            if profile is not None:
                curvature_values = profile["curvature"][
                    profile["valid"]
                ]
                profile_count = len(curvature_values)

                if profile_count:
                    curvature_p90 = float(
                        np.quantile(curvature_values, .9)
                    )
                    valid_curvatures.append(curvature_values)

                for key in [
                    "arc", "xy", "tangent",
                    "theta_axial", "curvature", "valid",
                ]:
                    archive[f"branch_{branch_id}_{key}"] = (
                        profile[key]
                    )

            s23_branch_rows.append({
                "entry_id": entry_id,
                "annotation_id": annotation_id,
                "graph_key": graph_key,
                "branch_id": branch_id,
                "length_px": branch_length,
                "chord_px": chord,
                "length_over_chord": (
                    branch_length / chord if chord > 0 else np.nan
                ),
                "closed_pixel_path": bool(path[0] == path[-1]),
                "profile_valid_points": profile_count,
                "curvature_p90": curvature_p90,
                "start_junction_zone": int(zones[path[0]]),
                "end_junction_zone": int(zones[path[-1]]),
            })

        for endpoint_index, endpoint in enumerate(endpoints):
            s23_endpoint_rows.append({
                "entry_id": entry_id,
                "annotation_id": annotation_id,
                "graph_key": graph_key,
                "endpoint_id": endpoint_index,
                **endpoint,
            })

        all_curvature = (
            np.concatenate(valid_curvatures)
            if valid_curvatures else np.array([])
        )

        s23_instance_rows.append({
            "entry_id": entry_id,
            "annotation_id": annotation_id,
            "graph_key": graph_key,
            "status": "ok",
            "area_px": int(mask.sum()),
            "nodes": len(coords),
            "edges": int(edges.nnz),
            "graph_components": int(component_count),
            "cycle_rank": int(
                edges.nnz - len(coords) + component_count
            ),
            "total_edge_length_px": float(edges.data.sum()),
            "endpoint_count": len(endpoints),
            "reliable_endpoints": sum(
                row["direction_reliable"] for row in endpoints
            ),
            "junction_pixels": int((degree >= 3).sum()),
            "junction_zones": zone_count,
            "junction_internal_length_px": internal_length,
            "traced_paths": len(paths),
            "curvature_p90": (
                float(np.quantile(all_curvature, .9))
                if len(all_curvature) else np.nan
            ),
        })

        temporary = S23_GRAPH_DIR / f"{graph_key}.npz.tmp"

        with open(temporary, "wb") as handle:
            np.savez_compressed(handle, **archive)

        temporary.replace(
            S23_GRAPH_DIR / f"{graph_key}.npz"
        )

        del mask, archive, coords, graph

    if position % 20 == 0 or position == len(s23_meta):
        print(
            f"Graph entries: {position}/{len(s23_meta)}",
            flush=True,
        )

        pd.DataFrame(s23_instance_rows).to_csv(
            S23_DIR / "instance_graph_geometry.csv",
            index=False,
        )
        pd.DataFrame(s23_endpoint_rows).to_csv(
            S23_DIR / "endpoint_geometry.csv",
            index=False,
        )
        pd.DataFrame(s23_branch_rows).to_csv(
            S23_DIR / "ordered_path_geometry.csv",
            index=False,
        )

s23_instances_df = pd.DataFrame(s23_instance_rows)
s23_endpoints_df = pd.DataFrame(s23_endpoint_rows)
s23_branches_df = pd.DataFrame(s23_branch_rows)

expected_annotations = sum(
    len(s23_annotations[entry_id])
    for entry_id in s23_meta["image_id"]
)

assert len(s23_instances_df) == expected_annotations
assert not s23_instances_df.duplicated(
    ["entry_id", "annotation_id"]
).any()

display(
    s23_instances_df.select_dtypes(
        include="number"
    ).describe().T
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for axis, column, title in [
    (axes[0, 0], "total_edge_length_px", "Total graph-edge length"),
    (axes[0, 1], "endpoint_count", "Endpoints per annotation"),
    (axes[0, 2], "junction_zones", "Connected junction zones"),
    (axes[1, 0], "cycle_rank", "Graph cycle rank"),
]:
    values = s23_instances_df[column].dropna().to_numpy()

    axis.hist(
        np.log1p(values),
        bins=35,
        color="#496b8a",
        edgecolor="white",
    )
    axis.set_title(title)
    axis.set_xlabel("log(1 + measurement)")
    axis.set_ylabel("Annotations")

axes[1, 1].scatter(
    s23_instances_df["total_edge_length_px"],
    s23_instances_df["curvature_p90"],
    s=10, alpha=.35,
)
axes[1, 1].set_xscale("symlog", linthresh=1)
axes[1, 1].set_title("Length versus supported curvature")
axes[1, 1].set_xlabel("Total graph-edge length, pixels")
axes[1, 1].set_ylabel("90th-percentile curvature, 1/pixel")

reliable = s23_endpoints_df[
    s23_endpoints_df["direction_reliable"]
]

axes[1, 2].scatter(
    reliable["support_px"],
    reliable["anisotropy"],
    s=10, alpha=.35,
    color="#3a9d70",
)
axes[1, 2].set_title("Endpoint direction support")
axes[1, 2].set_xlabel("Inward chain length, pixels")
axes[1, 2].set_ylabel("Directional anisotropy")

for axis in axes.flat:
    axis.grid(alpha=.2)

fig.tight_layout()
s23_finish(fig, "01_graph_geometry_distributions.png")

# Representative long and curved paths.
gallery_rows = pd.concat([
    s23_branches_df.nlargest(2, "length_px"),
    s23_branches_df.dropna(
        subset=["curvature_p90"]
    ).nlargest(2, "curvature_p90"),
]).drop_duplicates(["graph_key", "branch_id"])

for row in gallery_rows.itertuples(index=False):
    with np.load(
        S23_GRAPH_DIR / f"{row.graph_key}.npz",
        allow_pickle=False,
    ) as archive:
        prefix = f"branch_{row.branch_id}_"

        if prefix + "xy" not in archive:
            continue

        xy = archive[prefix + "xy"].copy()
        arc = archive[prefix + "arc"].copy()
        curvature = archive[prefix + "curvature"].copy()
        valid = archive[prefix + "valid"].copy()
        tangent = archive[prefix + "tangent"].copy()

    image = load_solar_image(
        s23_raw_ids[str(row.entry_id)]
    )

    x0 = max(0, int(np.floor(xy[:, 0].min())) - 20)
    x1 = min(image.shape[1], int(np.ceil(xy[:, 0].max())) + 21)
    y0 = max(0, int(np.floor(xy[:, 1].min())) - 20)
    y1 = min(image.shape[0], int(np.ceil(xy[:, 1].max())) + 21)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    axes[0].imshow(
        image[y0:y1, x0:x1],
        cmap="gray", vmin=0, vmax=255,
        extent=(x0, x1, y1, y0),
    )

    points = axes[0].scatter(
        xy[valid, 0], xy[valid, 1],
        c=curvature[valid],
        cmap="magma",
        s=10,
    )
    fig.colorbar(
        points, ax=axes[0],
        label="Curvature, inverse pixels",
    )

    selected = np.arange(0, len(xy), 8)
    axes[0].quiver(
        xy[selected, 0], xy[selected, 1],
        tangent[selected, 0], tangent[selected, 1],
        color="cyan",
        angles="xy",
        scale_units="xy",
        scale=.15,
    )
    axes[0].set_title("Smoothed centerline and local tangent")

    axes[1].plot(
        arc[valid], curvature[valid],
        color="#a64d79",
    )
    axes[1].set_title("Supported curvature profile")
    axes[1].set_xlabel("Arc length, pixels")
    axes[1].set_ylabel("Curvature, inverse pixels")
    axes[1].grid(alpha=.2)

    fig.suptitle(
        f"{row.entry_id} | annotation {row.annotation_id} | "
        f"path {row.branch_id}"
    )
    fig.tight_layout()

    s23_finish(
        fig,
        f"02_profile_{row.graph_key}_{row.branch_id}.png",
    )

### 23.3 Candidate endpoint links

A plausible link needs more than nearby endpoints.

We calculate:

- Endpoint distance.
- Axial tangent agreement.
- Whether both outward directions face the proposed gap.
- Endpoint direction reliability.
- Image-derived darkness along a narrow corridor joining the endpoints.

Image evidence uses the observed grayscale image:

    local_darkness = GaussianBackground(image) − image

This is a contrast response, not a probability or calibrated confidence.

We also report how many competing candidate links each endpoint has.
Several plausible links sharing an endpoint are an ambiguity that must
later be resolved jointly.

At this stage, links between distinct annotations are candidate negatives
under that annotator's labels. They are not automatically rejected by a
handwritten geometric threshold, and no mask is modified.

In [ ]:
def s23_link_features(a, b, darkness):
    pa = np.array([a["x"], a["y"]], dtype=float)
    pb = np.array([b["x"], b["y"]], dtype=float)

    displacement = pb - pa
    distance = float(np.linalg.norm(displacement))

    if distance <= 0:
        return None

    gap_direction = displacement / distance

    ua = np.array(
        [a["outward_x"], a["outward_y"]],
        dtype=float,
    )
    ub = np.array(
        [b["outward_x"], b["outward_y"]],
        dtype=float,
    )

    if not (
        np.isfinite(ua).all()
        and np.isfinite(ub).all()
    ):
        return None

    axial_agreement = float(
        np.clip(abs(np.dot(ua, ub)), 0, 1)
    )

    facing_a = float(np.dot(ua, gap_direction))
    facing_b = float(np.dot(ub, -gap_direction))

    # Sample a narrow image corridor, excluding the endpoints.
    fraction = np.linspace(
        .1, .9, S23_CFG["link_samples"]
    )
    center = pa[None, :] + fraction[:, None] * displacement

    normal = np.array(
        [-gap_direction[1], gap_direction[0]]
    )

    corridor_values = []

    for offset in np.linspace(
        -S23_CFG["link_band_half_width_px"],
        S23_CFG["link_band_half_width_px"],
        5,
    ):
        points = center + offset * normal

        corridor_values.append(
            ndi.map_coordinates(
                darkness,
                [points[:, 1], points[:, 0]],
                order=1,
                mode="nearest",
            )
        )

    evidence = np.asarray(corridor_values)

    return {
        "distance_px": distance,
        "axial_agreement": axial_agreement,
        "facing_a": facing_a,
        "facing_b": facing_b,
        "minimum_facing": min(facing_a, facing_b),
        "minimum_anisotropy": min(
            float(a["anisotropy"]),
            float(b["anisotropy"]),
        ),
        "minimum_support_px": min(
            float(a["support_px"]),
            float(b["support_px"]),
        ),
        "corridor_darkness_mean": float(evidence.mean()),
        "corridor_darkness_p25": float(
            np.quantile(evidence, .25)
        ),
        "corridor_positive_fraction": float(
            (evidence > 0).mean()
        ),
    }

# Use one annotation entry per physical image in this pilot.
# This prevents repeatedly evaluating the same image through annotators.
pilot_meta = s23_meta.copy()

pilot_meta["physical_image_id"] = pilot_meta["image_id"].map(
    lambda entry: Path(s23_info[entry]["file_name"]).stem
)

pilot_meta = (
    pilot_meta.sort_values("image_id")
    .drop_duplicates("physical_image_id")
)

pilot_meta = pilot_meta.sample(
    n=min(S23_CFG["pilot_entries"], len(pilot_meta)),
    random_state=S23_CFG["seed"],
).sort_values("image_id")

s23_link_rows = []
s23_cut_log = []

# The next cell adds controlled positive cuts using the same image maps.
def s23_darkness(entry_id):
    image = load_solar_image(
        s23_raw_ids[str(entry_id)]
    ).astype(np.float32) / 255.0

    return ndi.gaussian_filter(image, sigma=15) - image

for metadata in pilot_meta.itertuples(index=False):
    entry_id = str(metadata.image_id)

    endpoints = s23_endpoints_df[
        s23_endpoints_df["entry_id"].astype(str).eq(entry_id)
        & s23_endpoints_df["direction_reliable"]
    ].copy()

    if len(endpoints) < 2:
        continue

    darkness = s23_darkness(entry_id)
    endpoint_records = endpoints.to_dict("records")
    positions = endpoints[["x", "y"]].to_numpy()

    pairs = sorted(
        cKDTree(positions).query_pairs(
            S23_CFG["maximum_link_gap_px"]
        )
    )

    degree = defaultdict(int)

    for i, j in pairs:
        if (
            str(endpoint_records[i]["annotation_id"])
            != str(endpoint_records[j]["annotation_id"])
        ):
            degree[i] += 1
            degree[j] += 1

    for i, j in pairs:
        a, b = endpoint_records[i], endpoint_records[j]

        if str(a["annotation_id"]) == str(b["annotation_id"]):
            # Same-annotation disconnected parts are not assumed
            # to be known positive or negative continuations.
            continue

        features = s23_link_features(a, b, darkness)

        if features is None:
            continue

        s23_link_rows.append({
            "entry_id": entry_id,
            "physical_image_id": metadata.physical_image_id,
            "temporal_block": int(metadata.temporal_block),
            "capture_datetime": str(metadata.capture_datetime),
            "label": 0,
            "source": "distinct_annotations",
            "annotation_a": str(a["annotation_id"]),
            "annotation_b": str(b["annotation_id"]),
            "a_x": a["x"], "a_y": a["y"],
            "b_x": b["x"], "b_y": b["y"],
            "candidate_degree_a": degree[i],
            "candidate_degree_b": degree[j],
            **features,
        })

    del darkness

print("Distinct-annotation candidate links:", len(s23_link_rows))

### 23.4 Controlled positive links

Observed labels do not tell us which missing fragment links a future
model will produce.

We therefore create controlled positive examples:

- Start with a connected training annotation.
- Remove a small disk around an interior point of a long centerline.
- Require the modified mask to contain exactly two components.
- Find reliable nearby endpoints belonging to different cut components.
- Treat the closest eligible endpoint pair as a synthetic continuation.

These examples teach a limited question: can geometry and image evidence
distinguish an artificial gap in one annotated structure from nearby
endpoints belonging to different annotations?

Limitations:

- Artificial cuts need not resemble real model errors.
- Cut-created endpoints can be cleaner than predicted endpoints.
- Class balance is generated by this experiment.
- Link-classifier scores are not calibrated acceptance probabilities.
- Useful pilot discrimination does not establish improved PQ.

In [ ]:
for metadata in pilot_meta.itertuples(index=False):
    entry_id = str(metadata.image_id)
    info = s23_info[entry_id]
    shape = (int(info["height"]), int(info["width"]))

    eligible = (
        s23_branches_df[
            s23_branches_df["entry_id"].astype(str).eq(entry_id)
            & s23_branches_df["profile_valid_points"].gt(0)
            & ~s23_branches_df["closed_pixel_path"]
        ]
        .sort_values("length_px", ascending=False)
        .drop_duplicates("annotation_id")
        .head(S23_CFG["maximum_positive_cuts_per_entry"])
    )

    if eligible.empty:
        continue

    darkness = s23_darkness(entry_id)

    for branch in eligible.itertuples(index=False):
        annotation_id = str(branch.annotation_id)

        annotation = s23_annotation_lookup[
            (entry_id, annotation_id)
        ]

        original_mask = np.asarray(
            rasterize_annotation(annotation, shape),
            dtype=bool,
        )

        _, original_components = ndi.label(
            original_mask,
            structure=np.ones((3, 3)),
        )

        if original_components != 1:
            s23_cut_log.append({
                "entry_id": entry_id,
                "annotation_id": annotation_id,
                "status": "skipped_original_multipart",
            })
            continue

        with np.load(
            S23_GRAPH_DIR / f"{branch.graph_key}.npz",
            allow_pickle=False,
        ) as archive:
            xy = archive[
                f"branch_{branch.branch_id}_xy"
            ].copy()

        midpoint = xy[len(xy) // 2]
        successful = False

        for radius in S23_CFG["cut_radii_px"]:
            cut = original_mask.copy()

            cx, cy = midpoint
            x0 = max(0, int(np.floor(cx - radius)))
            x1 = min(shape[1], int(np.ceil(cx + radius)) + 1)
            y0 = max(0, int(np.floor(cy - radius)))
            y1 = min(shape[0], int(np.ceil(cy + radius)) + 1)

            yy, xx = np.mgrid[y0:y1, x0:x1]
            disk = (
                (xx - cx)**2 + (yy - cy)**2
            ) <= radius**2

            local = cut[y0:y1, x0:x1]
            local[disk] = False

            labels, components = ndi.label(
                cut,
                structure=np.ones((3, 3)),
            )

            if components != 2:
                continue

            coords, graph = s23_graph(cut)
            endpoints = [
                item for item in s23_endpoints(coords, graph)
                if item["direction_reliable"]
                and np.linalg.norm(
                    np.array([item["x"], item["y"]]) - midpoint
                ) <= radius + S23_CFG["endpoint_window_px"]
            ]

            options = []

            for i in range(len(endpoints)):
                for j in range(i + 1, len(endpoints)):
                    a, b = endpoints[i], endpoints[j]

                    ca = labels[
                        int(round(a["y"])),
                        int(round(a["x"])),
                    ]
                    cb = labels[
                        int(round(b["y"])),
                        int(round(b["x"])),
                    ]

                    if ca == 0 or cb == 0 or ca == cb:
                        continue

                    features = s23_link_features(a, b, darkness)

                    if (
                        features is not None
                        and features["distance_px"]
                        <= S23_CFG["maximum_link_gap_px"]
                    ):
                        options.append((features["distance_px"], a, b, features))

            if not options:
                continue

            _, a, b, features = min(
                options, key=lambda item: item[0]
            )

            s23_link_rows.append({
                "entry_id": entry_id,
                "physical_image_id": metadata.physical_image_id,
                "temporal_block": int(metadata.temporal_block),
                "capture_datetime": str(metadata.capture_datetime),
                "label": 1,
                "source": "synthetic_cut",
                "annotation_a": annotation_id,
                "annotation_b": annotation_id,
                "cut_radius_px": radius,
                "a_x": a["x"], "a_y": a["y"],
                "b_x": b["x"], "b_y": b["y"],
                **features,
            })

            successful = True
            break

        s23_cut_log.append({
            "entry_id": entry_id,
            "annotation_id": annotation_id,
            "status": (
                "positive_created"
                if successful else "no_eligible_two_component_cut"
            ),
        })

    del darkness

s23_links_df = pd.DataFrame(s23_link_rows)
s23_cuts_df = pd.DataFrame(s23_cut_log)

s23_links_df.to_csv(
    S23_DIR / "candidate_link_dataset.csv",
    index=False,
)
s23_cuts_df.to_csv(
    S23_DIR / "synthetic_cut_accounting.csv",
    index=False,
)

if s23_links_df.empty:
    raise RuntimeError("No candidate-link examples were generated.")

display(
    s23_links_df.groupby(["source", "label"])
    .size()
    .rename("examples")
    .reset_index()
)

display(s23_cuts_df["status"].value_counts().to_frame("count"))

### 23.5 Training-only grouped pilot

We compare:

1. Distance only.
2. Distance and endpoint geometry.
3. Geometry plus observed-image corridor evidence.

All models use identical test examples.

Splits are grouped by temporal block and training observations within
the configured purge distance of held-out observations are removed.

Scaling is fitted inside each training fold.

This experiment reports link-example ROC-AUC and average precision.
It does not report competition PQ, calibrated probabilities or an
unbiased final test result.

Candidate examples from one image are dependent. We retain image and
fold identifiers and summarize performance by image as well as globally.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if "s23_links_df" not in globals():
    raise RuntimeError("Step 23.4 must produce s23_links_df before scoring.")

s23_geometry = [
    "distance_px", "axial_agreement", "minimum_facing",
    "minimum_anisotropy", "minimum_support_px",
]
s23_models = {
    "distance": ["distance_px"],
    "geometry": s23_geometry,
    "geometry_and_image": s23_geometry + [
        "corridor_darkness_mean", "corridor_darkness_p25",
        "corridor_positive_fraction",
    ],
}

s23_images = s23_links_df[
    ["physical_image_id", "temporal_block", "capture_datetime"]
].drop_duplicates()

if not s23_images["physical_image_id"].is_unique:
    raise ValueError("Each physical image must have one block and timestamp.")

timestamps = pd.to_datetime(s23_images["capture_datetime"], utc=True)
if timestamps.isna().any():
    raise ValueError("The candidate links contain missing capture timestamps.")

times_ns = timestamps.to_numpy(dtype="datetime64[ns]").astype("int64")
purge_ns = pd.Timedelta(days=S23_CFG["purge_days"]).value


def s23_link_metrics(labels, scores):
    if labels.nunique() < 2:
        return {"roc_auc": np.nan, "average_precision": np.nan}
    return {
        "roc_auc": roc_auc_score(labels, scores),
        "average_precision": average_precision_score(labels, scores),
    }


fold_scores, image_scores = [], []
n_splits = min(S23_CFG["folds"], s23_images["temporal_block"].nunique())

if n_splits < 2:
    print("Link pilot skipped: fewer than two temporal blocks contain examples.")
else:
    cv = GroupKFold(n_splits=n_splits)
    for fold, (fit_idx, test_idx) in enumerate(
        cv.split(s23_images, groups=s23_images["temporal_block"]), start=1
    ):
        fit_times = times_ns[fit_idx]
        held_out_times = np.sort(times_ns[test_idx])
        positions = np.searchsorted(held_out_times, fit_times)
        left = np.clip(positions - 1, 0, len(held_out_times) - 1)
        right = np.clip(positions, 0, len(held_out_times) - 1)
        gaps = np.minimum(
            np.abs(fit_times - held_out_times[left]),
            np.abs(fit_times - held_out_times[right]),
        )
        keep = gaps > purge_ns

        fit_ids = s23_images.iloc[fit_idx[keep]]["physical_image_id"]
        test_ids = s23_images.iloc[test_idx]["physical_image_id"]
        fit_rows = s23_links_df[s23_links_df["physical_image_id"].isin(fit_ids)]
        test_rows = s23_links_df[s23_links_df["physical_image_id"].isin(test_ids)]

        if fit_rows["label"].nunique() < 2 or test_rows["label"].nunique() < 2:
            print(f"Fold {fold} skipped: both labels are required after purging.")
            continue

        for name, features in s23_models.items():
            model = make_pipeline(
                StandardScaler(),
                LogisticRegression(max_iter=1000, random_state=S23_CFG["seed"]),
            )
            model.fit(fit_rows[features], fit_rows["label"])
            scores = model.predict_proba(test_rows[features])[:, 1]
            fold_scores.append(dict(
                fold=fold, model=name, fit_examples=len(fit_rows),
                test_examples=len(test_rows), purged_images=int((~keep).sum()),
                **s23_link_metrics(test_rows["label"], scores),
            ))

            scored = test_rows[["physical_image_id", "label"]].assign(score=scores)
            for image_id, rows in scored.groupby("physical_image_id"):
                image_scores.append(dict(
                    fold=fold, model=name, physical_image_id=image_id,
                    test_examples=len(rows), positives=int(rows["label"].sum()),
                    **s23_link_metrics(rows["label"], rows["score"]),
                ))

s23_scores_df = pd.DataFrame(fold_scores, columns=[
    "fold", "model", "fit_examples", "test_examples", "purged_images",
    "roc_auc", "average_precision",
])
s23_image_scores_df = pd.DataFrame(image_scores, columns=[
    "fold", "model", "physical_image_id", "test_examples", "positives",
    "roc_auc", "average_precision",
])

s23_scores_df.to_csv(S23_DIR / "link_pilot_scores.csv", index=False)
s23_image_scores_df.to_csv(S23_DIR / "link_pilot_by_image.csv", index=False)
print("Evaluated link-pilot model/fold combinations:", len(s23_scores_df))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for label, color, description in [
    (0, "#c45a5a", "Different annotations"),
    (1, "#3a9d70", "Synthetic continuation"),
]:
    rows = s23_links_df[s23_links_df["label"] == label]

    axes[0].scatter(
        rows["distance_px"],
        rows["minimum_facing"],
        s=12, alpha=.3,
        color=color, label=description,
    )
    axes[1].scatter(
        rows["axial_agreement"],
        rows["corridor_darkness_mean"],
        s=12, alpha=.3,
        color=color,
    )

axes[0].set(
    xlabel="Endpoint gap, pixels",
    ylabel="Minimum outward-facing alignment",
    title="Distance and facing geometry",
)
axes[0].legend(fontsize=8)

axes[1].set(
    xlabel="Axial tangent agreement",
    ylabel="Mean corridor darkness",
    title="Geometry versus observed-image evidence",
)

negative_rows = s23_links_df[
    s23_links_df["source"] == "distinct_annotations"
]

if len(negative_rows):
    ambiguity = np.maximum(
        negative_rows["candidate_degree_a"].fillna(0),
        negative_rows["candidate_degree_b"].fillna(0),
    )
    axes[2].hist(
        ambiguity,
        bins=np.arange(0, int(ambiguity.max()) + 2) - .5,
        color="#a64d79",
        edgecolor="white",
    )

axes[2].set(
    xlabel="Larger endpoint candidate degree",
    ylabel="Candidate pairs",
    title="Competing links — not independent decisions",
)

for axis in axes:
    axis.grid(alpha=.2)

fig.tight_layout()
s23_finish(fig, "03_candidate_link_diagnostics.png")

if not s23_scores_df.empty:
    summary = s23_scores_df.groupby("model")[
        ["roc_auc", "average_precision"]
    ].mean()

    fig, axis = plt.subplots(figsize=(10, 5))
    summary.plot.bar(ax=axis)
    axis.set_ylim(0, 1.05)
    axis.set_ylabel("Mean held-out fold score")
    axis.set_title(
        "Controlled link pilot\n"
        "Synthetic positives; these are not segmentation/PQ results"
    )
    axis.grid(axis="y", alpha=.2)
    fig.tight_layout()
    s23_finish(fig, "04_link_pilot_comparison.png")

gallery = pd.concat([
    s23_links_df[
        s23_links_df["label"] == 1
    ].head(2),
    s23_links_df[
        s23_links_df["label"] == 0
    ].sort_values(
        ["minimum_facing", "axial_agreement"],
        ascending=False,
    ).head(2),
])

for number, row in enumerate(gallery.itertuples(index=False)):
    image = load_solar_image(
        s23_raw_ids[str(row.entry_id)]
    )

    x0 = max(0, int(min(row.a_x, row.b_x)) - 35)
    x1 = min(
        image.shape[1],
        int(max(row.a_x, row.b_x)) + 36,
    )
    y0 = max(0, int(min(row.a_y, row.b_y)) - 35)
    y1 = min(
        image.shape[0],
        int(max(row.a_y, row.b_y)) + 36,
    )

    fig, axis = plt.subplots(figsize=(8, 7))
    axis.imshow(
        image[y0:y1, x0:x1],
        cmap="gray", vmin=0, vmax=255,
        extent=(x0, x1, y1, y0),
    )
    axis.plot(
        [row.a_x, row.b_x],
        [row.a_y, row.b_y],
        "--o",
        color="cyan",
        linewidth=1.5,
    )
    axis.set_title(
        f"{row.source} | label={row.label}\n"
        f"gap={row.distance_px:.1f}px | "
        f"minimum facing={row.minimum_facing:.2f}\n"
        "Candidate shown for inspection; no link has been applied"
    )
    axis.set_xlabel("Original x coordinate")
    axis.set_ylabel("Original y coordinate")

    fig.tight_layout()
    s23_finish(fig, f"05_candidate_link_{number}.png")

### 23.6 Exploratory graph spectra

We examine a bounded selection of native pixel graphs.

The graph stored earlier contains edge lengths. For a Laplacian,
we explicitly convert lengths to positive conductances:

    adjacency conductance = 1 / edge length

We then use the symmetric normalized graph Laplacian.

Important limitations:

- Pixel count, sampling density and topology affect spectra.
- Graphs are not automatically comparable physical objects.
- The second eigenvalue is not a validated filament-quality score.
- Graph spectra do not establish instance identity.
- Large graphs are excluded from dense eigendecomposition and recorded
  as excluded, not silently approximated.

Spectral results remain exploratory until a controlled experiment
demonstrates value beyond simpler geometric descriptors.

In [ ]:
from scipy.sparse import csr_matrix

s23_spectrum_rows = []

spectrum_candidates = (
    s23_instances_df[
        s23_instances_df["status"] == "ok"
    ]
    .sort_values(
        ["nodes", "entry_id", "annotation_id"],
        ascending=[False, True, True],
    )
)

eligible_spectra = spectrum_candidates[
    spectrum_candidates["nodes"]
    <= S23_CFG["maximum_spectrum_nodes"]
].head(S23_CFG["spectrum_examples"])

for row in eligible_spectra.itertuples(index=False):
    with np.load(
        S23_GRAPH_DIR / f"{row.graph_key}.npz",
        allow_pickle=False,
    ) as archive:
        graph = csr_matrix(
            (
                archive["graph_data"],
                archive["graph_indices"],
                archive["graph_indptr"],
            ),
            shape=tuple(archive["graph_shape"]),
        )

    conductance = graph.copy()
    conductance.data = 1.0 / conductance.data

    normalized_laplacian = laplacian(
        conductance,
        normed=True,
    ).toarray()

    eigenvalues = np.linalg.eigvalsh(
        normalized_laplacian
    )

    if eigenvalues.min(initial=0) < -1e-7:
        raise AssertionError("Unexpected negative Laplacian eigenvalue.")

    zero_count = int(
        np.isclose(
            eigenvalues, 0,
            rtol=0, atol=1e-7,
        ).sum()
    )

    assert zero_count == int(row.graph_components)

    s23_spectrum_rows.append({
        "entry_id": row.entry_id,
        "annotation_id": row.annotation_id,
        "graph_key": row.graph_key,
        "nodes": row.nodes,
        "components": row.graph_components,
        "zero_eigenvalues": zero_count,
        "lambda2": (
            float(eigenvalues[1])
            if len(eigenvalues) > 1 else np.nan
        ),
    })

    np.save(
        S23_GRAPH_DIR / f"{row.graph_key}_eigenvalues.npy",
        eigenvalues,
    )

s23_spectra_df = pd.DataFrame(s23_spectrum_rows)

s23_spectra_df.to_csv(
    S23_DIR / "exploratory_graph_spectra.csv",
    index=False,
)

if not s23_spectra_df.empty:
    fig, axis = plt.subplots(figsize=(11, 5))

    for row in s23_spectra_df.itertuples(index=False):
        values = np.load(
            S23_GRAPH_DIR / f"{row.graph_key}_eigenvalues.npy"
        )

        axis.plot(
            np.arange(min(20, len(values))),
            values[:20],
            marker=".",
            alpha=.7,
            label=(
                f"{row.annotation_id}: "
                f"N={row.nodes}, C={row.components}"
            ),
        )

    axis.set_title(
        "First normalized-Laplacian eigenvalues\n"
        "Small native graphs only; exploratory comparison"
    )
    axis.set_xlabel("Sorted eigenvalue index")
    axis.set_ylabel("Eigenvalue")
    axis.legend(fontsize=7)
    axis.grid(alpha=.2)

    fig.tight_layout()
    s23_finish(fig, "06_graph_spectra.png")

s23_summary = {
    "step": 23,
    "expected_training_annotations": expected_annotations,
    "accounted_annotations": len(s23_instances_df),
    "nonempty_graphs": int(
        s23_instances_df["status"].eq("ok").sum()
    ),
    "endpoint_records": len(s23_endpoints_df),
    "ordered_path_records": len(s23_branches_df),
    "link_pilot_physical_images": len(pilot_meta),
    "candidate_link_examples": len(s23_links_df),
    "synthetic_positive_links": int(
        s23_links_df["label"].eq(1).sum()
    ),
    "distinct_annotation_negative_links": int(
        s23_links_df["label"].eq(0).sum()
    ),
    "evaluated_model_fold_combinations": len(s23_scores_df),
    "spectra_computed": len(s23_spectra_df),
    "graphs_above_dense_spectrum_limit": int(
        spectrum_candidates["nodes"].gt(
            S23_CFG["maximum_spectrum_nodes"]
        ).sum()
    ),
    "actual_model_fragmentation_evaluated": False,
    "segmentation_pq_improvement_demonstrated": False,
    "links_applied_to_masks": False,
    "validation_used_for_fitting": False,
    "limitations": [
        "Synthetic cuts differ from actual model errors.",
        "Link scores are not calibrated acceptance probabilities.",
        "Candidate links sharing endpoints require joint resolution.",
        "Topology and geometric smoothness do not prove common identity.",
        "Real fragmentation analysis requires saved model predictions.",
    ],
}

with open(
    S23_DIR / "step23_summary.json", "w"
) as handle:
    json.dump(s23_summary, handle, indent=2)

print(json.dumps(s23_summary, indent=2))
print("\nSaved Step 23 outputs:", S23_DIR)